# Chapter 8 — Laminar Flow

*Fluid Mechanics, 5th edition — Pijush K. Kundu, Ira M. Cohen, David R. Dowling (2012).*
This notebook teaches every new idea of the chapter with plain words, step-by-step mathematics, small worked examples, tested Python from the `fluidpy` package, figures, animations and interactive explainers. It does not reproduce the book's text or figures; every equation is shown in full together with its number, so you can follow along in your copy.

---

**The big idea in one paragraph**

Pour honey, drag a spoon through syrup, watch a cloud droplet drift down at about a centimetre a second: when viscosity
matters everywhere, the Navier–Stokes equations can often be solved exactly. The trick is always the same — a symmetry or
a thin geometry kills the nonlinear term, and what is left is a balance between pressure (or wall motion) and viscous
friction. This chapter solves that balance between plates, in pipes and between rotating cylinders; in the thin film under
a bearing and under a spreading drop; above a plate that starts or shakes, where momentum *diffuses* a distance
$\sqrt{\nu t}$ — the scale behind every boundary layer, every Ekman layer and every spin-up time of the ocean and
atmosphere; and round a sphere so small that inertia vanishes, which gives Stokes' drag $D=6\pi\mu aU$ *(Eq. 8.51)* and
the fall speed of cloud droplets, aerosols and silt.

**Road map** — the new ideas of this chapter, in the order they build on each other:

1. §8.1 laminar vs turbulent flow and ν as a momentum diffusivity, $\mathrm{Re}=Ud/\nu\sim2000$–$3000$ (C01)
2. §8.2 plane Couette–Poiseuille flow $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$ *(8.5)* and backflow (C02); Poiseuille pipe flow, $Q\propto a^4$ (C03); circular Couette flow $u_\varphi=AR+B/R$ *(8.9)* and its two limits (C04)
3. §8.3 the lubrication approximation, weighed by $\varepsilon^2\mathrm{Re}_L$ (C05); the lubrication profile and the Reynolds equation (C06); the slider bearing (C07); the thin-film (viscous gravity current) equation (C08)
4. §8.4 Stokes' first problem $\frac uU=1-\mathrm{erf}\big(\frac{y}{2\sqrt{\nu t}}\big)$ *(8.30)* and similarity (C09); the similarity ansatz and its exponents (C10)
5. §8.5 Stokes' second problem: the oscillating plate and its layer of depth $\sqrt{2\nu/\omega}$ (C11)
6. §8.6 creeping flow: the Stokes equations $\nabla p=\mu\nabla^2\mathbf u$ *(8.43)* (C12); Stokes' flow round a sphere (C13); Stokes drag $D=6\pi\mu aU$ *(8.51)* and settling (C14); where Stokes fails and Oseen's fix (C15)
7. §8.7 final remarks: where exact solutions stop

**What you should already know** (each is recapped where it is used):

- Navier–Stokes and the no-slip condition (Ch. 4 §4.5–4.10)
- Reynolds number and scaling (Ch. 4 §4.11)
- viscosity, ν = μ/ρ, plane Couette start-up (Ch. 1 §1.5)
- dimensional analysis (Ch. 1 §1.11)
- vorticity diffusion, the ideal vortex, solid-body rotation (Ch. 5 §5.1–5.4)
- the Gaussian vortex (Ch. 3 §3.5)
- the Stokes stream function and the ideal-flow sphere (Ch. 6 §6.8)
- complex amplitudes (Ch. 7 §7.7)

## 🎮 Interactive explainers in this chapter

| # | Explainer | What it makes clear |
|---|---|---|
| 1 | **When does fluid flow backwards in a channel?** | C02: a line plus a parabola — wall drag and pressure add, and backflow opens once dp/dx > 2μU/h² |
| 2 | **Why can a thin film ignore inertia?** | C05: term bars of the scaled momentum equations reorder with ε and Re_L; only ε²Re_L must be small |
| 3 | **How does a film thinner than a hair carry a load?** | C06 C07: the same flux through a narrowing gap needs a pressure hump — and it turns into suction if the pad slides backwards |
| 4 | **Why does a honey drop spread like t^(1/5)?** | C08: a nonlinear diffusion that chokes itself; every initial shape ends on one profile |
| 5 | **How can profiles at all times be one curve?** | C09: momentum diffuses √(νt) — plot against y/√(νt) and every profile collapses |
| 6 | **Where do similarity exponents come from?** | C10: guess At^(−n)F(ξ/δ); the equation and one conserved quantity fix n and δ(t) |
| 7 | **How deep does a shaking plate reach?** | C11: a 'wave' made only of diffusion — each layer lags by y/δ and shrinks as e^(−y/δ), δ = √(2ν/ω) |
| 8 | **What does creeping flow round a sphere look like?** | C13 C15: fore–aft symmetric, reaching tens of radii, until inertia returns far away: Oseen's wake |
| 9 | **Where does 6πμaU come from?** | C14: one third pressure, two thirds friction; drag ∝ U makes a droplet's fall speed grow as a² |

Each one opens right where its idea is taught and fills the window (no scrolling). Start with the **Walkthrough**, then **Explore** with the sliders and presets, read **Explain** (every number worked out with your settings, and what it means), step through the **Derivation** where there is one, read the **Equations** and the **Code**, and finish with **Check yourself**. On the web page they are full-window; in Colab and Jupyter they appear in the cell output (use ⤢ *Full screen* or *Open in new tab* for more room).

## ⚙️ Setup — run this cell first

It works in three places without changes:
* **Google Colab** — clones the public repository into `/content/fluidpy` (the `fluidpy` package, its tests and the
  interactive explainers) and installs the one package Colab lacks (`pint`). No GPU is needed.
* **Jupyter / VS Code on your computer** — finds the repository root (the folder with `book.yaml`) and imports from it.
* **The published web page** — already executed; nothing to run.

In [ ]:
# ── Setup: find (or fetch) the fluidpy repository, then load the house style ─────────────────────────────
import os, sys, subprocess, pathlib                     # standard library only, so this cell never fails on imports

IN_COLAB = "google.colab" in sys.modules                 # True when this notebook runs inside Google Colab
REPO_URL = "https://github.com/shammun/fluidpy.git"               # the public repository with the fluidpy package

if IN_COLAB:
    ROOT = pathlib.Path("/content/fluidpy")              # where the repository is cloned on Colab
    if not ROOT.exists():                                 # first run in this Colab session: clone it
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:                                                 # later runs: fetch the newest version
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
else:
    ROOT = pathlib.Path.cwd().resolve()                   # start where Jupyter was launched …
    while not (ROOT / "book.yaml").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent                                # … and walk up until we find the repository root

os.chdir(ROOT)                                            # relative paths (outputs/, viz/) now resolve from the root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))                         # make `import fluidpy` work

from fluidpy.core.style import setup_notebook            # matplotlib + plotly style shared with the explainers
from fluidpy.core.embed import show_viz                  # shows an interactive explainer in the notebook
from fluidpy.core.anim import show_animation             # plays a matplotlib animation inline
FAST = setup_notebook()                                   # FAST=True (env FLUIDPY_FAST=1) shrinks grids for quick runs
print(f"IN_COLAB = {IN_COLAB} | repository root: {ROOT} | FAST = {FAST}")

In [ ]:
import numpy as np                                       # arrays and maths (Ch. 1 primer P03)
import sympy as sp                                       # symbolic algebra (Ch. 1 primer P40)
import matplotlib.pyplot as plt                          # static figures (Ch. 1 primer P01)
from scipy import integrate, special, optimize, linalg   # quadrature, erf/erfc, root finding, banded solves
from fluidpy import ch08_laminar_flow as ch08            # the tested chapter-8 module (re-exports the three new toolkits)
from fluidpy.core import laminar as LAM, lubrication as LUB, creeping as CRP, diffusion   # the new toolkits + solvers
from fluidpy import ch01_introduction as ch01            # Ch. 1: property functions, Π groups
from fluidpy import ch03_kinematics as ch03              # Ch. 3: the developing pipe profile
from fluidpy import ch04_conservation_laws as ch04       # Ch. 4: plane Poiseuille, Stokes' first problem (earlier code)
from fluidpy import ch05_vorticity_dynamics as ch05      # Ch. 5: the rotating cylinder, the diffusing vortex sheet
from fluidpy.core import navier_stokes as NS, similarity as SIM, vortices as VX   # Ch. 4–5 tools: NS terms, Re, vortices
from fluidpy.core import potential as PF, curvilinear as CL                       # Ch. 6 potential flows, curvilinear calculus
from fluidpy.core.interact import slider_figure, animate_figure, live   # plotly sliders (P17), time players, widgets (P47)
from fluidpy.core.anim import animate, ffmpeg_path       # matplotlib animations (P16); show_animation came with setup
from fluidpy.core.style import COLORS, savefig           # the house palette and a helper that saves PNGs to outputs/ch08
from tools.convergence import observed_order             # slope of log(error) vs log(step) (P13)
sys.path.insert(0, str(ROOT / "scripts"))                # the chapter's drawing helpers live in scripts/
from ch08_drawings import channel, pad, sphere, profile_arrows   # drawing only: walls, the bearing pad, a sphere, arrows
ffmpeg_path()                                            # find ffmpeg once, for the MP4 animations
MU_W, RHO_W, NU_W = 1.0e-3, 1000.0, 1.0e-6              # water: μ [Pa s], ρ [kg/m³], ν = μ/ρ [m²/s] (our default)
plt.rcParams["contour.negative_linestyle"] = "solid"     # negative ψ levels are streamlines too: draw them solid
import logging                                           # Python's message system (matplotlib reports font fallbacks through it)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # hide harmless "font weight not found" notes


def recolor(fig, colors, dashes=None):                   # give plotly traces the notebook's colours by trace name
    for tr in fig.data:                                  # every trace of every slider step
        if tr.name in colors:                            # a name we assigned a colour to
            tr.line.color = colors[tr.name]              # same colour meaning as in the matplotlib figures
        if dashes and tr.name in dashes:                 # optional dash pattern ("dash", "dot")
            tr.line.dash = dashes[tr.name]
    return fig                                           # the same figure, restyled


def step_titles(fig, titles):                            # give each slider position of a slider_figure its own title
    old = fig.layout.sliders[0].steps                    # the steps slider_figure made (they switch curves on and off)
    fig.layout.sliders[0].steps = [dict(method=s.method, label=s.label, args=[s.args[0], {"title.text": t}])
                                   for s, t in zip(old, titles)]   # same switch + a new title text per step
    fig.layout.title.text = titles[fig.layout.sliders[0].active]   # the title of the step shown first
    return fig                                           # the same figure, now with numbers that follow the slider


print(len([n for n in dir(ch08) if not n.startswith("_")]), "public names in fluidpy.ch08_laminar_flow")
print("ch08.channel_flow is LAM.channel_flow:", ch08.channel_flow is LAM.channel_flow)   # the re-export

**What does the code above do?**

1. Numerical, symbolic and plotting libraries (all primed in Ch. 1); from `scipy` we use `integrate` (`quad`,
   `cumulative_trapezoid`, `solve_bvp`, `solve_ivp`), `special` (`erf`, `erfc` and their inverses), `optimize` (`brentq`,
   `minimize_scalar`) and `linalg` (`solve_banded`) — each is reminded or primed where it is first used.
2. `ch08` is the chapter module. It **re-exports the three new toolkits** `core.laminar` (exact parallel and unsteady
   flows), `core.lubrication` (thin gaps and films) and `core.creeping` (low-Reynolds-number flow), plus the diffusion
   solvers, so `ch08.channel_flow` and `LAM.channel_flow` are the same function (the last line prints `True`). Every
   function cites its § and equation and is tested in `tests/test_ch08.py`.
3. Earlier chapters' modules are used in the recaps and for parity checks (the Ch. 3 pipe profile, the Ch. 4 Poiseuille and
   Stokes-problem functions, the Ch. 5 rotating cylinder and vortex sheet, Ch. 6's potential-flow sphere).
4. `scripts/ch08_drawings.py` only draws (channel walls, the pad, a sphere, profile arrows); it computes no physics.
5. `MU_W, RHO_W, NU_W`: water at room temperature, our default fluid (numbers ours, not the book's).
6. `recolor` only restyles plotly figures.

### ⚠️ Conventions in this chapter (read once; each is repeated where it bites)

| Symbol | Meaning here | Before (and where it bites) |
|---|---|---|
| dp/dx | the book's pressure gradient: **favourable < 0** (pressure falls downstream) | Ch. 4's code used G = −dp/dx; new functions take `dpdx` (and accept `G=`) |
| y = 0, y = h | channel walls: y = 0 **fixed**, y = h moving at U | Ch. 1's Couette start-up moved the upper plate too |
| R, r | **R** = cylindrical radius (§8.2), **r** = spherical radius (§8.6) | Ch. 3–6 used r for both |
| θ | in §8.6 measured from the **downstream** axis: rear stagnation point θ = 0, front θ = π | Ch. 6's sphere also had θ from the axis; check the direction |
| ω | **vorticity** in §8.1, §8.4, §8.6 ($\omega_z$, $\omega_\varphi$) but the plate's **frequency** [rad/s] in §8.5 | as in Ch. 7 (frequency) and Ch. 5 (vorticity) |
| η | the **similarity variable** $\eta=y/\sqrt{\nu t}$ *(8.25)*; the book's figures plot $y/(2\sqrt{\nu t})=\eta/2$ | the surface elevation in Ch. 7 |
| ε | the fineness ratio h/L of a thin gap | a density ratio in Ch. 7, the dissipation rate in Ch. 4 |
| Λ | the bearing number $\mu UL/(P_ah^2)$ | new |
| δ₉₉, δ_e, δ(t) | 99 % thickness of Stokes' first problem · e-folding depth of the Stokes layer · the width in a similarity form | three different thicknesses |
| Re | pipe $Ud/\nu$ (diameter, mean speed) · lubrication $\mathrm{Re}_L=\rho UL/\mu$ (passage length) · generic $\rho UL/\mu$ · sphere $\mathrm{Re}=2aU/\nu$ (**diameter**) | the radius-based $\mathrm{Re}_a=\mathrm{Re}/2$ appears in C15 |
| D | an integration constant (C13) **and** the drag (C14) | say which |
| g | `ch08`'s settling and film functions use the SI standard g = 9.80665 m/s² (`G0`) | Ch. 4, 5, 7 used 9.81; the book prints no g-dependent number here |

We compute with the book's conventions and name the other one wherever it differs.

**Colours** (one meaning each, also in the explainers): pressure and the pressure-driven (Poiseuille) part **orange** ·
viscous friction and the wall-driven (Couette) part **rose** · inertia **teal** · velocity profiles **blue** · vorticity
**purple** · backflow and warnings **amber** · rescaled (similarity) curves purple dashed · an analytic ghost grey dashed ·
numerical dots teal · a printed slip grey dotted, labelled "as printed".

**Book slips we correct in this notebook** (each is explained where it is used):

| The book prints | We use | Where |
|---|---|---|
| §8.1 calls μ "the kinematic viscosity" | μ is the **dynamic** viscosity; ν = μ/ρ is the kinematic one | C01 |
| channel mean velocity written $V=\int_0^hu\,dy$ in the middle expression | $V\equiv Q/h$ (the 1/h is missing; units m²/s ≠ m/s) | C02 (N10) |
| (8.13b), $\frac{\partial v}{\partial t}+u\frac{\partial v}{\partial x}+v\frac{\partial v}{\partial y}=-\frac1\rho\frac{\partial p}{\partial y}+\frac\mu\rho\Big(\frac{\partial^2v}{\partial x^2}+\frac{\partial^2v}{\partial y^2}\Big)$ with $-\frac1\rho\frac{\partial p}{\partial x}$ | $-\frac1\rho\frac{\partial p}{\partial y}$ in the y-momentum equation | C05 (N24, D10) |
| (8.17a) as $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\frac{\partial^2u}{\partial y^2}$ | $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$ (units) | C05 (D11) |
| "(8.16a), $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$ can be integrated twice" | it is the simplified (8.17a) that is integrated | C06 (N33) |
| (8.19), $u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+U_h\frac yh+U_0\Big(1-\frac yh\Big)$ ending in $+U_0$ | $+U_0\big(1-\frac yh\big)$ (otherwise u(h) = U_h + U₀) | C06 (N34, D12) |
| Example 8.1: intermediate integrals with $(1-\alpha x/L)$; final pressure with the denominator to the **first** power | $(1+\alpha x/L)$; the denominator **squared** | C07 (D14) |
| Example 8.2's lubrication equations without ν and walls at "y = 0, h" | with ν; the gap coordinate is z | C06 (N36) |
| "the final equation of Example 8.2" (in Example 8.7) | Example 8.3 | C10 (N63) |
| $\int_0^\infty\omega\,dy=-U$ | $+U$ (u falls upward, so ω > 0) | C09 (N53) |
| Example 8.5's ±2.76 | ±2.772 (its own width 5.54 is right) | C10 (N59) |
| "substitution of (8.33), $u(y=0,t)=U\cos(\omega t)$ into (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$" | it is (8.35), $u(y,t)=\mathrm{Re}\{e^{i\omega t}f(y)\}$ that is substituted | C11 (N68) |
| rear pressure minimum +3μU/2a | **−**3μU/2a | C14 (N88, D30) |
| "(9.63)" and "(9.68)" in §8.6 | (8.43), $\nabla p=\mu\nabla^2\mathbf u$ and (8.48), $\psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)$ | C13 (N93), C15 (N98) |
| Oseen's equation with $+\partial p/\partial x_i$ | $-\partial p/\partial x_i$ | C15 (N95) |
| power into the fluid $(2\pi R_1)\tau_{R\varphi}u_\varphi$ | $-2\pi R_1\sigma_{R\varphi}u_\varphi$ (positive) | C04 (R11) |
| the bearing number Λ "near unity" with p* = p/P_a | Λ ≈ 49 for our film (≈ 10³ for thinner, longer films); the natural scale is μUL/h² | C05 (N31) |

> 🔁 **Tools from earlier chapters used in this one** (one line each where first used; the full primers are in
> `knowledge/primers.md`): partial derivative (P25), first-order Taylor (P26), definite integral (P27), the linear
> second-order ODE (P44), separation of variables (P42), chain rule (P49/P91), exponent rules (P43), power laws and
> log–log plots (P13), limits and orders of smallness (P68), order-of-magnitude scaling (P130), scaled variables and the
> chain rule (P133), product rule (P38), fundamental theorem of calculus (P84), differentiation under the integral sign
> (P109), substitution in an integral (P106), erfc (P123), Euler's formula (P45), complex square roots (P159), complex
> amplitudes (P176), curl of a curl (P122), Schwarz's theorem (P121), surface integrals on a sphere (P163), line integrals
> (P35), cylindrical and spherical unit vectors (P88), polar coordinates as a moving basis (P105), boundary conditions
> (P20), `np.linalg.solve` (P57), sympy (P40) and `expand`/`series` (P117), `quad` (P87), `brentq` (P108), `expm1` (P107),
> `minimize_scalar` (P170), `meshgrid` (P76), contour and streamline plots (P78), finite differences (P21), explicit
> stepping (P30), the trapezoid rule (P37), `np.random.default_rng` (P10), `animate` (P16), `slider_figure` (P17),
> `show_viz` (P18), live widgets (P47), `assert np.allclose` (P15). New primers of this chapter: P185–P199.

---

## 8.1 Introduction

**What is this section about?** Where this chapter lives: flows in which viscosity acts everywhere, not just in thin
layers. We meet Reynolds's experiment (smooth versus chaotic flow in a pipe), learn to read the kinematic viscosity ν as a
*diffusivity* of momentum and vorticity, and restate the equations and wall conditions every solution in the chapter must
satisfy.

> 🔁 **Recap — The Reynolds number as inertia over viscosity** (Ch. 4 §4.11). Scaling the Navier–Stokes equation gives inertia $\sim U^2/L$ and viscous acceleration $\sim\mu U/(\rho L^2)$; their ratio
$\frac{U^2/L}{\mu U/\rho L^2}=\frac{\rho UL}{\mu}=\mathrm{Re}$ is the Reynolds number. Earlier chapters dropped viscosity
where Re ≫ 1 (ideal flow, Ch. 6); in this chapter viscosity is kept everywhere.

In [ ]:
s = ch08.inertia_viscous_scales(0.1, 0.01, NU_W)         # U = 10 cm/s, L = 1 cm, water: the two accelerations [m/s²]
print({k: round(v, 6) for k, v in s.items()})            # inertia U²/L, viscous νU/L², and their ratio
print("Re =", round(SIM.reynolds_number(0.1, 0.01, nu=NU_W), 3))   # Ch. 4's Re = UL/ν — the same ratio

> 🔁 **Recap — Vorticity diffuses with ν** (Ch. 5 §5.4). In a plane flow the vorticity equation $\frac{D\boldsymbol\omega}{Dt}=(\boldsymbol\omega\cdot\nabla)\mathbf u+\nu\nabla^2
\boldsymbol\omega$ *(5.13)* loses its stretching term, because $\boldsymbol\omega=\omega_z\mathbf e_z$ and nothing depends on
z, so $(\boldsymbol\omega\cdot\nabla)\mathbf u=\omega_z\,\partial\mathbf u/\partial z=0$; what is left is
$D\omega_z/Dt=\nu\nabla^2\omega_z$ — the same form as the heat equation. A fluid with larger ν smooths out a vorticity
pattern faster.

### 🧩 Laminar vs turbulent flow, and ν as a momentum diffusivity `C01`

*The question:* When does a flow stay smooth — and what does the kinematic viscosity ν actually measure?

*In one line:* $\mathrm{Re}=\frac{Ud}{\nu}\sim2000$–$3000$ separates smooth from chaotic pipe flow, and $\nu=\mu/\rho$ is the diffusivity of momentum: motion spreads a distance $\sqrt{\nu t}$ in a time t.

#### The problem in plain words

Open a tap a little and the stream is glassy; open it fully and it turns frothy. Reynolds put a thread of dye into water
flowing through a glass tube and saw the same thing: at low flow the dye stayed a straight thread, at high flow it broke up
and filled the tube. Every exact solution in this chapter is of the first, *laminar* kind — so we need to know when it
applies, and what sets how fast viscosity spreads the motion.

#### The idea

```
laminar  (Re < 2000):   ─────────────── dye stays a thread; layers slide past each other
turbulent (Re > 3000):  ~~~≈≈≈∿∿∿≈≈~~~ dye fills the tube; eddies mix it
Re = U d / ν   (U = mean speed, d = diameter, ν = μ/ρ)
```

| quantity | water (20 °C) | air (20 °C) |
|---|---|---|
| μ [Pa s] | 1.0 × 10⁻³ | 1.8 × 10⁻⁵ |
| ρ [kg/m³] | 1000 | 1.2 |
| ν = μ/ρ [m²/s] | 1.0 × 10⁻⁶ | 1.5 × 10⁻⁵ |
| time to diffuse 1 cm, L²/ν | 100 s | 6.7 s |

**ν, not μ, says how fast motion spreads** — air is 55 times less viscous than water but spreads momentum 15 times faster.
(Rounded textbook values in the table; the code below computes them from Ch. 1's property functions.)

> ⚠️ **Common confusion:** μ (dynamic viscosity, Pa s) is the friction coefficient in $\tau=\mu\,du/dy$; ν = μ/ρ (kinematic viscosity, m²/s) is the
diffusivity. The book's §8.1 once calls μ "the kinematic viscosity" — it means the dynamic one.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **dynamic vs kinematic viscosity μ, ν = μ/ρ** — Newton's law $\tau=\mu\,du/dy$ defines μ as the stress per unit shear rate (Ch. 1 P05, stress); dividing by ρ gives ν.

> 📎 **Primer — diffusivity and the diffusion time L²/ν.** `P185` · A diffusivity D (units m²/s) says how fast something spreads by random molecular exchange: in a time t it spreads a
distance of order $\sqrt{Dt}$, so crossing a distance L takes about $L^2/D$. Heat has $\kappa=k/\rho C_p$, dye its molecular
diffusivity, and momentum has ν. Double the distance and the time quadruples.

In [ ]:
L = [0.001, 0.01, 0.1]                   # distances [m]: 1 mm, 1 cm, 10 cm
nu = 1.0e-6                              # water's kinematic viscosity [m²/s]
print([l**2/nu for l in L])              # diffusion times L²/ν [s]: [1.0, 100.0, 10000.0]

📝 **Note.** **N01 [B] · Heat and momentum spread the same way.** The Boussinesq heat equation below, with the thermal diffusivity $\kappa\equiv k/\rho C_p$, has the same form as
$\frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u$ *(8.1)*; both κ and ν are in m²/s, so ν is the *momentum
diffusivity*. The analogy is not complete: the pressure gradient in (8.1) also moves momentum around, so velocity is not
simply diffused.

$$ \frac{DT}{Dt}=\kappa\nabla^2T \qquad \text{(4.89)} $$

#### 🧮 Derivation — Air diffuses momentum ≈ 15× faster than water: ν = μ/ρ and t ~ L²/ν `D01`

**What we want to show.** Show that the rate at which motion spreads is set by ν = μ/ρ, not μ, and that air — 55 times less viscous than water — spreads momentum about 15 times faster.

**Assumptions.** Room temperature and pressure property values (step 4).

**The plan.**

1. Identify the diffusivity in (8.1), $\frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u$.
2. Turn a diffusion balance into a time scale.
3. Compare air and water with numbers.

**Tools we use** (each explained before this point): diffusivity and the diffusion time (primer P185) · order-of-magnitude scaling (P130) · ratios.

**We start from**

$$ DT/Dt=\kappa\nabla^2T\ \text{(4.89) and}\ D\mathbf u/Dt=-(1/\rho)\nabla p+\nu\nabla^2\mathbf u\ \text{(8.1)} $$

*In words:* in both, a diffusivity multiplies the Laplacian.

---

**Step 1 of 5 — Read the Laplacian's coefficient.**

$$ \nu\equiv\frac{\mu}{\rho}\ \ [\mathrm{m^2/s}] $$

- *Why we can do this:* (8.1), $\frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u$ is the momentum equation divided by ρ; the viscous term μ∇²u becomes (μ/ρ)∇²u, with the units m²/s of a diffusivity like κ.
- *In words:* the diffusivity of momentum is ν, not μ.

---

**Step 2 of 5 — Balance change against diffusion.**

$$ \frac{U}{t_d}\sim\nu\frac{U}{L^2}\ \Rightarrow\ t_d\sim\frac{L^2}{\nu} $$

- *Why we can do this:* Order-of-magnitude scaling: ∂u/∂t ~ U/t and ∂²u/∂y² ~ U/L²; equating them gives the time diffusion needs to cross a distance L.
- *In words:* spreading over L takes about L²/ν.

---

**Step 3 of 5 — Write the ratio of the two ν.**

$$ \frac{\nu_{air}}{\nu_{w}}=\frac{\mu_{air}}{\mu_{w}}\cdot\frac{\rho_{w}}{\rho_{air}} $$

- *Why we can do this:* Divide ν = μ/ρ for air by the same for water; the ratio splits into a viscosity ratio and an inverse density ratio.
- *In words:* two effects compete: viscosity and density.

---

**Step 4 of 5 — Insert room-temperature values.**

$$ \frac{\nu_{air}}{\nu_w}=\frac{1.8\times10^{-5}}{10^{-3}}\cdot\frac{1000}{1.2}= 0.018\times833=15 $$

- *Why we can do this:* μ and ρ at 20 °C (Ch. 1 property functions); air's density is 833 times smaller, which beats its 55 times smaller viscosity.
- *In words:* air's ν is 15 times water's.

---

**Step 5 of 5 — Compare diffusion times across 1 cm.**

$$ t_w=\frac{10^{-4}}{10^{-6}}=100\ \mathrm s,\ \ t_{air}=\frac{10^{-4}}{1.5\times 10^{-5}}=6.7\ \mathrm s $$

- *Why we can do this:* Use step 2 with L = 1 cm; the ratio of times is the inverse ratio of ν.
- *In words:* momentum crosses a centimetre of air 15 times sooner.

---

**Result**

$$ \nu=\frac{\mu}{\rho},\qquad t_d\sim\frac{L^2}{\nu},\qquad \frac{\nu_{air}}{\nu_w}\approx15 $$

*In words:* ν sets how fast motion spreads.

**What it means.** "More viscous" and "more diffusive" are different: honey is both, air is little viscous yet quickly diffusive. Every thickness in this chapter grows like √(νt).

**Check it.** Units: (Pa s)/(kg/m³) = (kg/(m s))/(kg/m³) = m²/s ✓; t = m²/(m²/s) = s ✓. The code cell after the tiny example below computes the ratio from Ch. 1's property functions (Sutherland air, IAPWS water) at 20 °C and prints it.

> ⚠️ **Common confusion (traps in `D01`):** Comparing μ instead of ν; forgetting that the density ratio (≈ 830) is larger than the viscosity ratio (≈ 55).

#### ✏️ Tiny example: water in a 1 cm tube at 10 cm/s

1. $\mathrm{Re}=Ud/\nu=0.1\times0.01/10^{-6}=1000$ — below 2000, laminar.
2. Double the speed: Re = 2000, the bottom of the transition band.
3. Diffusion across the tube: $t=d^2/\nu=10^{-4}/10^{-6}=100$ s — much longer than the 0.1 s the water needs to travel
   1 cm.
4. Same tube with air at 10 cm/s: $\mathrm{Re}=0.1\times0.01/1.5\times10^{-5}=67$, $t=6.7$ s.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **Python dictionaries (fluidpy results)** — fluidpy returns several named results at once as a dictionary: `s["ratio"]` (Ch. 1 P23).
- **f-strings** — `f"{x:.3g}"` puts a number with 3 significant figures into a sentence (Ch. 1 P04).
- **tuple unpacking** — `Re, label = ch08.pipe_flow_regime(...)` gives the two returned values their own names (Ch. 1 P14).
- **assert np.allclose** — stops the notebook if two results differ beyond a tolerance — our proof that a from-scratch version matches the library (Ch. 1 P15).

In [ ]:
for U in (0.1, 0.25, 0.5):                               # three mean speeds [m/s] in a 1 cm tube of water
    Re, label = ch08.pipe_flow_regime(U, 0.01, NU_W)     # Re = Ud/ν and the 2000–3000 band
    print(f"U = {U} m/s: Re = {Re:.0f} → {label}")
nu_air = ch08.momentum_diffusivity("air", 293.15)        # ν of air at 20 °C [m²/s] (Sutherland μ, ideal-gas ρ)
nu_water = ch08.momentum_diffusivity("water", 293.15)    # ν of water at 20 °C [m²/s] (IAPWS μ and ρ)
print(f"ν_air = {nu_air:.4g} m²/s, ν_water = {nu_water:.4g} m²/s, ratio = {nu_air/nu_water:.4g}")   # D01's ratio
print(f"time to diffuse 1 cm: water {ch08.diffusion_time(0.01, nu_water):.1f} s, air {ch08.diffusion_time(0.01, nu_air):.2f} s")

**What does the code above do?**

1. `pipe_flow_regime` computes $\mathrm{Re}=Ud/\nu$ and applies the 2000–3000 band: 1000 laminar, 2500 transitional, 5000
   turbulent.
2. `momentum_diffusivity` takes μ and ρ from Ch. 1's property functions (Sutherland's law for air, IAPWS for water) and
   returns ν = μ/ρ. The ratio printed here — **15.01** — is the number D01's step 4 estimated by hand as 15.
3. `diffusion_time` is $L^2/\nu$: about 100 s for a centimetre of water, 6.6 s for a centimetre of air.

**From scratch — the same regime labels and diffusion time by hand**, then a check that the library does exactly this:

In [ ]:
Us = (0.1, 0.25, 0.5)                                    # the same three speeds [m/s]
mine = [U*0.01/NU_W for U in Us]                         # Re = U d / ν with d = 1 cm
labels = ["laminar" if R < 2000 else ("turbulent" if R > 3000 else "transitional") for R in mine]   # the band
print(mine, labels)
assert np.allclose(mine, [ch08.pipe_flow_regime(U, 0.01, NU_W)[0] for U in Us], rtol=1e-12)   # same Re
assert labels == [ch08.pipe_flow_regime(U, 0.01, NU_W)[1] for U in Us]                         # same verdicts
assert np.allclose(0.01**2/nu_air, ch08.diffusion_time(0.01, nu_air), rtol=1e-12)              # L²/ν by hand

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **np.random.default_rng (dye sketch)** — `rng = np.random.default_rng(1)` makes reproducible random numbers; we use it to draw a schematic chaotic dye streak (Ch. 1 P10).
- **matplotlib figures** — `fig, ax = plt.subplots()`, `ax.plot`, labels with units; every figure is followed by What you see / How to read it / What would change if (Ch. 1 P01).
- **power laws and log–log plots** — on log–log axes a power law $t\propto L^2$ is a straight line of slope 2 (Ch. 1 P13).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw=dict(width_ratios=[1.3, 1]))   # two panels
x = np.linspace(0, 1, 400)                               # position along the tube [m]
for yc, U, lab in ((0.6, 0.1, "laminar"), (-0.6, 0.5, "turbulent")):     # two tubes, one above the other [schematic units]
    a1.plot([0, 1], [yc + 0.3, yc + 0.3], color=COLORS["ink"], lw=1.5)   # upper tube wall
    a1.plot([0, 1], [yc - 0.3, yc - 0.3], color=COLORS["ink"], lw=1.5)   # lower tube wall
    Re, verdict = ch08.pipe_flow_regime(U, 0.01, NU_W)   # the regime from the real Re of a 1 cm water pipe
    if verdict == "laminar":                             # a straight dye thread on the axis
        a1.plot(x, yc + 0*x, color=COLORS["blue"], lw=2)
    else:                                                # a seeded random walk that fills the tube (schematic only)
        rng = np.random.default_rng(1)                   # reproducible "chaos"
        yy = np.clip(np.cumsum(rng.normal(0, 0.03, x.size)) * (x > 0.25), -0.28, 0.28)   # starts to wander at x = 0.25 m
        a1.plot(x, yc + yy, color=COLORS["blue"], lw=1)
    a1.text(0.02, yc + 0.34, f"Re = {Re:.0f}: {verdict}", fontsize=9)    # label with the computed Re
a1.set_xlim(0, 1); a1.set_ylim(-1.1, 1.2); a1.set_yticks([])            # schematic: no vertical scale
a1.set_xlabel("distance along the tube [m]"); a1.set_title("Reynolds's dye (schematic — not a simulation)", fontsize=10)
L = np.geomspace(1e-5, 1.0, 50)                          # distances from 10 µm to 1 m [m]
for nu, c, name in ((nu_water, COLORS["blue"], "water"), (nu_air, COLORS["teal"], "air")):
    a2.loglog(L, ch08.diffusion_time(L, nu), color=c, lw=2, label=f"{name}, ν = {nu:.2g} m²/s")   # t = L²/ν
    a2.plot(0.01, ch08.diffusion_time(0.01, nu), "o", color=c)           # the 1 cm point
a2.loglog(L, 1e3*L**2, color=COLORS["muted"], ls="--", lw=1, label="slope 2")   # guide line t ∝ L²
a2.set_xlabel("distance L [m]"); a2.set_ylabel("diffusion time L²/ν [s]"); a2.legend(fontsize=8)
a2.set_title("Momentum spreads as √(νt): time ∝ L²", fontsize=10)
fig.suptitle("Smooth below Re ≈ 2000; momentum spreads as √(νt)", fontsize=11)
savefig(fig, "ch08", "c01_reynolds_diffusion"); plt.show()

**What you see.** (a) a straight dye thread at Re = 1000 and a mixed tube at Re = 5000 — our synthetic sketch of Reynolds's
apparatus `N103`, labelled schematic; (b) two parallel lines of slope 2, air's 15 times lower than water's.

**How to read it.** On (b) read the time to diffuse a distance: 1 cm takes about 100 s in water and 6.6 s in air (the dots); the slope 2
says ten times farther takes a hundred times longer.

**What would change if…** …the fluid were glycerine (ν ≈ 10⁻³ m²/s): its line drops by 1000 below water's — 1 cm in 0.1 s — and Re at
10 cm/s in the 1 cm tube is 1, deeply laminar.

📝 **Note.** **N02 [C] · Where this leads.** When viscosity matters only in thin layers next to walls, we get boundary layers (Ch. 9); whether these laminar flows
survive small disturbances is Ch. 11's question; fully turbulent flow is Ch. 12; viscous layers in a rotating fluid — the
Ekman layers of the ocean and atmosphere — are Ch. 13. The $\sqrt{\nu t}$ spreading of this chapter reappears in all four.

> 🔁 **Recap — The Navier–Stokes equation for constant ρ and μ** (Ch. 4 §4.6). Every solution below satisfies continuity $\nabla\cdot\mathbf u=0$ *(4.10)* and
$\frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u$ *(8.1)*, which is Ch. 4's
$\rho\frac{D\mathbf u}{Dt}=-\nabla p'+\mu\nabla^2\mathbf u$ *(4.85)* divided by ρ (gravity absorbed into p, R06). We check
each one with Ch. 4's term calculator: plug the velocity and pressure in and the residual must vanish.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **functions as arguments and lambda** — `lambda x, t=0.0: …` is a one-line function; Ch. 4's calculator takes the velocity and pressure *as functions* and differentiates them numerically (Ch. 1 P29).
- **component-first array layout (x[0], x[1])** — a point is an array whose first index is the component: `x[0]` is x, `x[1]` is y, `x[2]` is z (Ch. 2 P76, the project grid layout).

In [ ]:
u_fn = lambda x, t=0.0: np.stack([ch08.channel_flow(x[1], 0.01, U=0.1, dpdx=-4.0), 0*x[0], 0*x[0]])   # C02's channel flow u(y)
p_fn = lambda x, t=0.0: -4.0 * x[0]                      # a linear pressure p = −4x [Pa]: dp/dx = −4 Pa/m
terms = NS.ns_incompressible_terms(u_fn, p_fn, np.array([0.3, 0.004, 0.0]), mu=MU_W, g=(0.0, 0.0, 0.0))   # at one point
print("advective ", terms.advective)                    # u·∇u [m/s²]: exactly zero (C02 explains why)
print("pressure  ", terms.pressure)                     # −(1/ρ)∂p/∂x = +4 × 10⁻³ m/s²
print("viscous   ", terms.viscous)                      # ν ∂²u/∂y² = −4 × 10⁻³ m/s²
print(f"residual   max |sum| = {np.max(np.abs(terms.residual)):.1e} m/s²")   # their sum: ≈ 0 (round-off)

**What does the code above do?**

1. `u_fn` is the channel flow of C02 written as a function of position (component-first: `x[1]` is y); `p_fn` is a pressure
   falling 4 Pa per metre.
2. `ns_incompressible_terms` (Ch. 4) evaluates every term of (8.1), $\frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u$ at the point (0.3, 0.004, 0) m by finite differences.
3. The pressure term $+4\times10^{-3}$ m/s² and the viscous term $-4\times10^{-3}$ m/s² cancel; the advective term is exactly
   zero; the residual is at round-off (10⁻¹⁵). C02 derives this profile; here we only see that it passes the test.

> 🔁 **Recap — No flow through a wall** (Ch. 4 §4.10). At a solid surface moving with velocity $\mathbf U_s$ the normal velocities agree:
$\mathbf n\cdot\mathbf U_s=(\mathbf n\cdot\mathbf u)_{\text{on the surface}}$ *(8.2)*, with **n** the unit normal (length 1,
perpendicular to the surface).

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **unit normal and tangent vectors** — a unit vector has length 1; the normal **n** points out of the wall, a tangent **t** lies along it, and $\mathbf n\cdot\mathbf u$ picks the component of **u** along **n** (Ch. 2 P65, projection).

> 🔁 **Recap — No slip** (Ch. 4 §4.10). The tangential velocities agree too: $\mathbf t\cdot\mathbf U_s=(\mathbf t\cdot\mathbf u)_{\text{on the surface}}$ *(8.3)* —
viscous fluid sticks to walls. Every constant of integration in this chapter is fixed by (8.2), $\mathbf n\cdot\mathbf U_s=(\mathbf n\cdot\mathbf u)_{\text{on the surface}}$ and (8.3).

In [ ]:
n = np.array([0.0, 1.0, 0.0])                            # unit normal of a floor y = 0 (pointing into the fluid)
wall = np.array([0.1, 0.0, 0.0])                         # the floor slides at 0.1 m/s along x
print(ch08.wall_bc_residuals(np.array([0.1, 0, 0]), wall, n))      # fluid moves with the wall: (normal, tangential) = (0, 0)
print(ch08.wall_bc_residuals(np.array([0.08, 0.01, 0]), wall, n))  # a leaky, slipping wall: both residuals non-zero [m/s]

**What does the code above do?**

`wall_bc_residuals(u_wall, U_s, n)` returns how much (8.2), $\mathbf n\cdot\mathbf U_s=(\mathbf n\cdot\mathbf u)_{\text{on the surface}}$ and (8.3), $\mathbf t\cdot\mathbf U_s=(\mathbf t\cdot\mathbf u)_{\text{on the surface}}$ are violated: the normal mismatch
$\mathbf n\cdot(\mathbf u-\mathbf U_s)$ and the size of the tangential mismatch. The first call obeys both; the second has
fluid leaking through (0.01 m/s) and slipping (0.02 m/s).

> 🔁 **Recap — Standing assumptions** (Ch. 4 §4.9). Constant density, an inertial frame, and gravity absorbed into the pressure ($p\to p+\rho gz$) whenever there is no free
surface — only the spreading film (C08) and the settling sphere (C14) bring g back.

---

## 8.2 Exact Solutions for Steady Incompressible Viscous Flow

**What is this section about?** Three flows where symmetry removes the nonlinear term $\mathbf u\cdot\nabla\mathbf u$
exactly, so the Navier–Stokes equation becomes a linear ODE that we can integrate by hand: flow between parallel plates
driven by a moving wall and a pressure gradient, flow in a round pipe, and flow between two rotating cylinders. The same
short argument — continuity kills one velocity, "a function of x equals a function of y" makes the pressure gradient
constant — solves all three.

> 🔁 **Recap — Plane Couette flow** (Ch. 1 §1.5). A fluid between a fixed plate and a plate sliding at U, with no pressure gradient, moves with the straight profile
$u=Uy/h$ and a uniform shear stress $\tau=\mu\,du/dy=\mu U/h$ (Ch. 1). C02 shows it is one half of a two-part formula.

> 🔁 **Recap — Plane Poiseuille flow** (Ch. 4 §4.6). With both plates at rest and a pressure gradient, the profile is the parabola $u(y)=-\frac1{2\mu}\frac{dp}{dx}y(h-y)$, the
stress $\tau=\mu\frac{du}{dy}=-\big(\frac h2-y\big)\frac{dp}{dx}$ is linear and its magnitude at both walls is
$\frac h2\lvert dp/dx\rvert$. Ch. 4's `plane_poiseuille` takes $G=-dp/dx$.

In [ ]:
y = np.linspace(0, 0.01, 5)                              # five heights across a 1 cm channel [m]
print(ch04.plane_poiseuille(y, G=2.0, h=0.01))           # Ch. 4: G = −dp/dx = +2 Pa/m  → u [m/s]
print(ch08.channel_flow(y, 0.01, U=0.0, dpdx=-2.0))      # Ch. 8: the book's dp/dx = −2 Pa/m, fixed walls → the same u

> ⚠️ **Common confusion:** the book's $dp/dx=-2$ Pa/m (pressure falling downstream, a *favourable* gradient) is Ch. 4's $G=+2$ Pa/m. Same flow,
opposite sign of the argument — the two identical rows above pin it.

### 🧩 Plane Couette–Poiseuille flow `C02`

*The question:* What profile results from a moving wall plus a pressure gradient — and when does fluid near the fixed wall flow backwards?

*In one line:* $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$ *(8.5)*

#### The problem in plain words

A belt drags oil along a channel while a pump pushes back. Near the belt the oil is carried forward; near the fixed floor
the pump may win and push it backwards. Journal bearings, extrusion dies, the wind-driven surface layer of a lake with a
return current underneath — all are this flow. We want the exact profile and the condition for the reversed current.

#### The idea

```
y=h ═══════════► U        y=h ═════════        y=h ═══════════► U
     ─────────►                 ──►                    ─────────►
     ──────►        +           ────►        =         ──────►
     ───►                       ──►                    ─►      (backflow if the parabola
y=0 ═════════             y=0 ═════════        y=0 ═══◄══       is pushed the other way)
  Couette: line            Poiseuille: parabola          sum (8.5)
```

*The numbered equations in the sketch:* $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$ *(8.5)*

Because the equation turns out linear, the two causes simply **add**.

📝 **Note.** **N03 [B] · Fully developed flow.** Near the channel inlet, boundary layers grow on both walls and the profile changes with x — inside this *entrance length*
$\partial u/\partial x\ne0$, so continuity $\partial u/\partial x+\partial v/\partial y=0$ forces $v\ne0$ and the flow is not
parallel. Downstream the layers merge and the profile stops changing: u = u(y) alone. The figure below draws the wall-layer
edge with the diffusion estimate $\delta_{99}\sim3.64\sqrt{\nu t}$ *(8.31)* and t = x/U — **our estimate** (C09 derives the
3.64), not the book's; Ch. 9 treats entry flow properly.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **scipy.optimize.brentq** — `optimize.brentq(f, a, b)` finds the root of f between a and b where f changes sign (Ch. 3 P108); here: where the wall layer reaches mid-gap.

In [ ]:
h, Uin = 0.01, 0.01                                       # gap 1 cm, inflow speed 1 cm/s [m, m/s]
edge = lambda x: ch08.diffusion_thickness(x/Uin, NU_W)    # wall-layer edge δ(x) = 3.643√(ν x/U) [m] (our estimate)
x_e = optimize.brentq(lambda x: edge(x) - h/2, 1e-6, 1.0)  # where the two layers meet at mid-gap [m]
print(f"layers meet at x_e = {x_e:.4f} m (about {x_e/h:.1f} gap widths)")
x = np.linspace(1e-5, 0.05, 400)                          # distance from the inlet [m]
fig, ax = plt.subplots(figsize=(8, 3.2))
channel(ax, h, U=0.0, L=0.05)                             # two fixed walls (drawing only)
xl = x[x <= x_e]                                          # the edges exist only until they meet
ax.plot(xl, edge(xl), color=COLORS["accent"], ls="--", lw=1.5, label="wall-layer edge (our estimate)")   # lower wall's layer
ax.plot(xl, h - edge(xl), color=COLORS["accent"], ls="--", lw=1.5)                                     # the upper wall's layer
ax.axvline(x_e, color=COLORS["muted"], lw=0.8); ax.text(x_e + 5e-4, 0.0105, "x_e", fontsize=8)
yy = np.linspace(0, h, 41)                                # heights for the three profiles [m]
V = Uin                                                   # the mean speed is conserved along the channel [m/s]
for x0 in (0.003, 0.012):                                 # two developing stations: plug core, erf-shaped wall layers
    d = np.sqrt(NU_W*x0/Uin)                              # diffusion length √(ν x/U) there [m]
    shape = special.erf(yy/(2*d)) * special.erf((h - yy)/(2*d))   # schematic developing profile (0 at both walls)
    prof = V * shape / np.trapezoid(shape, yy) * h        # scaled to the same mean speed V
    profile_arrows(ax, yy, prof, x0=x0, scale=0.5, every=4, color=COLORS["blue"])
u_dev = ch08.channel_flow(yy, h, U=0.0, dpdx=-12*MU_W*V/h**2)   # fully developed: Poiseuille with mean V ((8.5), U = 0)
profile_arrows(ax, yy, u_dev, x0=0.035, scale=0.5, every=4, color=COLORS["orange"])
ax.set_xlim(0, 0.05); ax.set_xlabel("distance from the inlet x [m]"); ax.set_ylabel("y [m]")
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Entrance region: wall layers grow like √x and meet; then u = u(y) only", fontsize=10)
savefig(fig, "ch08", "c02_entrance"); plt.show()

**What you see.** Two dashed edges growing like √x from the walls and meeting at $x_e$ (printed above, about 1.9 cm — about twice the
gap); blue profiles still developing (flat core, thin wall layers), then the orange parabola of the fully developed flow.

**How to read it.** Left of $x_e$ the core still moves as a plug; right of it the profile no longer changes with x — it is the parabola of
(8.5), $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$ with U = 0 — and every result of this section applies.

**What would change if…** …the inflow were 10 times faster: the edge $\delta\propto\sqrt{\nu x/U}$ grows √10 times more slowly, so the entrance
length grows tenfold ($x_e\propto Uh^2/\nu$).

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **partial derivative** — ∂u/∂x is the rate of change of u with x while y, z, t are held fixed (Ch. 1 P25).

📝 **Note.** **N04 [B] · Continuity kills v.** With $\partial u/\partial x=0$, continuity gives $\partial v/\partial y=0$; v is then the same at every height and equals
its wall value 0: $\mathbf u=(u(y),0,0)$. This is steps 1–3 of the derivation below.

📝 **Note.** **N05 [B], N06 [B] · The reduced equations.** What is left of the momentum equations is (8.4a) and (8.4b) below: no acceleration at all — the advective term vanishes
*exactly*, which is why the problem is linear.

$$ 0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2},\qquad 0=-\frac1\rho\frac{\partial p}{\partial y} \qquad \text{(8.4a, 8.4b)} $$

#### 🧮 Derivation — v ≡ 0 and the reduced equations $0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2}$, $0=-\frac1\rho\frac{\partial p}{\partial y}$ `D02` (Eq. 8.4a: $0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2}$)

**What we want to show.** Show that in steady, fully developed flow between plates the vertical velocity vanishes and Navier–Stokes shrinks to two short linear equations.

**Assumptions.** Steady (step 5); fully developed ∂u/∂x = 0 (step 1); 2-D, w = 0, ∂/∂z = 0 (step 1); impermeable walls (step 3).

**The plan.**

1. Use "fully developed" in continuity.
2. Fix v with the walls.
3. Drop every term that vanishes.

**Tools we use** (each explained before this point): partial derivative (P25) · continuity (R03) · no through-flow (8.2), $\mathbf n\cdot\mathbf U_s=(\mathbf n\cdot\mathbf u)_{\text{on the surface}}$ (R04).

**We start from**

$$ \frac{\partial u}{\partial x}+\frac{\partial v}{\partial y}=0\ \text{((4.10) in 2-D)},\qquad \frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u\ \text{(8.1)} $$

*In words:* mass and momentum for the fluid between the plates.

---

**Step 1 of 7 — Use the fully developed condition.**

$$ \frac{\partial u}{\partial x}=0\ \Rightarrow\ u=u(y) $$

- *Why we can do this:* Far past the entrance the profile no longer changes along the channel; with w = 0 and ∂/∂z = 0 the flow is 2-D.
- *In words:* u depends on height only.

---

**Step 2 of 7 — Put it into continuity.**

$$ \frac{\partial v}{\partial y}=0 $$

- *Why we can do this:* (4.10), $\nabla\cdot\mathbf u=0$ with ∂u/∂x = 0 leaves only the v-term; we want to learn what this says about v.
- *In words:* v does not change with height.

---

**Step 3 of 7 — Use the impermeable walls.**

$$ v(x,0)=0\ \Rightarrow\ v\equiv0 $$

- *Why we can do this:* A function of y with zero y-derivative is the same at every height, and (8.2), $\mathbf n\cdot\mathbf U_s=(\mathbf n\cdot\mathbf u)_{\text{on the surface}}$ makes it zero at the wall — so it is zero everywhere.
- *In words:* the flow is parallel to the walls: u = (u(y), 0, 0).

---

**Step 4 of 7 — Evaluate the advective acceleration.**

$$ u\frac{\partial u}{\partial x}+v\frac{\partial u}{\partial y}=u\cdot0+0\cdot \frac{du}{dy}=0 $$

- *Why we can do this:* Both factors vanish by steps 1 and 3 — exactly, not approximately; this is why the problem becomes linear.
- *In words:* no fluid particle accelerates.

---

**Step 5 of 7 — Write the steady x-momentum.**

$$ 0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\Big(\frac{\partial^2u}{\partial x^2}+ \frac{\partial^2u}{\partial y^2}\Big) $$

- *Why we can do this:* The x-component of (8.1), $\frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u$ with ∂u/∂t = 0 (steady) and step 4.
- *In words:* pressure and friction balance.

---

**Step 6 of 7 — Drop the x-curvature.**

$$ 0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2}\ \text{(8.4a)} $$

- *Why we can do this:* u depends on y only, so ∂²u/∂x² = 0 and the partial y-derivative is an ordinary one.
- *In words:* (8.4a), $0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2}$: the pressure gradient is balanced by cross-channel friction.

---

**Step 7 of 7 — Write the y-momentum.**

$$ 0=-\frac1\rho\frac{\partial p}{\partial y}\ \text{(8.4b)} $$

- *Why we can do this:* Every v-term in the y-component of (8.1), $\frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u$ is zero because v ≡ 0; only the pressure term survives.
- *In words:* (8.4b), $0=-\frac1\rho\frac{\partial p}{\partial y}$: pressure does not change across the channel.

---

**Result**

$$ 0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2}\ \text{(8.4a)},\qquad 0=-\frac1\rho\frac{\partial p}{\partial y}\ \text{(8.4b)} $$

*In words:* a steady parallel flow is a pure pressure–friction balance.

**What it means.** Because the nonlinear term vanishes identically, the solution will be exact, not approximate. In the entrance region (∂u/∂x ≠ 0) this fails: v ≠ 0 (N03).

**Check it.** Units of (8.4a), $0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2}$: (1/ρ)∂p/∂x in (m³/kg)(Pa/m) = m/s², ν d²u/dy² in (m²/s)(1/(m s)) = m/s² ✓. The same steps applied in `ch08.parallel_flow_sympy("channel")` (the code cell after the tiny example of C02) return these two equations.

> ⚠️ **Common confusion (traps in `D02`):** ∂v/∂y = 0 alone only says v is constant across the gap — the wall value fixes it to zero; the advective term is zero exactly, not "small".

📝 **Note.** **N07 [B] · Separation of variables in one line.** From $0=-\frac1\rho\frac{\partial p}{\partial y}$ *(8.4b)* p depends on x only; then the balance below says "a function of
x = a function of y", so both equal one constant: the pressure falls **linearly** along the channel. The same argument
returns in the pipe (C03), the rotating cylinders (C04) and the lubrication gap (C06).

$$ \frac{1}{\mu}\frac{dp}{dx}=\frac{d^2u}{dy^2}=\text{const}  $$

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **separation of variables** — if f(x) = g(y) for all x and y, both sides must equal the same constant (Ch. 1 P42).

#### 🧮 Derivation — The pressure gradient is constant: $\frac1\mu\frac{dp}{dx}=\frac{d^2u}{dy^2}=\text{const}$ `D03` (Eq. 8.4a: $0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2}$)

**What we want to show.** Show that in fully developed channel flow the pressure falls linearly along the channel.

**Assumptions.** Steady (p does not depend on t).

**The plan.**

1. Learn from (8.4b), $0=-\frac1\rho\frac{\partial p}{\partial y}$ what p depends on.
2. Separate (8.4a), $0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2}$ into an x-side and a y-side.
3. Differentiate each side by the other variable.

**Tools we use** (each explained before this point): partial derivative (P25) · separation of variables (P42).

**We start from**

$$ 0=-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{d^2u}{dy^2}\ \text{(8.4a) and}\ 0=-\frac1\rho\frac{\partial p}{\partial y}\ \text{(8.4b)} $$

*In words:* the reduced equations of D02.

---

**Step 1 of 5 — Read (8.4b).**

$$ p=p(x)\ \Rightarrow\ \frac{\partial p}{\partial x}=\frac{dp}{dx}(x) $$

- *Why we can do this:* ∂p/∂y = 0 means p is the same at every height, so its x-derivative depends on x only.
- *In words:* the pressure gradient can vary only along the channel.

---

**Step 2 of 5 — Multiply (8.4a) by ρ and rearrange.**

$$ \frac{dp}{dx}(x)=\mu\frac{d^2u}{dy^2}(y) $$

- *Why we can do this:* ρν = μ; we isolate the pressure side, a function of x alone, and the velocity side, a function of y alone.
- *In words:* an x-function equals a y-function at every point.

---

**Step 3 of 5 — Differentiate both sides by x.**

$$ \frac{d^2p}{dx^2}=0 $$

- *Why we can do this:* The right side does not contain x, so its x-derivative is zero; hence so is the left side's.
- *In words:* the pressure gradient does not change along x.

---

**Step 4 of 5 — Differentiate both sides by y.**

$$ \mu\frac{d^3u}{dy^3}=0 $$

- *Why we can do this:* The left side does not contain y; so the y-derivative of the right side vanishes too — it is also a constant.
- *In words:* the curvature of the profile is the same at every height.

---

**Step 5 of 5 — Name the common constant.**

$$ \frac1\mu\frac{dp}{dx}=\frac{d^2u}{dy^2}=\text{const},\ \ p=p_0+\frac{dp}{dx}x $$

- *Why we can do this:* Both sides are constants and equal each other; integrating d²p/dx² = 0 once more gives a linear pressure.
- *In words:* pressure falls (or rises) linearly, the profile has constant curvature.

---

**Result**

$$ \frac1\mu\frac{dp}{dx}=\frac{d^2u}{dy^2}=\text{const} $$

*In words:* one constant sets both the pressure slope and the profile's curvature.

**What it means.** The same "x-function = y-function" argument will fix the pressure gradient in the pipe (D06), between rotating cylinders (no pressure in the φ-equation, D08) and across the lubrication gap (D12).

**Check it.** A pressure p = x² would make the left side 2x/μ, which cannot equal a function of y at every x — `parallel_flow_sympy` confirms no solution exists for non-constant dp/dx.

> ⚠️ **Common confusion (traps in `D03`):** Differentiate each side by the *other* variable; the constant can have either sign (favourable or adverse).

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **integrating an ODE twice and fixing two constants** — a second-order linear ODE has a two-parameter family of solutions; two conditions pick one (Ch. 1 P44, linear second-order ODE).
- **fundamental theorem of calculus** — integrating a derivative gives back the function plus a constant: $\int u''\,dy=u'+c_1$ (Ch. 2 P84).
- **sympy and sympy.dsolve** — `sp.dsolve(sp.Eq(lhs, rhs))` solves an ODE symbolically; `C1`, `C2` in its answer are the free constants (Ch. 1 P40).
- **two linear equations in two unknowns, np.linalg.solve** — `np.linalg.solve(M, b)` returns x with M x = b (Ch. 2 P57); we use it for two wall conditions.

*Integrating an ODE twice.* Integrating $u''=c$ twice brings two constants, $u=cy^2/2+c_1y+c_2$; two wall conditions fix them (a 2 × 2 linear system).
sympy's `dsolve` does it in one line:

In [ ]:
y_, c_ = sp.symbols("y c")                               # height and the constant curvature (symbols)
u_ = sp.Function("u")                                    # the unknown profile u(y)
print(sp.dsolve(sp.Eq(u_(y_).diff(y_, 2), c_)))          # u'' = c  →  u = C1 + C2·y + c·y²/2

📝 **Note.** **N08 [B] · The book's constants.** The book integrates to the line below with B = 0 and $A=\frac h2\frac{dp}{dx}-\frac{\mu U}{h}$; its A and B are minus our
$c_1$ and $c_2$ (D04 step 3).

$$ 0=-\frac{y^2}{2}\frac{dp}{dx}+\mu u+Ay+B  $$

#### 🧮 Derivation — Couette–Poiseuille profile $u(y)=\frac Uhy-\frac1{2\mu}\frac{dp}{dx}y(h-y)$ `D04` (Eq. 8.5: $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$)

**What we want to show.** Find the velocity at every height between a fixed floor and a top plate moving at U, with a constant pressure gradient.

**Assumptions.** dp/dx constant (D03); no slip at both plates (steps 4–5).

**The plan.**

1. Integrate twice.
2. Apply the two no-slip conditions.
3. Regroup into the book's form.

**Tools we use** (each explained before this point): integrating an ODE twice (gloss in C02) · 2 × 2 linear system (P57) · no slip (8.3), $\mathbf t\cdot\mathbf U_s=(\mathbf t\cdot\mathbf u)_{\text{on the surface}}$ (R05).

**We start from**

$$ \mu\frac{d^2u}{dy^2}=\frac{dp}{dx}=\text{const (D03)},\qquad u(0)=0,\ \ u(h)=U $$

*In words:* a constant curvature and two wall speeds.

---

**Step 1 of 9 — Integrate once in y.**

$$ \mu\frac{du}{dy}=\frac{dp}{dx}y+c_1 $$

- *Why we can do this:* Fundamental theorem of calculus; dp/dx is a constant (D03), so its antiderivative is (dp/dx)y plus a constant c₁.
- *In words:* the shear stress changes linearly with height.

---

**Step 2 of 9 — Integrate again.**

$$ \mu u=\frac{dp}{dx}\frac{y^2}{2}+c_1y+c_2 $$

- *Why we can do this:* Integrate each term once more; a second constant c₂ appears.
- *In words:* the profile is a parabola plus a straight line.

---

**Step 3 of 9 — Compare with the book's constants.**

$$ 0=-\frac{y^2}{2}\frac{dp}{dx}+\mu u+Ay+B,\ \ A=-c_1,\ B=-c_2 $$

- *Why we can do this:* Moving everything to one side gives the book's line; its A and B are minus our constants — worth saying to avoid sign confusion.
- *In words:* same equation, constants named differently.

---

**Step 4 of 9 — Apply no slip at the floor.**

$$ u(0)=0\ \Rightarrow\ c_2=0 $$

- *Why we can do this:* (8.3), $\mathbf t\cdot\mathbf U_s=(\mathbf t\cdot\mathbf u)_{\text{on the surface}}$ on the fixed plate; at y = 0 the first two terms vanish, leaving c₂ = 0.
- *In words:* the fluid at the floor is at rest.

---

**Step 5 of 9 — Apply no slip at the top.**

$$ \mu U=\frac{dp}{dx}\frac{h^2}{2}+c_1h $$

- *Why we can do this:* (8.3), $\mathbf t\cdot\mathbf U_s=(\mathbf t\cdot\mathbf u)_{\text{on the surface}}$ on the moving plate: at y = h the fluid moves at U.
- *In words:* one equation for c₁.

---

**Step 6 of 9 — Solve for c₁.**

$$ c_1=\frac{\mu U}{h}-\frac h2\frac{dp}{dx} $$

- *Why we can do this:* Divide step 5 by h and subtract; the book's A = −c₁ = (h/2)dp/dx − μU/h matches the page.
- *In words:* the wall stress constant mixes drag and pressure.

---

**Step 7 of 9 — Substitute the constants.**

$$ \mu u=\frac{dp}{dx}\frac{y^2}{2}+\Big(\frac{\mu U}{h}-\frac h2\frac{dp}{dx}\Big)y $$

- *Why we can do this:* Insert c₁ and c₂ = 0 into step 2 to get the profile.
- *In words:* the profile with both constants fixed.

---

**Step 8 of 9 — Divide by μ and group.**

$$ u=\frac Uhy+\frac{1}{2\mu}\frac{dp}{dx}(y^2-hy) $$

- *Why we can do this:* Collect the two dp/dx terms: y²/2 − hy/2 = (y² − hy)/2.
- *In words:* a line plus a parabola.

---

**Step 9 of 9 — Rewrite the parabola.**

$$ u(y)=\frac Uhy-\frac1{2\mu}\frac{dp}{dx}y(h-y)\ \text{(8.5)} $$

- *Why we can do this:* y² − hy = −y(h − y); this form makes the parabola positive inside the gap when dp/dx < 0.
- *In words:* (8.5), $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$.

---

**Result**

$$ u(y)=\frac Uhy-\frac1{2\mu}\frac{dp}{dx}y(h-y) $$

*In words:* wall drag (a line) plus pressure push (a parabola).

**What it means.** Because the equation is linear, causes simply add; the book's Fig. 8.4 is four cases of one formula.

**Check it.** u(0) = 0 ✓, u(h) = U ✓; units of (1/μ)(dp/dx)y² : (1/(Pa s))(Pa/m)m² = m/s ✓; dp/dx = 0 gives Couette, U = 0 gives Poiseuille (D05). Numbers: h = 1 cm, U = 0.1 m/s, dp/dx = 4 Pa/m at y = h/4: 0.025 − 0.0375 = −0.0125 m/s.

> ⚠️ **Common confusion (traps in `D04`):** The book's A, B have the opposite sign to c₁, c₂; dp/dx < 0 (favourable) gives a *forward* parabola because of the minus sign; check both walls at the end.

#### ✏️ Tiny example: a 1 cm channel, belt at 10 cm/s, adverse gradient 4 Pa/m

Take h = 0.01 m, U = 0.1 m/s, μ = 10⁻³ Pa s, dp/dx = +4 Pa/m (pressure rising downstream), at the quarter height
y = h/4 = 0.0025 m.
1. Couette part: $Uy/h=0.1\times0.25=0.025$ m/s.
2. Poiseuille part: $-\frac{1}{2\mu}\frac{dp}{dx}y(h-y)=-\frac{4}{2\times10^{-3}}\times0.0025\times0.0075=-0.0375$ m/s.
3. Sum: u = −0.0125 m/s — the fluid moves **backwards** at the quarter height.
4. Threshold: $2\mu U/h^2=2\times10^{-3}\times0.1/10^{-4}=2$ Pa/m; we are at twice it.
5. Flow rate $Q=\frac{Uh}2-\frac{h^3}{12\mu}\frac{dp}{dx}=5\times10^{-4}-3.33\times10^{-4}=1.67\times10^{-4}$ m²/s: still
   forward overall.

In [ ]:
h, U = 0.01, 0.1                                         # gap [m] and wall speed [m/s] of the tiny example
y = np.linspace(0, h, 9)                                 # nine heights, y = 0, h/8, …, h [m]
u = ch08.channel_flow(y, h, U=U, dpdx=4.0)               # (8.5) with an adverse gradient dp/dx = +4 Pa/m
print("u(y) =", np.round(u, 4), "m/s   (u at y = h/4 is the third entry)")
Q, V = ch08.channel_flow_rate(h, U=U, dpdx=4.0)          # flow rate per unit width [m²/s] and mean speed Q/h [m/s]
print(f"Q = {Q:.4g} m²/s, V = {V:.4g} m/s, threshold 2μU/h² = {ch08.channel_backflow_threshold(U, h):.3g} Pa/m")
s = ch08.couette_poiseuille_state(h, U, 4.0)             # everything the explainer shows, in one dictionary
print({k: (round(s[k], 6) if not isinstance(s[k], bool) else s[k]) for k in
       ("Q_couette", "Q_poiseuille", "tau_bottom", "tau_top", "backflow", "y_reversal", "zero_flow_dpdx")})
ps = ch08.parallel_flow_sympy("channel")                 # D02–D04 done symbolically
print("sympy profile:", ps["profile"], "   (G here is the symbol for dp/dx)")
print("reduced x-momentum (8.4a):", ps["momentum_x"])   # with p = p(x): the y-equation (8.4b) is satisfied identically

**What does the code above do?**

1. `channel_flow` evaluates (8.5), $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$; at y = h/4 it gives −0.0125 m/s, the tiny example's number.
2. `channel_flow_rate` integrates (8.5) exactly: Q = 1.667 × 10⁻⁴ m²/s and V = Q/h.
3. `couette_poiseuille_state` bundles the flow rate split into its two parts ($Uh/2$ and $-h^3(dp/dx)/12\mu$), the signed
   wall stresses (−0.01 Pa at the floor, +0.03 Pa at the top), the verdict `backflow = True`, where u changes sign —
   y = 5 mm, the reversed layer fills the lower half — and the gradient that stops the net flow, $6\mu U/h^2$ = 6 Pa/m.
4. `parallel_flow_sympy("channel")` repeats D02–D04 symbolically: its profile, written with the symbol G for dp/dx, is
   $\frac{Uy}{h}+\frac{G}{2\mu}(y^2-hy)$ — exactly (8.5).

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **finite differences (tridiagonal solve from scratch)** — $u''\approx(u_{i+1}-2u_i+u_{i-1})/\Delta y^2$ turns the ODE into one linear equation per grid point (Ch. 1 P21).

*Two numpy helpers.* `np.diag(v, k)` builds a matrix with the vector v on its k-th diagonal (k = 0 main, ±1 just above or
below); `np.r_[a, b, c]` glues numbers and arrays end to end into one array.

In [ ]:
N = 101; yy = np.linspace(0, h, N); dy = yy[1] - yy[0]   # 101 grid points across the gap [m]
Amat = (np.diag(-2*np.ones(N-2)) + np.diag(np.ones(N-3), 1) + np.diag(np.ones(N-3), -1)) / dy**2   # u'' on interior points
rhs = np.full(N-2, 4.0/MU_W)                             # μ u'' = dp/dx  →  u'' = (dp/dx)/μ at every interior point
rhs[-1] -= U/dy**2                                       # the known top-wall value u(h) = U moves to the right side
u_in = np.linalg.solve(Amat, rhs)                        # interior velocities [m/s]
u_fd = np.r_[0.0, u_in, U]                               # add the wall values u(0) = 0, u(h) = U
print("max |FD − (8.5)| =", np.max(np.abs(u_fd - ch08.channel_flow(yy, h, U=U, dpdx=4.0))), "m/s")
assert np.allclose(u_fd, ch08.channel_flow(yy, h, U=U, dpdx=4.0), rtol=1e-10, atol=1e-12)   # same profile

Central differences are exact for a parabola, so the hand-built solver reproduces $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$ *(8.5)* to round-off.

📝 **Note.** **N09 [B] · Favourable, adverse, backflow (Fig. 8.4).** $dp/dx<0$ pushes with the wall: a fuller profile. $dp/dx>0$ pushes against it; the fluid next to the fixed wall reverses as
soon as the slope du/dy at y = 0 turns negative, i.e. when the inequality below holds (ours — the book only draws it).
Ch. 9 meets the same fight between wall shear and an adverse gradient: that is how boundary layers separate.

$$ \frac{dp}{dx}>\frac{2\mu U}{h^{2}}  $$

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **shear stress τ = μ du/dy, signed** — τ is the force per area that the fluid above a level exerts on the fluid below, in the x-direction (Ch. 1 P05); here we keep its sign.
- **derivative as a slope** — du/dy at the floor is the slope of the profile there; its sign says which way the fluid next to the wall moves (Ch. 1 P19).
- **inequalities under division by a positive number** — multiplying or dividing both sides by a positive number keeps the direction of an inequality; a negative one flips it (Ch. 1 P48).
- **first-order Taylor expansion** — near a point, $u(y)\approx u(0)+u'(0)\,y$ (Ch. 1 P26).

#### 🧮 Derivation — Limits, stress, and the backflow threshold $\frac{dp}{dx}>\frac{2\mu U}{h^2}$ `D05` (Eq. 8.5: $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$)

**What we want to show.** Read the two classic special cases and the shear stress off (8.5), $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$, and find exactly when the fluid near the fixed floor flows backwards (the book only draws it).

**Assumptions.** U > 0 in step 7 (for U < 0 the inequality flips).

**The plan.**

1. Switch off one cause at a time.
2. Differentiate for the stress.
3. Ask when the floor slope turns negative.

**Tools we use** (each explained before this point): derivative as a slope (P19) · first-order Taylor (P26) · inequalities (P48).

**We start from**

$$ u(y)=\frac Uhy-\frac1{2\mu}\frac{dp}{dx}y(h-y)\ \text{(8.5).} $$

---

**Step 1 of 7 — Switch off the pressure gradient.**

$$ \frac{dp}{dx}=0\ \Rightarrow\ u=\frac{Uy}{h} $$

- *Why we can do this:* The parabola term is proportional to dp/dx; what is left is plane Couette flow (R07).
- *In words:* pure wall drag gives a straight line.

---

**Step 2 of 7 — Switch off the wall.**

$$ U=0\ \Rightarrow\ u=-\frac1{2\mu}\frac{dp}{dx}y(h-y),\ u_{max}=-\frac{h^2}{8\mu}\frac{dp}{dx} $$

- *Why we can do this:* Plane Poiseuille flow (R08); the parabola peaks at mid-gap y = h/2 where y(h − y) = h²/4.
- *In words:* pure pressure gives a symmetric parabola.

---

**Step 3 of 7 — Differentiate for the shear stress.**

$$ \tau=\mu\frac{du}{dy}=\frac{\mu U}{h}-\Big(\frac h2-y\Big)\frac{dp}{dx} $$

- *Why we can do this:* τ = μ du/dy (Newton's law); d/dy[y(h − y)] = h − 2y, times −(1/2μ)dp/dx·μ.
- *In words:* the stress is linear in y.

---

**Step 4 of 7 — Evaluate at the walls (U = 0).**

$$ \tau(0)=-\frac h2\frac{dp}{dx},\ \ \tau(h)=+\frac h2\frac{dp}{dx} $$

- *Why we can do this:* Put y = 0 and y = h in step 3; for Poiseuille flow the two wall stresses are equal in size (h/2)∣dp/dx∣.
- *In words:* both walls feel the same drag.

---

**Step 5 of 7 — State the backflow criterion.**

$$ u(y)\approx\frac{du}{dy}\Big\rvert_0\,y\ \ (y\to0) $$

- *Why we can do this:* First-order Taylor with u(0) = 0: near the floor the sign of u is the sign of the floor slope. We want when it turns negative.
- *In words:* backflow starts when the floor slope reverses.

---

**Step 6 of 7 — Evaluate the floor slope.**

$$ \frac{du}{dy}\Big\rvert_0=\frac Uh-\frac{h}{2\mu}\frac{dp}{dx} $$

- *Why we can do this:* Step 3 at y = 0, divided by μ.
- *In words:* wall drag pushes forward, an adverse gradient pushes back.

---

**Step 7 of 7 — Solve slope < 0 for dp/dx.**

$$ \frac Uh-\frac{h}{2\mu}\frac{dp}{dx}<0\iff\frac{dp}{dx}>\frac{2\mu U}{h^2} $$

- *Why we can do this:* Add (h/2μ)dp/dx and multiply by 2μ/h > 0, which keeps the inequality's direction.
- *In words:* an adverse gradient beyond 2μU/h² reverses the floor layer.

---

**Result**

$$ \begin{array}{l}\text{Couette: }u=\frac{Uy}{h},\qquad \text{Poiseuille: }u_{max}=-\frac{h^2}{8\mu}\frac{dp}{dx} \\ \tau=\frac{\mu U}{h}-\Big(\frac h2-y\Big)\frac{dp}{dx},\qquad \text{backflow}\iff\frac{dp}{dx}>\frac{2\mu U}{h^2}\end{array} $$

*In words:* the floor reverses when the pressure's pull there beats the wall's.

**What it means.** This is the channel version of boundary-layer separation: an adverse pressure gradient wins near a wall where the fluid is slow (Ch. 9).

**Check it.** Units 2μU/h²: (Pa s)(m/s)/m² = Pa/m ✓. Numbers: h = 1 cm, U = 0.1 m/s → 2 Pa/m; at 4 Pa/m the reversed layer reaches y = h − 2μU/(h dp/dx) = 5 mm. Poiseuille u_max = 1.5V (V = −(h²/12μ)dp/dx).

> ⚠️ **Common confusion (traps in `D05`):** Backflow starts when u′(0) < 0, not when the velocity maximum moves; τ is signed (the book draws ∣τ∣).

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **definite integral, scipy.integrate.quad** — `integrate.quad(f, a, b)` returns $\int_a^bf\,dy$ and an error estimate (Ch. 3 P87).

📝 **Note.** **N10 [B] · Flow rate and mean velocity.** Two polynomial integrals of (8.5), $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$ give Q and V below — a favourable gradient raises Q, an adverse one lowers it, and Q = 0 at
$dp/dx=6\mu U/h^2$ (6 Pa/m for the tiny example). ⚠️ The book writes the middle expression of V as $V=\int_0^hu\,dy$ without the
1/h; the units give it away (m²/s is not m/s).

$$ Q=\int_0^hu\,dy=\frac{Uh}{2}\Big[1-\frac{h^2}{6\mu U}\frac{dp}{dx}\Big],\qquad V\equiv\frac Qh=\frac U2\Big[1-\frac{h^2}{6\mu U}\frac{dp}{dx}\Big]  $$

In [ ]:
Q_num = integrate.quad(lambda yy: ch08.channel_flow(yy, h, U=U, dpdx=4.0), 0, h)[0]   # ∫₀ʰ u dy numerically [m²/s]
print(f"quad: {Q_num:.6g} m²/s   formula: {ch08.channel_flow_rate(h, U=U, dpdx=4.0)[0]:.6g} m²/s")   # the same

In [ ]:
h, U = 0.01, 0.1                                          # the tiny example's channel [m, m/s]
yy = np.linspace(0, h, 201)                               # heights [m]
cases = [-4.0, 0.0, 2.0, 4.0, 8.0]                        # dp/dx [Pa/m]: favourable, none, threshold, 2×, 4×
cols = [COLORS["blue"], COLORS["accent"], COLORS["teal"], COLORS["orange"], COLORS["amber"]]   # blue → amber
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
for G_, c in zip(cases, cols):
    u = ch08.channel_flow(yy, h, U=U, dpdx=G_)            # (8.5)
    Q_ = ch08.channel_flow_rate(h, U=U, dpdx=G_)[0]       # net flow rate [m²/s]: its sign matters beyond 6μU/h²
    tag = ", net flow reversed" if Q_ < 0 else ""
    a1.plot(u/U, yy/h, color=c, lw=2, label=f"dp/dx = {G_:+g} Pa/m (Q = {1e4*Q_:+.2f} cm²/s{tag})")
    if G_ > ch08.channel_backflow_threshold(U, h):       # shade the reversed layer (backflow) amber
        a1.fill_betweenx(yy/h, 0, u/U, where=u < 0, color=COLORS["amber"], alpha=0.35)
    a2.plot(1e3*ch08.channel_shear_stress(yy, h, U=U, dpdx=G_), yy/h, color=c, lw=2)   # signed τ(y) [mPa]
a1.plot(ch08.channel_flow(yy, h, U=0.0, dpdx=-4.0)/U, yy/h, color=COLORS["orange"], ls="--", lw=1.2,
        label="pure Poiseuille (U = 0, dp/dx = −4)")      # the parabola alone, for comparison
a1.axvline(0, color=COLORS["muted"], lw=0.8)
a1.set_xlabel("u/U [–]"); a1.set_ylabel("y/h [–]"); a1.legend(fontsize=7, loc="upper left")
a1.set_title("Profiles: the floor flow reverses past 2μU/h²", fontsize=10)
a2.axvline(0, color=COLORS["muted"], lw=0.8)
a2.set_xlabel("shear stress τ [mPa]"); a2.set_ylabel("y/h [–]"); a2.set_title("Stress: straight lines; τ(0) = 0 at the threshold", fontsize=10)
fig.suptitle("Adverse pressure beats the wall near the floor once dp/dx > 2μU/h²", fontsize=11)
savefig(fig, "ch08", "c02_couette_poiseuille"); plt.show()

**What you see.** Five profiles through (0, 0) and (1, 1); the two most adverse ones dip below zero near the floor (amber). At 4 Pa/m
the net flow is still forward (Q > 0 in the legend); at 8 Pa/m, beyond the zero-flow gradient $6\mu U/h^2$ (6 Pa/m here), the
net flow itself has reversed (Q < 0). The stresses are straight lines, and the dashed orange parabola is pure Poiseuille flow —
our remake of Fig. 8.4.

**How to read it.** The slope of a profile at y = 0 is proportional to the floor stress; it crosses zero exactly at the 2 Pa/m curve (vertical
at the floor), matching panel (b), where τ(0) = 0 for that case.

**What would change if…** …the gap were halved to 5 mm: the threshold $2\mu U/h^2$ quadruples to 8 Pa/m, so the 4 Pa/m case no longer reverses.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **slider_figure** — `slider_figure(fn, name, values, …)` precomputes one set of curves per slider position, so the plotly figure works on the web page without Python (Ch. 1 P17).

In [ ]:
h, U = 0.01, 0.1                                          # channel [m], wall speed [m/s]
yy = np.linspace(0, h, 81)                                # heights [m]


def frame(G_):                                            # the curves for one slider value dp/dx = G_ [Pa/m]
    u = ch08.channel_flow(yy, h, U=U, dpdx=G_)            # (8.5)
    return {"sum u(y)": (u, yy/h),                                                      # the profile
            "Couette part Uy/h": (U*yy/h, yy/h),                                        # the line
            "Poiseuille part": (ch08.channel_flow(yy, h, U=0.0, dpdx=G_), yy/h),        # the parabola
            "reversed layer (u < 0)": (np.where(u < 0, u, np.nan), yy/h)}               # backflow, if any


Gs = np.linspace(-8, 12, 21 if not FAST else 11)          # slider values dp/dx [Pa/m]
fig = slider_figure(frame, "dp/dx", Gs, unit="Pa/m", xlabel="u [m/s]", ylabel="y/h [–]", title="")
titles = []                                               # one title per slider value, with the verdict
for G_ in Gs:
    s = ch08.couette_poiseuille_state(h, U, G_)           # Q, backflow flag, threshold
    verdict = f"backflow below y = {1e3*s['y_reversal']:.1f} mm" if s["backflow"] else "all forward"
    titles.append(f"A line plus a parabola: Q = {1e4*s['Q']:.2f} cm²/s, {verdict}")
step_titles(fig, titles)
colors = {"sum u(y)": COLORS["blue"], "Couette part Uy/h": COLORS["rose"], "Poiseuille part": COLORS["orange"],
          "reversed layer (u < 0)": COLORS["amber"]}     # colours by meaning
recolor(fig, colors, {"Couette part Uy/h": "dash", "Poiseuille part": "dash"})
fig.show()

**What does the code above do?**

The slider re-evaluates (8.5), $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$ for 21 pressure gradients (11 in FAST mode). Watch the blue sum cross u = 0 at the floor as
dp/dx passes 2 Pa/m: the amber reversed layer appears and the title switches from "all forward" to "backflow".

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **show_viz and the explainer tabs** — `show_viz("ch08", slug)` embeds an interactive explainer; its tabs are Walkthrough, Explore, Explain, Derivation, Equations, Code and Check (Ch. 1 P18).

#### 🎮 Interactive: When does fluid flow backwards in a channel?

**Why interactive rather than a static figure:** Drag the pressure gradient through zero and past $2\mu U/h^2$: the straight Couette part and the parabolic Poiseuille part add
up in front of you, the reversed layer opens at the floor and the flow rate splits into its two contributions — a family of
curves that a static figure can only sample.

**What to try:**
- Start at pure Couette (dp/dx = 0) and slowly raise dp/dx — watch the floor-slope readout reach 0 at 2 Pa/m.
- Press the 'zero net flow' preset: the Couette and Poiseuille bars cancel exactly at $dp/dx=6\mu U/h^2$.
- Click at mid-height in the profile view and read the arithmetic of u from its two parts.
- Halve the gap h and predict the new threshold before you look.

In [ ]:
show_viz("ch08", "couette_poiseuille_backflow")   # full-width explainer; ⤢ Full screen for more room

📝 **Note.** **N11 [C] · Linear stress everywhere.** A constant pressure gradient and a *linear* stress τ(y) are general for any fully developed channel flow — they survive in
turbulent flow for the time-averaged stress (Ch. 12 uses exactly this).

**What would change if…** …the channel were a round pipe? The same three moves (continuity, "function of x = function of R", integrate twice) work,
but the Laplacian brings a 1/R — and a new rule: the solution must stay finite on the axis (C03).

> 🔁 **Recap — Shear stress in cylindrical coordinates** (Ch. 4 §4.5). In cylindrical coordinates (R, φ, z) the shear stress on a surface of constant R in the z-direction is
$\tau_{zR}=\mu\big(\frac{\partial u_R}{\partial z}+\frac{\partial u_z}{\partial R}\big)$ (Ch. 4's `strain_rate(..., "cylindrical")`
gives the same rate of strain).

### 🧩 Poiseuille pipe flow and Hagen–Poiseuille `C03`

*The question:* How much flows through a pipe for a given pressure drop — and why does halving the radius cut the flow sixteen-fold?

*In one line:* $u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}$ *(8.6)*

#### The problem in plain words

Blood squeezing through a capillary, water in a garden hose, oil in a pipeline, air in the lungs' smallest airways: a
pressure difference pushes fluid through a tube and viscosity at the wall holds it back. Doctors care because a slightly
narrowed artery needs a much larger pressure; engineers because pumping costs follow the same law.

#### The idea

```
R=a  ───────────────  u=0,  |τ| max = τ₀
       ─────────►
         ──────────────►    u_max = 2V on the axis
       ─────────►
R=a  ───────────────
```

A paraboloid of velocity, fastest on the axis, zero at the wall; the stress grows linearly from the axis to the wall.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **cylindrical coordinates (R, φ, z) and unit vectors** — R is the distance from the axis, φ the angle round it, z along it; the unit vectors $\mathbf e_R$, $\mathbf e_\varphi$ turn with φ (Ch. 3 P88).

📝 **Note.** **N12 [B] · Set-up in (R, φ, z).** Take $\mathbf u=(0,0,u_z(R))$: continuity is automatic, the R- and φ-equations give $0=\partial p/\partial\varphi$ and
$0=\partial p/\partial R$, so p = p(z). The z-equation is the balance below.

$$ 0=-\frac{dp}{dz}+\frac\mu R\frac{d}{dR}\Big(R\frac{du_z}{dR}\Big)  $$

> 📎 **Primer — Laplacian in cylindrical coordinates.** `P186` · For a function of R only, $\nabla^2u=\frac1R\frac{d}{dR}\big(R\frac{du}{dR}\big)$ — the R inside the bracket comes from the
circle's circumference growing with R. For an azimuthal velocity $u_\varphi(R)$ the φ-component of the *vector* Laplacian is
$\frac{d}{dR}\big[\frac1R\frac{d(Ru_\varphi)}{dR}\big]$ (it contains an extra $-u_\varphi/R^2$ because the unit vector
$\mathbf e_\varphi$ turns). Both come from `core.curvilinear`.

In [ ]:
import sympy as sp
R = sp.symbols('R', positive=True)
u = R**2                                                 # a test profile
print(sp.simplify(sp.diff(R*sp.diff(u, R), R)/R))        # (1/R)(R u')' = 4 for u = R²

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **natural logarithm and ln R → −∞** — ln R is the antiderivative of 1/R, and ln R → −∞ as R → 0 (Ch. 1 P36).
- **area element 2πR dR of a disc** — a thin ring of radius R and width dR has area 2πR dR, so a flow rate is $\int u\,2\pi R\,dR$ (Ch. 2 P83).

*Regularity at the axis.* A solution containing A ln R is infinite on the axis; a physical velocity is finite there, so
A = 0.

#### 🧮 Derivation — Poiseuille pipe profile $u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}$ `D06` (Eq. 8.6: $u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}$)

**What we want to show.** Find the velocity across a round pipe driven by a constant pressure gradient.

**Assumptions.** Steady, fully developed (∂/∂z of u = 0), axisymmetric (∂/∂φ = 0) — step 1; no slip at R = a (step 9).

**The plan.**

1. Show continuity and advection are trivial.
2. Separate the z-equation.
3. Integrate twice; the axis and the wall fix the constants.

**Tools we use** (each explained before this point): Laplacian in cylindrical coordinates (primer P186) · regularity at the axis (gloss) · integrating twice (gloss).

**We start from**

$$ \mathbf u=(u_R,u_\varphi,u_z)=(0,\,0,\,u_z(R)),\qquad \frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u\ \text{(8.1) in cylindrical form} $$

*In words:* steady, fully developed, axisymmetric flow along a round pipe

---

**Step 1 of 10 — Check continuity.**

$$ \nabla\cdot\mathbf u=\frac{\partial u_z}{\partial z}=0 $$

- *Why we can do this:* With u_R = u_φ = 0 only the z-term of the cylindrical divergence is left, and u_z does not depend on z (fully developed).
- *In words:* mass is conserved automatically.

---

**Step 2 of 10 — Evaluate the advective term.**

$$ (\mathbf u\cdot\nabla)u_z=u_z\frac{\partial u_z}{\partial z}=0 $$

- *Why we can do this:* Only the z-velocity is non-zero and nothing changes along z.
- *In words:* no acceleration: the equation is linear.

---

**Step 3 of 10 — Read the R- and φ-equations.**

$$ 0=\frac{\partial p}{\partial R},\ \ 0=\frac{\partial p}{\partial\varphi}\ \Rightarrow\ p=p(z) $$

- *Why we can do this:* All velocity terms in those components vanish, leaving only pressure.
- *In words:* pressure is uniform over each cross-section.

---

**Step 4 of 10 — Write the z-equation.**

$$ 0=-\frac{dp}{dz}+\frac\mu R\frac{d}{dR}\Big(R\frac{du_z}{dR}\Big) $$

- *Why we can do this:* The viscous term is μ times the axisymmetric Laplacian (1/R)d/dR(R du/dR) (P186).
- *In words:* pressure push balances friction across the pipe.

---

**Step 5 of 10 — Separate the variables.**

$$ \frac{dp}{dz}=\frac\mu R\frac{d}{dR}\Big(R\frac{du_z}{dR}\Big)=\text{const} $$

- *Why we can do this:* The left side depends on z only, the right on R only — the D03 argument makes both one constant.
- *In words:* the pressure falls linearly along the pipe.

---

**Step 6 of 10 — Multiply by R/μ and integrate.**

$$ R\frac{du_z}{dR}=\frac{R^2}{2\mu}\frac{dp}{dz}+A $$

- *Why we can do this:* d/dR(R u′) = (R/μ)dp/dz; its antiderivative is (R²/2μ)dp/dz plus a constant A. Keep the R in front of u′.
- *In words:* first integral.

---

**Step 7 of 10 — Divide by R and integrate again.**

$$ u_z(R)=\frac{R^2}{4\mu}\frac{dp}{dz}+A\ln R+B $$

- *Why we can do this:* u′ = (R/2μ)dp/dz + A/R; the antiderivative of A/R is A ln R.
- *In words:* the general solution has a parabola, a logarithm and a constant.

---

**Step 8 of 10 — Keep the velocity finite on the axis.**

$$ A=0 $$

- *Why we can do this:* ln R → −∞ as R → 0, but the velocity at the centre is finite; equivalently from step 6, R u′ → A must be 0 by symmetry.
- *In words:* the logarithm is removed.

---

**Step 9 of 10 — Apply no slip at the wall.**

$$ 0=\frac{a^2}{4\mu}\frac{dp}{dz}+B\ \Rightarrow\ B=-\frac{a^2}{4\mu}\frac{dp}{dz} $$

- *Why we can do this:* (8.3), $\mathbf t\cdot\mathbf U_s=(\mathbf t\cdot\mathbf u)_{\text{on the surface}}$: the fluid at R = a is at rest.
- *In words:* one condition fixes B.

---

**Step 10 of 10 — Combine.**

$$ u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}\ \text{(8.6)} $$

- *Why we can do this:* Insert A = 0 and B into step 7 and factor 1/(4μ).
- *In words:* (8.6), $u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}$, a paraboloid; positive when dp/dz < 0.

---

**Result**

$$ u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}\ \text{(8.6)} $$

*In words:* the axial velocity is a paraboloid, fastest on the axis, zero at the wall.

**What it means.** The pipe profile is the channel's parabola rotated about the axis, but the factor is 1/(4μ), not 1/(2μ), because a disc's circumference grows with R.

**Check it.** u_z(a) = 0 ✓; units (m²)/(Pa s)(Pa/m) = m/s ✓; a = 1 mm, dp/dz = −1000 Pa/m → u(0) = 0.25 m/s; parity with Ch. 3's `pipe_profile` far downstream and Ch. 4's `exact_solution("pipe_poiseuille")` (the code cell after the tiny example of C03).

> ⚠️ **Common confusion (traps in `D06`):** Forgetting the R in R du/dR before the second integration; keeping A ln R; writing 1/(2μ).

📝 **Note.** **N13 [B] · Two constants, two conditions.** The general solution $u_z(R)=\frac{R^2}{4\mu}\frac{dp}{dz}+A\ln R+B$ keeps two constants; the axis removes A, the wall fixes
$B=-\frac{a^2}{4\mu}\frac{dp}{dz}$ (steps 7–9 above).

📝 **Note.** **N14 [B], N15 [B], N16 [B] · Stress, wall stress, Hagen–Poiseuille.** Differentiating (8.6), $u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}$ gives the linear stress (8.7) — zero on the axis, largest at the wall. The **wall stress** (8.8) is
negative for forward flow (dp/dz < 0): the wall pulls the fluid back. A force balance on a slug of fluid gives the same line
without the profile, which is why (8.8) also holds for averaged turbulent pipe flow (D07 steps 4–5; Ch. 12 builds the
friction velocity on it). **Hagen–Poiseuille:** $Q=\int_0^au\,2\pi R\,dR=-\frac{\pi a^4}{8\mu}\frac{dp}{dz}$,
$V=\frac{Q}{\pi a^2}=-\frac{a^2}{8\mu}\frac{dp}{dz}$, $u_{max}=2V$, and the Darcy friction factor f = 64/Re (ours).

$$ \tau=\mu\frac{\partial u_z}{\partial R}=\frac R2\frac{dp}{dz}\ \ \text{(8.7)},\qquad \tau_0=\frac a2\frac{dp}{dz} \qquad \text{(8.8)} $$

#### 🧮 Derivation — Stress, wall stress as a force balance, Hagen–Poiseuille and f = 64/Re `D07` (Eq. 8.8: $\tau_0=\frac a2\frac{dp}{dz}$)

**What we want to show.** Get the shear stress, the wall stress (also from a force balance that holds for turbulent flow), the flow rate, the mean and maximum speeds and the friction factor.

**Assumptions.** Fully developed (momentum flux in = out, step 4); Darcy definition of f (step 9).

**The plan.**

1. Differentiate for τ.
2. Confirm τ₀ by a force balance on a slug.
3. Integrate over the cross-section.
4. Write the friction factor.

**Tools we use** (each explained before this point): cylindrical shear stress (R09) · control-volume force balance (Ch. 4 §4.4) · area element 2πR dR (gloss).

**We start from**

$$ u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}\ \text{(8.6).} $$

---

**Step 1 of 10 — Write the shear stress.**

$$ \tau=\tau_{zR}=\mu\frac{du_z}{dR} $$

- *Why we can do this:* $\tau_{zR}=\mu(\frac{\partial u_R}{\partial z} +\frac{\partial u_z}{\partial R})$ (R09) with u_R = 0.
- *In words:* only the radial slope of u_z matters.

---

**Step 2 of 10 — Differentiate (8.6).**

$$ \tau=\frac{R}{2}\frac{dp}{dz}\ \text{(8.7)} $$

- *Why we can do this:* d/dR(R² − a²) = 2R; 2R/(4μ) × μ = R/2.
- *In words:* (8.7), $\tau=\mu\frac{\partial u_z}{\partial R}=\frac R2\frac{dp}{dz}$: stress grows linearly from zero on the axis.

---

**Step 3 of 10 — Evaluate at the wall.**

$$ \tau_0=\frac a2\frac{dp}{dz}\ \text{(8.8)} $$

- *Why we can do this:* Put R = a; it is negative for forward flow (dp/dz < 0): the wall holds the fluid back.
- *In words:* (8.8), $\tau_0=\frac a2\frac{dp}{dz}$, the largest stress in magnitude.

---

**Step 4 of 10 — Balance forces on a fluid slug.**

$$ \pi a^2[p(0)-p(L)]+2\pi aL\,\tau_0=0 $$

- *Why we can do this:* Steady and fully developed, so the momentum flux in equals out (Ch. 4 CV momentum): pressure force on the ends plus wall shear on the side sum to zero.
- *In words:* the pressure drop is spent on wall friction.

---

**Step 5 of 10 — Solve for the wall stress.**

$$ \tau_0=\frac{a}{2}\frac{p(L)-p(0)}{L}=\frac a2\frac{dp}{dz} $$

- *Why we can do this:* Divide by 2πaL; the profile was never used, so (8.8), $\tau_0=\frac a2\frac{dp}{dz}$ holds for averaged turbulent pipe flow too.
- *In words:* same result, more general.

---

**Step 6 of 10 — Set up the flow rate.**

$$ Q=\int_0^au_z\,2\pi R\,dR $$

- *Why we can do this:* A thin ring of radius R and width dR has area 2πR dR; summing velocity × area gives the volume per second.
- *In words:* add up rings.

---

**Step 7 of 10 — Evaluate the integral.**

$$ \int_0^a(R^2-a^2)R\,dR=\frac{a^4}4-\frac{a^4}2=-\frac{a^4}{4}\ \Rightarrow\ Q=-\frac{\pi a^4}{8\mu}\frac{dp}{dz} $$

- *Why we can do this:* Polynomial antiderivatives; then multiply by 2π/(4μ) and dp/dz.
- *In words:* Q ∝ a⁴ (Hagen–Poiseuille).

---

**Step 8 of 10 — Mean and maximum speeds.**

$$ V=\frac{Q}{\pi a^2}=-\frac{a^2}{8\mu}\frac{dp}{dz},\ \ u_{max}=u_z(0)=-\frac{a^2}{4\mu} \frac{dp}{dz}=2V $$

- *Why we can do this:* Divide by the area; evaluate (8.6), $u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}$ on the axis.
- *In words:* the centre moves twice the average.

---

**Step 9 of 10 — Express the wall stress through V.**

$$ \lvert\tau_0\rvert=-\frac a2\frac{dp}{dz}=\frac{4\mu V}{a} $$

- *Why we can do this:* From step 8, dp/dz = −8μV/a²; insert in (8.8), $\tau_0=\frac a2\frac{dp}{dz}$. We want the Darcy factor f ≡ 8∣τ₀∣/(ρV²).
- *In words:* wall friction ∝ V.

---

**Step 10 of 10 — Form the friction factor.**

$$ f=\frac{8\lvert\tau_0\rvert}{\rho V^2}=\frac{32\mu}{\rho Va}=\frac{64}{\mathrm{Re}},\ \ \mathrm{Re}=\frac{V\,2a}{\nu} $$

- *Why we can do this:* Substitute step 9 and write 2a = d, the diameter the pipe Re uses.
- *In words:* the laminar line of the Moody chart.

---

**Result**

$$ \begin{array}{l}\tau=\mu\frac{\partial u_z}{\partial R}=\frac R2\frac{dp}{dz}\ \text{(8.7)},\qquad \tau_0=\frac a2\frac{dp}{dz}\ \text{(8.8)} \\ Q=-\frac{\pi a^4}{8\mu}\frac{dp}{dz},\quad V=-\frac{a^2}{8\mu}\frac{dp}{dz},\quad u_{max}=2V,\quad f=\frac{64}{\mathrm{Re}}\end{array} $$

*In words:* the pipe's resistance grows as 1/a⁴.

**What it means.** Halving a pipe's radius at fixed pressure drop cuts the flow 16-fold — why narrowed arteries matter. f = 64/Re is where Ch. 12's turbulent friction curves start.

**Check it.** Units: Q in m⁴/(Pa s) × Pa/m = m³/s ✓. Numbers a = 1 mm, dp/dz = −1000 Pa/m: V = 0.125 m/s, Q = 3.93 × 10⁻⁷ m³/s, τ₀ = −0.5 Pa, Re = 250, f = 0.256.

> ⚠️ **Common confusion (traps in `D07`):** τ₀ is negative for forward flow; ∫(R² − a²)R dR = −a⁴/4 (sign); Re uses the diameter and the mean velocity.

#### ✏️ Tiny example: a 2 mm tube at dp/dz = −1000 Pa/m

a = 1 mm, μ = 10⁻³ Pa s, water.
1. $V=-\frac{a^2}{8\mu}\frac{dp}{dz}=\frac{10^{-6}}{8\times10^{-3}}\times1000=0.125$ m/s.
2. $u_{max}=2V=0.25$ m/s.
3. $Q=\pi a^2V=\pi\times10^{-6}\times0.125=3.93\times10^{-7}$ m³/s (0.39 mL/s).
4. $\tau_0=\frac a2\frac{dp}{dz}=-0.5$ Pa.
5. $\mathrm{Re}=V\cdot2a/\nu=0.125\times0.002/10^{-6}=250$: laminar ✓.
6. f = 64/250 = 0.256, and the definition $8\lvert\tau_0\rvert/(\rho V^2)=8\times0.5/(1000\times0.0156)=0.256$ ✓.
7. Halve a: V falls 4×, the area 4×, so Q falls 16×.

In [ ]:
a, G = 1e-3, -1000.0                                     # radius [m], dp/dz [Pa/m] (favourable: pressure falls along z)
Q, V, umax = ch08.pipe_flow_rate(a, G)                   # Hagen–Poiseuille: Q [m³/s], mean V and axis speed [m/s]
print(f"Q = {Q:.4g} m³/s, V = {V:.4g} m/s, u_max = {umax:.4g} m/s")
print("τ₀ =", ch08.pipe_wall_stress(a, G), "Pa;  τ(R = a/2) =", ch08.pipe_shear_stress(0.5e-3, G), "Pa")   # (8.8), (8.7)
Re = V*2*a/NU_W                                          # pipe Reynolds number (diameter, mean speed)
print(f"Re = {Re:.0f}, f = {ch08.pipe_friction_factor(Re):.4g}")   # f = 64/Re
R = np.linspace(0, a, 5)                                 # five radii from the axis to the wall [m]
print("(8.6):         ", ch08.pipe_poiseuille(R, a, G))  # u_z(R) [m/s]
print("Ch. 3 far down:", ch03.pipe_profile(R, z=1e3, U_mean=V, R=a))   # Ch. 3's developing profile, 1 km downstream
print("Ch. 4 on axis: ", NS.exact_solution("pipe_poiseuille", np.array([0.0, 0.0, 0.0]), G=-G, R=a)[0][2])   # Ch. 4 (G = −dp/dz)
pp = ch08.parallel_flow_sympy("pipe")                    # D06 symbolically
print("sympy:", pp["profile"], "  unbounded term:", pp["unbounded_term"], "  (G here is the symbol for dp/dz)")

**What does the code above do?**

1. `pipe_flow_rate` returns Q, V and u_max = 2V: 3.93 × 10⁻⁷ m³/s, 0.125 m/s, 0.25 m/s (the tiny example).
2. `pipe_wall_stress` and `pipe_shear_stress` are (8.8), $\tau_0=\frac a2\frac{dp}{dz}$ and (8.7), $\tau=\mu\frac{\partial u_z}{\partial R}=\frac R2\frac{dp}{dz}$: −0.5 Pa at the wall, half of it at mid-radius.
3. Three independent earlier implementations agree with (8.6), $u_z(R)=\frac{R^2-a^2}{4\mu}\frac{dp}{dz}$: this chapter's formula, Ch. 3's developing profile evaluated
   far downstream, and Ch. 4's exact-solution preset on the axis (0.25 m/s).
4. `parallel_flow_sympy("pipe")` returns the profile and the term $C_1\ln R$ it discarded because it is unbounded on the axis.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **trapezoid rule (from scratch)** — `np.trapezoid(f, x)` adds up trapezoids under the sampled curve (Ch. 1 P37).

In [ ]:
RR = np.linspace(0, a, 2001)                             # 2001 radii [m]
Q_trap = np.trapezoid(ch08.pipe_poiseuille(RR, a, G) * 2*np.pi*RR, RR)   # Q = ∫ u 2πR dR by trapezoids [m³/s]
print(f"trapezoid Q = {Q_trap:.6g}, formula Q = {Q:.6g} m³/s")
assert np.allclose(Q_trap, Q, rtol=1e-6)                 # same flow rate
F_pressure = -np.pi*a**2*G*1.0                           # pressure force on a 1 m slug: πa²[p(0) − p(L)] [N]
F_wall = -2*np.pi*a*1.0*ch08.pipe_wall_stress(a, G)      # wall friction force on its side: −2πaL τ₀ [N]
assert np.isclose(F_pressure, F_wall)                    # D07 step 4: the pressure push equals the wall drag
print(f"force balance on a 1 m slug: pressure {F_pressure:.4g} N = wall friction {F_wall:.4g} N")

In [ ]:
a, G = 1e-3, -1000.0                                      # the tiny example's tube
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
R = np.linspace(-a, a, 201)                               # a diameter, from wall to wall [m]
u = ch08.pipe_poiseuille(np.abs(R), a, G)                 # (8.6) along the diameter [m/s]
a1.plot(1e3*R, u, color=COLORS["blue"], lw=2, label="u_z(R), (8.6)")
for Rk in np.linspace(-0.8*a, 0.8*a, 9):                  # nine velocity arrows (blue)
    a1.annotate("", xy=(1e3*Rk, ch08.pipe_poiseuille(abs(Rk), a, G)), xytext=(1e3*Rk, 0),
                arrowprops=dict(arrowstyle="->", color=COLORS["blue"], lw=0.8))
a1.axvline(-1, color=COLORS["muted"], lw=3); a1.axvline(1, color=COLORS["muted"], lw=3)   # the pipe walls
a1.set_xlabel("R [mm] (across a diameter)"); a1.set_ylabel("u_z [m/s]", color=COLORS["blue"])
b1 = a1.twinx()                                           # second axis for the stress (rose)
b1.plot(1e3*R, np.sign(R)*ch08.pipe_shear_stress(np.abs(R), G), color=COLORS["rose"], lw=1.5, ls="--")   # τ = (R/2) dp/dz
b1.set_ylabel("τ [Pa] (on the +R side: (R/2)dp/dz)", color=COLORS["rose"]); b1.axhline(0, color=COLORS["grid"], lw=0.8)
b1.text(0.55, ch08.pipe_wall_stress(a, G), "τ₀ = −0.5 Pa", color=COLORS["rose"], fontsize=8, va="bottom")
a1.set_title("Paraboloid and linear stress", fontsize=10)
aa = np.geomspace(1e-4, 1e-2, 60)                         # radii from 0.1 mm to 1 cm [m]
QV = np.array([ch08.pipe_flow_rate(a_, G)[:2] for a_ in aa])   # (Q, V) for each radius (the function takes one a)
Qs, Res = QV[:, 0], QV[:, 1]*2*aa/NU_W                    # Q(a) [m³/s] and Re(a) = V·2a/ν with the mean speed
lam = Res < 2000                                          # where the laminar law is trustworthy
a2.loglog(aa[lam], Qs[lam], color=COLORS["blue"], lw=2, label="Hagen–Poiseuille (laminar)")
a2.loglog(aa[~lam], Qs[~lam], color=COLORS["blue"], lw=1, ls="--", label="Re > 2000: turbulent beyond")
a2.loglog(aa, Qs[0]*(aa/aa[0])**4, color=COLORS["muted"], lw=1, ls=":", label="slope 4")
a_cross = aa[np.argmax(~lam)]                             # first radius where Re passes 2000
a2.axvline(a_cross, color=COLORS["amber"], lw=0.8); a2.text(a_cross*1.1, Qs[0]*10, f"Re = 2000\na ≈ {1e3*a_cross:.1f} mm", fontsize=8, color=COLORS["amber"])
a2.set_xlabel("radius a [m]"); a2.set_ylabel("Q [m³/s] at dp/dz = −1000 Pa/m"); a2.legend(fontsize=7, loc="lower right")
a2.set_title("Q ∝ a⁴", fontsize=10)
fig.suptitle("Flow rate grows like the fourth power of the radius", fontsize=11)
savefig(fig, "ch08", "c03_pipe"); plt.show()

**What you see.** (a) a parabola across the diameter with the stress line (rose dashed) growing from zero on the axis to −0.5 Pa at the
wall — our remake of the book's pipe sketch `N104`; (b) a straight log–log line of slope 4.

**How to read it.** Each decade of radius gives four decades of flow; past Re = 2000 (the amber line, a ≈ 2.5 mm here) the laminar line
is no longer trustworthy, so it is dashed.

**What would change if…** …the artery narrowed by 20 %: $0.8^4=0.41$ — the same pressure pushes 59 % less blood, so the heart must raise the
pressure 2.4× to keep Q.

In [ ]:
G = -1000.0                                               # dp/dz [Pa/m]
s_ = np.linspace(0, 1, 60)                                # R/a from the axis to the wall [–]


def frame(a_):                                            # curves for one radius a_ [m]
    return {"u_z(R), (8.6)": (s_, ch08.pipe_poiseuille(s_*a_, a_, G))}   # the paraboloid across the radius


radii = np.geomspace(2e-4, 2e-3, 13 if not FAST else 7)   # slider values a [m]
fig = slider_figure(frame, "a", radii, unit="m", xlabel="R/a [–]", ylabel="u_z [m/s]", title="")
titles = []                                               # Q and Re for each radius, in the title
for a_ in radii:
    Q_, V_, _ = ch08.pipe_flow_rate(a_, G)                # Hagen–Poiseuille flow rate and mean speed
    titles.append(f"a = {1e3*a_:.2f} mm: Q = {1e6*Q_:.3g} mL/s, Re = {V_*2*a_/NU_W:.0f} — Q ∝ a⁴")
step_titles(fig, titles)
recolor(fig, {"u_z(R), (8.6)": COLORS["blue"]})           # velocity profiles in blue
fig.show()

**What does the code above do?**

Each slider step is one radius between 0.2 and 2 mm at the same dp/dz; the title prints Q and Re. Doubling a multiplies the
centre speed by 4 and Q by 16 (compare the steps near a = 0.5 mm and 1 mm); past about 2.5 mm Re would exceed 2000.

**What would change if…** …the walls moved instead of a pressure pushing — sideways, round the axis? Then the velocity is azimuthal, $u_\varphi(R)$,
and the same reduction gives circular Couette flow (C04).

> 🔁 **Recap — A viscous flow with no net viscous force** (Ch. 5 §5.1). Outside a spinning cylinder the ideal-vortex velocity $u_\varphi=\Gamma/2\pi R$ has shear stress
$\sigma_{R\varphi}=\mu R\frac{d}{dR}\big(\frac{u_\varphi}R\big)=-\frac{2\mu\Omega_1R_1^2}{R^2}$ but no net viscous force on any
fluid element, because $\mu\nabla^2\mathbf u=-\mu\nabla\times\boldsymbol\omega=0$ when $\boldsymbol\omega=0$ (Ch. 5).

> 🔁 **Recap — Power in = dissipation** (Ch. 5 §5.1). The torque that keeps the cylinder spinning does work at the rate $-2\pi R_1\sigma_{R\varphi}u_\varphi$ per unit length
(positive: $\sigma_{R\varphi}<0$), and all of it is dissipated in the fluid: $4\pi\mu\Omega_1^2R_1^2$ (Ch. 5's
`dissipation_outside_cylinder`; Exercise 8.12). The book writes the power as $(2\pi R_1)\tau_{R\varphi}u_\varphi$, which is
negative with its own sign of σ_Rφ — the power delivered *to* the fluid carries a minus sign.

> 🔁 **Recap — Solid-body rotation** (Ch. 5 §5.1). A tank of fluid spun long enough rotates like a solid: $u_\theta=\omega r/2$ *(5.1)* with the vorticity ω = 2Ω everywhere,
i.e. $u_\varphi=\Omega R$ — no shear at all.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **curl of a curl identity (∇²u = −∇×ω)** — $\nabla\times(\nabla\times\mathbf u)=\nabla(\nabla\cdot\mathbf u)-\nabla^2\mathbf u$, so for ∇·u = 0 the viscous term is $\mu\nabla^2\mathbf u=-\mu\nabla\times\boldsymbol\omega$ (Ch. 4 P122).

### 🧩 Circular Couette flow and its two limits `C04`

*The question:* What swirl is set up between two rotating cylinders, and what happens when one of them is taken away?

*In one line:* $u_\varphi=\frac{1}{R_2^2-R_1^2}\Big\{\big[\Omega_2R_2^2-\Omega_1R_1^2\big]R-\big[\Omega_2-\Omega_1\big]\frac{R_1^2R_2^2}{R}\Big\}$ *(8.10)*

#### The problem in plain words

Stir tea between a spoon handle and the cup wall; spin the inner cylinder of a lab rig (a Taylor–Couette cell, the classic
apparatus of Ch. 11) or a viscometer. The fluid between two concentric rotating walls settles into a steady swirl. We want
that swirl, the pressure that holds it in its circle, and what it becomes when the outer wall is at infinity or the inner
one vanishes — the two vortices of Ch. 5.

#### The idea

| piece | u_φ | vorticity | shear | appears alone when |
|---|---|---|---|---|
| solid-body rotation | AR | 2A | none | no inner cylinder (R₁ → 0) |
| free (line) vortex | B/R | 0 | yes, but no net force | outer wall at infinity (R₂ → ∞, Ω₂ = 0) |

Only two swirls satisfy the viscous balance, and the walls choose the mix.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **centripetal acceleration in cylindrical coordinates** — a particle going round a circle of radius R at speed u_φ accelerates toward the axis at $u_\varphi^2/R$ even at constant speed, because $\mathbf e_\varphi$ turns (Ch. 3 P105).

📝 **Note.** **N17 [B] · Two equations left.** With $\mathbf u=(0,u_\varphi(R),0)$: the R-equation (first below) says pressure rises outward to supply the centripetal
acceleration; the φ-equation (second) is a pure viscous balance (no pressure, no advection).

$$ -\frac{u_\varphi^2}{R}=-\frac1\rho\frac{dp}{dR},\qquad 0=\mu\frac{d}{dR}\Big[\frac1R\frac{d}{dR}(Ru_\varphi)\Big]  $$

> 📎 **Primer — Euler–Cauchy (equidimensional) ODE.** `P187` · An ODE in which every term has the form $R^k\,d^ku/dR^k$ (each derivative comes with the same power of R) is solved by trying
$u=R^\lambda$: every term becomes a constant times $R^\lambda$, so λ must solve a polynomial. For $R^2u''+Ru'-u=0$ the
polynomial is $\lambda^2-1=0$, so $u=AR+B/R$.

In [ ]:
import sympy as sp
R, lam = sp.symbols('R lambda')
u = R**lam                                               # trial power
print(sp.factor(sp.simplify((R**2*sp.diff(u, R, 2) + R*sp.diff(u, R) - u)/u)))   # (lambda - 1)*(lambda + 1)

#### 🧮 Derivation — Circular Couette flow $u_\varphi=AR+B/R$ `D08` (Eq. 8.10: $u_\varphi=\frac{1}{R_2^2-R_1^2}\Big\{\big[\Omega_2R_2^2-\Omega_1R_1^2\big]R-\big[\Omega_2-\Omega_1\big]\frac{R_1^2R_2^2}{R}\Big\}$)

**What we want to show.** Find the swirl between an inner cylinder (R₁, Ω₁) and an outer cylinder (R₂, Ω₂).

**Assumptions.** Steady, axisymmetric (∂/∂φ = 0), no axial flow, infinitely long (steps 1–4); no slip (step 8).

**The plan.**

1. Reduce the momentum equations.
2. Integrate the φ-equation twice.
3. Fix A and B at the walls.

**Tools we use** (each explained before this point): φ-component of the cylindrical vector Laplacian (primer P186) · Euler–Cauchy ODE (primer P187) · 2 × 2 system (P57).

**We start from**

$$ \mathbf u=(u_R,u_\varphi,u_z)=(0,\,u_\varphi(R),\,0) $$

*In words:* steady swirl between two infinitely long concentric cylinders

---

**Step 1 of 10 — Check continuity.**

$$ \nabla\cdot\mathbf u=\frac1R\frac{\partial u_\varphi}{\partial\varphi}=0 $$

- *Why we can do this:* Only the φ-term of the divergence could survive and nothing depends on φ.
- *In words:* mass is conserved automatically.

---

**Step 2 of 10 — Evaluate the advective acceleration.**

$$ (\mathbf u\cdot\nabla)\mathbf u=-\frac{u_\varphi^2}{R}\mathbf e_R $$

- *Why we can do this:* For a purely azimuthal flow the moving basis leaves only the centripetal part (P105, Ch. 4 §4.7).
- *In words:* the fluid accelerates toward the axis.

---

**Step 3 of 10 — Write the R-momentum.**

$$ -\frac{u_\varphi^2}{R}=-\frac1\rho\frac{dp}{dR} $$

- *Why we can do this:* The R-component of the viscous term, −(2/R²)∂u_φ/∂φ, vanishes; pressure supplies the centripetal force.
- *In words:* pressure rises outward.

---

**Step 4 of 10 — Write the φ-momentum.**

$$ 0=\mu\frac{d}{dR}\Big[\frac1R\frac{d}{dR}(Ru_\varphi)\Big] $$

- *Why we can do this:* ∂p/∂φ = 0 by symmetry and the advective term has no φ-part; the viscous term is the φ-component of the vector Laplacian (P186).
- *In words:* a pure friction balance.

---

**Step 5 of 10 — Integrate once.**

$$ \frac1R\frac{d}{dR}(Ru_\varphi)=2A $$

- *Why we can do this:* Fundamental theorem of calculus; naming the constant 2A keeps the final form clean.
- *In words:* first integral.

---

**Step 6 of 10 — Multiply by R and integrate.**

$$ Ru_\varphi=AR^2+B $$

- *Why we can do this:* d(Ru_φ)/dR = 2AR has antiderivative AR² plus a new constant B.
- *In words:* the angular momentum per mass is a quadratic in R.

---

**Step 7 of 10 — Divide by R.**

$$ u_\varphi(R)=AR+\frac BR\ \text{(8.9)} $$

- *Why we can do this:* Solves the Euler–Cauchy ODE R²u″ + Ru′ − u = 0 (trial R^λ gives λ = ±1, P187).
- *In words:* (8.9), $u_\varphi(R)=AR+\frac BR$: solid-body rotation plus a free vortex.

---

**Step 8 of 10 — Apply no slip on both cylinders.**

$$ \Omega_1R_1^2=AR_1^2+B,\ \ \Omega_2R_2^2=AR_2^2+B $$

- *Why we can do this:* The wall speed is ΩR (not Ω); we multiplied each condition by its radius so B stands alone.
- *In words:* two equations for A, B.

---

**Step 9 of 10 — Subtract to find A.**

$$ A=\frac{\Omega_2R_2^2-\Omega_1R_1^2}{R_2^2-R_1^2} $$

- *Why we can do this:* Subtracting eliminates B; divide by R₂² − R₁².
- *In words:* A is set by the difference of the walls' angular momenta.

---

**Step 10 of 10 — Back-substitute for B and insert.**

$$ B=-\frac{(\Omega_2-\Omega_1)R_1^2R_2^2}{R_2^2-R_1^2},\ \ u_\varphi=\frac{[\Omega_2 R_2^2-\Omega_1R_1^2]R-[\Omega_2-\Omega_1]R_1^2R_2^2/R}{R_2^2-R_1^2}\ \text{(8.10)} $$

- *Why we can do this:* B = Ω₁R₁² − AR₁²; over a common denominator the Ω₂R₂²R₁² terms combine; then put A, B into (8.9), $u_\varphi(R)=AR+\frac BR$.
- *In words:* (8.10), $u_\varphi=\frac{1}{R_2^2-R_1^2}\Big\{\big[\Omega_2R_2^2-\Omega_1R_1^2\big]R-\big[\Omega_2-\Omega_1\big]\frac{R_1^2R_2^2}{R}\Big\}$.

---

**Result**

$$ \begin{array}{l}u_\varphi=\frac{1}{R_2^2-R_1^2}\Big\{\big[\Omega_2R_2^2-\Omega_1R_1^2\big]R-\big[\Omega_2-\Omega_1\big]\frac{R_1^2R_2^2}{R}\Big\}\ \text{(8.10)} \\ A=\frac{\Omega_2R_2^2-\Omega_1R_1^2}{R_2^2-R_1^2},\quad B=-\frac{(\Omega_2-\Omega_1)R_1^2R_2^2}{R_2^2-R_1^2}\end{array} $$

*In words:* the walls choose how much rigid rotation and how much free vortex.

**What it means.** Vorticity is 2A everywhere (the B/R part is irrotational); Ch. 11's Rayleigh criterion for Taylor vortices is written in A and B.

**Check it.** u_φ(R₁) = Ω₁R₁ and u_φ(R₂) = Ω₂R₂ ✓; Ω₁ = Ω₂ = Ω gives A = Ω, B = 0 (rigid rotation) ✓; numbers R₁ = 1 cm, R₂ = 2 cm, Ω₁ = 1 rad/s, Ω₂ = 0: A = −1/3 s⁻¹, B = 1.33 × 10⁻⁴ m²/s, u_φ(1.5 cm) = 3.9 mm/s.

> ⚠️ **Common confusion (traps in `D08`):** The φ-equation contains no pressure and no advection; the first integral is (1/R)(Ru_φ)′ = 2A; the wall speed is ΩR.

📝 **Note.** **N18 [B], N19 [B] · The general swirl and its constants.** The general solution (8.9) is solid-body rotation plus a line vortex; the walls give
$A=\frac{\Omega_2R_2^2-\Omega_1R_1^2}{R_2^2-R_1^2}$ and $B=-\frac{(\Omega_2-\Omega_1)R_1^2R_2^2}{R_2^2-R_1^2}$ (D08 steps
8–10).

$$ u_\varphi(R)=AR+\frac BR \qquad \text{(8.9)} $$

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **limits and orders of smallness** — as R₂ → ∞ a ratio like R₁²/R₂² → 0; dividing top and bottom by the growing quantity first makes the limit visible (Ch. 2 P68).

#### 🧮 Derivation — The limits $u_\varphi=\Omega_1R_1^2/R$ and $u_\varphi=\Omega_2R$ `D09` (Eq. 8.11: $u_\varphi(R)=\frac{\Omega_1R_1^2}{R}$)

**What we want to show.** Show that circular Couette flow contains the ideal vortex and solid-body rotation as limits.

**Assumptions.** Ω₂ = 0 in the first limit (step 2); Ω₁ = 0 in the second (step 5).

**The plan.**

1. Send the outer wall to infinity at rest.
2. Remove the inner cylinder.

**Tools we use** (each explained before this point): limits (P68) · circulation (Ch. 5 §5.1).

**We start from**

$$ u_\varphi(R)=AR+\frac BR\ \text{(8.9)},\qquad A=\frac{\Omega_2R_2^2-\Omega_1R_1^2}{R_2^2-R_1^2},\ \ B=-\frac{(\Omega_2-\Omega_1)R_1^2R_2^2}{R_2^2-R_1^2} $$

*In words:* the general swirl and its constants from D08

---

**Step 1 of 6 — Divide through by R₂².**

$$ A=\frac{\Omega_2-\Omega_1R_1^2/R_2^2}{1-R_1^2/R_2^2},\ \ B=-\frac{(\Omega_2-\Omega_1)R_1^2} {1-R_1^2/R_2^2} $$

- *Why we can do this:* Dividing numerator and denominator by R₂² makes the limit R₂ → ∞ visible: every R₁²/R₂² → 0.
- *In words:* the same constants, ready for the limit.

---

**Step 2 of 6 — Let R₂ → ∞ with Ω₂ = 0.**

$$ A\to0,\ \ B\to\Omega_1R_1^2 $$

- *Why we can do this:* R₁²/R₂² → 0; with Ω₂ = 0 the numerator of A vanishes (with Ω₂ ≠ 0, A → Ω₂ instead).
- *In words:* only the vortex part survives.

---

**Step 3 of 6 — Insert into (8.9).**

$$ u_\varphi(R)=\frac{\Omega_1R_1^2}{R}\ \text{(8.11)} $$

- *Why we can do this:* u_φ = AR + B/R with A = 0.
- *In words:* (8.11), $u_\varphi(R)=\frac{\Omega_1R_1^2}{R}$: speed falls as 1/R.

---

**Step 4 of 6 — Compute the circulation.**

$$ \Gamma=\oint\mathbf u\cdot d\mathbf l=2\pi Ru_\varphi=2\pi\Omega_1R_1^2 $$

- *Why we can do this:* Around a circle u is tangent and constant in size; the result does not depend on R — the signature of the ideal vortex (5.2) u = Γ/(2πR).
- *In words:* a viscous flow identical to the ideal vortex.

---

**Step 5 of 6 — Let R₁ → 0 with Ω₁ = 0.**

$$ A=\frac{\Omega_2R_2^2}{R_2^2}=\Omega_2,\ \ B=-\frac{\Omega_2R_1^2R_2^2}{R_2^2-R_1^2}\to0 $$

- *Why we can do this:* Put Ω₁ = 0 and R₁ = 0 in D08's A and B.
- *In words:* only rigid rotation survives.

---

**Step 6 of 6 — Insert into (8.9).**

$$ u_\varphi(R)=\Omega_2R\ \text{(8.12)} $$

- *Why we can do this:* u_φ = AR with B = 0.
- *In words:* (8.12), $u_\varphi(R)=\Omega_2R$: the tank's fluid turns like a solid (5.1), $u_\theta=\omega r/2\ (=\Omega r\ \text{with vorticity}\ \omega=2\Omega)$.

---

**Result**

$$ u_\varphi(R)=\frac{\Omega_1R_1^2}{R}\ \text{(8.11)},\qquad u_\varphi(R)=\Omega_2R\ \text{(8.12)} $$

*In words:* remove one wall and you get one of the two basic vortices of Ch. 5.

**What it means.** The flow outside a spinning rod is viscous but irrotational: shear exists (R10), no net viscous force acts, and all the power put in is dissipated (R11).

**Check it.** (8.11), $u_\varphi(R)=\frac{\Omega_1R_1^2}{R}$ at R = R₁ gives Ω₁R₁ ✓; the R₂ = 10⁶R₁ value of (8.10), $u_\varphi=\frac{1}{R_2^2-R_1^2}\Big\{\big[\Omega_2R_2^2-\Omega_1R_1^2\big]R-\big[\Omega_2-\Omega_1\big]\frac{R_1^2R_2^2}{R}\Big\}$ agrees with (8.11) to 1e-10; Ch. 5's `rotating_cylinder_flow(r, a, omega=2Ω₁)` gives the same.

> ⚠️ **Common confusion (traps in `D09`):** Divide by R₂² before the limit; Ω₂ = 0 is needed or A → Ω₂; Ch. 5's function takes ω = 2Ω₁.

📝 **Note.** **N21 [B] · A viscous flow that is irrotational.** With R₂ → ∞ and Ω₂ = 0, (8.11), $u_\varphi(R)=\frac{\Omega_1R_1^2}{R}$ is the ideal vortex $u_\theta=\frac{\Gamma}{2\pi r}$ *(5.2)* with $\Gamma=2\pi\Omega_1R_1^2$ —
shear stress everywhere, yet zero net viscous force (R10): the only completely irrotational viscous solution in the book. If
gravity acts along the axis, the free surface dips toward the cylinder (Fig. 8.7, remade below). ⚠️ Ch. 5's
`rotating_cylinder_flow(r, a, omega)` takes the cylinder's *vorticity* ω = 2Ω₁. The other limit is R12's solid-body
rotation, $u_\varphi=\Omega_2R$ *(8.12)*.

#### ✏️ Tiny example: a 1 cm cylinder spinning at 1 rad/s inside a fixed 2 cm cup

R₁ = 0.01 m, R₂ = 0.02 m, Ω₁ = 1 rad/s, Ω₂ = 0.
1. $A=\frac{0-1\times10^{-4}}{4\times10^{-4}-10^{-4}}=-\frac13$ s⁻¹.
2. $B=-\frac{(0-1)\times10^{-4}\times4\times10^{-4}}{3\times10^{-4}}=1.333\times10^{-4}$ m²/s.
3. At mid-gap R = 0.015 m: $u_\varphi=-\frac13\times0.015+\frac{1.333\times10^{-4}}{0.015}=-0.0050+0.0089=0.0039$ m/s.
4. Checks: at R₁, −0.00333 + 0.01333 = 0.0100 = Ω₁R₁ ✓; at R₂, −0.00667 + 0.00667 = 0 ✓.

In [ ]:
R1, R2 = 0.01, 0.02                                      # cylinder radii [m]
u, A, B = ch08.circular_couette(0.015, R1, R2, 1.0, 0.0, return_coeffs=True)   # (8.10) at mid-gap, and A, B of (8.9)
print(f"u_φ(1.5 cm) = {u:.4g} m/s, A = {A:.4g} 1/s, B = {B:.4g} m²/s")
print("R₂ = ∞ (free vortex):", ch08.circular_couette(0.05, R1, np.inf, 1.0, 0.0),
      "  Ch. 5 with ω = 2Ω₁:", ch05.rotating_cylinder_flow(0.05, R1, 2*1.0)[0])   # (8.11) vs Ch. 5 at R = 5 cm
print("R₁ = 0 (solid body): ", ch08.circular_couette(0.01, 0.0, R2, 0.0, 3.0),
      "  Ch. 5 solid body:", VX.solid_body_rotation(0.01, 3.0))               # (8.12) vs Ch. 5 at R = 1 cm, Ω₂ = 3 rad/s
print(f"pressure rise across the gap: {ch08.circular_couette_pressure(R2, R1, R2, 1.0, 0.0):.4g} Pa")
pw = ch08.circular_couette_power(R1, R2, 1.0, 0.0, mu=MU_W)   # torques, power and dissipation per metre of length
print({k: f"{pw[k]:.4g}" for k in ("torque_inner", "torque_outer", "power_in", "power_out", "dissipation")})
print("sympy profile:", ch08.parallel_flow_sympy("circular_couette")["profile"])   # D08 symbolically

**What does the code above do?**

1. `circular_couette` evaluates (8.10), $u_\varphi=\frac{1}{R_2^2-R_1^2}\Big\{\big[\Omega_2R_2^2-\Omega_1R_1^2\big]R-\big[\Omega_2-\Omega_1\big]\frac{R_1^2R_2^2}{R}\Big\}$; with `return_coeffs=True` it also returns A = −1/3 s⁻¹ and B = 1.333 × 10⁻⁴ m²/s,
   the tiny example's numbers.
2. The exact limit branches (R₂ = ∞, R₁ = 0) agree with Ch. 5's vortex (called with ω = 2Ω₁) and solid-body functions.
3. The pressure rises by 0.0217 Pa from the inner to the outer wall (N20 below).
4. The torque is the same in size at both walls (opposite signs: the inner wall drives, the outer wall holds) — a steady
   angular-momentum balance; and every watt the inner cylinder puts in is dissipated (`power_in` = `dissipation`,
   1.676 × 10⁻⁶ W per metre), since the outer wall is at rest (`power_out` = 0).
5. The sympy profile is (8.10) over a common denominator.

In [ ]:
M = np.array([[R1, 1/R1], [R2, 1/R2]])                   # no slip: A R + B/R = Ω R at each wall
rhs = np.array([1.0*R1, 0.0*R2])                         # wall speeds Ω₁R₁, Ω₂R₂ [m/s]
A_, B_ = np.linalg.solve(M, rhs)                         # the 2 × 2 system of D08 step 8
print(A_, B_)
assert np.allclose([A_, B_], [A, B], rtol=1e-12)         # same constants as the library

📝 **Note.** **N20 [B] · The pressure that holds the swirl (ours; the book says only that it 'can be determined').** Integrating the R-balance $dp/dR=\rho u_\varphi^2/R$ with $u_\varphi=AR+B/R$ gives the pressure below. In the atmosphere this
centripetal balance is the *cyclostrophic* balance of tornadoes and dust devils (Ch. 13).

$$ p(R)=p_1+\rho\Big[\frac{A^2}{2}(R^2-R_1^2)+2AB\ln\frac{R}{R_1}-\frac{B^2}{2}\Big(\frac{1}{R^2}-\frac{1}{R_1^2}\Big)\Big]  $$

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **np.gradient** — `np.gradient(f, x)` returns df/dx by central differences on the samples (Ch. 1 P22).

In [ ]:
Rg = np.linspace(R1, R2, 2001)                           # radii across the gap [m]
pg = ch08.circular_couette_pressure(Rg, R1, R2, 1.0, 0.0)   # N20's pressure [Pa]
ug = ch08.circular_couette(Rg, R1, R2, 1.0, 0.0)         # (8.10) [m/s]
dpdR, cent = np.gradient(pg, Rg)[5:-5], (1000*ug**2/Rg)[5:-5]   # finite-difference dp/dR and ρu²/R [Pa/m], away from the ends
print(f"max |dp/dR − ρu²/R| = {np.max(np.abs(dpdR - cent)):.1e} Pa/m (the largest value of ρu²/R is {cent.max():.2f} Pa/m)")
assert np.allclose(dpdR, cent, rtol=1e-5, atol=1e-5)     # they agree to the finite-difference error

📝 **Note.** **N22 [B] · Why these three flows are solvable.** All three are confined between walls, and symmetry kills the advective acceleration: $\mathbf u\cdot\nabla\mathbf u=0$ in the
channel and the pipe, while in circular Couette flow only the centripetal part $-(u_\varphi^2/R)\mathbf e_R$ survives,
balanced by the pressure. Other exact solutions exist (§8.4 and the exercises); Ch. 10 uses such exact solutions to test
flow solvers.

In [ ]:
for case in ("channel", "pipe", "circular_couette"):     # u·∇u computed symbolically with core.curvilinear
    print(case, "→ (u·∇)u =", ch08.advective_acceleration_check(case)["advective"])   # v(R) is the symbol for u_φ(R)

In [ ]:
R1, R2 = 0.01, 0.02                                       # the annulus of the tiny example [m] (R₁/R₂ = 0.5)
Rr = np.linspace(R1, R2, 200)                             # radii across the gap [m]
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(12, 3.8), gridspec_kw=dict(width_ratios=[1.2, 1, 1.1]))
cases = [((1, 0), "inner only", COLORS["blue"]), ((0, 1), "outer only", COLORS["teal"]),
         ((1, 1), "co-rotating (solid body)", COLORS["accent"]), ((1, -1), "counter-rotating", COLORS["amber"])]
for (O1, O2), lab, c in cases:                            # four (Ω₁, Ω₂) pairs [rad/s]
    a1.plot(1e2*Rr, 1e3*ch08.circular_couette(Rr, R1, R2, O1, O2), color=c, lw=2, label=lab)   # (8.10) [mm/s]
_, A, B = ch08.circular_couette(Rr, R1, R2, 1, 0, return_coeffs=True)   # the two parts of the inner-only case
a1.plot(1e2*Rr, 1e3*A*Rr, color=COLORS["orange"], ls="--", lw=1, label="AR part (inner only)")
a1.plot(1e2*Rr, 1e3*B/Rr, color=COLORS["rose"], ls="--", lw=1, label="B/R part (inner only)")
a1.axhline(0, color=COLORS["muted"], lw=0.8)
a1.set_xlabel("R [cm]"); a1.set_ylabel("u_φ [mm/s]"); a1.legend(fontsize=7); a1.set_title("Every swirl is AR + B/R", fontsize=10)
ph = np.linspace(0, 2*np.pi, 13)[:-1]                     # 12 angles
for Rk in np.linspace(R1*1.1, R2*0.95, 5):                # 5 radii: arrows ∝ u_φ, tangent to circles (inner-only case)
    uk = ch08.circular_couette(Rk, R1, R2, 1, 0)
    a2.quiver(Rk*np.cos(ph), Rk*np.sin(ph), -uk*np.sin(ph), uk*np.cos(ph), color=COLORS["blue"], scale=0.06, width=0.006)
t_ = np.linspace(0, 2*np.pi, 200)
a2.fill(R1*np.cos(t_), R1*np.sin(t_), color=COLORS["muted"]); a2.plot(R2*np.cos(t_), R2*np.sin(t_), color=COLORS["ink"], lw=2)
a2.set_aspect("equal"); a2.axis("off"); a2.set_title("Seen from above (inner cylinder turning)", fontsize=10)
Rf = np.linspace(R1, 6*R1, 200)                           # outside a lone cylinder (R₂ = ∞) [m]
uf = ch08.circular_couette(Rf, R1, np.inf, 1.0, 0.0)      # (8.11): Ω₁R₁²/R
Gam = 2*np.pi*1.0*R1**2                                   # circulation Γ = 2πΩ₁R₁² [m²/s]
zs = -Gam**2/(8*np.pi**2*9.81*Rf**2)                      # free-surface height relative to far away (ours, Ch. 5's funnel) [m]
a3.plot(1e2*Rf, 1e3*uf, color=COLORS["blue"], lw=2, label="u_φ = Ω₁R₁²/R (8.11) [mm/s]")
a3.plot(1e2*Rf, 1e6*zs, color=COLORS["accent"], lw=2, label="free-surface dip z_s [µm] (ours)")
a3.axhline(0, color=COLORS["muted"], lw=0.8)
a3.set_xlabel("R [cm]"); a3.legend(fontsize=7); a3.set_title("R₂ → ∞: the ideal vortex and its dip", fontsize=10)
fig.suptitle("Every swirl between cylinders is AR + B/R", fontsize=11)
savefig(fig, "ch08", "c04_circular_couette"); plt.show()

**What you see.** (a) four curves joining the two wall speeds; the co-rotating one is a straight line; the counter-rotating one crosses
zero inside the gap; (b) the annulus from above with arrows growing toward the inner cylinder (our remake of Fig. 8.6
`N104`); (c) the free vortex outside a lone cylinder and the free surface dipping toward it (our Fig. 8.7).

**How to read it.** A straight line is pure AR (B = 0, rigid rotation); a curve bending like 1/R carries a vortex part (dashed rose); where
$u_\varphi$ changes sign the fluid turns the other way. In (c) the dip is tiny here (micrometres) because Ω₁ is only 1 rad/s.

**What would change if…** …the outer cylinder turned faster than the inner in the same sense: $(Ru_\varphi)^2$ grows outward and the flow is stable
(Rayleigh's criterion, Ch. 11); spin only the inner one and it becomes unstable above a critical speed — the Taylor vortices.

In [ ]:
R1, R2 = 0.01, 0.02                                       # annulus [m]
Rr = np.linspace(R1, R2, 80)                              # radii [m]


def frame(r_):                                            # r_ = Ω₂/Ω₁ with Ω₁ = 1 rad/s
    return {"u_φ(R), (8.10)": (1e2*Rr, 1e3*ch08.circular_couette(Rr, R1, R2, 1.0, r_)),   # the swirl [mm/s]
            "solid body Ω₂R": (1e2*Rr, 1e3*r_*Rr),                                          # rigid rotation reference
            "free vortex Ω₁R₁²/R": (1e2*Rr, 1e3*R1**2/Rr)}                                  # ideal-vortex reference


ratios = np.linspace(-2, 2, 21 if not FAST else 11)       # slider values Ω₂/Ω₁ [–]
fig = slider_figure(frame, "Ω₂/Ω₁", ratios, unit="", xlabel="R [cm]", ylabel="u_φ [mm/s]", title="")
titles = []                                               # A, B and the Rayleigh verdict for each ratio
for r_ in ratios:
    st = ch08.circular_couette_state(R1, R2, 1.0, r_)     # constants of (8.9) and (Ru_φ)² increasing outward?
    tag = "Rayleigh-stable" if st["rayleigh_stable"] else "can become unstable (Ch. 11)"
    titles.append(f"A = {st['A']:.2f} 1/s, B = {1e4*st['B']:.2f} cm²/s — {tag}")
step_titles(fig, titles)
recolor(fig, {"u_φ(R), (8.10)": COLORS["blue"], "solid body Ω₂R": COLORS["orange"], "free vortex Ω₁R₁²/R": COLORS["rose"]},
        {"solid body Ω₂R": "dash", "free vortex Ω₁R₁²/R": "dash"})
fig.show()

**What does the code above do?**

The slider sets the outer wall's rotation relative to the inner one (Ω₁ = 1 rad/s). The title prints A, B and whether
Rayleigh's criterion (angular momentum growing outward) holds; the dashed references are the two pure limits.

**What would change if…** …the walls were not exactly parallel — a thin gap that narrows slowly, as under a bearing pad? The flow is then *nearly*
parallel, the nonlinear term is small instead of zero, and we need a way to decide what may be dropped: the lubrication
approximation (C05).

---

## 8.3 Elementary Lubrication Theory

**What is this section about?** Real walls are rarely exactly parallel, but in a thin gap they are *nearly* parallel. We
learn how to decide, by scaling with two different lengths, which terms of the Navier–Stokes equations can be dropped; the
result is a local Couette-plus-Poiseuille profile and one equation for the pressure (the Reynolds equation). It explains why
an oil film under a bearing pad carries tonnes, why viscous flow between glass plates draws the streamlines of ideal flow,
and how a honey drop spreads under its own weight. The same scaling argument becomes the boundary-layer approximation in
Ch. 9.

> 🔁 **Recap — The field equations in the gap** (Ch. 4 §4.6). In two dimensions: continuity $\frac{\partial u}{\partial x}+\frac{\partial v}{\partial y}=0$ *(6.2)* and the x-component of
(8.1), $\frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u$, written out: $\frac{\partial u}{\partial t}+u\frac{\partial u}{\partial x}+v\frac{\partial u}{\partial y}=-\frac1\rho
\frac{\partial p}{\partial x}+\frac\mu\rho\big(\frac{\partial^2u}{\partial x^2}+\frac{\partial^2u}{\partial y^2}\big)$ *(8.13a)*
— nothing new, just written out.

### 🧩 The lubrication approximation `C05`

*The question:* Why can we throw away inertia in a thin film even when UL/ν is in the thousands?

*In one line:* $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$ *(8.17a)*

#### The problem in plain words

Under a sliding bearing pad the oil film is 50 µm thick and 5 cm long — a thousand times longer than thick. The oil moves
at metres per second and UL/ν ≈ 4000, which in a pipe would be turbulent. Yet lubrication engineers treat the film as
purely viscous. We want the argument that makes this legitimate — and the number that must be small instead of 1/Re.

📝 **Note.** **N23 [B] · The lubrication idea.** In a narrow passage of gap h and length L ≫ h (Fig. 8.8) the flow is nearly parallel; pressure and viscous forces dominate,
and a gentle curvature of the passage does not matter as long as its radius is much larger than h. Exactly the same
reasoning, with the gap replaced by a thin layer next to a wall, gives the boundary-layer approximation (Ch. 9 §9.1).

#### The idea

```
↑ y ~ h = εL (small)          ∂/∂x ~ 1/L   (slow along the gap)
│  ═══════════════════        ∂/∂y ~ 1/h   (fast across it)
│  ──► ──► ──► u ~ U          continuity: U/L ~ v/h  ⇒  v ~ εU
└──────────────────────► x ~ L
```

Two length scales, and continuity fixes the size of v.

📝 **Note.** **N24 [B] · The y-momentum equation.** The y-equation is written below in its correct form. ⚠️ The book prints $-\frac1\rho\frac{\partial p}{\partial x}$ here; the
y-equation needs ∂p/∂**y** — the scaled (8.16b), $\varepsilon^4\mathrm{Re}_L\Big(\frac{\partial v^*}{\partial t^*}+u^*\frac{\partial v^*}{\partial x^*}+v^*\frac{\partial v^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+\varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}$ confirms it, and `ch08.lubrication_nondim_sympy()` shows that the printed
form gives a coefficient set that does not match (8.16b) (D10 step 12, check cell below).

$$ \frac{\partial v}{\partial t}+u\frac{\partial v}{\partial x}+v\frac{\partial v}{\partial y}=-\frac1\rho\frac{\partial p}{\partial y}+\frac\mu\rho\Big(\frac{\partial^2v}{\partial x^2}+\frac{\partial^2v}{\partial y^2}\Big) \qquad \text{(8.13b)} $$

> 📎 **Primer — anisotropic scaling with two length scales.** `P188` · When a flow is long and thin, measure x in units of the length L and y in units of the thickness h = εL; derivatives then
carry different sizes, $\partial/\partial x=(1/L)\partial/\partial x^*$ but $\partial/\partial y=(1/\varepsilon L)\partial/
\partial y^*$. The velocity across the layer gets its own scale from continuity: U/L must balance v/h, so v ~ εU. Written
this way every starred quantity is of order one and the size of each term sits in its coefficient.

In [ ]:
L, eps, U = 0.05, 1e-3, 5.0              # length [m], fineness h/L, speed [m/s]
h = eps*L                                # the gap [m]
print(U/L, (eps*U)/h)                    # both 100 1/s: the two continuity terms balance

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **scaled variables and the chain rule** — if $x=Lx^*$ then $\partial/\partial x=(1/L)\,\partial/\partial x^*$ (Ch. 4 P133).
- **bookkeeping of a small parameter ε** — multiply the whole equation by the factor that gives the term you expect to dominate the coefficient 1; every other coefficient is then a power of ε times a group, and you read off which terms are small (Ch. 2 P68, orders of smallness).
- **order-of-magnitude scaling** — replace each derivative by (change)/(distance over which it happens), e.g. ∂u/∂y ~ U/h (Ch. 4 P130).
- **sympy symbols and simplify in check cells** — `sp.symbols(..., positive=True)` makes symbols; `sp.simplify(expr) == 0` tests an identity (Ch. 4 P117).

📝 **Note.** **N26 [B] · The scalings (8.14),** with the fineness ratio ε = h/L and the atmospheric pressure $P_a$ as the book's pressure scale.

$$ x^*=\frac xL,\ y^*=\frac yh=\frac{y}{\varepsilon L},\ t^*=\frac{Ut}{L},\ u^*=\frac uU,\ v^*=\frac{v}{\varepsilon U},\ p^*=\frac{p}{P_a} \qquad \text{(8.14)} $$

#### 🧮 Derivation — The scaled gap equations `D10` (Eq. 8.16a: $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$)

**What we want to show.** Rewrite the gap equations in variables of order one, so that the size of every term sits in its coefficient.

**Assumptions.** One length along (L), one across (h = εL); pressure scaled by the atmospheric P_a (as the book does).

**The plan.**

1. Turn the scalings into substitutions and derivative rules.
2. Scale continuity.
3. Scale x-momentum and divide by the cross-gap friction's coefficient.
4. Scale y-momentum and choose the factor that keeps the same 1/Λ.

**Tools we use** (each explained before this point): anisotropic scaling (primer P188) · scaled variables and the chain rule (P133) · bookkeeping of ε (gloss).

**We start from**

$$ \begin{array}{l}\frac{\partial u}{\partial x}+\frac{\partial v}{\partial y}=0\ \text{(6.2),} \\ \frac{\partial u}{\partial t}+u\frac{\partial u}{\partial x}+ v\frac{\partial u}{\partial y}=-\frac1\rho\frac{\partial p}{\partial x}+\frac\mu\rho\big(\frac{\partial^2u}{\partial x^2}+\frac{\partial^2u} {\partial y^2}\big) \\ \text{(8.13a), its v-twin (8.13b) with ∂p/∂y, and the scalings} \\ x^*=\frac xL,\ y^*=\frac y{\varepsilon L},\ t^*=\frac {Ut}L,\ u^*=\frac uU,\ v^*=\frac v{\varepsilon U},\ p^*=\frac p{P_a}\ \text{(8.14).}\end{array} $$

---

**Step 1 of 14 — Invert the scalings.**

$$ x=Lx^*,\ y=\varepsilon Ly^*,\ t=\tfrac LU t^*,\ u=Uu^*,\ v=\varepsilon Uv^*,\ p=P_ap^* $$

- *Why we can do this:* We will substitute these into the equations; each starred variable is of order one by design.
- *In words:* dimensional quantities = scale × order-one variable.

---

**Step 2 of 14 — Write the derivative rules.**

$$ \frac{\partial}{\partial x}=\frac1L\frac{\partial}{\partial x^*},\ \frac{\partial} {\partial y}=\frac{1}{\varepsilon L}\frac{\partial}{\partial y^*},\ \frac{\partial}{\partial t}=\frac UL\frac{\partial}{\partial t^*} $$

- *Why we can do this:* Chain rule for scaled variables (P133): ∂x*/∂x = 1/L, and so on.
- *In words:* derivatives across the gap are 1/ε times larger.

---

**Step 3 of 14 — Scale continuity.**

$$ \frac UL\frac{\partial u^*}{\partial x^*}+\frac{\varepsilon U}{\varepsilon L}\frac{\partial v^*} {\partial y^*}=0 $$

- *Why we can do this:* Substitute u = Uu*, v = εUv* and the rules of step 2 into (6.2), $\frac{\partial u}{\partial x}+\frac{\partial v}{\partial y}=0$.
- *In words:* both terms carry U/L.

---

**Step 4 of 14 — Divide by U/L.**

$$ \frac{\partial u^*}{\partial x^*}+\frac{\partial v^*}{\partial y^*}=0\ \text{(8.15)} $$

- *Why we can do this:* The ε cancels because v was scaled with εU — chosen exactly so that continuity balances (if v ~ U the u-term would drop).
- *In words:* (8.15), $\frac{\partial u^*}{\partial x^*}+\frac{\partial v^*}{\partial y^*}=0$: mass is conserved with no approximation.

---

**Step 5 of 14 — Scale the x-inertia terms.**

$$ \frac{\partial u}{\partial t}+u\frac{\partial u}{\partial x}+v\frac{\partial u}{\partial y}= \frac{U^2}{L}\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big) $$

- *Why we can do this:* Each term: (U/L)U, U(U/L), (εU)(U/εL) — all U²/L.
- *In words:* inertia is of size U²/L.

---

**Step 6 of 14 — Scale the x-pressure term.**

$$ -\frac1\rho\frac{\partial p}{\partial x}=-\frac{P_a}{\rho L}\frac{\partial p^*}{\partial x^*} $$

- *Why we can do this:* p = P_a p* and ∂/∂x = (1/L)∂/∂x*.
- *In words:* pressure term of size P_a/(ρL).

---

**Step 7 of 14 — Scale the two x-viscous terms.**

$$ \frac\mu\rho\frac{\partial^2u}{\partial x^2}=\frac{\mu U}{\rho L^2}\frac{\partial^2u^*} {\partial x^{*2}},\ \ \frac\mu\rho\frac{\partial^2u}{\partial y^2}=\frac{\mu U}{\rho\varepsilon^2L^2}\frac{\partial^2u^*}{\partial y^{*2}} $$

- *Why we can do this:* Two derivatives along the gap bring 1/L², two across bring 1/(εL)².
- *In words:* friction across the gap is 1/ε² larger.

---

**Step 8 of 14 — Multiply by ρε²L²/(μU).**

$$ \times\frac{\rho\varepsilon^2L^2}{\mu U} $$

- *Why we can do this:* This makes the cross-gap friction — the term we expect to dominate — have coefficient 1, so every other coefficient reads directly as a relative size.
- *In words:* measure everything against cross-gap friction.

---

**Step 9 of 14 — Read the inertia coefficient.**

$$ \frac{U^2}{L}\cdot\frac{\rho\varepsilon^2L^2}{\mu U}=\varepsilon^2\frac{\rho UL}{\mu}= \varepsilon^2\mathrm{Re}_L $$

- *Why we can do this:* Simplify the product; Re_L = ρUL/μ uses the passage length.
- *In words:* inertia is weighed by ε²Re_L.

---

**Step 10 of 14 — Read the other x-coefficients.**

$$ \varepsilon^2\mathrm{Re}_L(\ldots)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+ \varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}\ \text{(8.16a)} $$

- *Why we can do this:* Pressure: (P_a/ρL)(ρε²L²/μU) = P_a h²/(μUL) = 1/Λ; along-gap friction: ε²; across: 1.
- *In words:* (8.16a), $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$.

---

**Step 11 of 14 — Scale the y-inertia terms.**

$$ \frac{\partial v}{\partial t}+u\frac{\partial v}{\partial x}+v\frac{\partial v}{\partial y}= \frac{\varepsilon U^2}{L}(\ldots)^* $$

- *Why we can do this:* Each term now carries one factor εU from v: (U/L)εU, U(εU/L), (εU)(εU/εL).
- *In words:* vertical inertia is ε times smaller.

---

**Step 12 of 14 — Scale the y-pressure and friction.**

$$ -\frac{P_a}{\rho\varepsilon L}\frac{\partial p^*}{\partial y^*},\ \ \frac{\mu\varepsilon U}{\rho L^2}\frac{\partial^2v^*}{\partial x^{*2}},\ \ \frac{\mu U}{\rho\varepsilon L^2}\frac{\partial^2v^*}{\partial y^{*2}} $$

- *Why we can do this:* Use ∂/∂y = (1/εL)∂/∂y* on the pressure — the corrected (8.13b), $\frac{\partial v}{\partial t}+u\frac{\partial v}{\partial x}+v\frac{\partial v}{\partial y}=-\frac1\rho\frac{\partial p}{\partial y}+\frac\mu\rho\Big(\frac{\partial^2v}{\partial x^2}+\frac{\partial^2v}{\partial y^2}\Big)$ has ∂p/∂y; the book prints ∂p/∂x, which would give a different set.
- *In words:* pressure across the gap is large.

---

**Step 13 of 14 — Multiply by ρε³L²/(μU).**

$$ \times\frac{\rho\varepsilon^3L^2}{\mu U} $$

- *Why we can do this:* We choose the factor that gives the pressure term the same coefficient 1/Λ as in (8.16a), $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$: (P_a/ρεL)(ρε³L²/μU) = ε²LP_a/(μU) = 1/Λ.
- *In words:* compare against the same pressure yardstick.

---

**Step 14 of 14 — Read the y-coefficients.**

$$ \varepsilon^4\mathrm{Re}_L(\ldots)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+ \varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}\ \text{(8.16b)} $$

- *Why we can do this:* Inertia (εU²/L)(ρε³L²/μU) = ε⁴Re_L; friction ε⁴ and ε² likewise.
- *In words:* (8.16b), $\varepsilon^4\mathrm{Re}_L\Big(\frac{\partial v^*}{\partial t^*}+u^*\frac{\partial v^*}{\partial x^*}+v^*\frac{\partial v^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+\varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}$: only pressure lacks a power of ε.

---

**Result**

$$ \begin{array}{l}\frac{\partial u^*}{\partial x^*}+\frac{\partial v^*}{\partial y^*}=0\ \text{(8.15)} \\ \varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}\ \text{(8.16a)} \\ \varepsilon^4\mathrm{Re}_L\Big(\frac{\partial v^*}{\partial t^*}+u^*\frac{\partial v^*}{\partial x^*}+v^*\frac{\partial v^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+\varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}\ \text{(8.16b)}\end{array} $$

*In words:* in a thin gap inertia carries ε²Re_L and all y-terms except pressure carry ε² or less.

**What it means.** "Large Re" is not the right test in a thin gap: inertia competes with ε²Re_L. The same bookkeeping becomes the boundary-layer approximation in Ch. 9.

**Check it.** Every coefficient is dimensionless ✓. sympy (`lubrication_nondim_sympy`, the check cell below) returns the sets {ε²Re_L, 1/Λ, ε², 1} and {ε⁴Re_L, 1/Λ, ε⁴, ε²}; the printed ∂p/∂x in (8.13b), $\frac{\partial v}{\partial t}+u\frac{\partial v}{\partial x}+v\frac{\partial v}{\partial y}=-\frac1\rho\frac{\partial p}{\partial y}+\frac\mu\rho\Big(\frac{\partial^2v}{\partial x^2}+\frac{\partial^2v}{\partial y^2}\Big)$ gives a set that does not match (8.16b), $\varepsilon^4\mathrm{Re}_L\Big(\frac{\partial v^*}{\partial t^*}+u^*\frac{\partial v^*}{\partial x^*}+v^*\frac{\partial v^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+\varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}$. Numbers (engine film): ε = 10⁻³, Re_L = 4350 → ε²Re_L = 4.35 × 10⁻³.

In [ ]:
sol = ch08.lubrication_nondim_sympy()                    # D10 steps 1–14 repeated symbolically (cached)
L_, h_, U_, rho_, mu_, Pa_, eps, ReL, Lam = sol["symbols"]   # the symbols it used (ε, Re_L, Λ among them)
print("scaled continuity:", sol["continuity_star"])      # (8.15): no coefficient at all
print("x-coefficients:", sol["x_coeffs"])                # (8.16a): ε²Re_L, 1/Λ, ε², 1
print("y-coefficients:", sol["y_coeffs"])                # (8.16b): ε⁴Re_L, 1/Λ, ε⁴, ε²
assert sp.simplify(sol["x_coeffs"]["inertia"] - eps**2*ReL) == 0      # step 9: inertia weighed by ε²Re_L
assert sp.simplify(sol["x_coeffs"]["pressure"] - 1/Lam) == 0          # step 10: pressure 1/Λ
assert sp.simplify(sol["x_coeffs"]["diff_across"] - 1) == 0           # step 8's choice: cross-gap friction has 1
assert sp.simplify(sol["y_coeffs"]["inertia"] - eps**4*ReL) == 0      # step 14: y-inertia ε⁴Re_L
assert sp.simplify(sol["y_coeffs"]["pressure"] - 1/Lam) == 0          # step 13's choice: the same 1/Λ yardstick
assert sp.simplify(sol["y_coeffs"]["diff_across"] - eps**2) == 0      # step 14: y-friction across the gap ε²
print("with the printed ∂p/∂x in (8.13b):", sol["printed_13b_y_coeffs"])   # pressure ε/Λ: not the set of (8.16b)
assert sp.simplify(sol["printed_13b_y_coeffs"]["pressure"] - 1/Lam) != 0   # the printed slip fails

> ⚠️ **Common confusion (traps in `D10`):** v scales with εU (from continuity), not U; the two momentum equations are multiplied by different factors (ε²L²ρ/μU and ε³L²ρ/μU) so that the pressure keeps 1/Λ; the printed ∂p/∂x in (8.13b), $\frac{\partial v}{\partial t}+u\frac{\partial v}{\partial x}+v\frac{\partial v}{\partial y}=-\frac1\rho\frac{\partial p}{\partial y}+\frac\mu\rho\Big(\frac{\partial^2v}{\partial x^2}+\frac{\partial^2v}{\partial y^2}\Big)$ must be ∂p/∂y.

📝 **Note.** **N27 [B], N28 [B], N29 [B] · What D10 delivered.** **N27** The scaled continuity equation (8.15) has no coefficient: mass is conserved exactly — v is small *because* u varies
slowly. **N28** In (8.16a), $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$ the inertia carries $\varepsilon^2\mathrm{Re}_L$ with $\mathrm{Re}_L=\rho UL/\mu$, and the pressure
carries $1/\Lambda$ with the bearing number $\Lambda=\mu UL/(P_ah^2)$. **N29** In (8.16b), $\varepsilon^4\mathrm{Re}_L\Big(\frac{\partial v^*}{\partial t^*}+u^*\frac{\partial v^*}{\partial x^*}+v^*\frac{\partial v^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+\varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}$ only the pressure term lacks a
power of ε — the seed of "pressure is constant across a thin layer" (Ch. 9, Ch. 13).

$$ \frac{\partial u^*}{\partial x^*}+\frac{\partial v^*}{\partial y^*}=0 \qquad \text{(8.15)} $$

#### 🧮 Derivation — The lubrication balance $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$, $0\cong-\frac1\rho\frac{\partial p}{\partial y}$ `D11` (Eq. 8.17a: $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$)

**What we want to show.** Drop the small terms of (8.16), $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$ and return to dimensional form — restoring the ν the printed (8.17a), $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$ omits.

**Assumptions.** ε ≪ 1, ε²Re_L ≪ 1, Λ = O(1) with a suitable pressure scale (step 1).

**The plan.**

1. State the orders.
2. Drop the small terms in each equation.
3. Undo the scaling.

**Tools we use** (each explained before this point): orders of smallness (P68).

**We start from**

$$ \varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}\ \text{(8.16a)},\qquad \varepsilon^4\mathrm{Re}_L\Big(\frac{\partial v^*}{\partial t^*}+u^*\frac{\partial v^*}{\partial x^*}+v^*\frac{\partial v^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+\varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}\ \text{(8.16b)} $$

*In words:* the scaled momentum equations of D10

---

**Step 1 of 7 — State the orders.**

$$ \varepsilon\ll1,\ \ \varepsilon^2\mathrm{Re}_L\ll1,\ \ \Lambda=O(1) $$

- *Why we can do this:* All starred derivatives are O(1) by construction; only the coefficients decide sizes.
- *In words:* thin gap, small reduced Reynolds number.

---

**Step 2 of 7 — Drop the x-inertia.**

$$ 0=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+ \frac{\partial^2u^*}{\partial y^{*2}}+O(\varepsilon^2\mathrm{Re}_L) $$

- *Why we can do this:* The left side of (8.16a), $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$ is ε²Re_L × O(1).
- *In words:* acceleration is negligible — even the unsteady part.

---

**Step 3 of 7 — Drop the along-gap friction.**

$$ 0\cong-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\frac{\partial^2u^*}{\partial y^{*2}} $$

- *Why we can do this:* ε²∂²u*/∂x*² is ε² times an order-one derivative, so it is as small as the inertia we just dropped.
- *In words:* friction acts across the gap only.

---

**Step 4 of 7 — Reduce the y-equation.**

$$ \frac1\Lambda\frac{\partial p^*}{\partial y^*}=O(\varepsilon^2)\ \Rightarrow\ 0\cong-\frac{\partial p^*}{\partial y^*} $$

- *Why we can do this:* In (8.16b), $\varepsilon^4\mathrm{Re}_L\Big(\frac{\partial v^*}{\partial t^*}+u^*\frac{\partial v^*}{\partial x^*}+v^*\frac{\partial v^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+\varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}$ every term except the pressure carries ε² or smaller.
- *In words:* pressure is the same across the gap to relative order ε².

---

**Step 5 of 7 — Undo the x-scaling.**

$$ 0\cong-\frac1\rho\frac{\partial p}{\partial x}+\frac\mu\rho\frac{\partial^2u}{\partial y^2} $$

- *Why we can do this:* Multiply by μU/(ρε²L²) (the inverse of D10 step 8) and use p = P_a p*, ∂/∂y = (1/εL)∂/∂y*.
- *In words:* back in dimensions.

---

**Step 6 of 7 — Write μ/ρ as ν.**

$$ 0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.17a)} $$

- *Why we can do this:* ν = μ/ρ; the book prints this without the ν, which fails the units (m/s² vs 1/(m s)).
- *In words:* (8.17a), $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$: pressure gradient = cross-gap friction.

---

**Step 7 of 7 — Undo the y-scaling.**

$$ 0\cong-\frac1\rho\frac{\partial p}{\partial y}\ \text{(8.17b)} $$

- *Why we can do this:* Multiply the reduced y-equation by P_a/(ρεL).
- *In words:* (8.17b), $0\cong-\frac1\rho\frac{\partial p}{\partial y}$.

---

**Result**

$$ 0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.17a)},\qquad 0\cong-\frac1\rho\frac{\partial p}{\partial y}\ \text{(8.17b)} $$

*In words:* the lubrication approximation: a local Poiseuille balance with pressure uniform across the gap.

**What it means.** With P_a the bearing number is Λ = 49 for our film, not O(1); rescaling p by μUL/h² sets Λ = 1 and changes nothing in the ordering — the physical pressures are tens to hundreds of atmospheres.

**Check it.** Units of (8.17a), $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$: both terms m/s² ✓. Time still appears as a parameter (no ∂/∂t term, yet h, p may vary slowly). Numbers: engine film ε²Re_L = 4.35 × 10⁻³.

> ⚠️ **Common confusion (traps in `D11`):** Dropping inertia needs ε²Re_L ≪ 1, not Re_L ≪ 1; ∂p*/∂y* is O(ε²), not exactly 0; re-dimensionalising restores ν.

📝 **Note.** **N30 [B] · Pressure is uniform across the gap,** with an error of relative size ε². The same statement across a thin atmospheric or oceanic layer is the hydrostatic
approximation of Ch. 13.

$$ 0\cong-\frac1\rho\frac{\partial p}{\partial y} \qquad \text{(8.17b)} $$

> ⚠️ **Common confusion:** the book prints (8.17a), $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$ as $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\frac{\partial^2u}{\partial y^2}$ — without the ν.
The units do not balance (m/s² against 1/(m s)); re-dimensionalising (D11 step 5) restores $\nu\frac{\partial^2u}{\partial y^2}$.
Its own next line, (8.18), $u(x,y,t)\cong\frac1\mu\frac{\partial p(x,t)}{\partial x}\frac{y^2}{2}+Ay+B$, carries the 1/μ correctly.

#### ✏️ Tiny example: an engine-bearing oil film

h = 50 µm, L = 5 cm, U = 5 m/s, a light oil with μ = 0.05 Pa s and ρ = 870 kg/m³, so ν = μ/ρ = 5.75 × 10⁻⁵ m²/s (numbers
ours).
1. $\varepsilon=h/L=5\times10^{-5}/0.05=10^{-3}$.
2. $\mathrm{Re}_L=UL/\nu=5\times0.05/5.75\times10^{-5}=4350$ — in a pipe this would be turbulent.
3. $\varepsilon^2\mathrm{Re}_L=10^{-6}\times4350=4.35\times10^{-3}$ — small: inertia is under half a percent of friction.
4. The natural pressure scale $\mu UL/h^2=0.05\times5\times0.05/(2.5\times10^{-9})=5\times10^6$ Pa ≈ 49 atm.
5. With the book's atmospheric scale, $\Lambda=\mu UL/(P_ah^2)=5\times10^6/101\,325=49$ — not "near unity".

In [ ]:
L, h, U, rho, mu = 0.05, 50e-6, 5.0, 870.0, 0.05          # the engine film: length, gap [m], speed [m/s], ρ, μ
s = ch08.lubrication_scales(L, h, U, rho, mu)             # ε, Re_L, ε²Re_L, Λ (with P_a) and μUL/h²
print({k: f"{v:.4g}" for k, v in s.items()})
keys = ("x_inertia", "x_pressure", "x_diff_along", "x_diff_across", "y_inertia", "y_pressure", "y_diff_along", "y_diff_across")
for scale in ("viscous", "atm"):                          # pressure scaled by μUL/h² (Λ = 1) or by P_a (the book's)
    t = ch08.lubrication_term_magnitudes(L, h, U, rho, mu, p_scale=scale)   # the coefficients of (8.16a), (8.16b)
    print(scale, {k: f"{t[k]:.3g}" for k in keys})

**What does the code above do?**

1. `lubrication_scales` returns ε = 10⁻³, Re_L = 4350, ε²Re_L = 4.35 × 10⁻³, Λ = 49.3 and the natural pressure scale
   μUL/h² = 5 × 10⁶ Pa.
2. `lubrication_term_magnitudes` returns exactly the coefficients of (8.16a), $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$ and (8.16b), $\varepsilon^4\mathrm{Re}_L\Big(\frac{\partial v^*}{\partial t^*}+u^*\frac{\partial v^*}{\partial x^*}+v^*\frac{\partial v^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+\varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}$. With the viscous pressure scale
   μUL/h² every kept term is O(1): x-pressure 1, cross-gap friction 1, and inertia 4.35 × 10⁻³. With the book's P_a both
   pressure coefficients become 1/Λ = 0.0203 — only the bookkeeping changes, not the physics.

In [ ]:
eps = h/L; ReL = rho*U*L/mu; Lam = mu*U*L/(101325*h**2)   # ε, Re_L, Λ by hand (P_a = 101 325 Pa)
mine = [eps**2*ReL, 1/Lam, eps**2, 1.0, eps**4*ReL, 1/Lam, eps**4, eps**2]   # the eight coefficients of D10
lib = ch08.lubrication_term_magnitudes(L, h, U, rho, mu, p_scale="atm")
assert np.allclose(mine, [lib[k] for k in keys], rtol=1e-12)   # same numbers, in the order of `keys`
print("hand-made coefficients = library coefficients")

In [ ]:
L, U, rho, mu = 0.05, 5.0, 870.0, 0.05                    # fixed: Re_L = ρUL/μ = 4350
epss = np.geomspace(1e-4, 0.3, 60)                        # fineness ratios ε = h/L
T = [ch08.lubrication_term_magnitudes(L, e*L, U, rho, mu, p_scale="viscous") for e in epss]   # coefficients at each ε
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, pre, title in ((a1, "x", "x-momentum (8.16a)"), (a2, "y", "y-momentum (8.16b)")):
    ax.loglog(epss, [t_[pre + "_inertia"] for t_ in T], color=COLORS["teal"], lw=2, label="inertia")
    ax.loglog(epss, [t_[pre + "_pressure"] for t_ in T], color=COLORS["orange"], lw=2, label="pressure")
    ax.loglog(epss, [t_[pre + "_diff_along"] for t_ in T], color=COLORS["rose"], lw=1.2, ls=":", label="friction along the gap")
    ax.loglog(epss, [t_[pre + "_diff_across"] for t_ in T], color=COLORS["rose"], lw=2, label="friction across the gap")
    e_fail = 1/np.sqrt(4350.0)                            # where ε²Re_L = 1
    ax.axvline(e_fail, color=COLORS["muted"], ls="--", lw=1)
    ax.text(e_fail*1.1, 1e-9, "ε²Re_L = 1:\nlubrication\nfails to the right", fontsize=7)
    ax.axvline(1e-3, color=COLORS["amber"], lw=0.8); ax.text(1.1e-3, 1e2, "engine film", fontsize=7, color=COLORS["amber"])
    ax.set_xlabel("ε = h/L [–]"); ax.set_title(title, fontsize=10); ax.set_ylim(1e-14, 1e4)
a1.set_ylabel("coefficient (size of the term) [–]"); a1.legend(fontsize=7, loc="lower right")
fig.suptitle("In a thin gap inertia is weighed with ε²Re_L, not Re_L (Re_L = 4350, pressure scale μUL/h²)", fontsize=11)
savefig(fig, "ch08", "c05_term_magnitudes"); plt.show()

**What you see.** Two flat lines on the left (pressure and cross-gap friction) and falling lines for everything else; on the right only the
pressure stays at 1 — every other y-term falls like ε² or faster.

**How to read it.** Read each term's size at your ε: at ε = 10⁻³ (amber line) inertia is 4.35 × 10⁻³ of the kept terms; only near
ε ≈ 0.015 (dashed line) does it catch up.

**What would change if…** …Re_L were 100 times larger: the teal line moves up two decades and lubrication fails already at ε ≈ 0.0015.

📝 **Note.** **N31 [B] · How small is small?** For our engine film $\varepsilon^2\mathrm{Re}_L=4.35\times10^{-3}$, so the lubrication balance is excellent. (The book's own
example uses a different oil film; its number lives in our private test file.) ⚠️ With $p^*=p/P_a$ the bearing number
$\Lambda=\mu UL/(P_ah^2)$ is 49 here and ≈ 10³ for thinner, longer films — not "near unity" as the text assumes. Nothing in the
ordering changes (Λ only rescales p*), but the physical pressure scale is μUL/h², tens to hundreds of atmospheres.

#### 🎮 Interactive: Why can a thin film ignore inertia?

**Why interactive rather than a static figure:** Drag the gap, the length and the speed and watch every term of the scaled momentum equations (8.16a), $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$ and (8.16b), $\varepsilon^4\mathrm{Re}_L\Big(\frac{\partial v^*}{\partial t^*}+u^*\frac{\partial v^*}{\partial x^*}+v^*\frac{\partial v^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial y^*}+\varepsilon^4\frac{\partial^2v^*}{\partial x^{*2}}+\varepsilon^2\frac{\partial^2v^*}{\partial y^{*2}}$ as a bar on
a log scale: inertia falls like ε²Re_L, the pressure and the cross-gap friction stay, and the y-equation leaves only the
pressure — an ordering argument you can see move.

**What to try:**
- Start from the engine-film preset and double U three times — the teal inertia bar climbs but stays far below the pressure bar.
- Press 'thick gap': ε = 0.2 and Re_L = 10⁴ put inertia on top and the status turns amber.
- Switch the pressure scale to P_a (the book's) and read Λ — then back to μUL/h².
- Open the Derivation tab and step through D10: each substitution lights the bar it creates.

In [ ]:
show_viz("ch08", "lubrication_scaling")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …we now use (8.17a), $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$ to find the velocity everywhere in a gap of any slowly varying shape? Integrating twice across the gap
gives a local Couette-plus-Poiseuille profile (C06).

### 🧩 The lubrication profile, the gap flux and the Reynolds equation `C06`

*The question:* Given the shape of the gap, what is the velocity at every point — and what single equation decides the pressure?

*In one line:* $u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+U_h\frac yh+U_0\Big(1-\frac yh\Big)$ *(8.19)*

#### The problem in plain words

Under a tilted pad, in a knee joint, in the gap of an injection mould, the gap height changes slowly along the flow. At each
station the flow "thinks" it is between parallel plates — so its profile is the Couette–Poiseuille profile of C02 with the
local gap h(x) and the local pressure gradient. But the pressure gradient is unknown; mass conservation across the whole gap
has to decide it.

#### The idea

```
station 1 (wide)       station 2            station 3 (narrow)
══════════ U_h         ══════════ U_h       ══════════ U_h
 ─►                     ──►                   ───►
 ►  + Poiseuille        ─► + smaller          ──►      same flux q must pass every station
═══════════ U₀        ═══════════ U₀         ═══════════ U₀
```

The channel formula, applied station by station.

📝 **Note.** **N25 [B] · Gap boundary conditions.** $u=U_0(t)$ on the lower wall y = 0 and $u=U_h(t)$ on the upper wall y = h(x, t); the pressure may depend on t. Time only
enters as a parameter: the lubrication balance (8.17), $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2},\ 0\cong-\frac1\rho\frac{\partial p}{\partial y}$ has no ∂/∂t term, yet h, U₀, U_h and p may all change slowly.

#### 🧮 Derivation — The lubrication profile `D12` (Eq. 8.19: $u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+U_h\frac yh+U_0\Big(1-\frac yh\Big)$)

**What we want to show.** Find the velocity at every point of a slowly varying gap with moving walls.

**Assumptions.** Lubrication balance (D11); walls at y = 0 and y = h(x, t).

**The plan.**

1. Use (8.17b), $0\cong-\frac1\rho\frac{\partial p}{\partial y}$ to treat ∂p/∂x as a constant across the gap.
2. Integrate twice.
3. Fix the "constants" with the two walls and regroup.

**Tools we use** (each explained before this point): integrating twice (gloss in C02) · 2 × 2 system (P57).

**We start from**

$$ 0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.17a)},\qquad 0\cong-\frac1\rho\frac{\partial p}{\partial y}\ \text{(8.17b)},\qquad u(x,0,t)=U_0(t),\ \ u(x,h,t)=U_h(t) $$

*In words:* the lubrication balance, with the lower wall y = 0 moving at U₀ and the upper wall y = h(x, t) at U_h

---

**Step 1 of 8 — Use (8.17b).**

$$ p=p(x,t)\ \Rightarrow\ \frac{\partial p}{\partial x}\ \text{is independent of }y $$

- *Why we can do this:* Pressure does not change across the gap, so at fixed x and t its x-slope is one number.
- *In words:* each station has one pressure gradient.

---

**Step 2 of 8 — Rewrite (8.17a).**

$$ \frac{\partial^2u}{\partial y^2}=\frac1\mu\frac{\partial p}{\partial x} $$

- *Why we can do this:* Multiply (8.17a), $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$ by 1/ν and use ρν = μ.
- *In words:* constant curvature across the gap.

---

**Step 3 of 8 — Integrate twice in y.**

$$ u(x,y,t)\cong\frac1\mu\frac{\partial p(x,t)}{\partial x}\frac{y^2}{2}+Ay+B\ \text{(8.18)} $$

- *Why we can do this:* The integration is at fixed x and t, so the "constants" A and B may depend on x and t.
- *In words:* (8.18), $u(x,y,t)\cong\frac1\mu\frac{\partial p(x,t)}{\partial x}\frac{y^2}{2}+Ay+B$.

---

**Step 4 of 8 — Apply the lower wall.**

$$ u(x,0,t)=U_0\ \Rightarrow\ B=U_0 $$

- *Why we can do this:* No slip at y = 0.
- *In words:* the floor's speed.

---

**Step 5 of 8 — Apply the upper wall.**

$$ A=\frac{U_h-U_0}{h}-\frac{h}{2\mu}\frac{\partial p}{\partial x} $$

- *Why we can do this:* No slip at y = h: U_h = (1/μ)p_x h²/2 + Ah + U₀, solved for A.
- *In words:* the slope mixes the wall-speed difference and pressure.

---

**Step 6 of 8 — Substitute A and B.**

$$ u=\frac{1}{2\mu}\frac{\partial p}{\partial x}(y^2-hy)+(U_h-U_0)\frac yh+U_0 $$

- *Why we can do this:* Insert into (8.18), $u(x,y,t)\cong\frac1\mu\frac{\partial p(x,t)}{\partial x}\frac{y^2}{2}+Ay+B$ and collect the p_x terms.
- *In words:* parabola plus a line.

---

**Step 7 of 8 — Regroup into gap fractions.**

$$ u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+U_h\frac yh+ U_0\Big(1-\frac yh\Big) $$

- *Why we can do this:* y² − hy = −h²(y/h)(1 − y/h) and (U_h − U₀)y/h + U₀ = U_h y/h + U₀(1 − y/h).
- *In words:* local Poiseuille + local Couette.

---

**Step 8 of 8 — Compare with the printed (8.19).**

$$ U_0=0:\ \ u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+ U_h\frac yh $$

- *Why we can do this:* The printed (8.19), $u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+U_h\frac yh+U_0\Big(1-\frac yh\Big)$ adds U₀ instead of U₀(1 − y/h), which gives u(h) = U_h + U₀; with U₀ = 0 (every example) the two agree.
- *In words:* the book's form is right whenever the floor is still.

---

**Result**

$$ u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+U_h\frac yh+U_0\Big(1-\frac yh\Big)\ \text{(the consistent (8.19))} $$

*In words:* at every station, the channel profile of (8.5), $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$ with the local gap.

**What it means.** The velocity field is known once p(x, t) is; D13 finds the equation for p.

**Check it.** u(0) = U₀ ✓, u(h) = U_h ✓ (the printed form gives U_h + U₀ ✗ when U₀ ≠ 0); U₀ = 0, h constant reproduces (8.5), $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$.

> ⚠️ **Common confusion (traps in `D12`):** ∂p/∂x is constant only in the y-integration; A and B depend on x and t.

📝 **Note.** **N33 [B] · The intermediate form (8.18)** has "constants" A, B that may depend on x and t (step 3). ⚠️ The text says (8.16a), $\varepsilon^2\mathrm{Re}_L\Big(\frac{\partial u^*}{\partial t^*}+u^*\frac{\partial u^*}{\partial x^*}+v^*\frac{\partial u^*}{\partial y^*}\Big)=-\frac1\Lambda\frac{\partial p^*}{\partial x^*}+\varepsilon^2\frac{\partial^2u^*}{\partial x^{*2}}+\frac{\partial^2u^*}{\partial y^{*2}}$ "can be integrated twice" — it is the
simplified (8.17a), $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}{\partial y^2}$ that is integrated.

$$ u(x,y,t)\cong\frac1\mu\frac{\partial p(x,t)}{\partial x}\frac{y^2}{2}+Ay+B \qquad \text{(8.18)} $$

📝 **Note.** **N34 [B] · The U₀ term.** The printed (8.19), $u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+U_h\frac yh+U_0\Big(1-\frac yh\Big)$ adds $U_0$ to the Couette part, so at the upper wall it gives $u(h)=U_h+U_0$ instead of $U_h$. The
consistent profile is $U_h\frac yh+U_0\big(1-\frac yh\big)$ (D12 step 7). Every example of the chapter has U₀ = 0, so no
result changes; our code uses the consistent form and keeps the printed one as `form='book'` for a test that must fail.

In [ ]:
h = 50e-6                                                # a 50 µm gap [m]
print(ch08.lubrication_velocity(h, h, 0.0, U_h=2.0, U_0=1.0))               # consistent (8.19) at the top wall: U_h = 2 m/s ✓
print(ch08.lubrication_velocity(h, h, 0.0, U_h=2.0, U_0=1.0, form="book"))  # as printed: U_h + U₀ = 3 m/s ✗

> 📎 **Primer — Leibniz rule with a moving upper limit.** `P189` · Differentiating an integral whose upper limit moves adds a boundary term:
$\frac{\partial}{\partial x}\int_0^{h(x)}u\,dy=\int_0^{h}\frac{\partial u}{\partial x}dy+u(x,h)\frac{\partial h}{\partial x}$.
The second term counts what enters or leaves because the limit itself moved (Ch. 3 met the fixed-limit version).

In [ ]:
import sympy as sp
x, y = sp.symbols('x y')
h = 1 + x**2                                             # a moving upper limit
u = x*y                                                  # a test integrand
lhs = sp.diff(sp.integrate(u, (y, 0, h)), x)             # d/dx of the integral
rhs = sp.integrate(sp.diff(u, x), (y, 0, h)) + u.subs(y, h)*sp.diff(h, x)   # Leibniz: inside + boundary term
print(sp.simplify(lhs - rhs))                            # 0

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **differentiation under the integral sign** — with fixed limits, $\frac{d}{dx}\int_a^bf\,dy=\int_a^b\frac{\partial f}{\partial x}dy$ (Ch. 3 P109) — the primer above adds the moving-limit term.
- **kinematic condition at a moving wall** — a particle on the surface y = h(x, t) stays on it, so D(y − h)/Dt = 0 there: $v=\partial h/\partial t+u\,\partial h/\partial x$ at y = h (Ch. 4 P132).

#### 🧮 Derivation — The gap flux and the 1-D Reynolds equation $\frac{\partial h}{\partial t}+\frac{\partial q}{\partial x}=0$ `D13`

**What we want to show.** Turn mass conservation into one equation for the pressure in a gap of any slowly varying shape (the book leaves it to Exercises 8.19–8.20).

**Assumptions.** Flat, impermeable, lower wall (step 4); the upper wall is a material surface y = h(x, t) (step 5); the lubrication profile (step 8).

**The plan.**

1. Integrate continuity across the gap.
2. Move ∂/∂x outside with the Leibniz rule.
3. Use the kinematic conditions at the walls.
4. Compute the flux from the profile.

**Tools we use** (each explained before this point): Leibniz rule with a moving upper limit (primer P189) · kinematic condition at a moving wall (gloss) · polynomial integrals.

**We start from**

$$ \begin{array}{l}\frac{\partial u}{\partial x}+\frac{\partial v}{\partial y}=0\ \text{(6.2) in 0 < y < h(x, t), and the consistent profile of} \\ \text{D12.}\end{array} $$

---

**Step 1 of 12 — Integrate continuity from 0 to h.**

$$ \int_0^h\frac{\partial u}{\partial x}dy+v(x,h,t)-v(x,0,t)=0 $$

- *Why we can do this:* Integrate (6.2), $\frac{\partial u}{\partial x}+\frac{\partial v}{\partial y}=0$ over the gap; the v-term integrates exactly by the fundamental theorem of calculus.
- *In words:* the net outflow of a thin slice is zero.

---

**Step 2 of 12 — Apply the Leibniz rule.**

$$ \int_0^h\frac{\partial u}{\partial x}dy=\frac{\partial}{\partial x}\int_0^hu\,dy-u(x,h,t) \frac{\partial h}{\partial x} $$

- *Why we can do this:* The upper limit h depends on x, so differentiating the integral adds a boundary term (P189); we want the flux itself inside the derivative.
- *In words:* the x-change of the flux, corrected for the moving limit.

---

**Step 3 of 12 — Name the flux.**

$$ \frac{\partial q}{\partial x}-u(h)\frac{\partial h}{\partial x}+v(h)-v(0)=0,\ \ q\equiv\int_0^hu\,dy $$

- *Why we can do this:* Insert step 2 into step 1.
- *In words:* flux divergence plus wall terms.

---

**Step 4 of 12 — Use the flat, impermeable floor.**

$$ v(x,0,t)=0 $$

- *Why we can do this:* No flow through the lower wall (8.2), $\mathbf n\cdot\mathbf U_s=(\mathbf n\cdot\mathbf u)_{\text{on the surface}}$, which does not move vertically.
- *In words:* nothing enters through the floor.

---

**Step 5 of 12 — Use the kinematic upper wall.**

$$ v(x,h,t)=\frac{\partial h}{\partial t}+u(x,h,t)\frac{\partial h}{\partial x} $$

- *Why we can do this:* A particle on y = h(x, t) stays on it: D(y − h)/Dt = 0 there (P132).
- *In words:* the fluid at the top moves with the wall.

---

**Step 6 of 12 — Substitute both wall conditions.**

$$ \frac{\partial q}{\partial x}-u(h)\frac{\partial h}{\partial x}+\frac{\partial h} {\partial t}+u(h)\frac{\partial h}{\partial x}=0 $$

- *Why we can do this:* Steps 4 and 5 into step 3.
- *In words:* the boundary terms appear twice with opposite signs.

---

**Step 7 of 12 — Cancel the boundary terms.**

$$ \frac{\partial h}{\partial t}+\frac{\partial q}{\partial x}=0 $$

- *Why we can do this:* The Leibniz term and the kinematic term are equal and opposite — whatever u(h) is.
- *In words:* the gap fills where more flux enters than leaves.

---

**Step 8 of 12 — Insert the profile in the flux.**

$$ q=\int_0^h\Big[-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh \Big)+U_h\frac yh+U_0\Big(1-\frac yh\Big)\Big]dy $$

- *Why we can do this:* Insert the consistent profile (8.19), $u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+U_h\frac yh+U_0\Big(1-\frac yh\Big)$ of D12, the only place where the momentum balance enters.
- *In words:* three pieces to integrate.

---

**Step 9 of 12 — Integrate the Poiseuille piece.**

$$ \int_0^h\frac yh\Big(1-\frac yh\Big)dy=\frac h6\ \Rightarrow\ -\frac{h^3}{12\mu} \frac{\partial p}{\partial x} $$

- *Why we can do this:* ∫₀^h (y/h − y²/h²)dy = h/2 − h/3 = h/6, times −h²p_x/(2μ).
- *In words:* pressure- driven flux ∝ h³.

---

**Step 10 of 12 — Integrate the Couette pieces.**

$$ \int_0^h\frac yh\,dy=\int_0^h\Big(1-\frac yh\Big)dy=\frac h2\ \Rightarrow\ \frac{(U_0+U_h)h}{2} $$

- *Why we can do this:* Each linear profile has the mean of its end values.
- *In words:* drag flux = mean wall speed × gap.

---

**Step 11 of 12 — Add the pieces.**

$$ q=-\frac{h^3}{12\mu}\frac{\partial p}{\partial x}+\frac{(U_0+U_h)h}{2} $$

- *Why we can do this:* Sum of steps 9 and 10.
- *In words:* the gap flux.

---

**Step 12 of 12 — Combine into one pressure equation.**

$$ \frac{\partial h}{\partial t}+\frac{\partial}{\partial x}\Big[-\frac{h^3}{12\mu} \frac{\partial p}{\partial x}+\frac{(U_0+U_h)h}{2}\Big]=0 $$

- *Why we can do this:* Put step 11 into step 7; for a steady gap ∂h/∂t = 0, so q is the same at every station.
- *In words:* the 1-D Reynolds equation.

---

**Result**

$$ \frac{\partial h}{\partial t}+\frac{\partial q}{\partial x}=0\ \text{,}\ q=-\frac{h^3}{12\mu}\frac{\partial p}{\partial x}+ \frac{(U_0+U_h)h}{2} $$

*In words:* mass conservation of the gap; steady ⇒ constant flux.

**What it means.** One ODE for p(x) in a steady gap: given the flux and two end pressures, the pressure is fixed — the lubrication method in one line (C07 applies it).

**Check it.** Units of q: m³/(Pa s) × Pa/m = m²/s ✓ and (m/s)m ✓. h constant and ∂p/∂x = 0 → q = (U₀ + U_h)h/2 (Couette). Example 8.1's C₁ is this flux measured in the pad frame. sympy (the check cell below): Leibniz identity, cancellation and final residual all 0.

In [ ]:
r = ch08.reynolds_equation_sympy()                       # D13 built symbolically: profile → flux → integrated continuity
print("q =", r["q"])                                     # −h³p_x/(12μ) + (U₀ + U_h)h/2, over a common denominator
assert sp.simplify(r["leibniz_residual"]) == 0           # step 2: the Leibniz rule holds for the lubrication profile
assert sp.simplify(r["kinematic_cancellation"]) == 0     # steps 5–7: the u(h)·h_x terms cancel
assert sp.simplify(r["reynolds_residual"]) == 0          # step 12: h_t + q_x = 0 is the gap integral of u_x + v_y = 0
print("Reynolds equation:", r["reynolds_equation"])      # the 1-D Reynolds equation, expanded

> ⚠️ **Common confusion (traps in `D13`):** The Leibniz boundary term −u(h)h_x cancels against the kinematic v(h) — do not drop only one of them; v(0) = 0 only for a flat impermeable lower wall.

📝 **Note.** **N32 [B] · Completing a lubrication problem.** The profile alone is not a solution: we still need p(x). Integrating continuity across the gap gives
$\frac{\partial h}{\partial t}+\frac{\partial q}{\partial x}=0$ with the gap flux below — the 1-D **Reynolds equation** (the
book leaves it to Exercises 8.19–8.20). For a steady gap q is the same at every station, which is one ODE for p(x); two end
pressures close it (C07).

$$ q=\int_0^hu\,dy=-\frac{h^3}{12\mu}\frac{\partial p}{\partial x}+\frac{(U_0+U_h)h}{2}  $$

> 📎 **Primer — scipy.integrate.cumulative_trapezoid.** `P190` · `cumulative_trapezoid(f, x, initial=0)` returns the running integral $\int_{x_0}^{x_i}f\,dx$ at every sample — the numerical
antiderivative. We use it to turn a pressure gradient into a pressure profile.

In [ ]:
from scipy.integrate import cumulative_trapezoid
import numpy as np
x = np.linspace(0, 1, 5)
print(cumulative_trapezoid(2*x, x, initial=0))           # ≈ x²: [0, 0.0625, 0.25, 0.5625, 1.0]

#### ✏️ Tiny example: the flux through a 50 µm gap

h = 50 µm, upper wall $U_h$ = 5 m/s, lower wall fixed, μ = 0.05 Pa s.
1. With no pressure gradient: $q=U_hh/2=5\times5\times10^{-5}/2=1.25\times10^{-4}$ m²/s.
2. The gradient that halves it: $\frac{h^3}{12\mu}\frac{dp}{dx}=\frac{U_hh}{4}$ ⇒ $\frac{dp}{dx}=\frac{3\mu U_h}{h^2}=\frac{3\times0.05\times5}
   {2.5\times10^{-9}}=3\times10^8$ Pa/m — 3 bar per millimetre.
3. At $dp/dx=6\mu U_h/h^2=6\times10^8$ Pa/m the flux is zero: the pressure-driven backflow exactly cancels the drag.

In [ ]:
h, Uh, mu = 50e-6, 5.0, 0.05                             # gap [m], upper-wall speed [m/s], oil viscosity [Pa s]
G_half = 3*mu*Uh/h**2                                    # the gradient that halves the flux [Pa/m]
print(ch08.lubrication_flux(h, 0.0, U_h=Uh, mu=mu), ch08.lubrication_flux(h, G_half, U_h=Uh, mu=mu))   # q [m²/s]
yy = np.linspace(0, h, 5)                                # five heights across the gap [m]
print(ch08.lubrication_velocity(yy, h, G_half, U_h=Uh, mu=mu))   # (8.19) at the halving gradient [m/s]
x = np.linspace(0, 0.05, 401); hx = h*(1 + 0.5*x/0.05)   # a slider gap h₀(1 + αx/L) with α = 0.5, L = 5 cm
p, q = ch08.reynolds_pressure_1d(x, hx, U_0=-Uh, mu=mu)  # pad frame: floor slides at −U under the resting pad; p = 0 at both ends
print(f"p_max = {p.max()/1e6:.3f} MPa at x/L = {x[np.argmax(p)]/0.05:.3f}; flux q = {q:.4g} m²/s")

**What does the code above do?**

1. `lubrication_flux` is N32's q: 1.25 × 10⁻⁴ m²/s with no pressure gradient, half of it at the "halving" gradient.
2. The profile at that gradient is [0, −0.156, 0.625, 2.344, 5.0] m/s: a thin reversed layer at the fixed wall, because
   $3\mu U/h^2$ exceeds the backflow threshold $2\mu U/h^2$ of C02.
3. `reynolds_pressure_1d` solves the steady Reynolds equation for a gap given as samples: it finds the constant flux that makes
   the two end pressures equal and integrates $dp/dx=12\mu(\bar q-q)/h^3$ (with $\bar q=(U_0+U_h)h/2$) using
   `cumulative_trapezoid`. For the α = 0.5 pad of C07 in the pad frame it gives a 1.00 MPa hump at x/L = 0.40 and the pad-frame
   flux q = −1.5 × 10⁻⁴ m²/s — the $C_1=-\frac{1+\alpha}{2+\alpha}Uh_o$ of D14 (cross-checked there).

In [ ]:
q_num = integrate.quad(lambda y_: ch08.lubrication_velocity(y_, h, 1e8, U_h=Uh, mu=mu), 0, h)[0]   # ∫₀ʰ u dy numerically
print(f"quad: {q_num:.6g} m²/s, formula: {ch08.lubrication_flux(h, 1e8, U_h=Uh, mu=mu):.6g} m²/s")
assert np.allclose(q_num, ch08.lubrication_flux(h, 1e8, U_h=Uh, mu=mu), rtol=1e-10)   # N32's flux is the integral of (8.19)

In [ ]:
h0, al, L, U, mu = 50e-6, 1.5, 0.05, 5.0, 0.05            # the slider gap of C07 with a steep taper α = 1.5
k = 200.0                                                 # vertical exaggeration of the drawing
fig, ax = plt.subplots(figsize=(10, 3.4))
xs = np.linspace(0, L, 200)                               # along the pad [m]
ax.fill_between(1e2*xs, k*h0*(1 + al*xs/L)*1e2, k*h0*(1 + al*xs/L)*1e2 + 1.2, color=COLORS["muted"], alpha=0.6)   # the pad
ax.axhline(0, color=COLORS["ink"], lw=2)                  # the floor
for x0 in np.linspace(0.04*L, 0.96*L, 7):                 # seven stations
    hx = h0*(1 + al*x0/L)                                 # local gap [m]
    yy = np.linspace(0, hx, 25)                           # heights across the gap [m]
    u = ch08.slider_gap_velocity(x0, yy, h0, al, L, U, mu=mu, frame="ground")   # (8.19) with the exact slider dp/dx [m/s]
    profile_arrows(ax, 1e2*k*yy, 1e2*0.12*u/U*L, x0=1e2*x0, every=4, color=COLORS["blue"])   # arrows ∝ u
for x0 in (0.04*L, 0.96*L):                               # the two parts at the narrowest and widest stations
    hx = h0*(1 + al*x0/L); yy = np.linspace(0, hx, 25)
    uc = U*yy/hx                                          # Couette part (the pad drags at U) [m/s]
    up = ch08.slider_gap_velocity(x0, yy, h0, al, L, U, mu=mu, frame="ground") - uc   # Poiseuille part [m/s]
    ax.plot(1e2*(x0 + 0.12*uc/U*L), 1e2*k*yy, color=COLORS["rose"], ls="--", lw=1)
    ax.plot(1e2*(x0 + 0.12*up/U*L), 1e2*k*yy, color=COLORS["orange"], ls="--", lw=1)
ax.set_xlabel("x along the pad [cm]"); ax.set_ylabel(f"height × {k:.0f} [cm]")
ax.set_title("Couette plus Poiseuille, station by station (heights exaggerated 200×; pad moving at U, α = 1.5)", fontsize=10)
ax.plot([], [], color=COLORS["blue"], label="u(y) (ground frame)"); ax.plot([], [], color=COLORS["rose"], ls="--", label="Couette part")
ax.plot([], [], color=COLORS["orange"], ls="--", label="Poiseuille part"); ax.legend(fontsize=8, loc="upper left")
savefig(fig, "ch08", "c06_gap_profiles"); plt.show()

**What you see.** Profiles in a gap that widens from h₀ at the left (the pad's trailing edge) to 2.5h₀ at the right (its leading
edge), the pad moving right at U over a fixed floor: near the narrow end the profiles sag below the straight Couette line (the
orange Poiseuille part points backward), near the wide end they bulge forward. The dashed rose and orange curves are the two
parts at the end stations — our remake of Fig. 8.8.

**How to read it.** The pressure hump pushes fluid backward where it rises (from the narrow end up to the peak at x = L/(2 + α) ≈ 1.4 cm) and
forward where it falls. In the ground frame the flux grows with the gap, $q=C_1+Uh$, because the pad-frame flux $C_1$ is the
same at every station (D14).

**What would change if…** …the pad slid the other way (U < 0): every profile flips, and the pressure hump becomes a suction dip (C07).

📝 **Note.** **N36 [B] · Hele-Shaw flow (Example 8.2): viscous flow that draws ideal streamlines.** Between two plates z = 0 and z = h the lubrication balances $0\cong-\frac1\rho\frac{\partial p}{\partial x}+\nu\frac{\partial^2u}
{\partial z^2}$, $0\cong-\frac1\rho\frac{\partial p}{\partial y}+\nu\frac{\partial^2v}{\partial z^2}$,
$0\cong-\frac1\rho\frac{\partial p}{\partial z}$ give the velocity below, with the potential $\phi=-\frac{z(h-z)}{2\mu}p$.
Integrating continuity over the gap (the 2-D Reynolds equation with constant h, depth-averaged velocity
$\bar{\mathbf u}=-\frac{h^2}{12\mu}\nabla p$) leaves $\frac{\partial^2p}{\partial x^2}+\frac{\partial^2p}{\partial y^2}=0$ — so φ
obeys $u\equiv\partial\phi/\partial x,\ v\equiv\partial\phi/\partial y$ *(6.10)* and $\nabla^2\phi=0$ *(6.12)*, the equations of
2-D ideal flow (Ch. 6). Dye injected between glass plates therefore traces ideal-flow streamlines, except in layers of
thickness ~h at obstacles, where no slip holds (Exercise 8.34). ⚠️ The book prints these lubrication equations without the ν,
and writes the wall conditions at "y = 0, h" — the gap coordinate here is z.

$$ u\cong-\frac{1}{2\mu}\frac{\partial p}{\partial x}z(h-z)=\frac{\partial\phi}{\partial x},\qquad \phi=-\frac{z(h-z)}{2\mu}p  $$

In [ ]:
a, gap, Um = 0.01, 1e-3, 1e-3                             # disc radius [m], plate gap [m], mean speed [m/s]
xg = np.linspace(-0.04, 0.04, 241 if not FAST else 121)   # a 8 cm × 8 cm window [m]
X, Y = np.meshgrid(xg, xg)
hs = ch08.hele_shaw_cylinder(X, Y, None, Um, a, gap)      # depth-averaged Hele-Shaw flow (z = None → gap average)
psi_ideal = PF.stream_function(PF.cylinder(Um, a), X, Y)  # Ch. 6: ideal flow past a circular cylinder
inside = X**2 + Y**2 < a**2                               # the disc
lev = np.linspace(-3.5e-5, 3.5e-5, 15)                    # streamline levels [m²/s]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw=dict(width_ratios=[1.4, 1]))
a1.contour(X*1e2, Y*1e2, np.where(inside, np.nan, hs["psi_mean"]), lev, colors=COLORS["blue"], linewidths=1.6)
a1.contour(X*1e2, Y*1e2, np.where(inside, np.nan, psi_ideal), lev, colors=COLORS["muted"], linewidths=1, linestyles="--")
a1.add_patch(plt.Circle((0, 0), a*1e2, color=COLORS["muted"]))
a1.set_aspect("equal"); a1.set_xlabel("x [cm]"); a1.set_ylabel("y [cm]")
a1.set_title("Hele-Shaw dye lines (blue) on ideal streamlines (dashed)", fontsize=10)
z = np.linspace(0, gap, 50)                               # across the gap [m]
dpdx = -12*MU_W*Um/gap**2                                 # the far-field gradient that drives the mean speed Um [Pa/m]
u_z, _ = ch08.hele_shaw_velocity(z, gap, (dpdx, 0.0))     # the parabola across the gap [m/s]
a2.plot(1e3*u_z, 1e3*z, color=COLORS["blue"], lw=2); a2.axvline(1e3*Um, color=COLORS["muted"], ls="--", lw=1)
a2.text(1e3*Um*1.02, 0.1, "mean", fontsize=8, color=COLORS["muted"])
a2.set_xlabel("u [mm/s]"); a2.set_ylabel("z across the gap [mm]"); a2.set_title("Profile across the gap (far away)", fontsize=10)
savefig(fig, "ch08", "c06_hele_shaw"); plt.show()

**What you see.** The blue gap-averaged dye lines of Hele-Shaw flow lie exactly on the dashed ideal-flow streamlines round a cylinder,
while across the 1 mm gap the flow is a viscous parabola — our remake of Fig. 8.10.

**How to read it.** The pressure is harmonic, so the depth-averaged velocity is a potential flow; only a thin layer at the disc (too thin to
see here) differs, where no slip holds.

**What would change if…** …the gap were as wide as the disc: the ε ≪ 1 assumption fails, inertia and the no-slip layer spread, and the pattern
loses its fore–aft symmetry.

**A numerical check (our route, not the book's).** `hele_shaw_streamfunction_grid` solves Laplace's equation for the
gap-averaged stream function on a square grid with the disc cut out (Ch. 6's masked five-point solver) and compares it with
the ideal-flow value away from the disc. The disc boundary is a staircase of grid cells, so the scheme is only **first
order**: doubling the grid roughly halves the error.

In [ ]:
ns = (33, 65, 129) if not FAST else (33, 65)             # nodes per side of the square grid
errs, hs_ = [], []                                        # far-field error and grid step for each grid
for n_ in ns:
    g_ = ch08.hele_shaw_streamfunction_grid(n=n_, a=1.0, U_mean=1.0, box=4.0)   # disc radius 1, box ±4 (non-dimensional)
    errs.append(g_["err_far"]); hs_.append(g_["h"])        # max |ψ − ψ_ideal| where r ≥ 2a, and the step
    print(f"n = {n_}: step {g_['h']:.3f}, max error {g_['err_far']:.2e}")
print(f"observed order ≈ {observed_order(hs_, errs):.2f}  (first order: the staircase disc)")

**What would change if…** …the gap were the space under a real, tilted pad with the same pressure $p_e$ at both ends? The flux must be constant along
the pad, which forces a pressure hump inside — and the hump carries a load (C07).

### 🧩 The slider bearing (Example 8.1) `C07`

*The question:* How does a thin viscous film carry a load — and why only in one direction?

*In one line:* $p-p_e=\frac{6\mu LU}{h_o^2}\frac{\alpha(x/L)(1-x/L)}{(2+\alpha)(1+\alpha x/L)^2}$ and, for a small taper, $W=\frac{\alpha\mu L^2U}{2h_o^2}$ (Example 8.1, denominator corrected to the square).

#### The problem in plain words

A tilted pad skates on a film of oil over a flat surface, as in the thrust bearing of a ship's propeller shaft or a
hydroelectric turbine carrying hundreds of tonnes. The pad never touches the surface. Where does the upward force come from,
and why does the bearing fail — the pad is sucked down — if it slides the wrong way?

#### The idea

```
            W ↓            pad moves → at U, wide end leading   (pad frame: floor moves ← at U)
p_e  ┌────────────────────────┐  p_e
     │ h₀ (x = 0)  →  h₀(1+α) (x = L) │
═════════════════════════════════   the floor drags oil in at the wide end, the narrow end lets less out
p(x):  ____/‾‾‾\____                ⇒ p rises until Poiseuille backflow evens the flux; ∫(p − p_e)dx = W
```

Same flux through a narrowing gap ⇒ the pressure must rise inside.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **antiderivatives of (1 + αx/L)⁻ⁿ** — substitute s = 1 + αx/L, ds = (α/L)dx: $\int(1+\alpha x/L)^{-n}dx=\frac L\alpha\frac{(1+\alpha x/L)^{1-n}}{1-n}$ for n ≠ 1 (Ch. 3 P106, substitution in an integral).
- **Taylor expansion in a small parameter α** — for small α, $(1+\alpha x/L)^{-2}\approx1-2\alpha x/L$ and $2+\alpha\approx2$: keep the first power of α (Ch. 1 P26, first-order Taylor).

#### 🧮 Derivation — The slider bearing: $p-p_e=\frac{6\mu LU}{h_o^2}\frac{\alpha(x/L)(1-x/L)}{(2+\alpha)(1+\alpha x/L)^2}$ and $W=\frac{\alpha\mu L^2U}{2h_o^2}$ `D14`

**What we want to show.** Find the pressure under a sloped pad sliding at U over a flat surface, and the load it carries.

**Assumptions.** Steady in the pad frame (step 1); lubrication profile (step 3); α ≪ 1 used only at step 14.

**The plan.**

1. Conserve mass in a CV riding with the pad.
2. Turn it into an ODE for p.
3. Integrate with the two end pressures.
4. Simplify, then expand for small α and integrate for W.

**Tools we use** (each explained before this point): moving control volume (Ch. 4 §4.2) · gap flux (D13 steps 9–10) · antiderivatives of (1 + αx/L)⁻ⁿ (gloss) · 2 × 2 system · Taylor in α (P26).

**We start from**

$$ h(x)=h_o\Big(1+\frac{\alpha x}{L}\Big),\ \ u(x,0)=0,\ \ u(x,h)=U,\ \ p(0)=p(L)=p_e,\qquad \frac{d}{dt}\int_{V^*(t)}\rho\,dV+\int_{A^*(t)}\rho\,(\mathbf u-\mathbf b)\cdot\mathbf n\,dA=0\ \text{(4.5)} $$

*In words:* a sloped pad sliding at U over a floor at rest, the same pressure at both ends, and the control-volume mass balance

---

**Step 1 of 16 — Balance mass in a pad-fixed CV.**

$$ \rho B\Big[-\int_0^{h(x_1)}(u-U)dy+\int_0^{h(x_2)}(u-U)dy\Big]=0 $$

- *Why we can do this:* (4.5), $\frac{d}{dt}\int_{V^*(t)}\rho\,dV+\int_{A^*(t)}\rho\,(\mathbf u-\mathbf b)\cdot\mathbf n\,dA=0$ for a CV of fixed shape moving at b = U e_x: its mass is constant, and the flux through a face uses the relative speed u − U.
- *In words:* what enters the slice leaves it.

---

**Step 2 of 16 — Shrink the slice.**

$$ \frac{d}{dx}\int_0^{h}(u-U)dy=0\ \Rightarrow\ \int_0^h(u-U)dy=C_1 $$

- *Why we can do this:* Divide by ρB(x₂ − x₁) and let x₂ → x₁: a derivative of zero means a constant.
- *In words:* the pad-frame flux C₁ is the same everywhere.

---

**Step 3 of 16 — Insert the profile's flux.**

$$ \int_0^hu\,dy=-\frac{h^3}{12\mu}\frac{dp}{dx}+\frac{Uh}{2} $$

- *Why we can do this:* D13 steps 9–11 with U₀ = 0, U_h = U.
- *In words:* ground-frame flux.

---

**Step 4 of 16 — Subtract Uh.**

$$ C_1=-\frac{h^3}{12\mu}\frac{dp}{dx}-\frac{Uh}{2} $$

- *Why we can do this:* The pad speed U is the same at every height, so ∫₀^h U dy = Uh; subtracting it gives the pad-frame flux.
- *In words:* the book's C₁ equation.

---

**Step 5 of 16 — Solve for the pressure gradient.**

$$ \frac{dp}{dx}=-\frac{12\mu C_1}{h^3}-\frac{6\mu U}{h^2} $$

- *Why we can do this:* Rearrange step 4; this is a first-order ODE for p once h(x) is known.
- *In words:* the pressure slope depends on the local gap.

---

**Step 6 of 16 — Insert the gap shape.**

$$ \frac{dp}{dx}=-\frac{12\mu C_1}{h_o^3}\Big(1+\frac{\alpha x}{L}\Big)^{-3}-\frac{6\mu U}{h_o^2} \Big(1+\frac{\alpha x}{L}\Big)^{-2} $$

- *Why we can do this:* h = h₀(1 + αx/L). The book's intermediate line prints (1 − αx/L); the gap of the problem needs 1 + αx/L.
- *In words:* two powers of the gap.

---

**Step 7 of 16 — Integrate in x.**

$$ p=\frac{6\mu LC_1}{\alpha h_o^3}\Big(1+\frac{\alpha x}L\Big)^{-2}+\frac{6\mu UL}{\alpha h_o^2}\Big(1+ \frac{\alpha x}L\Big)^{-1}+C_2 $$

- *Why we can do this:* ∫(1 + αx/L)⁻ⁿdx = (L/α)(1 + αx/L)^(1−n)/(1 − n) for n = 3, 2 (substitute s = 1 + αx/L).
- *In words:* pressure with two unknown constants.

---

**Step 8 of 16 — Apply p(0) = p_e.**

$$ p_e=\frac{6\mu L}{\alpha h_o^2}\Big[\frac{C_1}{h_o}+U\Big]+C_2 $$

- *Why we can do this:* At x = 0 both powers equal 1.
- *In words:* first end condition.

---

**Step 9 of 16 — Apply p(L) = p_e.**

$$ p_e=\frac{6\mu L}{\alpha h_o^2}\Big[\frac{C_1}{h_o(1+\alpha)^2}+\frac{U}{1+\alpha}\Big]+C_2 $$

- *Why we can do this:* At x = L the gap is h₀(1 + α).
- *In words:* second end condition.

---

**Step 10 of 16 — Subtract to find C₁.**

$$ C_1=-\frac{1+\alpha}{2+\alpha}Uh_o $$

- *Why we can do this:* C₂ cancels: (C₁/h₀)[1 − (1+α)⁻²] + U[1 − (1+α)⁻¹] = 0, i.e. (C₁/h₀)α(2+α)/(1+α)² = −Uα/(1+α).
- *In words:* the pad-frame flux (negative: toward the narrow end).

---

**Step 11 of 16 — Back-substitute for C₂.**

$$ C_2=p_e-\frac{6\mu LU}{\alpha h_o^2}\cdot\frac{1}{2+\alpha} $$

- *Why we can do this:* Step 8 with C₁: U − (1 + α)U/(2 + α) = U/(2 + α).
- *In words:* the book's C₂.

---

**Step 12 of 16 — Substitute C₁ and C₂ into step 7.**

$$ p-p_e=\frac{6\mu LU}{\alpha h_o^2}\Big[-\frac{1+\alpha}{(2+\alpha)(1+s)^2}+\frac{1}{1+s}-\frac{1}{2+\alpha}\Big],\ \ s=\frac{\alpha x}{L} $$

- *Why we can do this:* Insert C₁ = −(1 + α)Uh₀/(2 + α) and C₂ from step 11 into step 7 and subtract p_e; the factor 6μLU/(αh₀²) comes out of all three terms. (Part F went straight to the common denominator; this step is added so that each line follows from the one above.)
- *In words:* three simple fractions in s = αx/L.

---

**Step 13 of 16 — Put p − p_e over one denominator.**

$$ p-p_e=\frac{6\mu LU}{\alpha h_o^2}\cdot\frac{s(\alpha-s)}{(2+\alpha)(1+s)^2},\ \ s=\frac{\alpha x}{L} $$

- *Why we can do this:* Multiply each fraction up to the common denominator (2 + α)(1 + s)²; the numerator −(1 + α) + (2 + α)(1 + s) − (1 + s)² expands to −1 − α + 2 + 2s + α + αs − 1 − 2s − s² = αs − s² = s(α − s).
- *In words:* the "some algebra" of the book, done.

---

**Step 14 of 16 — Write s back in terms of x.**

$$ p-p_e=\frac{6\mu LU}{h_o^2}\frac{\alpha(x/L)(1-x/L)}{(2+\alpha)(1+\alpha x/L)^2} $$

- *Why we can do this:* s(α − s) = α²(x/L)(1 − x/L); one α cancels. The printed result has the denominator to the first power; only the square satisfies step 5.
- *In words:* the exact pressure hump.

---

**Step 15 of 16 — Keep the linear term in α.**

$$ p-p_e\cong\frac{3\alpha\mu LU}{h_o^2}\frac xL\Big(1-\frac xL\Big) $$

- *Why we can do this:* For α ≪ 1, (2 + α)(1 + αx/L)² ≈ 2; (8.19), $u\cong-\frac{h^2}{2\mu}\frac{\partial p}{\partial x}\frac yh\Big(1-\frac yh\Big)+U_h\frac yh+U_0\Big(1-\frac yh\Big)$ is only valid for small slopes anyway.
- *In words:* a symmetric parabola.

---

**Step 16 of 16 — Integrate for the load.**

$$ W=\int_0^L(p-p_e)dx=\frac{3\alpha\mu LU}{h_o^2}\cdot\frac L6=\frac{\alpha\mu L^2U}{2h_o^2} $$

- *Why we can do this:* ∫₀^L (x/L)(1 − x/L)dx = L/2 − L/3 = L/6.
- *In words:* the load per unit width.

---

**Result**

$$ \begin{array}{l}p-p_e=\frac{6\mu LU}{h_o^2}\frac{\alpha(x/L)(1-x/L)}{(2+\alpha)(1+\alpha x/L)^2}\ \ \text{(exact)} \\ p-p_e\cong\frac{3\alpha\mu LU}{h_o^2}\frac xL\Big(1-\frac xL\Big),\qquad W=\frac{\alpha\mu L^2U}{2h_o^2}\ \ (\alpha\ll1)\end{array} $$

*In words:* a constant flux through a narrowing gap needs a pressure hump, and the hump carries the pad.

**What it means.** Load ∝ 1/h₀²: a heavier load squeezes the film and raises its capacity — the bearing is stable. It works only if the pad slides toward the narrow end (αU > 0).

**Check it.** p(0) = p(L) = p_e ✓ (the printed first-power form also meets them, but it fails step 5's ODE); units μLU/h₀²: (Pa s)(m)(m/s)/m² = Pa ✓; W in Pa·m = N/m ✓; αU < 0 ⇒ W < 0. Numbers (the code cell after the tiny example of C07 prints them): L = 5 cm, h₀ = 50 µm, U = 5 m/s, μ = 0.05 Pa s, α = 0.1 → W = 12.5 kN/m (linear), 10.8 kN/m (exact, N35).

In [ ]:
sb = ch08.slider_bearing_sympy()                         # D14 steps 5–15 rebuilt symbolically (cached)
x_, L_, h0_, mu_, U_, al_, pe_ = sb["symbols"]           # x, L, h₀, μ, U, α, p_e
print("C1 =", sp.simplify(sb["C1"]))                     # step 10: −(1 + α)U h₀/(2 + α)
print("C2 =", sp.simplify(sb["C2"]))                     # step 11
assert sp.simplify(sb["ode_residual_exact"]) == 0        # the squared denominator satisfies dp/dx = −12μC₁/h³ − 6μU/h² (step 5)
assert sp.simplify(sb["ode_residual_book"]) != 0         # the printed first-power form does not
assert all(sp.simplify(r_) == 0 for r_ in sb["bc_residuals"])   # p(0) = p(L) = p_e
print("W_linear =", sb["W_linear"])                      # step 15: αμL²U/(2h₀²)
print("W_exact  =", sb["W_exact_series"])                # N35's exact load expanded in α: starts with W_linear

> ⚠️ **Common confusion (traps in `D14`):** The pad frame makes the flow steady; the integrands carry (1 + αx/L), not the printed (1 − αx/L); the final denominator is squared; α ≪ 1 enters only at step 14.

> ⚠️ **Common confusion:** **two printing slips in Example 8.1.** The intermediate integrals are printed with $(1-\alpha x/L)$, although the gap is
$h_o(1+\alpha x/L)$; and the final exact pressure is printed with $(1+\alpha x/L)$ to the first power in the denominator.
Substituting back into $dp/dx=-12\mu C_1/h^3-6\mu U/h^2$ shows the denominator must be **squared** (D14 step 14, checked
above). The printed C₁, C₂, the linear-α pressure and the load W are right.

#### ✏️ Tiny example: a 5 cm pad on a 50 µm film

L = 0.05 m, h₀ = 50 µm, U = 5 m/s, μ = 0.05 Pa s, α = 0.1.
1. Scale: $6\mu LU/h_o^2=6\times0.05\times0.05\times5/(2.5\times10^{-9})=3\times10^7$ Pa.
2. At the middle x = L/2: $\frac{\alpha(x/L)(1-x/L)}{(2+\alpha)(1+\alpha/2)^2}=\frac{0.1\times0.25}{2.1\times1.1025}=0.0108$ ⇒
   p − p_e = 3.24 × 10⁵ Pa ≈ 3.2 atm.
3. Linear formula: $\frac{3\alpha\mu LU}{h_o^2}\times0.25=3.75\times10^5$ Pa.
4. Load $W=\alpha\mu L^2U/(2h_o^2)=0.1\times0.05\times0.0025\times5/(5\times10^{-9})=1.25\times10^4$ N/m — 12.5 kN per metre of
   pad width, about 1.3 tonnes.
5. Reverse the motion (U = −5 m/s): W = −12.5 kN/m, the pad is pulled down.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **scipy.optimize.minimize_scalar** — `optimize.minimize_scalar(f, bounds=(a, b), method="bounded")` finds the minimum of a one-variable function; to maximise W we minimise −W (Ch. 7 P170).
- **np.log1p and cancellation for small α** — `np.log1p(a)` computes ln(1 + a) accurately for tiny a, where 1 + a loses digits — the same cancellation problem as `np.expm1` (Ch. 3 P107); the exact load uses a series for |α| < 10⁻³.

In [ ]:
L, h0, U, mu = 0.05, 50e-6, 5.0, 0.05                    # the pad of the tiny example
for model in ("exact", "linear", "book"):                # corrected exact, O(α), and the printed slip (a ghost)
    print(f"{model:6s} p(L/2) − p_e = {ch08.slider_bearing(0.025, h0, 0.1, L, U, mu=mu, model=model):.4g} Pa")
st = ch08.slider_bearing_state(h0, 0.1, L, U, mu=mu)     # everything the explainer shows
print({k: (round(st[k], 7) if not isinstance(st[k], bool) else st[k]) for k in
       ("C1", "p_max", "x_pmax", "W_exact", "W_linear", "err_linear", "p_max_atm", "inlet_backflow")})
print(ch08.slider_optimum_taper())                       # the taper that maximises the exact load
print("reversed U:", ch08.slider_bearing_load(h0, 0.1, L, -U, mu=mu, model="linear"), "N/m")   # suction

**What does the code above do?**

1. The three models at mid-pad: exact 3.239 × 10⁵ Pa, linear 3.75 × 10⁵ Pa, and the printed first-power form 3.401 × 10⁵ Pa —
   5 % off, and (D14 check) it does not satisfy the pressure equation.
2. The state: the pad-frame flux C₁ = −1.310 × 10⁻⁴ m²/s, the peak 3.247 × 10⁵ Pa at x = 2.38 cm (x/L = 0.476, just before
   mid-pad), the exact load 10.8 kN/m against the linear 12.5 kN/m — the linear one 15.6 % high — the peak in atmospheres, and no inlet
   recirculation (α ≤ 1).
3. `slider_optimum_taper` maximises the exact load: α = 1.189, inlet/outlet gap ratio 1 + α = 2.189, dimensionless load
   W h₀²/(6μUL²) = 0.02671 (the San Andrés benchmark).
4. Reversing U turns the load negative: the pad is sucked down.

In [ ]:
h_ = lambda x_: h0*(1 + 0.1*x_/L)                        # the gap h₀(1 + αx/L), α = 0.1 [m]
dpdx = lambda x_, C1: -12*mu*C1/h_(x_)**3 - 6*mu*U/h_(x_)**2   # D14 step 5 (ground frame) [Pa/m]
C1 = optimize.brentq(lambda C: integrate.quad(lambda x_: dpdx(x_, C), 0, L)[0], -U*h0, 0.0)   # choose C₁ so p(L) = p(0)
x = np.linspace(0, L, 401)                               # stations [m]
p_mine = integrate.cumulative_trapezoid(dpdx(x, C1), x, initial=0)   # p − p_e by integrating the ODE
print(f"C1 by root finding = {C1:.6g} m²/s (library {st['C1']:.6g}); max |Δp| = {np.max(np.abs(p_mine - ch08.slider_bearing(x, h0, 0.1, L, U, mu=mu))):.2f} Pa")
assert np.allclose(C1, st["C1"], rtol=1e-8)              # same flux constant
assert np.allclose(p_mine, ch08.slider_bearing(x, h0, 0.1, L, U, mu=mu), rtol=1e-6, atol=1.0)   # same pressure hump

In [ ]:
L, h0, U, mu = 0.05, 50e-6, 5.0, 0.05
x = np.linspace(0, L, 400)                                # along the pad [m]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
for al, c in ((0.1, COLORS["orange"]), (0.5, COLORS["amber"]), (1.189, COLORS["rose"])):   # three tapers
    a1.plot(x/L, ch08.slider_bearing(x, h0, al, L, U, mu=mu)/1e6, color=c, lw=2, label=f"α = {al:g}, exact")
    a1.plot(x/L, ch08.slider_bearing(x, h0, al, L, U, mu=mu, model="linear")/1e6, color=c, lw=1, ls="--")
a1.plot(x/L, ch08.slider_bearing(x, h0, 0.5, L, U, mu=mu, model="book")/1e6, color=COLORS["muted"], lw=1.5, ls=":",
        label="α = 0.5 as printed (first power)")
a1.plot([], [], color=COLORS["ink"], ls="--", lw=1, label="linear in α")
a1.set_xlabel("x/L [–]"); a1.set_ylabel("p − p_e [MPa]"); a1.legend(fontsize=7); a1.set_title("Pressure humps", fontsize=10)
als = np.linspace(-0.9, 3.0, 200)                         # tapers, including negative (pad sliding the wrong way)
W_ex = np.array([ch08.slider_bearing_load(h0, a_, L, U, mu=mu) for a_ in als])       # N35 exact load [N/m]
W_li = np.array([ch08.slider_bearing_load(h0, a_, L, U, mu=mu, model="linear") for a_ in als])
opt = ch08.slider_optimum_taper()
a2.plot(als, W_ex/1e3, color=COLORS["blue"], lw=2, label="exact (N35)")
a2.plot(als, W_li/1e3, color=COLORS["blue"], lw=1, ls="--", label="linear αμL²U/(2h₀²)")
a2.axvline(opt["alpha_opt"], color=COLORS["muted"], lw=0.8)
a2.text(opt["alpha_opt"]*1.03, 5, f"optimum α = {opt['alpha_opt']:.3f}", fontsize=8)
a2.fill_between(als, W_ex/1e3, 0, where=als < 0, color=COLORS["amber"], alpha=0.3, label="W < 0: pad sucked down")
a2.axhline(0, color=COLORS["muted"], lw=0.8); a2.set_ylim(-60, 100)
a2.set_xlabel("taper α [–]"); a2.set_ylabel("load W [kN/m]"); a2.legend(fontsize=7); a2.set_title("Load against taper", fontsize=10)
fig.suptitle("A pressure hump carries the load — best at inlet/outlet gap ratio 2.19", fontsize=11)
savefig(fig, "ch08", "c07_slider_pressure"); plt.show()

**What you see.** (a) pressure humps peaking just before mid-pad and growing with α, their linear-α parabolas (dashed) and the printed
first-power form for α = 0.5 (dotted grey) missing the exact hump; (b) the exact load curve bending over and peaking near
α = 1.19, while the linear formula keeps rising — our remake of Fig. 8.9 `N105`.

**How to read it.** The load is the area under a hump; the linear formula is fine only for α ≲ 0.1 — 15.6 % high at 0.1, about 90 % high at 0.5;
negative α (shaded) means the pad slides with its narrow end leading and is pulled down.

**What would change if…** …the film were halved to 25 µm: every pressure and the load grow 4× (∝ 1/h₀²) — the bearing is "stiff": extra load
squeezes the film and raises the capacity.

📝 **Note.** **N35 [B] · The exact load for any taper (ours; the book stops at linear α).** Integrating the exact pressure gives W below, which tends to $\alpha\mu L^2U/(2h_o^2)$ as α → 0 (the series of ln(1 + α)). It
peaks at an inlet/outlet gap ratio 1 + α = 2.189 — the classic optimum of lubrication texts (San Andrés, benchmark V5). With
our numbers: α = 0.1 → 12.5 kN/m (linear) vs 10.8 kN/m (exact); α = 0.5 → 62.5 vs 32.8 kN/m.

$$ W=\frac{6\mu UL^2}{h_o^2\alpha^2}\Big[\ln(1+\alpha)-\frac{2\alpha}{2+\alpha}\Big]  $$

In [ ]:
W_formula = ch08.slider_bearing_load(h0, 0.5, L, U, mu=mu)     # N35 for α = 0.5 [N/m]
W_quad = integrate.quad(lambda xx: ch08.slider_bearing(xx, h0, 0.5, L, U, mu=mu), 0, L)[0]   # ∫(p − p_e)dx numerically
print(f"W (formula) = {W_formula:.5g} N/m, W (quad) = {W_quad:.5g} N/m")   # both 3.279e4

In [ ]:
L, h0, U, mu = 0.05, 50e-6, 5.0, 0.05
xs = np.linspace(0, L, 150)                              # stations [m]


def frame(al):                                            # curves for one taper α
    return {"exact": (xs/L, ch08.slider_bearing(xs, h0, al, L, U, mu=mu)/1e6),
            "linear in α": (xs/L, ch08.slider_bearing(xs, h0, al, L, U, mu=mu, model="linear")/1e6),
            "as printed (first power)": (xs/L, ch08.slider_bearing(xs, h0, al, L, U, mu=mu, model="book")/1e6)}


als = np.linspace(-0.8, 3.0, 20 if not FAST else 10)      # slider values α [–]
fig = slider_figure(frame, "α", als, unit="", xlabel="x/L [–]", ylabel="p − p_e [MPa]", title="", yrange=[-2.0, 2.5])
titles = []                                               # the two loads in each title
for al in als:
    s_ = ch08.slider_bearing_state(h0, al, L, U, mu=mu)
    titles.append(f"α = {al:.2f}: W exact = {s_['W_exact']/1e3:.1f} kN/m, W linear = {s_['W_linear']/1e3:.1f} kN/m")
step_titles(fig, titles)
recolor(fig, {"exact": COLORS["orange"], "linear in α": COLORS["orange"], "as printed (first power)": COLORS["muted"]},
        {"linear in α": "dash", "as printed (first power)": "dot"})
fig.show()

**What does the code above do?**

Each slider step is one taper α from −0.8 to 3; the title prints the exact and linear loads. For α < 0 the hump becomes a
suction dip; for large α the linear formula wildly overestimates the load; the grey dotted "as printed" curve meets both end
pressures but is the wrong shape.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **live widgets** — `live(fn, name=(min, max, step), …)` makes sliders that call fn when released — only with a running kernel; the plotly figure above carries the same idea on the web page (Ch. 1 P47).

> ▶️ **Live cell.** The widget below needs a running Python kernel. On the web page it is frozen — open this notebook in Colab (button at the top) or run it locally to drag the sliders.

In [ ]:
def bearing(h0_um=50.0, alpha=0.5, U=5.0, mu=0.05):      # film thickness [µm], taper, sliding speed [m/s], oil viscosity [Pa s]
    s_ = ch08.slider_bearing_state(h0_um*1e-6, alpha, 0.05, U, mu=mu)   # a 5 cm pad
    print(f"p_max = {s_['p_max']/1e5:.3g} bar at x/L = {s_['x_pmax']/0.05:.3f}; W exact = {s_['W_exact']/1e3:.3g} kN/m, "
          f"W linear = {s_['W_linear']/1e3:.3g} kN/m ({s_['err_linear']:.1f} % off); inlet backflow: {s_['inlet_backflow']}")


live(bearing, h0_um=(10, 200, 5), alpha=(-0.8, 3.0, 0.05), U=(-10, 10, 0.5), mu=(0.005, 0.5, 0.005))

#### 🎮 Interactive: How does a film thinner than a hair carry a load?

**Why interactive rather than a static figure:** Drag the taper and flip the sliding direction: the pressure hump, the station-by-station Couette-plus-Poiseuille profiles and
the load change together, the printed-slip ghost visibly departs from the exact hump, and past α = 1 a small recirculation
appears at the wide inlet — the whole of Example 8.1 in one picture.

**What to try:**
- Set α = 0.1 and compare the exact and linear humps; then α = 1.19 (the optimum) and read the load.
- Flip U to negative: the hump turns into suction and the status says the pad is sucked down.
- Choose 'compare': the printed curve meets both end pressures but sits above the exact one — it fails the pressure equation.
- Push α past 1 and look at the inlet profiles in the pad frame: a backward-moving layer appears under the pad.

In [ ]:
show_viz("ch08", "slider_bearing")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …there were no upper wall at all — a free surface — and gravity, not a pressure applied at the ends, pushed the fluid? The
thin-layer balance still holds, with the hydrostatic pressure of the layer itself (C08).

### 🧩 The thin-film equation (Example 8.3) `C08`

*The question:* What equation governs a thin viscous layer spreading under its own weight?

*In one line:* $\frac{\partial h}{\partial t}=\frac{\rho g}{3\mu}\frac{\partial}{\partial x}\Big(h^3\frac{\partial h}{\partial x}\Big)$ (Example 8.3) — a diffusion equation whose diffusivity $\rho gh^3/3\mu$ depends on the thickness itself.

#### The problem in plain words

Honey poured on a plate, paint on a wall, a lava lobe, a mud flow, and — with a stiffer, nonlinear rheology — an ice sheet: a
thin layer of very viscous fluid spreads because its own weight presses harder where it is thicker. We want one equation for
the thickness h(x, t) that predicts the spreading.

#### The idea

```
     h ↑   ____
          /    \      p = ρg(h − y): higher under the crest
         /      \     flux q = −(ρg/3μ) h³ ∂h/∂x   ← h³: a thin film barely moves
════════╱════════╲════════
```

Thick places have higher hydrostatic pressure, so fluid is pushed from thick to thin; the flux is choked as the layer thins.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **hydrostatic pressure in a thin layer, stress-free surface** — at a free surface with negligible air drag the shear stress vanishes, μ∂u/∂y = 0; in a thin, slowly varying layer the vertical balance is hydrostatic, p = p_a + ρg(h − y) — valid because the slope is small (the lubrication ordering of C05), not merely because accelerations are ignored (Ch. 1 P20, boundary conditions).

#### 🧮 Derivation — The thin-film equation $\frac{\partial h}{\partial t}=\frac{\rho g}{3\mu}\frac{\partial}{\partial x}\big(h^3\frac{\partial h}{\partial x}\big)$ (Example 8.3) `D15`

**What we want to show.** Find one equation for the thickness of a thin viscous layer spreading under its own weight.

**Assumptions.** Small slope and no inertia (lubrication, steps 3, 5); no surface tension; 2-D; air drag negligible (step 7).

**The plan.**

1. Conserve mass in a slice.
2. Get the pressure gradient from hydrostatics.
3. Fix A and B with no slip below and no stress on top.
4. Integrate the profile and insert.

**Tools we use** (each explained before this point): CV mass (Ch. 4 §4.2) · hydrostatic thin layer and stress-free surface (gloss in C08) · polynomial integral.

**We start from**

$$ \frac{d}{dt}\int_{V^*(t)}\rho\,dV+\int_{A^*(t)}\rho\,(\mathbf u-\mathbf b)\cdot\mathbf n\,dA=0\ \text{(4.5)},\qquad u(x,y,t)\cong\frac1\mu\frac{\partial p(x,t)}{\partial x}\frac{y^2}{2}+Ay+B\ \text{(8.18)} $$

*In words:* the control-volume mass balance on a slice of width dx, and the generic lubrication profile

---

**Step 1 of 10 — Balance mass in a slice.**

$$ \rho\frac{\partial h}{\partial t}dx-\int_0^{h(x)}\rho u\,dy+\int_0^{h(x+dx)}\rho u\,dy=0 $$

- *Why we can do this:* (4.5), $\frac{d}{dt}\int_{V^*(t)}\rho\,dV+\int_{A^*(t)}\rho\,(\mathbf u-\mathbf b)\cdot\mathbf n\,dA=0$ for a fixed slice of width dx and unit depth: storage change plus net outflow is zero.
- *In words:* the layer thickens where more enters than leaves.

---

**Step 2 of 10 — Divide by ρ dx and shrink.**

$$ \frac{\partial h}{\partial t}+\frac{\partial}{\partial x}\int_0^hu\,dy=0 $$

- *Why we can do this:* The difference of the two integrals over dx becomes an x-derivative as dx → 0.
- *In words:* thickness changes by flux divergence.

---

**Step 3 of 10 — Write the hydrostatic pressure.**

$$ p=p_a+\rho g(h-y) $$

- *Why we can do this:* In a thin layer with small slope the vertical balance is hydrostatic (the lubrication ordering of C05), with the air pressure p_a on top.
- *In words:* pressure grows with depth below the surface.

---

**Step 4 of 10 — Take its x-derivative.**

$$ \frac{\partial p}{\partial x}=\rho g\frac{\partial h}{\partial x} $$

- *Why we can do this:* p_a is uniform and y does not depend on x.
- *In words:* the surface slope drives the flow — same at every height.

---

**Step 5 of 10 — Put it into (8.18).**

$$ u\cong\frac{\rho g}{2\mu}\frac{\partial h}{\partial x}y^2+Ay+B $$

- *Why we can do this:* (8.18), $u(x,y,t)\cong\frac1\mu\frac{\partial p(x,t)}{\partial x}\frac{y^2}{2}+Ay+B$ with the pressure gradient of step 4.
- *In words:* a parabola in y.

---

**Step 6 of 10 — Apply no slip on the plate.**

$$ u(0)=0\ \Rightarrow\ B=0 $$

- *Why we can do this:* No slip (8.3), $\mathbf t\cdot\mathbf U_s=(\mathbf t\cdot\mathbf u)_{\text{on the surface}}$ on the stationary surface makes the velocity vanish at y = 0.
- *In words:* the bottom layer sticks.

---

**Step 7 of 10 — Apply no stress on the surface.**

$$ \mu\frac{\partial u}{\partial y}\Big\rvert_h=\rho g\frac{\partial h}{\partial x}h+\mu A=0\ \Rightarrow\ A=-\frac{\rho g}{\mu}h\frac{\partial h}{\partial x} $$

- *Why we can do this:* The air exerts no shear on the free surface.
- *In words:* the profile is flat at the top.

---

**Step 8 of 10 — Assemble the profile.**

$$ u\cong-\frac{\rho g}{2\mu}\frac{\partial h}{\partial x}y(2h-y) $$

- *Why we can do this:* Insert A, B: (ρg/2μ)h_x(y² − 2hy).
- *In words:* a half-parabola, fastest at the surface, down the slope.

---

**Step 9 of 10 — Integrate across the layer.**

$$ \int_0^hu\,dy=-\frac{\rho g}{2\mu}\frac{\partial h}{\partial x}\cdot\frac{2h^3}{3}=-\frac {\rho g}{3\mu}h^3\frac{\partial h}{\partial x} $$

- *Why we can do this:* ∫₀^h y(2h − y)dy = h³ − h³/3 = 2h³/3.
- *In words:* flux ∝ h³ × slope.

---

**Step 10 of 10 — Insert into the mass balance.**

$$ \frac{\partial h}{\partial t}=\frac{\rho g}{3\mu}\frac{\partial}{\partial x}\Big(h^3 \frac{\partial h}{\partial x}\Big) $$

- *Why we can do this:* Step 9 into step 2; the two minus signs cancel.
- *In words:* a nonlinear diffusion equation for h.

---

**Result**

$$ \frac{\partial h}{\partial t}=\frac{\rho g}{3\mu}\frac{\partial}{\partial x}(h^3\frac{\partial h}{\partial x}) $$

*In words:* a diffusion whose diffusivity ρgh³/3μ collapses as the layer thins.

**What it means.** The layer spreads ever more slowly; the exponent (t^(1/5)) comes in C10. Lava, mud flows and, with a nonlinear rheology, ice sheets obey the same structure.

**Check it.** Units: ρg/3μ is (kg/m³)(m/s²)/(Pa s) = 1/(m s); ∂(h³∂h/∂x)/∂x is m³/m = m²; product m/s = ∂h/∂t ✓. Volume ∫h dx is conserved (flux form). Glycerol: ρg/3μ = 4.12 × 10³ m⁻¹s⁻¹.

> ⚠️ **Common confusion (traps in `D15`):** The hydrostatic pressure needs a small slope, not only "no acceleration"; the top condition is ∂u/∂y = 0, not u = 0; ∫₀^h y(2h − y)dy = 2h³/3; the flux points down the slope.

> 📎 **Primer — nonlinear diffusion in flux form.** `P191` · $\partial h/\partial t=\partial/\partial x\,(D(h)\,\partial h/\partial x)$ is a diffusion equation whose diffusivity depends
on the unknown — here $D=\rho gh^3/(3\mu)$, large where the layer is thick, tiny where it is thin. Written as
$\partial h/\partial t+\partial q/\partial x=0$ with the flux $q=-D\,\partial h/\partial x$, integrating over x shows the total
volume ∫h dx never changes when no fluid leaves the ends.

In [ ]:
import numpy as np
rho, g, mu = 1260.0, 9.81, 1.0                           # glycerol
for h in (0.01, 0.005, 0.001):                           # layer thickness [m]
    print(h, rho*g*h**3/(3*mu))                          # effective diffusivity [m²/s] falls as h³

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **explicit stepping and its stability limit** — an explicit (forward-Euler) step of a diffusion equation is stable only for Δt ≤ Δx²/(2D) (Ch. 1 P30) — with D ∝ h³ that limit changes as the layer thins.

> 📎 **Primer — implicit time stepping with Picard iteration.** `P192` · Explicit steps of a stiff diffusion equation need a tiny time step. Backward Euler evaluates the right-hand side at the *new*
time, which is stable for any step; when the diffusivity depends on the unknown we guess it from the last iterate, solve the
linear system, update the guess and repeat until it stops changing (Picard iteration; our solver uses the faster Newton
version of the same fixed point). A thin "precursor film" $h_{min}$ keeps the diffusivity positive ahead of the front.

In [ ]:
import numpy as np
D, dt, dx = 1.0, 0.1, 0.1                                # an explicit step would need dt ≤ dx²/(2D) = 0.005
lam = D*dt/dx**2                                         # = 10: explicit would blow up
print(1/(1 + 4*lam))                                     # backward-Euler damping of the shortest mode: 0.024 < 1, stable

#### ✏️ Tiny example: a glycerol bead 1 cm high

ρ = 1260 kg/m³, μ = 1 Pa s, g = 9.81 m/s².
1. Coefficient $\rho g/3\mu=1260\times9.81/3=4.12\times10^3$ m⁻¹s⁻¹.
2. Flux where h = 1 cm and the slope is ∂h/∂x = −0.5: $q=-4.12\times10^3\times10^{-6}\times(-0.5)=2.06\times10^{-3}$ m²/s,
   flowing outward.
3. Where h = 1 mm with the same slope the flux is 1000 times smaller — the edge barely moves.
4. Effective diffusivity at the crest $\rho gh^3/3\mu=4.1\times10^{-3}$ m²/s, falling as h³ while the bead thins.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **np.where** — `np.where(cond, a, b)` picks a where the condition holds and b elsewhere — here a box-shaped initial bead (Ch. 1 P46).

In [ ]:
x = np.linspace(-0.2, 0.2, 400 if not FAST else 200)     # nodes across a 40 cm plate [m]
h0 = np.where(np.abs(x) < 0.01, 0.01, 0.0)               # a box 1 cm high and 2 cm wide: area 2 × 10⁻⁴ m² (half-area 10⁻⁴)
t_out = np.r_[0.0, np.geomspace(0.1, 1000, 24 if not FAST else 12)]   # output times on a log clock [s]
run = ch08.thin_film_spread(h0, x, t_out, rho=1260.0, mu=1.0)   # backward Euler + Newton, conservative flux form
print(f"volume: first {run['volume'][0]:.6e}, last {run['volume'][-1]:.6e} m² (relative change {abs(run['volume'][-1]/run['volume'][0] - 1):.1e})")
for t_ in (10, 100, 1000):                               # compare the front with the similarity law of C10
    i = np.argmin(np.abs(run["t"] - t_))                 # the output time closest to t_
    xN = ch08.viscous_current_similarity(0.0, run["t"][i], 1e-4, rho=1260.0, mu=1.0, return_front=True)[1]
    print(f"t = {run['t'][i]:7.1f} s: numerical front {100*run['x_front'][i]:.2f} cm, similarity x_N {100*xN:.2f} cm")
print("flux at h = 1 cm, slope −0.5:", ch08.thin_film_flux(0.01, -0.5, rho=1260.0, mu=1.0), "m²/s")

**What does the code above do?**

1. `thin_film_spread` marches $\partial h/\partial t=\frac{\rho g}{3\mu}\partial_x(h^3\partial_xh)$ in the conservative flux form
   (flux differences between cells), so the volume is conserved to round-off (the printed relative change); the precursor film
   of 1 µm adds a constant 4 × 10⁻⁷ m² to it.
2. The steps are implicit (backward Euler, Newton iterations), so large time steps are stable although the diffusivity is
   stiff.
3. The fronts approach the similarity law of C10, $x_N=\eta_N(\beta A^3t)^{1/5}$: a few percent ahead at 10 s (the box has not
   been forgotten yet), within 2 % at 1000 s.
4. `thin_film_flux` is the tiny example's 2.06 × 10⁻³ m²/s.

In [ ]:
xe = np.linspace(-0.2, 0.2, 200); dxe = xe[1] - xe[0]    # 200 cells for the explicit version [m]
he = np.where(np.abs(xe) < 0.01, 0.01, 0.0)              # the same box
beta = 1260.0*ch08.G0/(3*1.0)                            # ρg/3μ [1/(m s)] (ch08 uses g = 9.80665 m/s²)
dte = 0.2*dxe**2/(beta*he.max()**3)                      # explicit stability: well below Δx²/(2D_max) [s]
vol0 = he.sum()*dxe                                      # the starting volume (area per unit width) [m²]
for _ in range(2000):                                    # 2000 explicit finite-volume steps
    hf = 0.5*(he[1:] + he[:-1])                          # thickness at the cell faces
    F = -beta*hf**3*np.diff(he)/dxe                      # face flux q = −β h³ ∂h/∂x [m²/s]
    he[1:-1] -= dte/dxe*np.diff(F)                       # conservative update: flux in minus flux out
t_end = 2000*dte                                         # the time reached [s]
ref = ch08.thin_film_spread(np.where(np.abs(xe) < 0.01, 0.01, 0.0), xe, [t_end], rho=1260.0, mu=1.0)["h"][-1]
print(f"t = {t_end:.3f} s: volume by hand {he.sum()*dxe:.8e} m²; max |explicit − library| = {np.max(np.abs(he - ref)):.1e} m")
assert abs(he.sum()*dxe - vol0) < 1e-12*vol0             # the explicit scheme conserves the volume to round-off
assert np.max(np.abs(he - ref)) < 0.01*0.01              # agree to 1 % of the bead height (the library adds a 1 µm precursor film)

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **animate and show_animation** — `animate(update, frames, fig)` calls update(i) for every frame; `show_animation(anim, player="frames")` gives ◀ ▮▮ ▶ buttons to stop at any time (Ch. 1 P16).

In [ ]:
xN = ch08.viscous_current_similarity(0.0, run["t"][1:], 1e-4, rho=1260.0, mu=1.0, return_front=True)[1]   # similarity fronts [m]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.4), gridspec_kw=dict(width_ratios=[1.6, 1]))
fill = [a1.fill_between(100*x, 1000*run["h"][0], color=COLORS["blue"], alpha=0.5)]   # the bead (a list so update can replace it)
front, = a1.plot([], [], "|", color=COLORS["amber"], ms=18, mew=2)    # the front marker
a1.set_xlim(-15, 15); a1.set_ylim(0, 10.5); a1.set_xlabel("x [cm]"); a1.set_ylabel("h [mm]")
a2.loglog(run["t"][1:], 100*xN, color=COLORS["accent"], ls="--", lw=1.5, label="slope 1/5 (similarity)")
dots, = a2.loglog([], [], "o", color=COLORS["teal"], ms=4, label="numerical front")
a2.set_xlim(0.08, 1300); a2.set_ylim(0.8, 20); a2.set_xlabel("t [s]"); a2.set_ylabel("front x_N [cm]"); a2.legend(fontsize=7)


def update(i):                                            # frame i = output time run["t"][i]
    fill[0].remove()                                      # replace the filled bead
    fill[0] = a1.fill_between(100*x, 1000*run["h"][i], color=COLORS["blue"], alpha=0.5)
    front.set_data([100*run["x_front"][i]], [0.3])        # front marker on the plate
    dots.set_data(run["t"][1:i+1], 100*run["x_front"][1:i+1])   # the front history so far
    a1.set_title(f"t = {run['t'][i]:.3g} s · area ∫h dx = {run['volume'][i]*1e4:.4f} cm² (never changes)", fontsize=9)
    return []


show_animation(animate(update, frames=len(run["t"]), fig=fig, interval=200), player="frames")

**What you see.** A box of glycerol slumps into a dome and keeps spreading ever more slowly; the amber front marker creeps outward; on
the right the numerical front (teal dots) settles onto the purple slope-1/5 line; the volume in the title never changes — the
`A3` animation, our remake of Fig. 8.11.

**How to read it.** On the log–log panel ten times longer buys only $10^{1/5}=1.58$ times more spread; use the ◀ ▶ buttons to stop at any
time and compare the dome with the box.

**What would change if…** …the fluid were ten times more viscous: every time is ten times longer (t enters only as ρgt/μ), the curves are the
same.

📝 **Note.** **N37 [C] · A similarity solution** — a change of variables that turns this PDE into an ODE — exists for the spreading bead; §8.4 develops the idea (C09) and
delivers this one (C10: $h=At^{-1/5}F(x/Dt^{1/5})$).

#### 🎮 Interactive: Why does a honey drop spread like t^(1/5)?

**Why interactive rather than a static figure:** Play the spreading from any initial shape and watch it forget that shape and lock onto one profile, while the front marches
along a slope-1/5 line on log–log axes and the volume readout never moves — convergence to self-similarity is a process in
time, and C10 derives the exponent you measure here.

**What to try:**
- Start from 'two humps': they merge, then the profile turns into the same dome as the box.
- Watch the log–log front view: compare the slope-1/5 and slope-1/2 guides.
- Make the fluid 1000× more viscous (lava lobe) and predict how the curves change before you press play.
- Switch the rescaled view to the wrong exponent m = 1/2 and see the collapse fail.

In [ ]:
show_viz("ch08", "viscous_gravity_current")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …a flow had no length or time scale at all — a plate started suddenly in a fluid that fills all space? Then the only way to
make y dimensionless is to combine it with time, $y/\sqrt{\nu t}$, and the whole PDE collapses to an ODE (C09).

---

## 8.4 Similarity Solutions for Unsteady Incompressible Viscous Flow

**What is this section about?** A plate starts moving under still fluid: how far does the motion reach after a time t?
Because the problem has no length or time scale of its own, y and t can only appear together as $y/\sqrt{\nu t}$; the PDE
collapses to an ODE whose solution is the error function, and every profile at every time is one curve. We then turn this
into a recipe — guess a power-law form, demand that every term scales the same way with t — and use it for a thickening shear
layer, a decaying vortex and the spreading bead of C08. The $\sqrt{\nu t}$ scale is the thickness of every laminar boundary
layer (Ch. 9) and of the Ekman layers of Ch. 13.

> 🔁 **Recap — Couette start-up: a flow with a length scale** (Ch. 1 §1.5). Ch. 1 solved the start-up of plane Couette flow — a plate suddenly moving at U with a second, fixed plate at distance h — as
a Fourier series (`couette_startup_profile`, in which the plate at y = h moves). Impulsively started parallel flows keep
$u\,\partial u/\partial x=0$, so they are exact solutions (Exercise 8.31). The gap h is an imposed length: the profiles at
different times are *not* one curve. Remove the second plate and that changes (C09).

### 🧩 Stokes' first problem `C09`

*The question:* What is u(y, t) above a plate that suddenly starts moving — and why do all the profiles look the same when rescaled?

*In one line:* $\frac{u}{U}=1-\mathrm{erf}\Big(\frac{y}{2\sqrt{\nu t}}\Big)$ *(8.30)*

#### The problem in plain words

Yank a plate sideways under a still bath (or start the wind over a calm lake): at first only a thin layer moves; the "news"
that the wall moves spreads upward by viscous diffusion. How thick is the moving layer after one second, one minute, one
hour? This is the prototype of every boundary layer.

#### The idea

```
t = 1 s        t = 4 s          t = 16 s        rescaled: y/√(νt)
│▏             │▎                │▍               │ one curve F(η)
│▎ thin        │▌ 2× thicker     │█ 4× thicker    │ for every t, U, ν
══►U           ══►U              ══►U
```

No ruler in the problem except $\sqrt{\nu t}$, so the profile can only stretch.

📝 **Note.** **N38 [B], N39 [B] · Set-up (Fig. 8.12; often called Rayleigh's problem).** An infinite plate at y = 0 starts moving at U at t = 0 under fluid at rest in y > 0; nothing depends on x, so continuity gives
v = 0, and the pressure is the same everywhere because far from the plate the fluid is still at rest. What remains is the 1-D
diffusion equation (8.20) — the heat equation of Ch. 1 with ν in place of κ.

$$ \frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2} \qquad \text{(8.20)} $$

#### 🧮 Derivation — The diffusion equation $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$ for the impulsively started plate `D16` (Eq. 8.20: $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$)

**What we want to show.** Reduce Navier–Stokes to a diffusion equation for the fluid above a plate that suddenly starts moving.

**Assumptions.** Infinite plate (nothing depends on x); fluid at rest at infinity at every finite t (step 5).

**The plan.**

1. Use invariance in x.
2. Show the pressure gradient vanishes.
3. Divide by ρ.

**Tools we use** (each explained before this point): continuity (D02's move) · partial derivative (P25).

**We start from**

$$ u(x,y,t)\ \text{for}\ y>0,\ \ u(0,t)=U\ (t\ge0),\qquad \nabla\cdot\mathbf u=0\ \text{(4.10)},\qquad \frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u\ \text{(8.1)} $$

*In words:* an infinite plate at y = 0 set moving at U at t = 0 under fluid at rest

---

**Step 1 of 6 — Use invariance along the plate.**

$$ \frac{\partial}{\partial x}=0\ \Rightarrow\ \frac{\partial v}{\partial y}=0\ \Rightarrow\ v\equiv0 $$

- *Why we can do this:* Continuity with ∂u/∂x = 0, and v = 0 at the impermeable plate (the D02 move).
- *In words:* the flow is parallel to the plate.

---

**Step 2 of 6 — Drop the advective terms.**

$$ u\frac{\partial u}{\partial x}+v\frac{\partial u}{\partial y}=0 $$

- *Why we can do this:* Both factors vanish; the equation becomes linear although u is not small.
- *In words:* no advection.

---

**Step 3 of 6 — Write the two momentum equations.**

$$ \rho\frac{\partial u}{\partial t}=-\frac{\partial p}{\partial x}+\mu\frac{\partial^2u} {\partial y^2},\ \ 0=-\frac{\partial p}{\partial y} $$

- *Why we can do this:* The x- and y-components of (8.1), $\frac{D\mathbf u}{Dt}=-\frac1\rho\nabla p+\nu\nabla^2\mathbf u$ times ρ with steps 1–2.
- *In words:* unsteady friction balance.

---

**Step 4 of 6 — Read the y-equation.**

$$ p=p(x,t) $$

- *Why we can do this:* ∂p/∂y = 0: the x-gradient of pressure is the same at every height.
- *In words:* what holds far away holds at the plate.

---

**Step 5 of 6 — Evaluate far from the plate.**

$$ \frac{\partial p}{\partial x}=0 $$

- *Why we can do this:* At any finite t there is a height where the fluid is still at rest, so ∂u/∂t = ∂²u/∂y² = 0 there, forcing ∂p/∂x = 0 — and by step 4 everywhere.
- *In words:* no pressure gradient drives this flow.

---

**Step 6 of 6 — Divide by ρ.**

$$ \frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.20)} $$

- *Why we can do this:* Dividing by the constant ρ turns μ into the kinematic viscosity ν = μ/ρ, the diffusivity of C01.
- *In words:* (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$: velocity diffuses like heat.

---

**Result**

$$ \begin{array}{l}\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.20)} \\ u(y,t=0)=0\ \text{(8.21)} \\ u(y=0,t)=\begin{cases}0 & t<0\\ U & t\ge0\end{cases}\ \text{(8.22)} \\ u(y\to\infty,t)=0\ \text{(8.23)}\end{array} $$

*In words:* the plate's motion spreads upward by pure diffusion.

**What it means.** The problem is linear and has no length scale — the door to the similarity solution (D17).

**Check it.** Units: m/s² both sides ✓; the same equation as Ch. 1's Couette start-up (R14) without the upper wall.

> ⚠️ **Common confusion (traps in `D16`):** ∂p/∂x = 0 because a still region exists far away at every finite t; linearity comes from u∂u/∂x = 0, not from small u.

📝 **Note.** **N40 [B], N41 [B], N42 [B] · Conditions.** $u(y,t=0)=0$ *(8.21)*, the plate condition (8.22) below, and $u(y\to\infty,t)=0$ *(8.23)*. The problem is well posed: (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$
has one time derivative and two space derivatives, so it needs one condition in t and two in y (the order of a PDE in each
variable counts the conditions it needs in that variable).

$$ u(y=0,t)=\begin{cases}0 & t<0\\ U & t\ge0\end{cases} \qquad \text{(8.22)} $$

📝 **Note.** **N43 [B], N44 [B] · Dimensional analysis, then linearity.** **N43** u depends on U, y, t and ν; with two dimensions (L, T) there are three groups, (8.24). **N44** Linearity removes the
second group, (8.25) — one variable instead of two.

$$ u/U=f\big(y/\sqrt{\nu t},\ y/Ut\big)\ \ \text{(8.24)},\qquad u/U=F\big(y/\sqrt{\nu t}\big)\equiv F(\eta) \qquad \text{(8.25)} $$

#### 🧮 Derivation — From dimensional analysis to one variable: $u/U=f(y/\sqrt{\nu t},y/Ut)$ → $u/U=F(y/\sqrt{\nu t})$ `D17` (Eq. 8.25: $u/U=F\big(y/\sqrt{\nu t}\big)\equiv F(\eta)$)

**What we want to show.** Show that the velocity can depend on y and t only through the single combination y/√(νt).

**Assumptions.** Linear equation and conditions (step 5); no other parameters (no imposed length or time).

**The plan.**

1. Count variables and dimensions.
2. Form the groups.
3. Use linearity to drop one.

**Tools we use** (each explained before this point): Buckingham Π (Ch. 1 §1.11, `ch01.pi_groups`) · linearity (gloss).

**We start from**

$$ u=u(U,y,t,\nu)\ \text{solving}\ \frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.20) with (8.21)–(8.23)} $$

*In words:* the velocity can depend only on the plate speed, the height, the time and the viscosity

---

**Step 1 of 8 — List the variables.**

$$ u,\ U,\ y,\ t,\ \nu $$

- *Why we can do this:* The answer can depend only on what appears in (8.20)–(8.23), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2},\ u(y,t=0)=0,\ u(y=0,t)=\begin{cases}0 & t<0\\ U & t\ge0\end{cases},\ u(y\to\infty,t)=0$: the plate speed, height, time and viscosity.
- *In words:* five quantities.

---

**Step 2 of 8 — Count the dimensions.**

$$ [u]=[U]=\mathrm{L/T},\ [y]=\mathrm L,\ [t]=\mathrm T,\ [\nu]=\mathrm{L^2/T} $$

- *Why we can do this:* Only length and time appear, so there are two base dimensions.
- *In words:* two dimensions.

---

**Step 3 of 8 — Apply the Π theorem.**

$$ 5-2=3\ \text{groups}:\ \ \frac uU,\ \ \frac{y}{\sqrt{\nu t}},\ \ \frac{y}{Ut} $$

- *Why we can do this:* Buckingham: variables minus independent dimensions; each group is checked dimensionless.
- *In words:* three dimensionless groups.

---

**Step 4 of 8 — Write the functional relation.**

$$ u/U=f(y/\sqrt{\nu t},\ y/Ut)\ \text{(8.24)} $$

- *Why we can do this:* One group as a function of the other two.
- *In words:* (8.24), $u/U=f\big(y/\sqrt{\nu t},\ y/Ut\big)$.

---

**Step 5 of 8 — Use linearity.**

$$ u\ \text{solves with }U\ \Rightarrow\ \lambda u\ \text{solves with }\lambda U $$

- *Why we can do this:* (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$ and its conditions are linear; only the wall value contains U, and it scales with it.
- *In words:* doubling U doubles u.

---

**Step 6 of 8 — Conclude u/U cannot depend on U.**

$$ \frac{\partial}{\partial U}\Big(\frac uU\Big)=0 $$

- *Why we can do this:* By step 5, u/U is the same for every U; but y/Ut contains U, so f cannot depend on its second argument.
- *In words:* the second group must go.

---

**Step 7 of 8 — Drop the second group.**

$$ u/U=F(y/\sqrt{\nu t})\equiv F(\eta)\ \text{(8.25)} $$

- *Why we can do this:* What remains is a function of the first group only.
- *In words:* (8.25), $u/U=F\big(y/\sqrt{\nu t}\big)\equiv F(\eta)$: one similarity variable.

---

**Step 8 of 8 — Name the similarity variable.**

$$ \eta=\frac{y}{\sqrt{\nu t}} $$

- *Why we can do this:* At fixed t, η ∝ y, so F reads as a profile; another choice (νt/y²) would only change F, not u. The book's figures use η/2.
- *In words:* two variables became one.

---

**Result**

$$ u/U=F\big(y/\sqrt{\nu t}\big)\equiv F(\eta)\ \text{(8.25)} $$

*In words:* the PDE in y and t becomes an ODE in η.

**What it means.** No imposed scale exists, so y can only be measured in units of √(νt).

**Check it.** η dimensionless: m/√((m²/s)s) ✓. Numbers: y = 1 cm, t = 100 s, ν = 10⁻⁶ → η = 1.

> ⚠️ **Common confusion (traps in `D17`):** Five variables and two dimensions give three groups; it is linearity, not dimensional analysis, that removes y/Ut.

In [ ]:
groups = ch01.pi_groups({"u": "m/s", "U": "m/s", "y": "m", "t": "s", "nu": "m^2/s"})   # Ch. 1's Π-group finder
for g_ in groups:                                        # each group as exponents of the variables
    print(" · ".join(f"{k}^{v}" for k, v in g_.items() if v != 0))

**What does the code above do?**

`pi_groups` returns three independent groups, as D17 step 3 counts: $u/U$, $tU/y$ and $\nu/(Uy)$. They are equivalent to the
book's choice: $tU/y=(y/Ut)^{-1}$, and $\frac{\nu}{Uy}\cdot\frac{Ut}{y}=\frac{\nu t}{y^2}=\big(\frac{y}{\sqrt{\nu t}}\big)^{-2}$ —
any set of three independent combinations is as good as another; linearity then removes the one that contains U.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **chain rule** — if u depends on t only through η(y, t), then ∂u/∂t = (du/dη)(∂η/∂t) (Ch. 1 P49).
- **exponent rules** — $\eta=y\nu^{-1/2}t^{-1/2}$, so $\partial\eta/\partial t=-\tfrac12y\nu^{-1/2}t^{-3/2}=-\eta/(2t)$ (Ch. 1 P43).

📝 **Note.** **N45 [B], N46 [B], N47 [B], N48 [B] · Into the equation.** The chain rule gives $\frac{\partial u}{\partial t}=-\frac{U\eta}{2t}\frac{dF}{d\eta}$ and $\frac{\partial^2u}{\partial y^2}=
\frac{U}{\nu t}\frac{d^2F}{d\eta^2}$; they turn (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$ into the ODE (8.26) with $F(\eta=0)=1$ *(8.27)* and
$F(\eta\to\infty)=0$ *(8.28)*: the initial condition (8.21), $u(y,t=0)=0$ and the far condition (8.23), $u(y\to\infty,t)=0$ both become η → ∞, so three
conditions collapse into two — the test that the similarity form is right.

$$ -\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big) \qquad \text{(8.26)} $$

#### 🧮 Derivation — The similarity ODE $-\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\big(\frac{dF}{d\eta}\big)$ with $F(0)=1$, $F(\infty)=0$ `D18` (Eq. 8.26: $-\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big)$)

**What we want to show.** Turn (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$ into an ODE for F(η) and its conditions.

**Assumptions.** t > 0.

**The plan.**

1. Differentiate η.
2. Chain rule for ∂u/∂t and ∂²u/∂y².
3. Substitute, divide, check that t drops out.
4. Translate the conditions.

**Tools we use** (each explained before this point): chain rule (P49/P91) · exponent rules (P43).

**We start from**

$$ \text{u = U F(η), η = y/√(νt), in}\ \frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.20).} $$

---

**Step 1 of 9 — Differentiate η in time.**

$$ \frac{\partial\eta}{\partial t}=-\frac12\frac{y}{\sqrt\nu}t^{-3/2}=-\frac{\eta}{2t} $$

- *Why we can do this:* η = yν^(−1/2)t^(−1/2); power rule at fixed y.
- *In words:* η shrinks in time at a fixed height.

---

**Step 2 of 9 — Chain rule for ∂u/∂t.**

$$ \frac{\partial u}{\partial t}=U\frac{dF}{d\eta}\frac{\partial\eta}{\partial t}=-\frac{U\eta}{2t} \frac{dF}{d\eta} $$

- *Why we can do this:* u depends on t only through η.
- *In words:* the local rate of change.

---

**Step 3 of 9 — Differentiate η in y.**

$$ \frac{\partial\eta}{\partial y}=\frac{1}{\sqrt{\nu t}} $$

- *Why we can do this:* η is linear in y with the slope 1/√(νt), which does not depend on y.
- *In words:* one unit of η is √(νt) of height.

---

**Step 4 of 9 — First y-derivative.**

$$ \frac{\partial u}{\partial y}=\frac{U}{\sqrt{\nu t}}\frac{dF}{d\eta} $$

- *Why we can do this:* Chain rule: u depends on y only through η, so ∂u/∂y = U F′ ∂η/∂y.
- *In words:* the velocity slope.

---

**Step 5 of 9 — Second y-derivative.**

$$ \frac{\partial^2u}{\partial y^2}=\frac{U}{\nu t}\frac{d^2F}{d\eta^2} $$

- *Why we can do this:* 1/√(νt) does not depend on y; differentiate F′ again with the chain rule.
- *In words:* the curvature.

---

**Step 6 of 9 — Substitute into (8.20).**

$$ -\frac{U\eta}{2t}F'=\nu\frac{U}{\nu t}F''=\frac Ut F'' $$

- *Why we can do this:* Insert steps 2 and 5 into (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$ to see whether the time dependence can cancel.
- *In words:* every term carries U/t.

---

**Step 7 of 9 — Divide by U/t.**

$$ -\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big)\ \text{(8.26)} $$

- *Why we can do this:* U/t ≠ 0; t disappears — the test that the similarity form is right.
- *In words:* (8.26), $-\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big)$, an ODE.

---

**Step 8 of 9 — Translate the wall condition.**

$$ F(\eta=0)=1\ \text{(8.27)} $$

- *Why we can do this:* y = 0 gives η = 0 for every t > 0, and u = U there (8.22), $u(y=0,t)=\begin{cases}0 & t<0\\ U & t\ge0\end{cases}$.
- *In words:* fluid at the plate moves with it.

---

**Step 9 of 9 — Merge the other two conditions.**

$$ F(\eta\to\infty)=0\ \text{(8.28)} $$

- *Why we can do this:* y → ∞ at fixed t (8.23), $u(y\to\infty,t)=0$ and t → 0⁺ at fixed y > 0 (8.21), $u(y,t=0)=0$ both send η → ∞; three conditions become two, as a second-order ODE needs.
- *In words:* far away and initially, the fluid is at rest.

---

**Result**

$$ \begin{array}{l}-\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big)\ \text{(8.26)} \\ F(\eta=0)=1\ \text{(8.27)} \\ F(\eta\to\infty)=0\ \text{(8.28)}\end{array} $$

*In words:* a two-point boundary-value problem in η alone.

**What it means.** Had t not cancelled at step 7, the guess u = UF(y/√(νt)) would have been wrong.

**Check it.** `ch08.similarity_ode_solve("stokes1")` (solve_bvp) finds F with F(0) = 1, F(12) = 0 matching erfc(η/2) to 1e-7.

> ⚠️ **Common confusion (traps in `D18`):** ∂η/∂t = −η/(2t), not −η/t; t → 0 and y → ∞ both give η → ∞.

> 📎 **Primer — Gaussian integral.** `P194` · The bell curve $e^{-\zeta^2}$ has total area √π: $\int_{-\infty}^{\infty}e^{-\zeta^2}d\zeta=\sqrt\pi$, and by symmetry half of it,
√π/2, lies on each side. No elementary antiderivative exists, which is why its running integral gets a name (next primer).

In [ ]:
import numpy as np
from scipy.integrate import quad
print(quad(lambda z: np.exp(-z**2), 0, np.inf)[0], np.sqrt(np.pi)/2)   # 0.886227 0.886227

> 📎 **Primer — error function erf and its inverses.** `P195` · $\mathrm{erf}(\zeta)=\frac2{\sqrt\pi}\int_0^\zeta e^{-\xi^2}d\xi$ rises from 0 to 1; erfc = 1 − erf falls from 1 to 0 and is
computed directly without cancellation. Their inverses answer "where does the profile reach this level?":
`scipy.special.erfinv`, `erfcinv`.

In [ ]:
from scipy import special
print(special.erf(1.0), special.erfc(1.0))               # 0.8427 0.1573
print(2*special.erfcinv(0.01))                           # 3.643: where erfc(η/2) = 1 %

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **natural logarithm, exponentiating** — ∫dG/G = ln G, and $e^{\ln G}=G$: exponentiating turns a constant of integration into a factor (Ch. 1 P36).
- **substitution ξ = 2ζ in an integral** — with ξ = 2ζ, dξ = 2dζ and the limits change with it (Ch. 3 P106).
- **complementary error function erfc** — erfc = 1 − erf, computed without cancellation for large arguments (Ch. 4 P123).

📝 **Note.** **N49 [B], N50 [B], N51 [B] · Solving the ODE.** Separating gives $\frac{dF}{d\eta}=A\exp(-\eta^2/4)$, a second integration gives (8.29), and the conditions give B = 1 and, with
ξ = 2ζ and the Gaussian integral, $A=-1/\sqrt\pi$.

$$ F(\eta)=A\int_0^\eta\exp(-\xi^2/4)\,d\xi+B \qquad \text{(8.29)} $$

#### 🧮 Derivation — Solving the ODE: and $\frac uU=1-\mathrm{erf}\big(\frac{y}{2\sqrt{\nu t}}\big)$ `D19` (Eq. 8.30: $\frac{u}{U}=1-\mathrm{erf}\Big(\frac{y}{2\sqrt{\nu t}}\Big)$)

**What we want to show.** Solve (8.26), $-\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big)$ with its conditions and express the answer through the error function.

**Assumptions.** None beyond (8.26)–(8.28).

**The plan.**

1. Solve for F′ by separation.
2. Integrate again.
3. Fix the constants with the Gaussian integral.
4. Recognise erf.

**Tools we use** (each explained before this point): separation of variables (P42) · natural log and exponential (P36) · Gaussian integral (primer P194) · substitution (P106) · erf (primer P195).

**We start from**

$$ \begin{array}{l}-\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big)\ \text{(8.26)} \\ F(\eta=0)=1\ \text{(8.27)} \\ F(\eta\to\infty)=0\ \text{(8.28)}\end{array} $$

*In words:* the similarity ODE and its two conditions from D18

---

**Step 1 of 12 — Call the slope G.**

$$ G\equiv F'\ \Rightarrow\ -\frac\eta2G=\frac{dG}{d\eta} $$

- *Why we can do this:* F itself does not appear in (8.26), $-\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big)$, so it is a first-order ODE for F′.
- *In words:* first solve for the slope.

---

**Step 2 of 12 — Separate the variables.**

$$ -\frac\eta2d\eta=\frac{dG}{G} $$

- *Why we can do this:* Divide by G and multiply by dη (P42).
- *In words:* η on one side, G on the other.

---

**Step 3 of 12 — Integrate both sides.**

$$ -\frac{\eta^2}{4}=\ln G+\text{const} $$

- *Why we can do this:* ∫η dη = η²/2; ∫dG/G = ln G.
- *In words:* a logarithm of the slope.

---

**Step 4 of 12 — Exponentiate.**

$$ \frac{dF}{d\eta}=A\exp(-\eta^2/4) $$

- *Why we can do this:* $e^(ln G)$ = G; the constant becomes a factor A.
- *In words:* the slope is a Gaussian.

---

**Step 5 of 12 — Integrate from 0 to η.**

$$ F(\eta)=A\int_0^\eta\exp(-\xi^2/4)\,d\xi+B\ \text{(8.29)} $$

- *Why we can do this:* Fundamental theorem of calculus; ξ is a dummy variable, B = F(0).
- *In words:* (8.29), $F(\eta)=A\int_0^\eta\exp(-\xi^2/4)\,d\xi+B$.

---

**Step 6 of 12 — Apply F(0) = 1.**

$$ B=1 $$

- *Why we can do this:* The integral from 0 to 0 vanishes.
- *In words:* the plate condition.

---

**Step 7 of 12 — Apply F(∞) = 0.**

$$ 0=A\int_0^\infty e^{-\xi^2/4}d\xi+1 $$

- *Why we can do this:* Apply (8.28), $F(\eta\to\infty)=0$, F → 0 as η → ∞, with B = 1 already fixed.
- *In words:* one equation for A.

---

**Step 8 of 12 — Substitute ξ = 2ζ.**

$$ 0=2A\int_0^\infty e^{-\zeta^2}d\zeta+1 $$

- *Why we can do this:* dξ = 2dζ and ξ²/4 = ζ²; this turns the integral into the standard Gaussian.
- *In words:* the factor 2 that ends up in y/(2√(νt)).

---

**Step 9 of 12 — Use the Gaussian integral.**

$$ 0=2A\frac{\sqrt\pi}{2}+1\ \Rightarrow\ A=-\frac{1}{\sqrt\pi} $$

- *Why we can do this:* ∫₀^∞$e^(-ζ^2)$dζ = √π/2, half of the full √π by symmetry (P194).
- *In words:* A is negative: F falls.

---

**Step 10 of 12 — Insert A and B into (8.29).**

$$ F(\eta)=1-\frac{1}{\sqrt\pi}\int_0^\eta e^{-\xi^2/4}\,d\xi $$

- *Why we can do this:* Put B = 1 (step 6) and A = −1/√π (step 9) back into (8.29), $F(\eta)=A\int_0^\eta\exp(-\xi^2/4)\,d\xi+B$, so that the solution is written with both constants known. (Part F went straight to the substituted integral; this step is added.)
- *In words:* the solution, still in the variable ξ.

---

**Step 11 of 12 — Substitute again inside F.**

$$ F(\eta)=1-\frac{2}{\sqrt\pi}\int_0^{\eta/2}e^{-\zeta^2}d\zeta $$

- *Why we can do this:* Substitute ξ = 2ζ again (dξ = 2dζ, ξ²/4 = ζ²): the factor 2 turns 1/√π into 2/√π, and the upper limit ξ = η becomes ζ = η/2.
- *In words:* ready to name.

---

**Step 12 of 12 — Recognise the error function.**

$$ \frac{u}{U}=1-\mathrm{erf}\Big(\frac{y}{2\sqrt{\nu t}}\Big),\ \ \mathrm{erf}(\zeta)= \frac{2}{\sqrt\pi}\int_0^\zeta e^{-\xi^2}d\xi\ \text{(8.30)} $$

- *Why we can do this:* The definition of erf (P195); η/2 = y/(2√(νt)).
- *In words:* (8.30), $\frac{u}{U}=1-\mathrm{erf}\Big(\frac{y}{2\sqrt{\nu t}}\Big)$ = erfc.

---

**Result**

$$ \frac{u}{U}=1-\mathrm{erf}\Big(\frac{y}{2\sqrt{\nu t}}\Big),\qquad \mathrm{erf}(\zeta)=\frac2{\sqrt\pi}\int_0^\zeta e^{-\xi^2}d\xi\ \ \text{(8.30)} $$

*In words:* the velocity falls from U at the plate as the complementary error function of y/(2√(νt)).

**What it means.** One curve for all t, U, ν — the collapse of Fig. 8.13; the plate's influence reaches ~ 3.6√(νt) (D20).

**Check it.** F(0) = 1 − 0 = 1 ✓, F(∞) = 1 − 1 = 0 ✓; residual of (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$ zero (sympy); y = 1 cm, t = 100 s: erfc(0.5) = 0.4795; the CN solution converges to it at order 2.

> ⚠️ **Common confusion (traps in `D19`):** The substitution carries dξ = 2dζ; ∫₀^∞ is √π/2 (half); A < 0; compute with erfc, not 1 − erf, for large η.

> 📎 **Primer — scipy.integrate.solve_bvp.** `P196` · `solve_bvp` solves an ODE with conditions at two ends (here F(0) = 1 and F(η_max) = 0 on a truncated domain) by collocation on
a mesh; we give it the system as first-order equations (F′ = G, G′ = −ηG/2) and a guess.

In [ ]:
import numpy as np
from scipy.integrate import solve_bvp
eta = np.linspace(0, 12, 50)                             # the truncated η-domain and initial mesh
sol = solve_bvp(lambda e, Y: np.vstack([Y[1], -e/2*Y[1]]),   # Y = (F, F′): F′ = G, G′ = −(η/2)G  (8.26)
                lambda a, b: np.array([a[0] - 1, b[0]]),     # F(0) = 1, F(12) = 0
                eta, np.vstack([np.exp(-eta), -np.exp(-eta)]))   # a rough guess
print(sol.sol(2.0)[0])                                   # 0.1573 = erfc(1)

📝 **Note.** **N52 [B], N55 [B] · One curve (Figs. 8.12–8.13, remade below).** **N52** Profiles at several times (raw) and the same against the similarity variable. ⚠️ The book's figure axis is
$y/(2\sqrt{\nu t})=\eta/2$, while (8.25), $u/U=F\big(y/\sqrt{\nu t}\big)\equiv F(\eta)$ defines $\eta=y/\sqrt{\nu t}$; our axes say which. **N55** *Self-similarity:*
profiles at any t, for any U and ν, fall on one curve when u/U is plotted against η — we measure the spread below
(< 10⁻¹²).

#### 🧮 Derivation — The 99 % thickness $\delta_{99}\sim3.64\sqrt{\nu t}$ `D20` (Eq. 8.31: $\delta_{99}\sim3.64\sqrt{\nu t}$)

**What we want to show.** Compute (instead of reading off a figure) the height where u falls to 1 % of U.

**Assumptions.** The 1 % level is a convention.

**The plan.**

1. Set the level.
2. Invert erfc.
3. Convert to height.

**Tools we use** (each explained before this point): erf and its inverses (primer P195) · brentq (P108) as a cross-check.

**We start from**

$$ \frac uU=1-\mathrm{erf}\big(\frac{y}{2\sqrt{\nu t}}\big)=\mathrm{erfc}\big(\frac{\eta}{2}\big)\ \text{(8.30).} $$

---

**Step 1 of 5 — Choose the edge level.**

$$ \frac uU=0.01 $$

- *Why we can do this:* The profile never reaches zero; the book (and Ch. 9) define the edge at 99 % of the change.
- *In words:* the 1 % point.

---

**Step 2 of 5 — Write it with (8.30).**

$$ \mathrm{erfc}\Big(\frac{\eta_{99}}{2}\Big)=0.01 $$

- *Why we can do this:* Substitute the 1 % level into (8.30), $\frac{u}{U}=1-\mathrm{erf}\Big(\frac{y}{2\sqrt{\nu t}}\Big)$, written with η/2 as its argument.
- *In words:* one equation for η₉₉.

---

**Step 3 of 5 — Invert erfc.**

$$ \frac{\eta_{99}}{2}=\mathrm{erfc}^{-1}(0.01)=1.8214 $$

- *Why we can do this:* erfc is monotonic, so it has a unique inverse (`special.erfcinv`; brentq agrees).
- *In words:* on the book's axis η/2 = 1.82.

---

**Step 4 of 5 — Double it.**

$$ \eta_{99}=3.643\ \Rightarrow\ \delta_{99}=3.643\sqrt{\nu t}\ \text{(8.31)} $$

- *Why we can do this:* η = y/√(νt), so the height is η₉₉√(νt).
- *In words:* (8.31), $\delta_{99}\sim3.64\sqrt{\nu t}$.

---

**Step 5 of 5 — Put in numbers.**

$$ \nu=10^{-6},\ t=100\ \mathrm s:\ \ \delta_{99}=3.64\ \mathrm{cm};\ \ t=3600\ \mathrm s:\ 21.9\ \mathrm{cm} $$

- *Why we can do this:* √(νt) = 1 cm and 6 cm; the thickness grows as t^(1/2).
- *In words:* an hour of diffusion reaches only a hand's width in water.

---

**Result**

$$ \delta_{99}=2\,\mathrm{erfc}^{-1}(0.01)\sqrt{\nu t}=3.643\sqrt{\nu t}\ \ \text{(8.31)} $$

*In words:* the moving layer thickens like √(νt).

**What it means.** The √(νt) law is the thickness of every laminar boundary layer (Ch. 9, with t = x/U) and of spin-up layers in the ocean (Ch. 13).

**Check it.** Units m ✓; a 95 % level gives 2 erfc⁻¹(0.05) = 2.772 — the factor is a convention.

> ⚠️ **Common confusion (traps in `D20`):** The book's figure axis is η/2 = 1.82; (8.31), $\delta_{99}\sim3.64\sqrt{\nu t}$ quotes η = 3.64.

📝 **Note.** **N54 [B] · The 99 % thickness** grows like $t^{1/2}$: in water 3.64 cm after 100 s, 21.9 cm after an hour — diffusion is slow over large distances (the code
below prints $2\,\mathrm{erfc}^{-1}(0.01)=3.643$).

$$ \delta_{99}\sim3.64\sqrt{\nu t} \qquad \text{(8.31)} $$

#### ✏️ Tiny example: a plate in water after 100 s

ν = 10⁻⁶ m²/s, t = 100 s, U = 0.1 m/s.
1. $\sqrt{\nu t}=\sqrt{10^{-4}}=0.01$ m.
2. At y = 1 cm: η = 1, η/2 = 0.5, u/U = erfc(0.5) = 0.4795 — half the plate speed.
3. $\delta_{99}=3.643\times0.01=3.64$ cm.
4. After 400 s: $\sqrt{\nu t}$ doubles to 2 cm, $\delta_{99}=7.29$ cm, and u(2 cm) = 0.4795U again (same η).
5. Wall stress $\tau_w=\mu U/\sqrt{\pi\nu t}=10^{-3}\times0.1/\sqrt{\pi\times10^{-4}}=5.64\times10^{-3}$ Pa, falling like
   $t^{-1/2}$.

In [ ]:
t = 100.0; y = np.array([0.0, 0.005, 0.01, 0.02, 0.0364])   # a time [s] and five heights [m]
print("u =", ch08.stokes_first_problem(y, t, U=0.1, nu=NU_W), "m/s")   # (8.30) via erfc
print("η =", ch08.similarity_variable(y, t, NU_W), " η/2 =", ch08.similarity_variable(y, t, NU_W, half=True))
print(f"δ99 = {ch08.diffusion_thickness(t, NU_W):.5f} m at 100 s, {ch08.diffusion_thickness(3600.0, NU_W):.4f} m after an hour")
print("2 erfcinv(0.01) =", round(2*special.erfcinv(0.01), 4))   # the 3.643 of (8.31)
st = ch08.stokes_first_state(t, U=0.1, nu=NU_W)          # the numbers the explainer shows
print({k: round(float(st[k]), 6) for k in ("sqrt_nut", "eta_edge", "delta", "tau_w", "vorticity_content")})
print("∫ω dy =", ch08.vorticity_content(t, U=0.1, nu=NU_W), "m/s (= U)")   # N53 below, by quad
print("Ch. 4:", ch04.stokes_first_problem(y, t, 0.1, NU_W))   # the same u from Ch. 4's earlier function
eta, F = ch08.similarity_ode_solve("stokes1")             # solve_bvp on (8.26)–(8.28), knowing nothing about erf
print(f"max |F_bvp − erfc(η/2)| = {np.max(np.abs(F - special.erfc(eta/2))):.1e}")

**What does the code above do?**

1. `stokes_first_problem` is (8.30), $\frac{u}{U}=1-\mathrm{erf}\Big(\frac{y}{2\sqrt{\nu t}}\Big)$ computed with erfc: u = [0.1, 0.0724, 0.0480, 0.0157, 0.0010] m/s.
2. `similarity_variable` returns η = y/√(νt), and with `half=True` the book's figure axis η/2.
3. `diffusion_thickness` is (8.31), $\delta_{99}\sim3.64\sqrt{\nu t}$: 3.64 cm after 100 s, 21.9 cm after an hour.
4. `stokes_first_state` bundles √(νt) = 1 cm, the edge η₉₉ = 3.643, δ₉₉, the wall stress 5.64 × 10⁻³ Pa and the vorticity
   content; `vorticity_content` integrates ω over the whole layer and gets U (N53).
5. Ch. 4's earlier function gives the same numbers; the boundary-value solver, knowing nothing about erf, lands on the same
   curve to about 10⁻¹³.

> 📎 **Primer — Crank–Nicolson with scipy.linalg.solve_banded.** `P193` · Crank–Nicolson averages the diffusion term between the old and the new time level: second-order accurate in time and stable
for any step. Each step solves a tridiagonal system, which `scipy.linalg.solve_banded` does in O(N) from the three diagonals
stored as rows. An impulsive start (the plate jumps to U) makes pure Crank–Nicolson ring, so we take the first two steps
with backward Euler (fully implicit), which damps the ringing.

In [ ]:
import numpy as np
from scipy.linalg import solve_banded
ab = np.array([[0, -1, -1], [4, 4, 4], [-1, -1, 0]], float)   # upper, main, lower diagonals (as rows)
print(solve_banded((1, 1), ab, np.array([3.0, 2.0, 3.0])))    # [1. 1. 1.]

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **observed order of convergence** — if the error behaves like C·Δyᵖ, the slope of log(error) against log(Δy) is p; `observed_order` fits it (Ch. 1 P13, log–log slopes).

In [ ]:
U0, t_end = 0.1, 1000.0                                  # plate speed [m/s] and final time [s]


def cn_by_hand(N, nt):                                   # Crank–Nicolson march of (8.20) on N points, nt steps
    yy = np.linspace(0, 8*np.sqrt(NU_W*t_end), N)        # up to 8√(νt_end), where u ≈ 0 [m]
    dy, dt = yy[1] - yy[0], t_end/nt                     # grid step [m], time step [s]
    r = NU_W*dt/dy**2                                    # the diffusion number ν Δt/Δy²
    u = np.zeros(N)                                      # fluid at rest (8.21)
    for k in range(nt):
        th = 1.0 if k < 2 else 0.5                       # two backward-Euler steps, then Crank–Nicolson (θ = ½)
        ab = np.zeros((3, N - 2))                        # the three diagonals of the implicit part
        ab[0, 1:], ab[1, :], ab[2, :-1] = -th*r, 1 + 2*th*r, -th*r
        rhs = u[1:-1] + (1 - th)*r*(u[2:] - 2*u[1:-1] + u[:-2])   # the explicit part of the average
        rhs[0] += th*r*U0                                # the wall value u(0) = U (8.22) moves to the right side
        u = np.r_[U0, linalg.solve_banded((1, 1), ab, rhs), 0.0]   # new profile with the two boundary values
    return yy, u


N_ = 400 if not FAST else 200                            # grid points
yy, u_cn = cn_by_hand(N_, 200 if not FAST else 100)       # our march
exact = ch08.stokes_first_problem(yy, t_end, U0, NU_W)   # (8.30)
print(f"hand CN vs (8.30): max error {np.max(np.abs(u_cn - exact)):.1e} m/s")
assert np.allclose(u_cn, exact, rtol=1e-3, atol=2e-4)    # the march lands on the erfc curve
u_lib = ch08.crank_nicolson_1d(np.zeros(N_), yy, t_end/(200 if not FAST else 100), 200 if not FAST else 100, NU_W, U0, 0.0)
assert np.allclose(u_lib, u_cn, rtol=0, atol=1e-12)      # the library solver does exactly this
dyf = yy[1] - yy[0]; dtf = diffusion.stable_time_step(NU_W, dyf); nf = int(np.ceil(t_end/dtf))   # Ch. 1's explicit FTCS
u_ftcs = diffusion.ftcs_diffusion_1d(np.r_[U0, np.zeros(N_ - 1)], NU_W, dyf, t_end/nf, nf, values=(U0, 0.0), save_every=nf)[-1]
print(f"FTCS ({nf} small steps) vs (8.30): max error {np.max(np.abs(u_ftcs - exact)):.1e} m/s")
assert np.allclose(u_ftcs, exact, atol=1e-3*U0)          # the explicit scheme agrees too, with ~28× more steps
errs, steps = [], []                                     # a convergence table: refine space and time together
for N2 in (100, 200, 400):
    y2, u2 = cn_by_hand(N2, N2)
    errs.append(np.max(np.abs(u2 - ch08.stokes_first_problem(y2, t_end, U0, NU_W)))); steps.append(y2[1] - y2[0])
    print(f"N = {N2}: max error {errs[-1]:.2e} m/s")
print(f"observed order ≈ {observed_order(steps, errs):.2f}")   # ≈ 2: second order in space and time

In [ ]:
U0 = 0.1                                                  # plate speed [m/s]
ts = [10, 30, 100, 300, 1000]                             # five times [s]
shades = plt.cm.Blues(np.linspace(0.4, 1.0, len(ts)))     # darker blue for later times
y = np.linspace(0, 0.14, 400)                             # heights [m]
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(12, 3.8))
for t_, c in zip(ts, shades):
    u = ch08.stokes_first_problem(y, t_, U0, NU_W)        # (8.30)
    a1.plot(u/U0, 100*y, color=c, lw=2, label=f"t = {t_} s")
    d99 = ch08.diffusion_thickness(t_, NU_W)              # (8.31)
    a1.plot([0, 0.08], [100*d99]*2, color=COLORS["amber"], lw=2)   # δ99 tick
    a2.plot(u/U0, ch08.similarity_variable(y, t_, NU_W), color=c, lw=2)   # the same profile against η
a2.plot(u_cn[::8]/U0, ch08.similarity_variable(yy[::8], 1000.0, NU_W), "o", color=COLORS["teal"], ms=3, label="Crank–Nicolson, t = 1000 s")
a1.set_xlabel("u/U [–]"); a1.set_ylabel("y [cm]"); a1.legend(fontsize=7); a1.set_title("Raw profiles (amber: δ99)", fontsize=10)
a2.set_xlabel("u/U [–]"); a2.set_ylabel("η = y/√(νt) [–]"); a2.set_ylim(0, 6); a2.legend(fontsize=7)
a2.axhline(3.643, color=COLORS["amber"], lw=0.8); a2.text(0.5, 3.75, "η99 = 3.643", color=COLORS["amber"], fontsize=8)
sec = a2.secondary_yaxis("right", functions=(lambda e: e/2, lambda e: 2*e)); sec.set_ylabel("η/2 = y/(2√(νt)) (book's axis)")
a2.set_title("Against η: one curve", fontsize=10)
tt = np.geomspace(1, 1e4, 50)                             # times [s]
a3.loglog(tt, 100*ch08.diffusion_thickness(tt, NU_W), color=COLORS["blue"], lw=2, label="water")
a3.loglog(tt, 100*ch08.diffusion_thickness(tt, nu_air), color=COLORS["teal"], lw=2, label="air")
a3.set_xlabel("t [s]"); a3.set_ylabel("δ99 [cm]"); a3.legend(fontsize=8); a3.set_title("δ99 ∝ t^½", fontsize=10)
fig.suptitle("One curve for all times: the profile only stretches like √(νt)", fontsize=11)
savefig(fig, "ch08", "c09_stokes_first"); plt.show()

**What you see.** (a) five profiles of growing thickness with their 99 % heights (amber ticks); (b) the same five profiles against
η — they coincide exactly, and the Crank–Nicolson dots sit on them; (c) two straight lines of slope ½ — our remake of
Figs. 8.12–8.13 `N52`.

**How to read it.** (b): the 1 % point sits at η = 3.64 (η/2 = 1.82 on the book's axis, the right-hand scale); (c): 100 times longer gives
10 times thicker.

**What would change if…** …the fluid were air: every thickness grows √15 ≈ 3.9 times faster, but (b) is unchanged.

In [ ]:
eta_grid = np.linspace(0, 8, 161)                        # a common η grid
spread = 0.0                                             # the largest difference from erfc(η/2)
for U_, nu_, t_ in ((0.1, 1e-6, 10.0), (2.0, 1.5e-5, 3.0), (0.01, 1e-3, 500.0)):   # water, air, syrup at three times
    y_ = eta_grid*np.sqrt(nu_*t_)                        # heights with those η [m]
    spread = max(spread, np.max(np.abs(ch08.stokes_first_problem(y_, t_, U_, nu_)/U_ - special.erfc(eta_grid/2))))
print(f"collapse: max |u/U − erfc(η/2)| over three (U, ν, t) = {spread:.1e}")
assert spread < 1e-12                                    # N55: one curve for every U, ν and t

📝 **Note.** **N53 [B] · Reading it as vorticity.** At t = 0 the wall creates a vortex sheet (all the vorticity in an infinitely thin layer); afterwards it diffuses out:
$\omega=-\partial u/\partial y=\frac{U}{\sqrt{\pi\nu t}}e^{-y^2/4\nu t}>0$. The total below never changes, so no new vorticity is
made after t = 0. ⚠️ The book prints this integral as −U; since u decreases upward, ω is positive and the integral is +U
(checked above by `vorticity_content`). The heat analogue: a cold solid whose face is suddenly heated.

$$ \int_0^\infty\omega\,dy=u(0)-u(\infty)=U  $$

📝 **Note.** **N56 [B] · Similarity needs the absence of imposed scales.** Stop the plate at t = T (Exercise 8.30) and a time scale appears; by superposition (ours) the velocity is
$u=U\big[\mathrm{erfc}\frac{y}{2\sqrt{\nu t}}-\mathrm{erfc}\frac{y}{2\sqrt{\nu(t-T)}}\big]$ for t > T. Put a second plate at
y = h (R14) and a length scale appears. Either way the profiles stop collapsing — the spin-down of a stirred tank (Ch. 13) is
of this kind.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
eta_ = np.linspace(0, 6, 200)                             # η grid [–]
a1.plot(special.erfc(eta_/2), eta_, color=COLORS["accent"], ls="--", lw=1.5, label="erfc(η/2): plate never stops")
for t_, c in ((50, COLORS["blue"]), (150, COLORS["teal"]), (400, COLORS["orange"])):   # times [s]; the plate stops at T = 100 s
    y_ = eta_*np.sqrt(NU_W*t_)                            # heights at this time [m]
    a1.plot(ch08.stokes_first_stopped(y_, t_, 100.0, U=1.0, nu=NU_W), eta_, color=c, lw=2, label=f"t = {t_} s")
a1.set_xlabel("u/U [–]"); a1.set_ylabel("η = y/√(νt) [–]"); a1.legend(fontsize=7); a1.set_title("Plate stopped at T = 100 s", fontsize=10)
h_ = 0.02                                                 # a second, fixed plate 2 cm away [m]
for t_, c in ((10, COLORS["blue"]), (100, COLORS["teal"]), (400, COLORS["orange"])):
    yb = np.linspace(0, h_, 200)                          # the gap [m]; Ch. 1's function moves the plate at y = h
    ub = diffusion.couette_startup_profile(yb, t_, 0.1, h_, NU_W)   # Fourier series of the start-up [m/s]
    a2.plot(ub/0.1, (h_ - yb)/np.sqrt(NU_W*t_), color=c, lw=2, label=f"t = {t_} s")   # distance from the moving plate, in η
a2.plot(special.erfc(eta_/2), eta_, color=COLORS["accent"], ls="--", lw=1.5, label="erfc(η/2)")
a2.set_ylim(0, 6); a2.set_xlabel("u/U [–]"); a2.set_ylabel("η from the moving plate [–]"); a2.legend(fontsize=7)
a2.set_title("A second wall 2 cm away (Couette start-up)", fontsize=10)
fig.suptitle("An imposed time or length scale breaks the collapse", fontsize=11)
savefig(fig, "ch08", "c09_no_collapse"); plt.show()

**What you see.** (a) after the plate stops (T = 100 s) the curves no longer lie on the erfc line and the wall speed drops to zero;
(b) the Couette start-up curves follow erfc while the layer is thin (10 s) and leave it once it feels the far wall (400 s).

**How to read it.** In (b) η₉₉ = 3.64 corresponds to δ₉₉ = 3.64√(νt): 1.2 cm at 10 s (well inside the 2 cm gap), 7.3 cm at 400 s (far
beyond it) — the far wall's no-slip condition bends the curve back to the straight Couette line.

**What would change if…** …T → ∞ (the plate never stops): the stopped-plate curves rejoin the erfc curve.

In [ ]:
nfr = 60 if not FAST else 30                              # frames
tf = np.geomspace(1, 1000, nfr)                           # a log clock t = 1 … 1000 s
U0 = 0.1                                                  # plate speed [m/s]
yv = np.linspace(0, 0.12, 121)                            # heights [m]
sgrid = np.geomspace(1e-4, 1000, 3000)                    # fine times for the dye displacement [s]
X = integrate.cumulative_trapezoid(ch08.stokes_first_problem(yv[None, :], sgrid[:, None], U0, NU_W), sgrid, axis=0, initial=0)
disp = np.array([X[np.searchsorted(sgrid, t_)] / (U0*t_) for t_ in tf])   # dye displacement ÷ plate travel U t [–]
tcn, Ucn = ch08.crank_nicolson_1d(np.zeros(121), yv, 1000.0/400, 400, NU_W, U0, 0.0, return_all=True)   # numerical twin
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
dye, = a1.plot(disp[0], 100*yv, color=COLORS["blue"], lw=2)   # the dyed line, bent into the time-integrated profile
a1.axhline(0, color=COLORS["ink"], lw=4)                  # the plate
a1.set_xlim(0, 1.05); a1.set_xlabel("dye displacement ÷ plate travel Ut [–]"); a1.set_ylabel("y [cm]")
prof, = a2.plot([], [], color=COLORS["blue"], lw=2, label="(8.30)")
dots, = a2.plot([], [], "o", color=COLORS["teal"], ms=3, label="Crank–Nicolson")
mark = a2.axhline(0, color=COLORS["amber"], lw=1.5, label="δ99 = 3.64√(νt)")
a2.set_xlim(0, 1.05); a2.set_ylim(0, 12); a2.set_xlabel("u/U [–]"); a2.legend(fontsize=7, loc="upper right")


def update(i):                                            # frame i: time tf[i]
    t_ = tf[i]
    dye.set_xdata(disp[i])                                # the dye line
    prof.set_data(ch08.stokes_first_problem(yv, t_, U0, NU_W)/U0, 100*yv)   # the exact profile
    k = np.argmin(np.abs(tcn - t_))                       # the nearest Crank–Nicolson output
    dots.set_data(Ucn[k][::4]/U0, 100*yv[::4])
    mark.set_ydata([100*ch08.diffusion_thickness(t_, NU_W)]*2)   # the rising δ99 marker
    a1.set_title(f"t = {t_:6.1f} s", fontsize=10)
    return []


show_animation(animate(update, frames=nfr, fig=fig, interval=80))

**What you see.** On the left a dyed vertical line above the plate tilts and bends as the fluid is dragged along (drawn as a fraction of
the plate's own travel); on the right the profile grows, the Crank–Nicolson dots ride the erfc curve, and the amber δ₉₉ marker
rises ever more slowly — the `A1` animation.

**How to read it.** The marker height at time t is 3.64√(νt): it needs four times as long to double; the clock is logarithmic, so equal
steps of the movie are equal factors of time.

**What would change if…** …ν were ten times larger: the same movie, ten times faster.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **animate_figure (plotly time slider)** — `animate_figure(frame_fn, times, …)` is the time-player version of `slider_figure`: Play/Pause and a time slider, every frame precomputed, so it works on the web page (Ch. 1 P17).

In [ ]:
yv = np.linspace(0, 0.08, 120)                            # heights [m]


def frame(t_):                                            # curves at time t_ [s]
    u = ch08.stokes_first_problem(yv, t_, 0.1, NU_W)/0.1  # u/U
    return {"raw: u/U against y [cm]": (u, 100*yv),
            "rescaled: u/U against η = y/√(νt)": (u, ch08.similarity_variable(yv, t_, NU_W))}


fig = animate_figure(frame, np.geomspace(1, 1000, 25 if not FAST else 12), time_label="t", unit="s",
                     xlabel="u/U [–]", ylabel="y [cm] (blue) or η [–] (purple)", title="Raw profiles spread; rescaled ones do not",
                     yrange=[0, 8], xrange=[0, 1])
recolor(fig, {"raw: u/U against y [cm]": COLORS["blue"], "rescaled: u/U against η = y/√(νt)": COLORS["accent"]},
        {"rescaled: u/U against η = y/√(νt)": "dash"})
fig.show()

**What does the code above do?**

Press Play: the blue profile (plotted against y in centimetres) thickens as √t, while the purple one (the same profile plotted
against η) never moves — the collapse, without a kernel.

#### 🎮 Interactive: How can profiles at all times be one curve?

**Why interactive rather than a static figure:** Start the plate, watch the profile spread with a moving δ₉₉ marker, then flip to the rescaled mode: every earlier profile falls
on one curve, and changing ν or U does not break it — while stopping the plate or adding a second wall does.

**What to try:**
- Play with water, then switch to air: the layer grows faster but the rescaled curve does not move.
- Click a point in the profile view: read η and the erfc arithmetic of u/U.
- Choose 'stop the plate at T' and watch the collapse fail after T.
- Open the Derivation tab at D19 (the substitution ξ = 2ζ): the factor 2 in y/(2√(νt)) appears.

In [ ]:
show_viz("ch08", "stokes_first_problem")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …the initial state were a velocity jump inside the fluid (two streams), a line vortex, or the spreading bead of C08? The same
trick works, but the similarity form may need a power of t in front — and finding the exponents becomes the whole game (C10).

### 🧩 The similarity ansatz and exponent matching `C10`

*The question:* How do you find the exponents that make a problem self-similar?

*In one line:* $\gamma=At^{-n}F\big(\xi/\delta(t)\big)\equiv At^{-n}F(\eta)$ *(8.32a)*

#### The problem in plain words

Two streams slide past each other and the shear layer between them thickens; a vortex left alone decays; a honey bead spreads.
Each looks self-similar, but with different growth laws ($t^{1/2}$, $t^{1/5}$) and amplitudes that fall with time. We want a
procedure that finds those laws instead of guessing them.

#### The idea

| step | move |
|---|---|
| 1 | guess $\gamma=At^{-n}F(\xi/\delta(t))$ (or $A\xi^{-n}F(\xi/\delta)$ if ξ appears in the initial condition) |
| 2 | substitute into the PDE, divide by one coefficient |
| 3 | demand every bracketed coefficient is the same power of t → one relation between n and δ(t) |
| 4 | a conserved quantity (momentum jump, circulation, volume) → the second relation |

**Two unknown exponents, two conditions: nothing is left to choose.**

> 📎 **Primer — exponent matching for similarity forms.** `P197` · If $c_1t^aF(\eta)+c_2t^bG(\eta)=0$ must hold for every t and every η, the powers of t must agree (a = b) — otherwise dividing
by $t^a$ leaves a t that no function of η can cancel. Power laws $\delta=Dt^m$ turn every bracket into a power of t, and
matching powers gives linear equations for the exponents.

In [ ]:
import sympy as sp
n, m = sp.symbols('n m')
print(sp.solve([sp.Eq(-n - 1, -4*n - 2*m), sp.Eq(-n + m, 0)], [n, m]))   # {n: 1/5, m: 1/5}

📝 **Note.** **N57 [B] · The second form (8.32b)** puts a power of the space coordinate in front; use it when ξ appears in the initial or boundary condition (the line vortex
below, Γ/2πr).

$$ \gamma=A\xi^{-n}F\big(\xi/\delta(t)\big)\equiv A\xi^{-n}F(\eta) \qquad \text{(8.32b)} $$

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **separable first-order ODE for δ(t)** — δ dδ/dt = C: move all δ to one side and all t to the other, then integrate both (Ch. 1 P42).

#### 🧮 Derivation — Example 8.4: the ansatz with A = 1, n = 0 gives δ = √(2C₁νt) `D21` (Eq. 8.32a: $\gamma=At^{-n}F\big(\xi/\delta(t)\big)\equiv At^{-n}F(\eta)$)

**What we want to show.** Recover Stokes' first problem from the general similarity form, finding the growth law δ(t) instead of assuming it.

**Assumptions.** A similarity solution exists (no imposed scales).

**The plan.**

1. Fix A and n from the wall.
2. Differentiate.
3. Demand that the brackets scale alike.
4. Solve for δ.

**Tools we use** (each explained before this point): exponent matching (primer P197) · chain rule (P49) · separable ODE (P42).

**We start from**

$$ \gamma=At^{-n}F\big(\xi/\delta(t)\big)\equiv At^{-n}F(\eta)\ \text{(8.32a) with}\ \gamma=u/U,\ \xi=y,\ \text{in}\ \frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.20)} $$

*In words:* the general similarity form applied to Stokes' first problem

---

**Step 1 of 8 — Fix the amplitude law.**

$$ A=1,\ n=0:\ \ \frac uU=F\Big(\frac{y}{\delta(t)}\Big) $$

- *Why we can do this:* u/U = 1 at η = 0 for every t > 0; any factor t^(−n) with n ≠ 0 would change the wall speed.
- *In words:* only the width can change.

---

**Step 2 of 8 — Differentiate in time.**

$$ \frac{\partial u}{\partial t}=U\frac{dF}{d\eta}\Big(-\frac{y}{\delta^2}\Big)\frac{d\delta}{dt}=-U\eta \frac{\delta'}{\delta}F' $$

- *Why we can do this:* Chain rule through η = y/δ(t).
- *In words:* the profile stretches.

---

**Step 3 of 8 — Differentiate twice in y.**

$$ \frac{\partial^2u}{\partial y^2}=\frac{U}{\delta^2}F'' $$

- *Why we can do this:* Chain rule with ∂η/∂y = 1/δ, applied twice; δ does not depend on y.
- *In words:* curvature in η units.

---

**Step 4 of 8 — Substitute into (8.20).**

$$ -U\eta\frac{\delta'}{\delta}F'=\nu\frac{U}{\delta^2}F'' $$

- *Why we can do this:* Insert steps 2 and 3 into (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$ to see which factors depend on time.
- *In words:* one equation with t-dependent coefficients.

---

**Step 5 of 8 — Divide by U and bracket.**

$$ -\Big[\frac{1}{\delta}\frac{d\delta}{dt}\Big]\eta\frac{dF}{d\eta}=\Big[\frac{\nu}{\delta^2}\Big] \frac{d^2F}{d\eta^2} $$

- *Why we can do this:* Put everything that depends on t into brackets.
- *In words:* the book's bracketed form.

---

**Step 6 of 8 — Make the brackets proportional.**

$$ \frac1\delta\frac{d\delta}{dt}=C_1\frac{\nu}{\delta^2} $$

- *Why we can do this:* Exponent matching (P197): dividing by one bracket must leave an ODE in η alone, so their ratio is a constant C₁.
- *In words:* both sides scale alike in time.

---

**Step 7 of 8 — Solve for δ.**

$$ \delta\frac{d\delta}{dt}=C_1\nu\ \Rightarrow\ \frac{\delta^2}{2}=C_1\nu t+C_2 $$

- *Why we can do this:* Multiply by δ²; δδ′ = d(δ²/2)/dt, integrate in t.
- *In words:* δ² grows linearly in time.

---

**Step 8 of 8 — Start from zero thickness.**

$$ \delta(0)=0\ \Rightarrow\ \delta=\sqrt{2C_1\nu t};\ \ C_1=\tfrac12:\ \delta=\sqrt{\nu t} $$

- *Why we can do this:* The layer starts with zero thickness; C₁ = ½ reproduces η of (8.25), $u/U=F\big(y/\sqrt{\nu t}\big)\equiv F(\eta)$ and turns step 5 into (8.26), $-\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big)$.
- *In words:* the √(νt) law found, not assumed.

---

**Result**

$$ \delta=\sqrt{2C_1\nu t};\quad C_1=\tfrac12:\ \ -\frac\eta2\frac{dF}{d\eta}=\frac{d}{d\eta}\Big(\frac{dF}{d\eta}\Big)\ \text{(8.26)} $$

*In words:* the equation itself forces δ ∝ √t.

**What it means.** The recipe: guess, bracket, match powers. D22 and D23 add a conserved quantity for the second exponent.

**Check it.** Units: δδ′ = C₁ν in m²/s ✓. `similarity_reduce_sympy("stokes1_delta")` returns δ = √(2C₁νt).

> ⚠️ **Common confusion (traps in `D21`):** ∂F(y/δ)/∂t = −ηF′δ′/δ; δ(0) = 0 fixes C₂ = 0; A = 1, n = 0 come from the wall condition, not the ODE.

📝 **Note.** **N58 [B] · Example 8.4 in one line.** It recovers Stokes' first problem from the ansatz: A = 1, n = 0, δδ′ = C₁ν, $\delta=\sqrt{2C_1\nu t}$; C₁ = ½ gives the η of
(8.25), $u/U=F\big(y/\sqrt{\nu t}\big)\equiv F(\eta)$.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **product rule read backwards, (ηF)′** — $F+\eta\frac{dF}{d\eta}=\frac{d}{d\eta}(\eta F)$ — the product rule read from right to left (Ch. 1 P38).

#### 🧮 Derivation — Example 8.5: the viscous vortex sheet, n = ½, $\omega_z=-\frac{U}{\sqrt{\pi\nu t}}e^{-y^2/4\nu t}$, $u=U\,\mathrm{erf}\frac{y}{2\sqrt{\nu t}}$ `D22`

**What we want to show.** Find how a velocity jump (+U above, −U below) thickens, using the ansatz with an amplitude that decays in time.

**Assumptions.** Similarity (no imposed scales); ω and ηF vanish far away (step 11).

**The plan.**

1. Get the vorticity equation.
2. Insert the ansatz, bracket, take δ = √(νt).
3. Fix n by the conserved jump.
4. Integrate the ODE with an exact derivative.
5. Fix the amplitude; integrate for u.

**Tools we use** (each explained before this point): exponent matching (primer P197) · product rule read backwards (gloss) · Gaussian integral (primer P194) · erf (primer P195).

**We start from**

$$ \omega_z=-\frac{\partial u}{\partial y},\quad u(y,0)=\begin{cases}+U & y>0\\ -U & y<0\end{cases},\qquad \frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.20)} $$

*In words:* two streams sliding past each other, and the diffusion equation

---

**Step 1 of 14 — Differentiate (8.20) with respect to y.**

$$ \frac{\partial\omega_z}{\partial t}=\nu\frac{\partial^2\omega_z}{\partial y^2} $$

- *Why we can do this:* ω_z = −∂u/∂y; y- and t-derivatives commute (P121). The initial jump is a delta of vorticity.
- *In words:* vorticity diffuses too.

---

**Step 2 of 14 — Insert the ansatz.**

$$ \omega_z=At^{-n}F(\eta),\ \ \eta=\frac{y}{\delta(t)} $$

- *Why we can do this:* (8.32a), $\gamma=At^{-n}F\big(\xi/\delta(t)\big)\equiv At^{-n}F(\eta)$: y does not appear in the initial condition, so a time power is the right prefactor; the peak must fall as the sheet spreads.
- *In words:* an amplitude and a width that change in time.

---

**Step 3 of 14 — Differentiate in time.**

$$ \frac{\partial\omega_z}{\partial t}=-nAt^{-n-1}F-At^{-n}\eta\frac{\delta'}{\delta}F' $$

- *Why we can do this:* Product rule on t^(−n)F, chain rule through η.
- *In words:* amplitude decay plus stretching.

---

**Step 4 of 14 — Differentiate twice in y.**

$$ \frac{\partial^2\omega_z}{\partial y^2}=At^{-n}\frac{F''}{\delta^2} $$

- *Why we can do this:* Chain rule with ∂η/∂y = 1/δ, applied twice; the prefactor At^(−n) does not depend on y.
- *In words:* curvature.

---

**Step 5 of 14 — Substitute and divide by At^(−n).**

$$ -\Big[\frac nt\Big]F-\Big[\frac{\delta'}{\delta}\Big]\eta F'=\Big[\frac{\nu}{\delta^2}\Big] F'' $$

- *Why we can do this:* Insert steps 3 and 4 into the vorticity equation of step 1 and divide by At^(−n).
- *In words:* three brackets.

---

**Step 6 of 14 — Take δ = √(νt) and multiply by t.**

$$ -nF-\frac12\eta F'=F'' $$

- *Why we can do this:* From D21 (C₁ = ½): δ′/δ = 1/(2t), ν/δ² = 1/t — every bracket ∝ 1/t.
- *In words:* an ODE with n still free.

---

**Step 7 of 14 — State the conserved jump.**

$$ -\int_{-\infty}^{\infty}\omega_z\,dy=u(\infty)-u(-\infty)=2U $$

- *Why we can do this:* ω_z = −∂u/∂y integrates to the velocity difference, which diffusion cannot change.
- *In words:* the total vorticity is fixed.

---

**Step 8 of 14 — Insert the ansatz into it.**

$$ -At^{-n}\delta\int_{-\infty}^{\infty}F(\eta)\,d\eta=2U $$

- *Why we can do this:* Change the integration variable to η: y = δη, so dy = δ dη at fixed t.
- *In words:* amplitude × width × a number.

---

**Step 9 of 14 — Match the powers of t.**

$$ t^{-n}\delta\propto t^{-n+1/2}=\text{const}\ \Rightarrow\ n=\tfrac12 $$

- *Why we can do this:* ∫F dη is a number and the right side is constant for all t (P197).
- *In words:* the peak falls as t^(−1/2).

---

**Step 10 of 14 — Spot an exact derivative.**

$$ -\frac12\Big(F+\eta\frac{dF}{d\eta}\Big)=-\frac12\frac{d}{d\eta}(\eta F)=\frac{d}{d\eta}\Big(\frac {dF}{d\eta}\Big) $$

- *Why we can do this:* Step 6 with n = ½; F + ηF′ = (ηF)′ by the product rule read backwards.
- *In words:* both sides are derivatives.

---

**Step 11 of 14 — Integrate once.**

$$ \frac{dF}{d\eta}+\frac12\eta F=C=0 $$

- *Why we can do this:* F, F′ and ηF vanish as η → ∞ (checked once F is known), so the constant is zero.
- *In words:* a first-order ODE.

---

**Step 12 of 14 — Separate and integrate.**

$$ \frac{F'}{F}=-\frac\eta2\ \Rightarrow\ F=De^{-\eta^2/4} $$

- *Why we can do this:* ln F = −η²/4 + const; ηF → 0 indeed.
- *In words:* a Gaussian.

---

**Step 13 of 14 — Fix the amplitude product AD.**

$$ -At^{-1/2}\sqrt{\nu t}\,D\cdot2\sqrt\pi=2U\ \Rightarrow\ AD=-\frac{U}{\sqrt{\pi\nu}} $$

- *Why we can do this:* ∫$e^(-\eta ^2/4)$dη = 2√π (Gaussian with η = 2ζ) in step 8 with δ = √(νt).
- *In words:* the sign follows the jump.

---

**Step 14 of 14 — Assemble ω and integrate for u.**

$$ \omega_z=-\frac{U}{\sqrt{\pi\nu t}}e^{-y^2/4\nu t},\ \ u=-\int_0^y\omega_z\,dy'= U\,\mathrm{erf}\frac{y}{2\sqrt{\nu t}} $$

- *Why we can do this:* u(0) = 0 by odd symmetry; the integral is the erf of P195.
- *In words:* the sheet thickens like √(νt).

---

**Result**

$$ \omega_z=-\frac{U}{\sqrt{\pi\nu t}}e^{-y^2/4\nu t}\ \text{,}\ u=U\,\mathrm{erf}\frac{y}{2\sqrt{\nu t}} $$

*In words:* a Gaussian vorticity layer of fixed total strength, widening as √(νt).

**What it means.** The upper half is Stokes' first problem seen from a moving frame (N60): a temporally developing boundary layer with C_f ∝ Re_x^(−1/2).

**Check it.** −∫ω dy = 2U at every t ✓ (sympy, the check cell below); u → ±U far away ✓; ±0.95U at η = ±2 erfinv(0.95) = ±2.772 (the book prints 2.76) and width 5.544√(νt); U = 1 cm/s, t = 1 s: width 5.5 mm, peak ω −5.64 s⁻¹.

In [ ]:
vs = ch08.similarity_reduce_sympy("vortex_sheet")        # D22 in sympy: the ansatz in ∂ω/∂t = ν ∂²ω/∂y²
print("brackets:", vs["brackets"])                       # n/t, δ'/δ, ν/δ² with δ = √(νt): all ∝ 1/t (step 6)
print("ODE:", vs["ode"])                                 # −nF − ½ηF′ = F″
print("n from the conserved jump:", vs["n"])             # 1/2 (step 9)
assert sp.simplify(vs["residual"]) == 0                  # the final ω solves the reduced problem
y_, t_, U_, nu_ = sp.symbols("y t U nu", positive=True)  # fresh symbols for a direct check
omega = -U_/sp.sqrt(sp.pi*nu_*t_)*sp.exp(-y_**2/(4*nu_*t_))   # step 14's vorticity
assert sp.simplify(sp.diff(omega, t_) - nu_*sp.diff(omega, y_, 2)) == 0   # it solves the diffusion equation
print("−∫ω dy =", sp.simplify(-sp.integrate(omega, (y_, -sp.oo, sp.oo))))   # 2U at every t: the jump is conserved

> ⚠️ **Common confusion (traps in `D22`):** It is the conserved jump, not the ODE, that fixes n = ½; C = 0 because ηF → 0; AD is negative; 2.772, not 2.76.

📝 **Note.** **N59 [B] · Example 8.5, the viscous vortex sheet.** The solution is below. Define the layer's width by u = ±0.95U: η = ±2 erfinv(0.95) = ±2.772 and the width is 5.544√(νt).
⚠️ The book prints ±2.76 but also the width 5.54 — the 2.76 is a rounding slip. ⚠️ Ch. 5's `diffusing_vortex_sheet(y, t, gamma,
nu)` takes γ = u_below − u_above = −2U.

$$ \omega_z(y,t)=-\frac{U}{\sqrt{\pi\nu t}}\exp\Big\{-\frac{y^2}{4\nu t}\Big\},\qquad u(y,t)=U\,\mathrm{erf}\Big\{\frac{y}{2\sqrt{\nu t}}\Big\}  $$

In [ ]:
u, w = ch08.vortex_sheet_diffusion(0.001, 1.0, U=0.01, nu=NU_W)   # at y = 1 mm after 1 s, U = 1 cm/s
print(f"u = {u:.6f} m/s, ω_z = {w:.4f} 1/s;  Ch. 5 (γ = −2U):", ch05.diffusing_vortex_sheet(0.001, 1.0, -0.02, NU_W))
print(f"width = {ch08.transition_width(1.0, NU_W):.4e} m, 2 erfinv(0.95) = {2*special.erfinv(0.95):.4f}")

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **Galilean shift, temporally developing boundary layer, C_f** — adding a constant velocity to everything changes no stress or vorticity (Ch. 3 P96, frames of reference); the skin-friction coefficient is $C_f=\tau_w/(\tfrac12\rho U^2)$.

📝 **Note.** **N60 [B] · A temporally developing boundary layer.** Shift the upper half of Example 8.5 by −U and flip the sign: it is Stokes' first problem. Its wall stress and skin friction
are below; with Ut read as a distance x they become $C_f=1.128\,\mathrm{Re}_x^{-1/2}$ — the $\mathrm{Re}_x^{-1/2}$ law of a
laminar boundary layer on a plate (Ch. 9 finds 0.664 for the real, spatially growing layer: same exponent, different constant).

$$ \tau_w=\mu\Big(\frac{\partial u}{\partial y}\Big)_{y=0}=\frac{\mu U}{\sqrt{\pi\nu t}},\qquad C_f=\frac{\tau_w}{\frac12\rho U^2}=\frac{2}{\sqrt\pi}\sqrt{\frac{\nu}{U^2t}}  $$

In [ ]:
tb = ch08.temporal_bl_wall_stress(1e4*NU_W/1.0**2, 1.0, NU_W)   # U = 1 m/s at the time where Re_x = U²t/ν = 10⁴
print(f"Re_x = {tb['Re_x']:.0f}, C_f = {tb['Cf']:.5f} (1.128/√Re_x); Blasius (Ch. 9) would give {0.664/np.sqrt(1e4):.5f}")

📝 **Note.** **N61 [B], N62 [B] · Example 8.6, a decaying line vortex (stated; the same moves as D22 with the form (8.32b), Ar⁻ⁿ = Γ/2πr).** **N61** The similarity ODE $(\frac1\eta-\frac\eta2)F'=F''$ gives $F=1-e^{-\eta^2/4}$ and the velocity below — the Gaussian
(Lamb–Oseen) vortex $u_\theta=\frac{\Gamma}{2\pi r}\big(1-e^{-r^2/\sigma^2}\big)$ *(3.29)* with σ² = 4νt (Ch. 3, Ch. 5): rigid
rotation inside r ≈ 2.24√(νt), an ideal vortex outside. **N62** Exercise 8.26, a vortex switched on:
$u_\theta=\frac{\Gamma}{2\pi r}\exp\{-\frac{r^2}{4\nu t}\}$ — the difference between the steady ideal vortex and the decaying one.

$$ u_\theta(r,t)=\frac{\Gamma}{2\pi r}\Big[1-\exp\Big\{-\frac{r^2}{4\nu t}\Big\}\Big]  $$

In [ ]:
r_, t_, G_, nu_ = sp.symbols("r t Gamma nu", positive=True)   # symbols for the two line-vortex solutions
for name, u_th in (("decaying (N61)", G_/(2*sp.pi*r_)*(1 - sp.exp(-r_**2/(4*nu_*t_)))),
                   ("switched on (N62)", G_/(2*sp.pi*r_)*sp.exp(-r_**2/(4*nu_*t_)))):
    res = sp.diff(u_th, t_) - nu_*sp.diff(sp.diff(r_*u_th, r_)/r_, r_)   # ∂u/∂t − ν d/dr[(1/r) d(r u)/dr] (P186's azimuthal Laplacian)
    print(name, "residual:", sp.simplify(res))           # 0: both solve the diffusion of u_θ
print(np.allclose(ch08.line_vortex_decay(0.01, 100.0, 1e-3, NU_W), VX.gaussian_vortex(0.01, 1e-3, 2*np.sqrt(NU_W*100.0))[0]))   # = Ch. 3's Gaussian vortex, σ = 2√(νt)

#### 🧮 Derivation — Example 8.7: the spreading bead, n = m = 1/5, $h=At^{-1/5}F(x/Dt^{1/5})$ `D23`

**What we want to show.** Find the similarity form of the spreading bead of C08 from the equation and its conserved volume.

**Assumptions.** Constant volume (no source); similarity (step 1).

**The plan.**

1. Compute every derivative.
2. Bracket the t-dependence.
3. Two proportionality conditions and the volume.
4. Solve the exponent equations.

**Tools we use** (each explained before this point): exponent matching (primer P197) · product and chain rules (P38, P49).

**We start from**

$$ \begin{array}{l}\frac{\partial h}{\partial t}=\frac{\rho g}{3\mu}\frac{\partial}{\partial x}\big(h^3\frac{\partial h}{\partial x}\big)\ \text{(Example 8.3) and the ansatz}\ h=At^{-n}F(x/\delta(t)) \\ \text{(8.32a).}\end{array} $$

---

**Step 1 of 9 — Differentiate in time.**

$$ \frac{\partial h}{\partial t}=-nAt^{-n-1}F-At^{-n}\eta\frac{\delta'}{\delta}F' $$

- *Why we can do this:* Product rule on t^(−n)F and chain rule through η = x/δ(t), as in D22 step 3.
- *In words:* sinking plus stretching.

---

**Step 2 of 9 — Differentiate in space.**

$$ \frac{\partial h}{\partial x}=At^{-n}\frac{F'}{\delta} $$

- *Why we can do this:* Chain rule with ∂η/∂x = 1/δ; the prefactor At^(−n) does not depend on x.
- *In words:* the slope.

---

**Step 3 of 9 — Form the flux factor.**

$$ h^3\frac{\partial h}{\partial x}=A^4t^{-4n}\frac{F^3F'}{\delta} $$

- *Why we can do this:* Multiply the cube of the ansatz by step 2.
- *In words:* h³ brings A³t^(−3n).

---

**Step 4 of 9 — Differentiate the flux factor.**

$$ \frac{\partial}{\partial x}\Big(h^3\frac{\partial h}{\partial x}\Big)=\frac{A^4t^{-4n}} {\delta^2}(3F^2F'^2+F^3F'') $$

- *Why we can do this:* (F³F′)′ = 3F²F′² + F³F″ by the product rule; one more 1/δ.
- *In words:* the diffusion side.

---

**Step 5 of 9 — Assemble with brackets.**

$$ -[nAt^{-n-1}]F-\Big[At^{-n}\frac{\delta'}{\delta}\Big]\eta F'=\Big[\frac{\rho g}{3\mu}\frac{A^4 t^{-4n}}{\delta^2}\Big](3F^2F'^2+F^3F'') $$

- *Why we can do this:* Steps 1 and 4 in the equation.
- *In words:* three bracketed coefficients.

---

**Step 6 of 9 — Match the first two brackets.**

$$ \frac{\delta'}{\delta}=\frac{Cn}{t}\ \Rightarrow\ \delta=Dt^m,\ \ m=Cn $$

- *Why we can do this:* Their ratio must be constant (P197); δ′/δ ∝ 1/t integrates to a power law.
- *In words:* the width is a power of time.

---

**Step 7 of 9 — Match the second and third.**

$$ t^{-n-1}\propto t^{-4n}\delta^{-2}=t^{-4n-2m}\ \Rightarrow\ -1=-3n-2m $$

- *Why we can do this:* Equal powers of t on both sides (the constant factors are absorbed into F).
- *In words:* first exponent relation.

---

**Step 8 of 9 — Conserve the volume.**

$$ \int_{-\infty}^{\infty}h\,dx=At^{-n}Dt^m\int F\,d\eta=\text{const}\ \Rightarrow\ -n+m=0 $$

- *Why we can do this:* dx = δ dη; ∫F dη is a number, so the power of t must vanish.
- *In words:* second relation — from a conserved quantity, not a boundary condition.

---

**Step 9 of 9 — Solve the two relations.**

$$ n=m,\ \ 3n+2n=1\ \Rightarrow\ n=m=\tfrac15:\ \ h=At^{-1/5}F\big(x/Dt^{1/5}\big) $$

- *Why we can do this:* Two linear equations in n, m.
- *In words:* height falls and width grows like t^(1/5).

---

**Result**

$$ h(x,t)=At^{-1/5}F(x/Dt^{1/5}) $$

*In words:* the bead spreads as t^(1/5) and thins as t^(−1/5).

**What it means.** Not a diffusion length (t^(1/2)): the diffusivity ρgh³/3μ collapses as the layer thins. The book's last line says 'the final equation of Example 8.2'; it means Example 8.3.

**Check it.** Volume ∝ t^(−1/5)t^(1/5) = const ✓. The C08 run's fitted front slope is 0.200 ± 0.005; Huppert's closed form (N63) has the same exponents; `similarity_reduce_sympy("spreading")` returns n = m = 1/5.

> ⚠️ **Common confusion (traps in `D23`):** (h³h_x)_x brings δ^(−2) and A⁴t^(−4n); the second condition is the volume, not a boundary condition.

📝 **Note.** **N63 [B] · Example 8.7 solved.** The spreading bead of C08 is self-similar with n = m = 1/5: $h(x,t)=At^{-1/5}F(x/Dt^{1/5})$. Solving the ODE for F (not done in
the book) gives Huppert's (1982) dome $F\propto(1-\eta^2/\eta_N^2)^{1/3}$ and the front $x_N=\eta_N(\beta A^3t)^{1/5}$,
$\eta_N=1.411$ (benchmark V5) — the `viscous_current_similarity` used in C08. ⚠️ The book's last line says "the final equation of
Example 8.2"; it means Example 8.3.

#### ✏️ Tiny example: a 1 cm/s shear layer after one second

U = 0.01 m/s on each side, ν = 10⁻⁶ m²/s, t = 1 s.
1. $\sqrt{\nu t}=10^{-3}$ m.
2. Width 5.544 × 1 mm = 5.5 mm.
3. At y = 1 mm: η = 1, u = U erf(0.5) = 0.0052 m/s.
4. Peak vorticity $-U/\sqrt{\pi\nu t}=-0.01/1.772\times10^{-3}=-5.64$ s⁻¹.
5. After 100 s: width 5.5 cm, peak −0.564 s⁻¹, and $-\int\omega\,dy=2U$ still.

In [ ]:
for case in ("stokes1", "vortex_sheet", "line_vortex", "spreading"):   # four similarity reductions in sympy
    s = ch08.similarity_reduce_sympy(case)
    print(f"{case:13s} n = {s['n']},  δ(t) = {s['delta']}")
good = ch08.similarity_collapse_error("spreading", 0.2, 0.2, [10, 30, 100, 300, 1000])    # right exponents n = m = 1/5
bad = ch08.similarity_collapse_error("spreading", 0.25, 0.5, [10, 30, 100, 300, 1000])    # a diffusion-like guess m = 1/2
print(f"collapse spread of the bead: {good:.1e} at (n, m) = (1/5, 1/5), {bad:.2f} at (1/4, 1/2)")

**What does the code above do?**

1. `similarity_reduce_sympy` repeats Examples 8.4–8.7 symbolically: n = 0 and δ = √(νt) for Stokes' first problem, n = ½ for
   the sheet, n = 1 (a power of r) for the line vortex, and n = m = 1/5 with δ = Dt^(1/5) for the bead.
2. `similarity_collapse_error` samples the exact bead solution at five times, rescales with trial exponents and measures how
   far the curves are apart: round-off at the right pair, order one at a wrong pair.

In [ ]:
tw = np.array([1.0, 10.0, 100.0, 1000.0])                # times [s]
widths = [2*optimize.brentq(lambda y_: ch08.vortex_sheet_diffusion(y_, t_, 1.0, NU_W)[0] - 0.95, 1e-6, 1.0) for t_ in tw]   # u = 0.95U
slope_sheet = np.polyfit(np.log(tw), np.log(widths), 1)[0]   # log–log slope of the sheet width
m_ = run["t"] >= 10                                      # the bead of C08 once it has forgotten the box
slope_bead = np.polyfit(np.log(run["t"][m_]), np.log(run["x_front"][m_]), 1)[0]   # slope of the numerical front
print(f"sheet width slope = {slope_sheet:.4f} (expect 1/2); bead front slope = {slope_bead:.4f} (expect 1/5)")
assert abs(slope_sheet - 0.5) < 5e-3                     # D22's n = 1/2 measured
assert abs(slope_bead - 0.2) < 1e-2                      # D23's m = 1/5 measured (the front is found on a grid: ~1 %)

In [ ]:
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(12, 3.8))
yv = np.linspace(-0.012, 0.012, 400)                      # across the sheet [m]
for t_, c in ((1.0, COLORS["accent"]), (4.0, COLORS["blue"])):   # two times [s]
    u, w = ch08.vortex_sheet_diffusion(yv, t_, 0.01, NU_W)   # Example 8.5 with U = 1 cm/s
    a1.plot(w, 1e3*yv, color=c, lw=2, label=f"t = {t_:g} s")
    a2.plot(u/0.01, ch08.similarity_variable(yv, t_, NU_W, half=True), color=c, lw=2, ls="-" if t_ == 1 else "--")
a1.set_xlabel("ω_z [1/s]"); a1.set_ylabel("y [mm]"); a1.legend(fontsize=8); a1.set_title("Sheet vorticity: peak halves, width doubles", fontsize=9)
a2.set_ylim(-3, 3); a2.set_xlabel("u/U [–]"); a2.set_ylabel("y/(2√(νt)) (the book's axis)"); a2.set_title("One curve for both times", fontsize=9)
rr = np.linspace(1e-4, 0.03, 300)                          # radius [m]
G0_ = 1e-4                                                 # circulation Γ [m²/s] (ours)
a3.plot(100*rr, 1e3*G0_/(2*np.pi*rr), color=COLORS["muted"], ls="--", lw=1.5, label="νt = 0: ideal vortex")
for nt_, c in ((1e-5, COLORS["blue"]), (4e-5, COLORS["teal"]), (1.6e-4, COLORS["orange"])):   # νt [m²] (our values)
    a3.plot(100*rr, 1e3*ch08.line_vortex_decay(rr, nt_/NU_W, G0_, NU_W), color=c, lw=2, label=f"νt = {nt_:.1e} m²")
    rc = 2.24*np.sqrt(nt_)                                  # the solid-body core edge ≈ 2.24√(νt)
    a3.plot(100*rc, 1e3*ch08.line_vortex_decay(rc, nt_/NU_W, G0_, NU_W), "o", color=c)
a3.set_ylim(0, 10); a3.set_xlabel("r [cm]"); a3.set_ylabel("u_θ [mm/s]"); a3.legend(fontsize=7)
a3.set_title("Decaying line vortex (dots: core edge)", fontsize=9)
fig.suptitle("Three self-similar diffusions", fontsize=11)
savefig(fig, "ch08", "c10_vortex_diffusion"); plt.show()

**What you see.** (a) the Gaussian vorticity of the thickening sheet: from 1 s to 4 s the peak halves and the width doubles; (b) both
velocity profiles on one curve against the book's axis y/(2√(νt)); (c) the ideal vortex (dashed) smoothed into a solid-body
core that grows with νt — our remake of Figs. 8.14–8.15 `N106` (our values of νt, not the book's).

**How to read it.** In (c) the dots mark r ≈ 2.24√(νt), where the speed peaks: inside it the fluid turns like a solid, outside it the speed
still falls as Γ/2πr.

**What would change if…** …Γ doubled: every curve in (c) doubles; its shape and the core radius 2.24√(νt) do not change.

In [ ]:
nfr = 60 if not FAST else 30                              # frames
tf = np.geomspace(0.5, 50, nfr)                           # a log clock [s]
yv = np.linspace(-0.02, 0.02, 300); rr = np.linspace(1e-4, 0.03, 300)   # across the sheet, and radius [m]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
l1, = a1.plot([], [], color=COLORS["blue"], lw=2, label="u/U")
l2, = a1.plot([], [], color=COLORS["accent"], lw=2, label="ω/ω_peak(0.5 s)")
l3, = a2.plot([], [], color=COLORS["blue"], lw=2)
a1.legend(fontsize=7, loc="upper left"); a1.set_xlim(-1.1, 1.1)
nres = int(0.75*nfr)                                      # the last quarter of the frames switches to rescaled axes
w0 = abs(ch08.vortex_sheet_diffusion(0.0, tf[0], 0.01, NU_W)[1])   # the first frame's peak vorticity [1/s]


def update(i):                                            # frame i
    t_ = tf[i]
    u, w = ch08.vortex_sheet_diffusion(yv, t_, 0.01, NU_W)   # the sheet at t_
    ut = ch08.line_vortex_decay(rr, t_, 1e-4, NU_W)       # the vortex at t_
    if i < nres:                                          # raw axes
        l1.set_data(u/0.01, 1e3*yv); l2.set_data(w/w0, 1e3*yv); a1.set_ylim(-20, 20); a1.set_ylabel("y [mm]")
        l3.set_data(100*rr, 1e3*ut); a2.set_xlim(0, 3); a2.set_ylim(0, 3); a2.set_xlabel("r [cm]"); a2.set_ylabel("u_θ [mm/s]")
        a1.set_title(f"t = {t_:.2f} s: the sheet thickens", fontsize=9); a2.set_title("the vortex decays", fontsize=9)
    else:                                                 # rescaled axes: every profile falls on one curve
        s_ = np.sqrt(NU_W*t_)
        l1.set_data(u/0.01, yv/s_); l2.set_data(w/abs(w).max(), yv/s_); a1.set_ylim(-6, 6); a1.set_ylabel("y/√(νt)")
        l3.set_data(rr/s_, ut*2*np.pi*rr/1e-4); a2.set_xlim(0, 10); a2.set_ylim(0, 1.05)
        a2.set_xlabel("r/√(νt)"); a2.set_ylabel("2πr u_θ/Γ")
        a1.set_title(f"t = {t_:.2f} s: rescaled — one curve", fontsize=9); a2.set_title("rescaled — one curve", fontsize=9)
    return []


show_animation(animate(update, frames=nfr, fig=fig, interval=90))

**What you see.** Left: the vortex sheet's velocity (blue) and vorticity (purple) spreading; right: the line vortex losing its peak. In
the last quarter of the movie both switch to rescaled axes and stop changing — the `A5` animation.

**How to read it.** In the rescaled frames the sheet is plotted against y/√(νt) and the vortex as 2πr u_θ/Γ against r/√(νt): if the
curves still moved, the similarity form would be wrong.

**What would change if…** …ν were halved: the raw movie runs √2 times slower in space; the rescaled frames are identical.

#### 🎮 Interactive: Where do similarity exponents come from?

**Why interactive rather than a static figure:** Set trial exponents n and m with two sliders: the rescaled profiles collapse onto one curve only at the right values, and the
powers of t in the reduced equation turn green when they match — the exponent algebra of Examples 8.4–8.7 as a game you can
win or lose.

**What to try:**
- In 'vortex sheet' mode, start at n = 0.4: the curves fan out; slide to 0.5 and watch them merge.
- In 'spreading bead' mode try the diffusion guess m = 1/2 — the bracket chip turns red.
- Switch to 'line vortex': why is the prefactor now a power of r, not of t?
- Check the conserved-integral readout: it is flat only at the right n.

In [ ]:
show_viz("ch08", "similarity_exponents")   # full-width explainer; ⤢ Full screen for more room

📝 **Note.** **N64 [C] · Two kinds of length.** Diffusive lengths grow like $(\nu t)^{1/2}$; the bead's $t^{1/5}$ is not a diffusion length but an advective one — how far the
fluid itself has travelled.

**What would change if…** …the wall did not start once but oscillated forever? Then the problem has an imposed time 1/ω; no similarity variable exists,
but the diffusion length $\sqrt{\nu/\omega}$ sets the depth (C11).

---

## 8.5 Flow Due to an Oscillating Plate

**What is this section about?** A plate shaking back and forth in its own plane drags a thin layer of fluid with it. The
motion decays and lags with height, looking like a wave travelling away from the wall — but it is diffusion. The depth
$\sqrt{2\nu/\omega}$ and the phase structure $(1+i)y/\delta$ are exactly those of tidal bottom boundary layers and of the
Ekman layer of Ch. 13.

### 🧩 Stokes' second problem: the Stokes layer `C11`

*The question:* How deep does an oscillation reach into a viscous fluid — and why do the deeper layers lag behind?

*In one line:* $u=U\exp\Big\{-y\sqrt{\frac{\omega}{2\nu}}\Big\}\cos\Big(\omega t-y\sqrt{\frac{\omega}{2\nu}}\Big)$ *(8.38)*

#### The problem in plain words

Shake a plate under water at one cycle per second; the tide rubs the sea floor back and forth twice a day; a sound wave slides
air along a wall a thousand times a second. How far above the wall does the fluid feel the shaking, and how late does each
layer follow?

#### The idea

```
y = 3δ   ~~ 5 %,  3 rad late
y = 2δ   ~~~ 14 %, 2 rad late
y = δ    ~~~~ 37 %, 1 rad late
y = 0    ◄══►  U cos ωt           δ = δ_e = √(2ν/ω)
```

| shaking | ν [m²/s] | δ_e (computed below) |
|---|---|---|
| plate at 1 Hz in water | 10⁻⁶ | 0.56 mm |
| M₂ tide (12.42 h), molecular ν | 10⁻⁶ | 0.12 m |
| M₂ tide, eddy ν (turbulent) | 10⁻² | 12 m |
| sound at 1 kHz in air | 1.5 × 10⁻⁵ | 0.069 mm |

Each layer repeats the wall's motion, delayed by y/δ radians and shrunk by $e^{-y/\delta}$.

📝 **Note.** **N65 [B], N66 [B], N67 [B] · Set-up.** The plate at y = 0 oscillates in its own plane; we want only the periodic state after start-up transients have died, so there
is no initial condition. The equation is (8.20) again, $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$, but
now the wall imposes a time scale 1/ω, through the wall condition (8.33) below and $u(y\to\infty,t)=\text{bounded}$ *(8.34)*.

$$ u(y=0,t)=U\cos(\omega t) \qquad \text{(8.33)} $$

> ⚠️ **Common confusion:** ω is now a **frequency** [rad/s] (as in Ch. 7), not a vorticity.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **complex exponential solution, taking the real part** — write the wall motion as $\mathrm{Re}\{Ue^{i\omega t}\}$; for a linear equation with real coefficients we can solve with the complex form and take the real part at the end (Ch. 7 P176, complex amplitudes).
- **square root of i** — $\sqrt i=(1+i)/\sqrt2$, since $((1+i)/\sqrt2)^2=(1+2i-1)/2=i$ (Ch. 6 P159, complex square roots).
- **Euler's formula, i² = −1** — $e^{i\theta}=\cos\theta+i\sin\theta$, so $\mathrm{Re}\{e^{i\theta}\}=\cos\theta$ (Ch. 1 P45).

In [ ]:
print(np.sqrt(1j), (1 + 1j)/np.sqrt(2))                  # both (0.7071+0.7071j): the square root of i

📝 **Note.** **N68 [B], N69 [B], N70 [B], N71 [B] · The complex route.** **N68** The complex trial is (8.35) below. ⚠️ The text then says "substitution of (8.33), $u(y=0,t)=U\cos(\omega t)$ into (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$"; it is (8.35) that is
substituted. **N69** It turns (8.20) into $i\omega f=\nu\frac{d^2f}{dy^2}$ *(8.36)*. **N70** $f=e^{ky}$ with
$k=(i\omega/\nu)^{1/2}=\pm(i+1)(\omega/2\nu)^{1/2}$. **N71** The general solution is (8.37); bounded ⇒ B = 0, wall ⇒ A = U.

$$ u(y,t)=\mathrm{Re}\{e^{i\omega t}f(y)\}\ \ \text{(8.35)},\qquad f(y)=A\exp\{-(i+1)y\sqrt{\omega/2\nu}\}+B\exp\{+(i+1)y\sqrt{\omega/2\nu}\}\ \ \text{(8.37)}  $$

#### 🧮 Derivation — Stokes' second problem: $u=Ue^{-y\sqrt{\omega/2\nu}}\cos(\omega t-y\sqrt{\omega/2\nu})$ `D24` (Eq. 8.38: $u=U\exp\Big\{-y\sqrt{\frac{\omega}{2\nu}}\Big\}\cos\Big(\omega t-y\sqrt{\frac{\omega}{2\nu}}\Big)$)

**What we want to show.** Find the periodic flow above a plate oscillating as U cos ωt.

**Assumptions.** Periodic steady state (transients gone) — step 1.

**The plan.**

1. Complex form.
2. ODE in y.
3. Exponential roots with √i.
4. Pick the bounded root, fix the amplitude, take the real part.

**Tools we use** (each explained before this point): complex exponential solution (gloss, P176) · complex square roots (P159) · linear ODE and $e^(\lambda y)$ (P44).

**We start from**

$$ \frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}\ \text{(8.20),}\ u(y=0,t)=U\cos(\omega t)\ \text{(8.33), u bounded as y → ∞ (8.34).} $$

---

**Step 1 of 10 — Try the complex form.**

$$ u(y,t)=\mathrm{Re}\{e^{i\omega t}f(y)\}\ \text{(8.35)} $$

- *Why we can do this:* The wall motion is Re{$Ue^(i\omega t)$}; (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$ is linear with real coefficients, so we solve for ũ = $e^(i\omega t)$f and take the real part at the end.
- *In words:* every layer oscillates at the wall's frequency.

---

**Step 2 of 10 — Differentiate ũ.**

$$ \frac{\partial\tilde u}{\partial t}=i\omega e^{i\omega t}f,\ \ \frac{\partial^2\tilde u}{\partial y^2}=e^{i\omega t} f'' $$

- *Why we can do this:* d/dt $e^(i\omega t)$ = iω$e^(i\omega t)$; y enters only through f.
- *In words:* time derivative → multiplication by iω.

---

**Step 3 of 10 — Substitute and cancel $e^(i\omega t)$.**

$$ i\omega f=\nu\frac{d^2f}{dy^2}\ \text{(8.36)} $$

- *Why we can do this:* $e^(i\omega t)$ ≠ 0. The text says "substitution of (8.33), $u(y=0,t)=U\cos(\omega t)$" — it is (8.35), $u(y,t)=\mathrm{Re}\{e^{i\omega t}f(y)\}$ that is substituted.
- *In words:* (8.36), $i\omega f=\nu\frac{d^2f}{dy^2}$, an ODE with constant coefficients.

---

**Step 4 of 10 — Try f = $e^(ky)$.**

$$ i\omega=\nu k^2\ \Rightarrow\ k^2=\frac{i\omega}{\nu} $$

- *Why we can do this:* The standard trial for constant coefficients (P44).
- *In words:* k is a complex square root.

---

**Step 5 of 10 — Take the square root of i.**

$$ k=\pm\frac{1+i}{\sqrt2}\sqrt{\frac\omega\nu}=\pm(1+i)\sqrt{\frac{\omega}{2\nu}} $$

- *Why we can do this:* ((1 + i)/√2)² = (1 + 2i − 1)/2 = i (P159).
- *In words:* decay and oscillation at the same rate.

---

**Step 6 of 10 — Write the general solution.**

$$ f=A\exp\{-(i+1)y\sqrt{\omega/2\nu}\}+B\exp\{+(i+1)y\sqrt{\omega/2\nu}\}\ \text{(8.37)} $$

- *Why we can do this:* The ODE is linear, so any combination of the two exponential solutions is a solution.
- *In words:* (8.37), $f(y)=A\exp\{-(i+1)y\sqrt{\omega/2\nu}\}+B\exp\{+(i+1)y\sqrt{\omega/2\nu}\}$.

---

**Step 7 of 10 — Discard the growing root.**

$$ B=0 $$

- *Why we can do this:* The second exponential grows like $e^(+y\sqrt (\omega /2\nu ))$, violating (8.34), $u(y\to\infty,t)=\text{bounded}$.
- *In words:* only the decaying layer survives.

---

**Step 8 of 10 — Match the wall motion.**

$$ \tilde u(0,t)=Ae^{i\omega t}\ \Rightarrow\ A=U $$

- *Why we can do this:* Re{$Ae^(i\omega t)$} must be U cos ωt for all t.
- *In words:* amplitude fixed.

---

**Step 9 of 10 — Name the depth.**

$$ \tilde u=Ue^{-y/\delta_e}e^{i(\omega t-y/\delta_e)},\ \ \delta_e=\sqrt{2\nu/\omega} $$

- *Why we can do this:* Split $e^(-(1+i)y/\delta _e)$ into a real decay and a phase.
- *In words:* amplitude and phase both change with y.

---

**Step 10 of 10 — Take the real part.**

$$ u(y,t)=Ue^{-y\sqrt{\omega/2\nu}}\cos\big(\omega t-y\sqrt{\omega/2\nu}\big)\ \text{(8.38)} $$

- *Why we can do this:* Take the real part as promised in step 1: Re{e^(iθ)} = cos θ with θ = ωt − y/δ_e.
- *In words:* (8.38), $u=U\exp\Big\{-y\sqrt{\frac{\omega}{2\nu}}\Big\}\cos\Big(\omega t-y\sqrt{\frac{\omega}{2\nu}}\Big)$.

---

**Result**

$$ u=U\exp\Big\{-y\sqrt{\frac{\omega}{2\nu}}\Big\}\cos\Big(\omega t-y\sqrt{\frac{\omega}{2\nu}}\Big)\ \text{(8.38)} $$

*In words:* each layer repeats the wall's motion, shrunk by $e^(-y/\delta _e)$ and delayed by y/δ_e radians.

**What it means.** The same (1 + i)/δ structure gives the Ekman spiral (Ch. 13) and tidal bottom layers.

**Check it.** u(0, t) = U cos ωt ✓; bounded ✓; residual of (8.20), $\frac{\partial u}{\partial t}=\nu\frac{\partial^2u}{\partial y^2}$ zero (`stokes_second_sympy`); the CN twin after 10 periods agrees to 2 × 10⁻³U.

> ⚠️ **Common confusion (traps in `D24`):** ∂/∂t of $e^(i\omega t)$f is iω$e^(i\omega t)$f; keep only the decaying root; the phase lag carries a minus sign.

In [ ]:
s2 = ch08.stokes_second_sympy()                          # D24 in sympy
print("roots k:", s2["k_roots"], "  bounded:", s2["bounded_root"])
print("residual of (8.20) for (8.38):", s2["residual"])  # 0

📝 **Note.** **N72 [B] · Reading (8.38), $u=U\exp\Big\{-y\sqrt{\frac{\omega}{2\nu}}\Big\}\cos\Big(\omega t-y\sqrt{\frac{\omega}{2\nu}}\Big)$.** The cosine moves toward +y — it looks like a damped transverse wave — but there is no restoring force: it is diffusion driven
by a periodic wall. At $y=4(\nu/\omega)^{1/2}$ the amplitude is $Ue^{-4/\sqrt2}$ (0.05911U, printed by the code below), so the
book calls $\delta\sim4(\nu/\omega)^{1/2}$ the layer thickness; the literature's e-folding depth is $\delta_e=\sqrt{2\nu/\omega}$,
2√2 = 2.83 times smaller (ours: crests move at $\sqrt{2\nu\omega}$, "wavelength" $2\pi\delta_e$).

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **phase of a cosine and a time lag** — in cos(ωt − φ) the phase lag φ [rad] means the peak comes φ/ω seconds later (Ch. 7 P165, phase of a wave).

#### 🧮 Derivation — Reading: 0.059U at 4√(ν/ω), crest speed √(2νω), e-folding depth √(2ν/ω) `D25`

**What we want to show.** Turn (8.38), $u=U\exp\Big\{-y\sqrt{\frac{\omega}{2\nu}}\Big\}\cos\Big(\omega t-y\sqrt{\frac{\omega}{2\nu}}\Big)$ into numbers a reader can picture: how deep, how late, how fast the apparent wave moves.

**Assumptions.** None.

**The plan.**

1. Envelope.
2. The book's depth.
3. Phase lag and crest speed.

**Tools we use** (each explained before this point): phase of a wave (P165) · exponent rules (P43).

**We start from**

$$ u=Ue^{-y/\delta_e}\cos(\omega t-y/\delta_e),\quad \delta_e=\sqrt{2\nu/\omega}\ \ \text{(8.38)} $$

*In words:* Stokes' second problem written with the e-folding depth

---

**Step 1 of 6 — Read the envelope.**

$$ \lvert u\rvert\le Ue^{-y/\delta_e} $$

- *Why we can do this:* The cosine is at most 1; δ_e is the e-folding depth.
- *In words:* the motion shrinks by e every δ_e.

---

**Step 2 of 6 — Evaluate at the book's depth.**

$$ y=4\sqrt{\nu/\omega}:\ \ \frac{y}{\delta_e}=\frac{4}{\sqrt2}=2\sqrt2,\ \ e^{-2\sqrt2}= 0.0591 $$

- *Why we can do this:* 4√(ν/ω)/√(2ν/ω) = 4/√2. The book's δ ~ 4√(ν/ω) is 2√2 = 2.83 δ_e.
- *In words:* 6 % of the wall's motion left.

---

**Step 3 of 6 — Read the phase lag.**

$$ \phi(y)=\frac{y}{\delta_e}\ \text{rad},\ \ \Delta t=\frac{y}{\omega\delta_e} $$

- *Why we can do this:* The cosine's argument is ωt − y/δ_e: a layer at y reaches the same phase y/δ_e radians later.
- *In words:* deeper layers lag.

---

**Step 4 of 6 — Follow a crest.**

$$ \omega t-\frac{y}{\delta_e}=\text{const}\ \Rightarrow\ \frac{dy}{dt}=\omega\delta_e=\sqrt{2\nu\omega} $$

- *Why we can do this:* A point of fixed phase moves; differentiate in t.
- *In words:* crests climb at √(2νω).

---

**Step 5 of 6 — Read the 'wavelength'.**

$$ \lambda=2\pi\delta_e $$

- *Why we can do this:* The phase changes by 2π over Δy = 2πδ_e — but the amplitude has fallen by $e^(-2π)$ ≈ 0.2 % by then.
- *In words:* barely one 'wave' is ever visible.

---

**Step 6 of 6 — Put in numbers.**

$$ \omega=2\pi,\ \nu=10^{-6}:\ \ \delta_e=0.56\ \mathrm{mm},\ \ 4\sqrt{\nu/\omega}=1.6\ \mathrm{mm},\ \ \sqrt{2\nu \omega}=3.5\ \mathrm{mm/s} $$

- *Why we can do this:* Direct evaluation with ω = 2π rad/s (1 Hz) and water's ν = 10⁻⁶ m²/s.
- *In words:* a plate shaken once a second in water moves only a millimetre of fluid.

---

**Result**

$$ \delta_e=\sqrt{\frac{2\nu}{\omega}},\quad \frac{\lvert u\rvert_{max}}{U}\Big\rvert_{y=4\sqrt{\nu/\omega}}=e^{-2\sqrt2}=0.0591,\quad \phi=\frac{y}{\delta_e},\quad \frac{dy}{dt}\Big\rvert_{crest}=\sqrt{2\nu\omega} $$

*In words:* a diffusion layer with a phase delay.

**What it means.** It looks like a wave, but there is no restoring force: the pattern is diffusion from a periodic boundary.

**Check it.** Units: √(m²/s ÷ 1/s) = m ✓; √(m²/s × 1/s) = m/s ✓.

> ⚠️ **Common confusion (traps in `D25`):** Book depth (4√(ν/ω)) and e-folding depth (√(2ν/ω)) differ by 2√2; calling it a real wave.

#### ✏️ Tiny example: a plate shaken once a second in water

ω = 2π rad/s, ν = 10⁻⁶ m²/s, U = 0.1 m/s.
1. $\delta_e=\sqrt{2\nu/\omega}=\sqrt{2\times10^{-6}/6.283}=5.64\times10^{-4}$ m = 0.56 mm.
2. Book depth $4\sqrt{\nu/\omega}=1.60$ mm.
3. At y = 1 mm: amplitude $e^{-1/0.564}=e^{-1.77}=0.17$ → 1.7 cm/s; phase lag 1.77 rad, i.e. 1.77/(2π) × 1 s = 0.28 s late.
4. Crest speed $\omega\delta_e=3.5$ mm/s.
5. Shake 100× faster: every length shrinks 10×.

In [ ]:
y = np.array([0.0, 0.5e-3, 1e-3, 2e-3])                  # four heights [m]
print("u(t = 0) =", ch08.stokes_second_problem(y, 0.0, U=0.1, omega=2*np.pi, nu=NU_W), "m/s")   # (8.38)
sl = ch08.stokes_layer(NU_W, 2*np.pi)                    # depths and speeds of the 1 Hz layer
print({k: f"{sl[k]:.4g}" for k in ("delta_e", "delta_book", "phase_speed", "wavelength", "amplitude_at_delta_book")})
ss = ch08.stokes_layer_state(NU_W, 2*np.pi, 1e-3)        # what the layer does at y = 1 mm
print({k: f"{ss[k]:.4g}" for k in ("amplitude", "phase_lag", "time_lag")})
print(f"e^(−2√2) = {np.exp(-2*np.sqrt(2)):.5f}")          # N72's amplitude at the book's depth
wM2 = 2*np.pi/(12.42*3600)                               # the M₂ tide's frequency [rad/s]
print(f"M2 tide: δ_e = {ch08.stokes_layer(1e-6, wM2)['delta_e']:.3f} m (molecular ν), {ch08.stokes_layer(1e-2, wM2)['delta_e']:.1f} m (eddy ν)")
print(f"sound at 1 kHz in air: δ_e = {1e3*ch08.stokes_layer(1.5e-5, 2*np.pi*1e3)['delta_e']:.3f} mm")
for g_ in ch01.pi_groups({"u": "m/s", "U": "m/s", "y": "m", "t": "s", "nu": "m^2/s", "omega": "1/s"}):   # N73's groups
    print("  group:", " · ".join(f"{k}^{v}" for k, v in g_.items() if v != 0))

**What does the code above do?**

1. `stokes_second_problem` is (8.38), $u=U\exp\Big\{-y\sqrt{\frac{\omega}{2\nu}}\Big\}\cos\Big(\omega t-y\sqrt{\frac{\omega}{2\nu}}\Big)$: at t = 0 the wall moves at U and the layer at 1 mm already moves backward (the lag).
2. `stokes_layer` gives δ_e = 0.564 mm, the book's depth 1.596 mm, crest speed 3.545 mm/s, "wavelength" 2πδ_e = 3.545 mm
   and the amplitude 0.05911 at the book's depth.
3. `stokes_layer_state` at 1 mm: amplitude 0.170, phase lag 1.772 rad, time lag 0.282 s — the tiny example.
4. The tide numbers fill the table above: 12 cm with molecular viscosity, 12 m with an eddy viscosity.
5. **N73 [B]** `pi_groups` finds four groups for (u, U, y, t, ν, ω); linearity removes the one that contains U, leaving three,
   u/U, ωt and $y(\omega/\nu)^{1/2}$: y and t never combine, so there is **no similarity variable**; the extent $(\nu/\omega)^{1/2}$
   is viscosity times the imposed period.

In [ ]:
de = np.sqrt(2*NU_W/(2*np.pi)); t_ = 0.3                  # e-folding depth [m] and a time [s]
u_mine = np.real(0.1*np.exp(1j*2*np.pi*t_)*np.exp(-(1 + 1j)*y/de))   # Re{U e^{iωt} e^{−(1+i)y/δ_e}} (D24 steps 9–10)
assert np.allclose(u_mine, ch08.stokes_second_problem(y, t_, U=0.1, omega=2*np.pi, nu=NU_W), rtol=1e-12, atol=1e-15)
T = 1.0; nper = 10 if not FAST else 5                    # period [s] and number of periods to march from rest
yy = np.linspace(0, 20*de, 201)                          # 20 e-folding depths: the far end sees nothing [m]
u_num = ch08.crank_nicolson_1d(np.zeros(201), yy, T/200, 200*nper, NU_W, lambda tt: 0.1*np.cos(2*np.pi*tt), 0.0)   # from rest
err = np.max(np.abs(u_num - ch08.stokes_second_problem(yy, nper*T, U=0.1, omega=2*np.pi, nu=NU_W))[yy <= 6*de])
print(f"Crank–Nicolson from rest after {nper} periods vs (8.38): max error {err/0.1:.1e} U within 6δ_e")
assert err < 2e-3*0.1                                    # the start-up transient has died: the periodic state is (8.38)

In [ ]:
yd = np.linspace(0, 6, 300)                               # y/δ_e = y√(ω/2ν), the book's axis [–]
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for k, wt in enumerate((0, np.pi/2, np.pi, 3*np.pi/2)):  # four phases of the wall
    u = ch08.stokes_second_problem(yd*de, wt/(2*np.pi), U=1.0, omega=2*np.pi, nu=NU_W)   # (8.38) with U = 1
    ax.plot(u, yd, color=plt.cm.Blues(0.45 + 0.18*k), lw=2, label=f"ωt = {['0', 'π/2', 'π', '3π/2'][k]}")
ax.plot(np.exp(-yd), yd, color=COLORS["muted"], ls="--", lw=1, label="envelope ±e^(−y/δ_e)")
ax.plot(-np.exp(-yd), yd, color=COLORS["muted"], ls="--", lw=1)
yb = 2*np.sqrt(2)                                          # the book's depth 4√(ν/ω) in units of δ_e
ax.axhline(yb, color=COLORS["amber"], lw=1.2)
ax.text(0.35, yb + 0.1, f"book depth 4√(ν/ω) = 2√2 δ_e: amplitude {np.exp(-yb):.4f}", color=COLORS["amber"], fontsize=8)
ax.set_xlabel("u/U [–]"); ax.set_ylabel("y/δ_e = y√(ω/2ν) [–]"); ax.legend(fontsize=8, loc="upper right")
ax.set_title("Each layer is the wall's motion, delayed and shrunk", fontsize=10)
savefig(fig, "ch08", "c11_stokes_layer"); plt.show()

**What you see.** Four S-shaped profiles (one per quarter period) inside a narrowing funnel, the envelope $\pm e^{-y/\delta_e}$; the amber
line is the book's depth 4√(ν/ω), where 0.0591 of the motion is left — our remake of Fig. 8.16 (part of `N107`).

**How to read it.** At any height the spread of the curves is the local amplitude; the height where a curve crosses zero moves up with ωt —
that is the apparent "wave".

**What would change if…** …the fluid were ten times more viscous: all lengths grow √10 ≈ 3.2 times; in these rescaled axes nothing moves.

In [ ]:
nfr = 60 if not FAST else 30                              # frames (two periods)
tf = np.linspace(0, 2.0, nfr)                             # time [s], 1 Hz plate
de = np.sqrt(2*NU_W/(2*np.pi))                            # e-folding depth [m]
yv = np.linspace(0, 5*de, 150)                            # heights [m]
ytr = de*np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0])         # six tracer heights [m]
xs0 = np.linspace(0, 1, 6)                                # their resting x positions [schematic units]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
trac = a1.scatter(xs0, 1e3*ytr, c=[COLORS["teal"]]*6, s=40, zorder=3)   # tracers
dye, = a1.plot(np.zeros(150), 1e3*yv, color=COLORS["blue"], lw=2)      # a dyed vertical line
plate, = a1.plot([-0.2, 1.2], [0, 0], color=COLORS["ink"], lw=5)       # the plate
a1.set_xlim(-0.5, 1.5); a1.set_ylim(-0.1, 1e3*5*de); a1.set_xlabel("x (displacement × 50, schematic)"); a1.set_ylabel("y [mm]")
prof, = a2.plot([], [], color=COLORS["blue"], lw=2)
a2.plot(np.exp(-yv/de), 1e3*yv, color=COLORS["muted"], ls="--", lw=1); a2.plot(-np.exp(-yv/de), 1e3*yv, color=COLORS["muted"], ls="--", lw=1)
probe, = a2.plot([], [], "o", color=COLORS["orange"], ms=8)            # probe at y = δ_e
a2.set_xlim(-1.1, 1.1); a2.set_xlabel("u/U [–]")
Dp = lambda y_, t_: np.imag(np.exp(1j*2*np.pi*t_)*np.exp(-(1 + 1j)*y_/de))/(2*np.pi)   # displacement ∫u dt for U = 1 [m·s/m]


def update(i):                                            # frame i
    t_ = tf[i]
    u = ch08.stokes_second_problem(yv, t_, U=1.0, omega=2*np.pi, nu=NU_W)   # (8.38) with U = 1
    prof.set_data(u, 1e3*yv)
    probe.set_data([ch08.stokes_second_problem(de, t_, U=1.0, omega=2*np.pi, nu=NU_W)], [1e3*de])
    dye.set_xdata(0.5 + 3*Dp(yv, t_))                     # the dye line bends with the lagging layers (displacement × 3)
    trac.set_offsets(np.c_[xs0 + 3*Dp(ytr, t_), 1e3*ytr]) # tracers oscillate with growing lag
    plate.set_xdata(np.array([-0.2, 1.2]) + 3*Dp(0.0, t_))   # the plate itself
    a1.set_title(f"t = {t_:.2f} s (1 Hz plate)", fontsize=10); a2.set_title("profile and probe at y = δ_e", fontsize=10)
    return []


show_animation(animate(update, frames=nfr, fig=fig, interval=60))

**What you see.** Left: tracers at six heights and a dyed line oscillating with the plate, each layer later and smaller than the one
below; right: the profile swinging inside its envelope, with an orange probe at y = δ_e — the `A2` animation.

**How to read it.** The probe moves e⁻¹ = 37 % as far as the plate and peaks 1 rad (0.16 s) later; the dye line is an S that travels
upward — the apparent wave of (8.38), $u=U\exp\Big\{-y\sqrt{\frac{\omega}{2\nu}}\Big\}\cos\Big(\omega t-y\sqrt{\frac{\omega}{2\nu}}\Big)$.

**What would change if…** …the probe moved to 2δ_e: its dot would move e⁻² = 14 % as far and peak 2 rad later.

In [ ]:
de = np.sqrt(2*NU_W/(2*np.pi))                            # e-folding depth [m]
yd = np.linspace(0, 6, 120)                               # y/δ_e [–]


def frame(wt):                                            # curves at phase ωt
    return {"u/U, (8.38)": (ch08.stokes_second_problem(yd*de, wt/(2*np.pi), U=1.0, omega=2*np.pi, nu=NU_W), yd),
            "envelope +": (np.exp(-yd), yd), "envelope −": (-np.exp(-yd), yd)}


fig = animate_figure(frame, np.linspace(0, 2*np.pi, 25 if not FAST else 12), time_label="ωt", unit="rad",
                     xlabel="u/U [–]", ylabel="y/δ_e [–]", title="The Stokes layer through one period", xrange=[-1.05, 1.05], yrange=[0, 6])
recolor(fig, {"u/U, (8.38)": COLORS["blue"], "envelope +": COLORS["muted"], "envelope −": COLORS["muted"]},
        {"envelope +": "dash", "envelope −": "dash"})
fig.show()

**What does the code above do?**

The page-surviving version of the animation: Play runs one period of (8.38), $u=U\exp\Big\{-y\sqrt{\frac{\omega}{2\nu}}\Big\}\cos\Big(\omega t-y\sqrt{\frac{\omega}{2\nu}}\Big)$ inside its envelope.

#### 🎮 Interactive: How deep does a shaking plate reach?

**Why interactive rather than a static figure:** The animation shows the phase lag and the envelope together; drag ω and the layer shrinks while the 0.059U marker at
4√(ν/ω) follows, and a probe clock shows a deep layer peaking later — none of which a four-phase static figure makes vivid.

**What to try:**
- Play the 1 Hz preset and click at y = 1 mm: read the amplitude and the time lag.
- Switch to the M₂ tide with molecular and then eddy viscosity: 12 cm versus 12 m.
- Double ω and predict δ_e before you look (it falls by √2).
- Open the 'Right now' notes: which real Stokes layer is closest to your setting?

In [ ]:
show_viz("ch08", "oscillating_plate")   # full-width explainer; ⤢ Full screen for more room

📝 **Note.** **N74 [C] · Where the Stokes layer reappears.** The same layer explains why sound is weakly absorbed at flat walls: the viscous acoustic boundary layer (Ch. 15). For the
climate thread: replace the plate's oscillation by the Coriolis force and the (1 + i)/δ structure of (8.37), $f(y)=A\exp\{-(i+1)y\sqrt{\omega/2\nu}\}+B\exp\{+(i+1)y\sqrt{\omega/2\nu}\}$ becomes the Ekman
spiral (Ch. 13).

**What would change if…** …the plate stood still and a sphere moved slowly through the fluid instead? Then there is no single wall-normal direction and
the flow is 3-D; for tiny Reynolds numbers inertia vanishes altogether and the equations become linear again (C12).

---

## 8.6 Low Reynolds Number Viscous Flow Past a Sphere

**What is this section about?** Tiny, slow things — cloud droplets, silt, bacteria, a bead sinking in syrup — live at Reynolds
numbers far below one. We find the right way to drop inertia (pressure must be rescaled, not dropped), solve the resulting
linear Stokes equations for a sphere, integrate the surface stresses to Stokes' drag $D=6\pi\mu aU$, and use it for the fall
speed of droplets and for Millikan's measurement of the electron's charge. Finally we see why the solution fails far from the
sphere and how Oseen repaired it.

> 🔁 **Recap — Steady Navier–Stokes** (Ch. 4 §4.6). For steady flow round a body of size L at speed U: $\rho\mathbf u\cdot\nabla\mathbf u+\nabla p=\mu\nabla^2\mathbf u$ *(8.39)* —
Ch. 4's $\rho\frac{D\mathbf u}{Dt}=-\nabla p+\rho\mathbf g+\mu\nabla^2\mathbf u$ *(4.39b)* without the time derivative and with
gravity absorbed into p.

> 🔁 **Recap — The high-Re scaling** (Ch. 4 §4.11). With $u^*=u/U$, $x^*=x/L$ and $p^*=(p-p_\infty)/\rho U^2$ *(4.100)*, (8.39), $\rho\mathbf u\cdot\nabla\mathbf u+\nabla p=\mu\nabla^2\mathbf u$ becomes
$\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*=\frac{1}{\mathrm{Re}}\nabla^{*2}\mathbf u^*$ *(8.40)*, Re = ρUL/μ; for Re ≫ 1 the
viscous term looks negligible (Euler, Ch. 6).

In [ ]:
print(SIM.nondimensional_ns_coefficients("dynamic", "advective"))   # pressure scaled by ρU²: viscous coefficient μ/(ρUl) = 1/Re
print(SIM.nondimensional_ns_coefficients("viscous", "advective"))   # pressure scaled by μU/l: pressure coefficient μ/(ρUl) too

### 🧩 Creeping flow: the Stokes equations `C12`

*The question:* Which terms survive when Re → 0 — and why does pressure not disappear together with inertia?

*In one line:* $\nabla p=\mu\nabla^2\mathbf u$ *(8.43)*

#### The problem in plain words

A bacterium swims at a few body lengths per second; a grain of silt sinks through a river; a mist droplet drifts down. For
them Re = UL/ν ≪ 1. Inertia is negligible — but if we simply multiply (8.40), $\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*=\frac{1}{\mathrm{Re}}\nabla^{*2}\mathbf u^*$ by Re and let it go to zero, the pressure
disappears too, and the resulting equation cannot hold a sphere in a stream. What went wrong, and what is the right limit?

#### The idea

| regime | pressure balances | pressure scale | dimensionless form |
|---|---|---|---|
| Re ≫ 1 | inertia | ρU² | $\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*=\frac{1}{\mathrm{Re}}\nabla^{*2}\mathbf u^*$ *(8.40)* |
| Re ≪ 1 | viscous stress | μU/L | $\mathrm{Re}\,(\mathbf u^*\cdot\nabla^*\mathbf u^*)=-\nabla^*p^*+\nabla^{*2}\mathbf u^*$ *(8.42)* |

The size of pressure differences is set by whatever they balance.

📝 **Note.** **N75 [C], N76 [C] · Expansions in a parameter.** **N75** Many problems are solved as expansions in a small or large parameter (here Re); Ch. 9 (1/Re) and Ch. 11 (small
disturbances) do the same. **N76** At high Re the expansion in 1/Re fails near walls, where the inviscid solution cannot satisfy
no slip — the boundary layer of Ch. 9; the two-length scaling (8.14), $x^*=\frac xL,\ y^*=\frac yh=\frac{y}{\varepsilon L},\ t^*=\frac{Ut}{L},\ u^*=\frac uU,\ v^*=\frac{v}{\varepsilon U},\ p^*=\frac{p}{P_a}$ of C05 was the hint.

> 📎 **Primer — dominant balance.** `P198` · Choose the scale of a quantity from the term it must balance. If pressure differences are pushed by viscous stresses, then
$p-p_\infty\sim\mu U/L$, not ρU²; picking the wrong scale makes a term look negligible when it is not.

In [ ]:
mu, U, L, rho = 1e-3, 1e-3, 1e-5, 1000.0                 # a 10 µm particle at 1 mm/s in water
print(rho*U**2, mu*U/L)                                  # dynamic 1e-3 Pa vs viscous 0.1 Pa: pressure is viscous here

📝 **Note.** **N77 [B], N78 [B], N79 [B] · The two scalings.** **N77** Multiplying (8.40), $\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*=\frac{1}{\mathrm{Re}}\nabla^{*2}\mathbf u^*$ by Re gives (8.41): as Re → 0 it leaves 0 = μ∇²u — pressure lost too. **N78** The low-Re pressure
scale is $p^*=(p-p_\infty)L/(\mu U)$. **N79** It gives (8.42), whose Re → 0 limit is (8.43), $\nabla p=\mu\nabla^2\mathbf u$.

$$ \mathrm{Re}\,(\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*)=\nabla^{*2}\mathbf u^*\ \ \text{(8.41)},\qquad \mathrm{Re}\,(\mathbf u^*\cdot\nabla^*\mathbf u^*)=-\nabla^*p^*+\nabla^{*2}\mathbf u^*\ \ \text{(8.42)}  $$

#### 🧮 Derivation — From to the Stokes equations $\nabla p=\mu\nabla^2\mathbf u$ `D26` (Eq. 8.43: $\nabla p=\mu\nabla^2\mathbf u$)

**What we want to show.** Show how to drop inertia correctly when Re → 0, and why pressure must be rescaled.

**Assumptions.** Steady, no body force; Re → 0 at the end.

**The plan.**

1. Scale with the high-Re pressure (4.100), $x_i^*=\frac{x_i}{l},\ u_j^*=\frac{u_j}{U},\ p^*=\frac{p-p_\infty}{\rho U^2}$.
2. Multiply by Re and see the failure.
3. Rescale pressure by dominant balance.
4. Take the limit.

**Tools we use** (each explained before this point): scaled variables (P133) · dominant balance (primer P198).

**We start from**

$$ \rho\mathbf u\cdot\nabla\mathbf u+\nabla p=\mu\nabla^2\mathbf u\ \text{(8.39).} $$

---

**Step 1 of 9 — Scale with the dynamic pressure.**

$$ \mathbf u=U\mathbf u^*,\ \mathbf x=L\mathbf x^*,\ p-p_\infty=\rho U^2p^* $$

- *Why we can do this:* The scaling (4.100), $x_i^*=\frac{x_i}{l},\ u_j^*=\frac{u_j}{U},\ p^*=\frac{p-p_\infty}{\rho U^2}$, natural when inertia balances pressure.
- *In words:* the high-Re yardsticks.

---

**Step 2 of 9 — Size each term.**

$$ \frac{\rho U^2}{L}\mathbf u^*\cdot\nabla^*\mathbf u^*+\frac{\rho U^2}{L}\nabla^*p^*=\frac{\mu U}{L^2}\nabla^{*2} \mathbf u^* $$

- *Why we can do this:* ∇ = ∇*/L; two derivatives on the viscous term.
- *In words:* inertia and pressure ~ ρU²/L, friction ~ μU/L².

---

**Step 3 of 9 — Divide by ρU²/L.**

$$ \mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*=\frac{1}{\mathrm{Re}}\nabla^{*2}\mathbf u^*\ \text{(8.40)} $$

- *Why we can do this:* Divide every term by ρU²/L; the viscous coefficient (μU/L²)/(ρU²/L) = μ/(ρUL) = 1/Re.
- *In words:* (8.40), $\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*=\frac{1}{\mathrm{Re}}\nabla^{*2}\mathbf u^*$.

---

**Step 4 of 9 — Multiply by Re.**

$$ \mathrm{Re}(\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*)=\nabla^{*2}\mathbf u^*\ \text{(8.41)} $$

- *Why we can do this:* Look at the low-Re end: the viscous term now has coefficient 1.
- *In words:* (8.41), $\mathrm{Re}\,(\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*)=\nabla^{*2}\mathbf u^*$.

---

**Step 5 of 9 — Let Re → 0 naively.**

$$ 0=\nabla^{*2}\mathbf u^*\ \Leftrightarrow\ 0=\mu\nabla^2\mathbf u $$

- *Why we can do this:* The pressure term carries Re too and disappears — leaving an equation with no force balance.
- *In words:* wrong: pressure cannot vanish.

---

**Step 6 of 9 — Rescale pressure by the viscous stress.**

$$ p-p_\infty=\frac{\mu U}{L}p^* $$

- *Why we can do this:* Dominant balance (P198): at low Re pressure differences push against viscous stresses ~ μU/L, not ρU².
- *In words:* the right yardstick.

---

**Step 7 of 9 — Divide (8.39) by μU/L².**

$$ \mathrm{Re}\,\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*=\nabla^{*2}\mathbf u^* $$

- *Why we can do this:* Inertia (ρU²/L)/(μU/L²) = Re; pressure (μU/L²)/(μU/L²) = 1.
- *In words:* only inertia carries Re now.

---

**Step 8 of 9 — Rearrange.**

$$ \mathrm{Re}(\mathbf u^*\cdot\nabla^*\mathbf u^*)=-\nabla^*p^*+\nabla^{*2}\mathbf u^*\ \text{(8.42)} $$

- *Why we can do this:* Move the pressure term to the right.
- *In words:* (8.42), $\mathrm{Re}\,(\mathbf u^*\cdot\nabla^*\mathbf u^*)=-\nabla^*p^*+\nabla^{*2}\mathbf u^*$.

---

**Step 9 of 9 — Let Re → 0 and restore dimensions.**

$$ \nabla p=\mu\nabla^2\mathbf u\ \text{(8.43)} $$

- *Why we can do this:* The left side vanishes; multiply by μU/L² and undo the scalings.
- *In words:* (8.43), $\nabla p=\mu\nabla^2\mathbf u$: pressure gradient balances friction everywhere.

---

**Result**

$$ \nabla p=\mu\nabla^2\mathbf u\ \text{(8.43)},\qquad \nabla\cdot\mathbf u=0\ \text{(4.10)} $$

*In words:* creeping flow is linear: pressure and viscosity balance.

**What it means.** Linearity: solutions add and reversing U reverses the flow (no wake). The italic rule — scales come from the terms that balance — is how Ch. 9 and Ch. 13 build their approximations.

**Check it.** `low_re_scaling_sympy`: dynamic {1, 1, 1/Re}, ×Re {Re, Re, 1}, viscous {Re, 1, 1}. Droplet: μU/a = 0.022 Pa vs ρU² = 1.7 × 10⁻⁴ Pa.

> ⚠️ **Common confusion (traps in `D26`):** Multiplying (8.40), $\mathbf u^*\cdot\nabla^*\mathbf u^*+\nabla^*p^*=\frac{1}{\mathrm{Re}}\nabla^{*2}\mathbf u^*$ by Re and letting Re → 0 kills the pressure too; re-dimensionalise at the end.

#### ✏️ Tiny example: a 10 µm cloud droplet

a = 10 µm, falling at U ≈ 1.2 cm/s in air (ν = 1.5 × 10⁻⁵ m²/s, μ = 1.81 × 10⁻⁵ Pa s; the speed is derived in C14).
1. $\mathrm{Re}=2aU/\nu=2\times10^{-5}\times0.012/1.5\times10^{-5}=0.016$.
2. Dynamic pressure $\rho U^2=1.2\times1.44\times10^{-4}=1.7\times10^{-4}$ Pa.
3. Viscous pressure $\mu U/a=1.81\times10^{-5}\times0.012/10^{-5}=0.022$ Pa — 125 times larger (= 2/Re). The viscous scale is
   the right one.

In [ ]:
lr = ch08.low_re_scaling_sympy()                         # D26's coefficients in sympy (in terms of Re)
for k in ("dynamic", "dynamic_times_Re", "viscous_times_Re"):
    print(f"{k:17s}", {t_: lr[k][t_] for t_ in ("inertia", "pressure", "viscous")})
u_fn = lambda x, t=0.0: np.stack(ch08.stokes_sphere_velocity_xyz(x[0], x[1], x[2], U=1e-3, a=1e-5))   # C13's field (8.49)
p_fn = lambda x, t=0.0: ch08.stokes_sphere_pressure(np.sqrt((x**2).sum(0)), np.arccos(x[0]/np.sqrt((x**2).sum(0))),
                                                    U=1e-3, a=1e-5, mu=MU_W)   # C14's pressure (8.50)
xp = np.array([3e-5, 1e-5, 0.5e-5])                      # a point 3.3 radii from the centre [m]
res = ch08.stokes_residual(u_fn, p_fn, xp, mu=MU_W, h=1e-8)   # ∇p − μ∇²u by central differences (step a/1000)
scale = 3*MU_W*1e-3*1e-5/np.sqrt((xp**2).sum())**3       # the size of ∇p there, 3μUa/r³ [Pa/m]
print(f"Stokes residual |∇p − μ∇²u| = {np.max(np.abs(res)):.1e} Pa/m against ∇p ~ {scale:.0f} Pa/m (relative {np.max(np.abs(res))/scale:.0e})")

**What does the code above do?**

1. `low_re_scaling_sympy` gives the coefficients of inertia, pressure and viscous terms: with the dynamic pressure scale
   {1, 1, 1/Re}; multiplied by Re {Re, Re, 1} — the pressure dies with inertia; with the viscous pressure scale (times Re)
   {Re, 1, 1} — only inertia dies.
2. `stokes_residual` evaluates ∇p − μ∇²u for the sphere solution of C13–C14 by finite differences: about 10⁻⁷ of the pressure
   gradient, i.e. the field satisfies (8.43), $\nabla p=\mu\nabla^2\mathbf u$ to the accuracy of the differences.

In [ ]:
Re_ = 0.016                                              # the droplet's Reynolds number
dyn, visc = [1, 1, 1/Re_], [Re_, 1, 1]                   # by hand: dynamic scaling, and viscous scaling × Re
Re_sym = sp.Symbol("Re", positive=True)                  # the symbol the engine uses
assert np.allclose(dyn, [float(lr["dynamic"][k].subs(Re_sym, Re_)) if hasattr(lr["dynamic"][k], "subs") else float(lr["dynamic"][k])
                         for k in ("inertia", "pressure", "viscous")])
assert np.allclose(visc, [float(lr["viscous_times_Re"][k].subs(Re_sym, Re_)) if hasattr(lr["viscous_times_Re"][k], "subs") else float(lr["viscous_times_Re"][k])
                          for k in ("inertia", "pressure", "viscous")])
print("hand coefficients = engine coefficients at Re = 0.016")

In [ ]:
Re = np.geomspace(1e-3, 1e3, 100)                          # Reynolds numbers
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
a1.loglog(Re, Re, color=COLORS["teal"], lw=2, label="inertia (Re)")
a1.loglog(Re, Re, color=COLORS["orange"], lw=2, ls="--", label="pressure (Re)")
a1.loglog(Re, 1 + 0*Re, color=COLORS["rose"], lw=2, label="viscous (1)")
a1.set_title("(8.41): dynamic pressure scale × Re — pressure dies with inertia", fontsize=9)
a2.loglog(Re, Re, color=COLORS["teal"], lw=2, label="inertia (Re)")
a2.loglog(Re, 1 + 0*Re, color=COLORS["orange"], lw=2, label="pressure (1)")
a2.loglog(Re, 1.05 + 0*Re, color=COLORS["rose"], lw=2, ls="--", label="viscous (1)")
a2.set_title("(8.42): viscous pressure scale — only inertia dies", fontsize=9)
for ax in (a1, a2):
    ax.axvline(1, color=COLORS["muted"], lw=0.8); ax.set_xlabel("Re [–]"); ax.legend(fontsize=7)
a1.set_ylabel("coefficient of the term [–]")
fig.suptitle("Rescale pressure, not just inertia", fontsize=11)
savefig(fig, "ch08", "c12_low_re_scaling"); plt.show()

**What you see.** Left: with the pressure scaled by ρU² and the equation multiplied by Re, the pressure coefficient falls together with
inertia; right: with the pressure scaled by μU/L, pressure and viscous terms stay at 1 and only inertia falls.

**How to read it.** Read off which terms survive as Re → 0 (the left end): on the right, pressure and friction balance — the Stokes equations
$\nabla p=\mu\nabla^2\mathbf u$ *(8.43)*.

**What would change if…** …Re = 1: the two scalings coincide — neither term is negligible and the full equation is needed.

In [ ]:
def frame(lr10):                                          # coefficients at Re = 10^lr10
    R_ = 10**lr10
    return {"dynamic scale × Re: inertia, pressure, viscous": ([1, 2, 3], np.log10([R_, R_, 1.0])),
            "viscous scale × Re: inertia, pressure, viscous": ([1, 2, 3], np.log10([R_, 1.0, 1.0]))}


fig = slider_figure(frame, "log10 Re", np.linspace(-3, 3, 25 if not FAST else 13), unit="",
                    xlabel="term: 1 = inertia, 2 = pressure, 3 = viscous", ylabel="log10(coefficient)",
                    title="Which terms survive as Re → 0?", modes={"dynamic scale × Re: inertia, pressure, viscous": "lines+markers",
                                                                  "viscous scale × Re: inertia, pressure, viscous": "lines+markers"})
recolor(fig, {"dynamic scale × Re: inertia, pressure, viscous": COLORS["muted"], "viscous scale × Re: inertia, pressure, viscous": COLORS["orange"]})
fig.show()

**What does the code above do?**

Slide log₁₀ Re down to −3: with the dynamic scale (grey) both inertia and pressure drop below the viscous term; with the
viscous scale (orange) the pressure stays level with the viscous term.

📝 **Note.** **N80 [C] · The italic rule.** The right length and time scales depend on the region of the flow and are found by balancing the terms that matter there.
Ch. 9 (boundary layers) and Ch. 13 (rotating, stratified scales) live by it.

**What would change if…** …we solve (8.43), $\nabla p=\mu\nabla^2\mathbf u$ round a sphere? The equation is linear, so reversing the stream reverses the whole flow: no wake (C13).

> 🔁 **Recap — Vorticity in spherical coordinates** (Ch. 3 §3.4). For an axisymmetric flow without swirl only the azimuthal vorticity survives,
$\omega_\varphi=\frac1r\big[\frac{\partial(ru_\theta)}{\partial r}-\frac{\partial u_r}{\partial\theta}\big]$ (the spherical curl of
`core.curvilinear`).

> 🔁 **Recap — Stokes' stream function** (Ch. 6 §6.8). Axisymmetric velocities come from one function: $u_r=\frac1{r^2\sin\theta}\frac{\partial\psi}{\partial\theta},\
u_\theta=-\frac1{r\sin\theta}\frac{\partial\psi}{\partial r}$ *(6.83)*.

> 🔁 **Recap — The uniform stream far away** (Ch. 6 §6.8). A uniform stream U along the axis has $\psi=\tfrac12Ur^2\sin^2\theta$ *(6.86)*; the far-field condition is
$\psi(r\to\infty,\theta)=\tfrac12Ur^2\sin^2\theta$ *(8.47)*.

### 🧩 Stokes' solution for the sphere `C13`

*The question:* What is the flow round a sphere when viscosity dominates everywhere?

*In one line:* $\psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)$ *(8.48)*

#### The problem in plain words

A tiny bead sinks through syrup. Drawn in the bead's frame, the fluid streams past it; drawn in the syrup's frame, the fluid is
pushed aside in front and closes in behind. We want the whole velocity field — it gives the pressure, the stresses and the
drag (C14), and it is the reference for every settling particle and swimming micro-organism.

#### The idea

```
─────────►        ψ = ½Ur² sin²θ  −  (3a/4)Ur sin²θ  +  (a³/4)(U/r) sin²θ
────╮ ● ╭─►          stream          'Stokeslet' ~ a/r      'doublet' ~ a³/r³
─────────►        the a/r part decays so slowly that the sphere is felt tens of radii away
```

A uniform stream minus a slowly decaying disturbance, fore–aft symmetric.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **spherical coordinates with θ from the downstream axis** — r from the centre, θ the angle from the +x axis — here the **downstream** direction, so the rear stagnation point is θ = 0 and the front θ = π — and φ round the axis (Ch. 3 P88).
- **curl of a gradient is zero, curl commutes with the Cartesian Laplacian** — ∇ × ∇p = 0 because mixed partial derivatives are equal (Schwarz, Ch. 4 P121); in Cartesian components curl and ∇² are both built from constant-coefficient derivatives, so their order can be swapped.
- **index notation ε_ijk** — $(\nabla\times\mathbf A)_k=\varepsilon_{kli}\partial_lA_i$ with the permutation symbol $\varepsilon_{kli}$ = +1 for cyclic, −1 for anticyclic orders, 0 otherwise (Ch. 2 §2.7, P72).

📝 **Note.** **N81 [B] · Take the curl.** The curl of (8.43), $\nabla p=\mu\nabla^2\mathbf u$ kills the pressure (curl of a gradient) and leaves ∇²ω = 0 — in Cartesian components; in spherical
coordinates the operator means −∇×∇×ω (the book's footnote).

#### 🧮 Derivation — Curl of the Stokes equations: ∇²ω = 0 `D27`

**What we want to show.** Remove the pressure from (8.43), $\nabla p=\mu\nabla^2\mathbf u$ and get an equation for the vorticity alone.

**Assumptions.** Constant μ; Cartesian components for steps 1–5.

**The plan.**

1. Take the curl.
2. The pressure drops out.
3. Swap the curl and the Laplacian.
4. Interpret in spherical coordinates.

**Tools we use** (each explained before this point): index notation ε_ijk (Ch. 2 §2.7) · Schwarz's theorem (P121) · curl of a curl (P122).

**We start from**

$$ \nabla p=\mu\nabla^2\mathbf u\ \text{(8.43)},\qquad \partial_ip=\mu\,\partial_j\partial_ju_i\ \text{(Cartesian components)} $$

*In words:* the Stokes equations, written component by component

---

**Step 1 of 6 — Apply the curl.**

$$ \varepsilon_{kli}\partial_l\partial_ip=\mu\,\varepsilon_{kli}\partial_l\partial_j\partial_ju_i $$

- *Why we can do this:* Multiply both sides by ε_kli∂_l and sum; this is the curl in index notation. We do it to eliminate p.
- *In words:* take the curl of both sides.

---

**Step 2 of 6 — Drop the pressure.**

$$ \varepsilon_{kli}\partial_l\partial_ip=0 $$

- *Why we can do this:* ∂_l∂_i is symmetric in l, i (Schwarz, P121) while ε_kli is antisymmetric: the sum vanishes — the curl of a gradient is zero.
- *In words:* pressure has no curl.

---

**Step 3 of 6 — Move the Laplacian outside.**

$$ 0=\mu\,\partial_j\partial_j(\varepsilon_{kli}\partial_lu_i) $$

- *Why we can do this:* In Cartesian components ε is constant and partial derivatives commute.
- *In words:* curl and Laplacian can swap order.

---

**Step 4 of 6 — Recognise the vorticity.**

$$ \omega_k=\varepsilon_{kli}\partial_lu_i\ \Rightarrow\ 0=\mu\nabla^2\omega_k $$

- *Why we can do this:* ω = ∇ × u in index form.
- *In words:* each Cartesian component of ω is harmonic.

---

**Step 5 of 6 — Write it as a vector statement.**

$$ \nabla^2\boldsymbol\omega=0 $$

- *Why we can do this:* Divide by the constant viscosity μ, which is not zero.
- *In words:* vorticity neither diffuses nor is advected here — it is simply harmonic.

---

**Step 6 of 6 — Say what it means in spherical form.**

$$ \nabla^2\boldsymbol\omega=\nabla(\nabla\cdot\boldsymbol\omega)-\nabla\times\nabla\times \boldsymbol\omega=-\nabla\times\nabla\times\boldsymbol\omega=0 $$

- *Why we can do this:* The curl-of-curl identity (P122) with ∇·ω = 0; in spherical components this, not the scalar Laplacian of each component, is meant (the book's footnote).
- *In words:* the form used in D28.

---

**Result**

$$ -\nabla\times\nabla\times\boldsymbol\omega=0\ \text{(∇²ω = 0 in Cartesian components)} $$

*In words:* in creeping flow the vorticity satisfies a pressure-free equation.

**What it means.** The pressure is recovered afterwards from ∇p = −μ∇×ω (D30).

**Check it.** For Stokes' solution ω_φ = −(3Ua/2r²) sin θ (D30 step 3), the check cell below applies the spherical curl of `core.curvilinear` twice and gets 0 in every component.

In [ ]:
S = ch08.stokes_sphere_sympy()                           # the Stokes-sphere results in sympy (cached; shared by D28–D31)
r, th, a, U, mu = S["symbols"]                           # its symbols: r, θ, a, U, μ
X = (r, th, sp.Symbol("phi", real=True))                 # spherical coordinates for core.curvilinear
w = [0, 0, S["omega_phi"]]                               # Stokes' vorticity ω = ω_φ e_φ, ω_φ = −(3Ua/2r²) sin θ
cc = CL.curl(CL.curl(w, "spherical", X), "spherical", X) # ∇ × (∇ × ω), component by component
print("ω_φ =", S["omega_phi"])
print("∇×∇×ω =", [sp.simplify(c) for c in cc])          # [0, 0, 0]: −∇×∇×ω = 0 holds for Stokes' solution
assert all(sp.simplify(c) == 0 for c in cc)

> ⚠️ **Common confusion (traps in `D27`):** Curl and Laplacian commute only in Cartesian components; applying the scalar Laplacian to ω_φ alone is wrong.

> 📎 **Primer — the Stokes operator E² applied twice.** `P199` · For axisymmetric flow the operator $E^2=\frac{\partial^2}{\partial r^2}+\frac{\sin\theta}{r^2}\frac{\partial}{\partial\theta}\big(
\frac1{\sin\theta}\frac{\partial}{\partial\theta}\big)$ (Ch. 6 used it once, (6.77), $E^2\psi=\frac{\partial^2\psi}{\partial r^2}+\frac{\sin\theta}{r^2}\frac{\partial}{\partial\theta}\Big(\frac1{\sin\theta}\frac{\partial\psi}{\partial\theta}\Big)$) turns ψ into the vorticity:
$\omega_\varphi=-E^2\psi/(r\sin\theta)$. Applying it twice, $E^2(E^2\psi)$, is **not** the biharmonic ∇⁴ψ — a sympy test in D28
shows the difference.

In [ ]:
import sympy as sp
r, th = sp.symbols('r theta', positive=True)
E2 = lambda f: sp.diff(f, r, 2) + sp.sin(th)/r**2*sp.diff(sp.diff(f, th)/sp.sin(th), th)   # the Stokes operator
print(sp.simplify(E2(r**2*sp.sin(th)**2)))               # 0: the uniform stream has no vorticity

📝 **Note.** **N82 [B], N83 [B] · Vorticity from ψ, and the ψ-equation.** **N82** $\omega_\varphi=-\frac1r\big[\frac{1}{\sin\theta}\frac{\partial^2\psi}{\partial r^2}+\frac1{r^2}\frac{\partial}{\partial\theta}
\big(\frac1{\sin\theta}\frac{\partial\psi}{\partial\theta}\big)\big]=-\frac{E^2\psi}{r\sin\theta}$. **N83** Combining with ∇²ω_φ = 0
in the footnote's sense gives (8.44). ⚠️ It is the square of E², not the biharmonic (footnote 2).

$$ \Big[\frac{\partial^2}{\partial r^2}+\frac{\sin\theta}{r^2}\frac{\partial}{\partial\theta}\Big(\frac1{\sin\theta}\frac{\partial}{\partial\theta}\Big)\Big]^2\psi=0 \qquad \text{(8.44)} $$

#### 🧮 Derivation — The stream-function equation $\big[\frac{\partial^2}{\partial r^2}+\frac{\sin\theta}{r^2}\frac{\partial}{\partial\theta}\big(\frac{1}{\sin\theta}\frac{\partial}{\partial\theta}\big)\big]^2\psi=0$ `D28` (Eq. 8.44: $\Big[\frac{\partial^2}{\partial r^2}+\frac{\sin\theta}{r^2}\frac{\partial}{\partial\theta}\Big(\frac1{\sin\theta}\frac{\partial}{\partial\theta}\Big)\Big]^2\psi=0$)

**What we want to show.** Write the vorticity equation of D27 as one scalar equation for Stokes' stream function ψ.

**Assumptions.** Axisymmetric (∂/∂φ = 0), no swirl (u_φ = 0); θ from the downstream axis.

**The plan.**

1. Express ω_φ through ψ with the operator E².
2. Show the curl–curl of any azimuthal field is an E² operation.
3. Combine.

**Tools we use** (each explained before this point): Stokes stream function (R18) · spherical vorticity (R17) · the Stokes operator E² (primer P199) · sympy.

**We start from**

$$ u_r=\frac1{r^2\sin\theta}\frac{\partial\psi}{\partial\theta},\ u_\theta=-\frac1{r\sin\theta}\frac{\partial\psi}{\partial r}\ \text{(6.83)},\qquad \boldsymbol\omega=\omega_\varphi\mathbf e_\varphi,\qquad -\nabla\times\nabla\times\boldsymbol\omega=0\ \text{(D27)} $$

*In words:* Stokes' stream function, the azimuthal vorticity, and the vorticity equation of D27

---

**Step 1 of 12 — Keep only the azimuthal vorticity.**

$$ \boldsymbol\omega=\omega_\varphi(r,\theta)\,\mathbf e_\varphi $$

- *Why we can do this:* With u_φ = 0 and ∂/∂φ = 0 the r- and θ-components of the spherical curl vanish.
- *In words:* vorticity circles the axis.

---

**Step 2 of 12 — Write ω_φ from the velocity.**

$$ \omega_\varphi=\frac1r\Big[\frac{\partial(ru_\theta)}{\partial r}-\frac{\partial u_r}{\partial \theta}\Big] $$

- *Why we can do this:* The φ-component of the spherical curl (R17).
- *In words:* a rotation rate in the meridian plane.

---

**Step 3 of 12 — Insert u_θ from (6.83).**

$$ \frac{\partial(ru_\theta)}{\partial r}=-\frac{1}{\sin\theta}\frac{\partial^2\psi}{\partial r^2} $$

- *Why we can do this:* ru_θ = −(1/sin θ)∂ψ/∂r, and sin θ does not depend on r.
- *In words:* first half.

---

**Step 4 of 12 — Insert u_r from (6.83).**

$$ \frac{\partial u_r}{\partial\theta}=\frac{1}{r^2}\frac{\partial}{\partial\theta}\Big(\frac{1}{\sin\theta} \frac{\partial\psi}{\partial\theta}\Big) $$

- *Why we can do this:* u_r = (1/r²)(1/sin θ)∂ψ/∂θ; differentiate in θ at fixed r.
- *In words:* second half.

---

**Step 5 of 12 — Combine.**

$$ \omega_\varphi=-\frac1r\Big[\frac{1}{\sin\theta}\frac{\partial^2\psi}{\partial r^2}+\frac1{r^2}\frac{\partial} {\partial\theta}\Big(\frac1{\sin\theta}\frac{\partial\psi}{\partial\theta}\Big)\Big] $$

- *Why we can do this:* Insert steps 3 and 4 into the curl formula of step 2 and collect the bracket.
- *In words:* the book's line.

---

**Step 6 of 12 — Factor out 1/sin θ.**

$$ \omega_\varphi=-\frac{E^2\psi}{r\sin\theta},\ \ E^2=\frac{\partial^2}{\partial r^2}+\frac{\sin\theta}{r^2} \frac{\partial}{\partial\theta}\Big(\frac{1}{\sin\theta}\frac{\partial}{\partial\theta}\Big) $$

- *Why we can do this:* Multiply and divide the bracket by sin θ; the operator is Ch. 6's E² (P199).
- *In words:* vorticity = −E²ψ/(r sin θ).

---

**Step 7 of 12 — Curl an azimuthal field.**

$$ \nabla\times(A\,\mathbf e_\varphi)=\frac{\partial_\theta(\sin\theta A)}{r\sin\theta}\mathbf e_r-\frac {\partial_r(rA)}{r}\mathbf e_\theta $$

- *Why we can do this:* Spherical curl of A = A(r, θ)e_φ (`core.curvilinear`). We need the curl twice.
- *In words:* an azimuthal field curls into a meridional one.

---

**Step 8 of 12 — Introduce Ψ = r sin θ A.**

$$ \nabla\times(A\,\mathbf e_\varphi)=\frac{1}{r^2\sin\theta}\frac{\partial\Psi}{\partial\theta} \mathbf e_r-\frac{1}{r\sin\theta}\frac{\partial\Psi}{\partial r}\mathbf e_\theta $$

- *Why we can do this:* sin θ A = Ψ/r and rA = Ψ/sin θ turn step 7 into exactly the (6.83), $u_r=\frac1{r^2\sin\theta}\frac{\partial\psi}{\partial\theta},\ u_\theta=-\frac1{r\sin\theta}\frac{\partial\psi}{\partial r}$ pattern with Ψ in place of ψ.
- *In words:* the curl looks like a velocity from a stream function Ψ.

---

**Step 9 of 12 — Curl again using step 6.**

$$ \nabla\times\nabla\times(A\,\mathbf e_\varphi)=-\frac{E^2\Psi}{r\sin\theta}\mathbf e_\varphi $$

- *Why we can do this:* Steps 2–6 showed: the curl of a (6.83), $u_r=\frac1{r^2\sin\theta}\frac{\partial\psi}{\partial\theta},\ u_\theta=-\frac1{r\sin\theta}\frac{\partial\psi}{\partial r}$-type field from Ψ is −E²Ψ/(r sin θ) e_φ.
- *In words:* curl–curl is an E² operation.

---

**Step 10 of 12 — Apply to the vorticity.**

$$ \Psi=r\sin\theta\,\omega_\varphi=-E^2\psi $$

- *Why we can do this:* A = ω_φ, and step 6.
- *In words:* the 'stream function' of the vorticity is −E²ψ.

---

**Step 11 of 12 — Insert into D27's equation.**

$$ -\nabla\times\nabla\times\boldsymbol\omega=\frac{E^2\Psi}{r\sin\theta}\mathbf e_\varphi=-\frac {E^2(E^2\psi)}{r\sin\theta}\mathbf e_\varphi=0 $$

- *Why we can do this:* Insert step 10 into step 9, with A = ω_φ, into D27's equation.
- *In words:* one scalar equation left.

---

**Step 12 of 12 — Divide by r sin θ.**

$$ E^2(E^2\psi)=0\ \text{(8.44)} $$

- *Why we can do this:* r sin θ ≠ 0 off the axis. It is the square of E², not the biharmonic ∇⁴ (the book's footnote).
- *In words:* (8.44), $\Big[\frac{\partial^2}{\partial r^2}+\frac{\sin\theta}{r^2}\frac{\partial}{\partial\theta}\Big(\frac1{\sin\theta}\frac{\partial}{\partial\theta}\Big)\Big]^2\psi=0$.

---

**Result**

$$ (E^2)^2\psi=0 $$

*In words:* creeping axisymmetric flow obeys the squared Stokes operator.

**What it means.** A fourth-order equation needs four conditions: two on the sphere, the stream far away (and regularity) — D29 uses them.

**Check it.** Units: E² is 1/m², so E⁴ψ ~ ψ/m⁴ ✓; sympy: E⁴ψ = 0 for (8.48), $\psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)$, ∇⁴ψ ≠ 0 for (8.48), and the identity of step 9 holds for a generic A(r, θ) (the check cell below).

In [ ]:
A_ = sp.Function("A")(r, th)                             # a generic azimuthal field A(r, θ) e_φ
cc = CL.curl(CL.curl([0, 0, A_], "spherical", X, False), "spherical", X, False)   # ∇×∇×(A e_φ), not simplified yet
E2 = lambda f: sp.diff(f, r, 2) + sp.sin(th)/r**2*sp.diff(sp.diff(f, th)/sp.sin(th), th)   # the Stokes operator E²
assert sp.simplify(cc[0]) == 0 and sp.simplify(cc[1]) == 0   # steps 7–9: the curl–curl of an azimuthal field is azimuthal …
assert sp.simplify(cc[2] + E2(r*sp.sin(th)*A_)/(r*sp.sin(th))) == 0   # … and equals −E²(r sinθ A)/(r sinθ) e_φ
print("E²(E²ψ) for (8.48):", sp.simplify(E2(E2(S["psi"]))))   # 0: Stokes' ψ satisfies (8.44)
print("∇⁴ψ for (8.48):   ", S["biharmonic_psi"])          # not 0: the scalar biharmonic is a different operator
assert sp.simplify(S["E4psi"]) == 0 and sp.simplify(S["biharmonic_psi"]) != 0

> ⚠️ **Common confusion (traps in `D28`):** (E²)² is not ∇⁴; θ from the downstream axis; the identity of steps 7–9 must be checked, not assumed.

📝 **Note.** **N84 [B], N85 [B], N86 [B] · Conditions and the separated ODE.** **N84, N85** On the sphere: $\psi(r=a,\theta)=0$ *(8.45)* (no flow through it) and $\partial\psi(r=a,\theta)/\partial r=0$ *(8.46)*
(no slip); far away (8.47), $\psi(r\to\infty,\theta)=\tfrac12Ur^2\sin^2\theta$. **N86** (8.47) suggests ψ = f(r) sin²θ; (8.44), $\Big[\frac{\partial^2}{\partial r^2}+\frac{\sin\theta}{r^2}\frac{\partial}{\partial\theta}\Big(\frac1{\sin\theta}\frac{\partial}{\partial\theta}\Big)\Big]^2\psi=0$ becomes the ODE below, solved by
$f=Ar^4+Br^2+Cr+D/r$ with A = 0, B = U/2, C = −3Ua/4, D = Ua³/4.

$$ f^{iv}-\frac{4f''}{r^2}+\frac{8f'}{r^3}-\frac{8f}{r^4}=0  $$

#### 🧮 Derivation — Separable solution, f = Ar⁴ + Br² + Cr + D/r and Stokes' ψ `D29` (Eq. 8.48: $\psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)$)

**What we want to show.** Solve (8.44), $\Big[\frac{\partial^2}{\partial r^2}+\frac{\sin\theta}{r^2}\frac{\partial}{\partial\theta}\Big(\frac1{\sin\theta}\frac{\partial}{\partial\theta}\Big)\Big]^2\psi=0$ with the sphere's conditions.

**Assumptions.** The angular dependence of (8.47), $\psi(r\to\infty,\theta)=\tfrac12Ur^2\sin^2\theta$ persists everywhere (step 1).

**The plan.**

1. Separate ψ = f(r) sin²θ.
2. Get the ODE for f.
3. Euler–Cauchy roots.
4. Four constants from the conditions.

**Tools we use** (each explained before this point): E² (primer P199) · Euler–Cauchy ODE (primer P187) · 2 × 2 system (P57).

**We start from**

$$ (E^2)^2\psi=0\ \text{(8.44),}\ \psi(a,\theta)=0\ \text{(8.45),}\ \partial\psi(a,\theta)/\partial r=0\ \text{(8.46),}\ \psi\to\frac12Ur^2\sin^2\theta\ \text{(8.47).} $$

---

**Step 1 of 10 — Try a separable form.**

$$ \psi=f(r)\sin^2\theta $$

- *Why we can do this:* The far field (8.47), $\psi(r\to\infty,\theta)=\tfrac12Ur^2\sin^2\theta$ has sin²θ; if E² keeps sin²θ, the whole problem reduces to r.
- *In words:* guess the angular shape from infinity.

---

**Step 2 of 10 — Apply the angular part.**

$$ \sin\theta\frac{\partial}{\partial\theta}\Big(\frac{1}{\sin\theta}\frac{\partial\sin^2\theta}{\partial\theta} \Big)=\sin\theta\frac{\partial}{\partial\theta}(2\cos\theta)=-2\sin^2\theta $$

- *Why we can do this:* ∂_θ sin²θ = 2 sin θ cos θ; divided by sin θ it is 2 cos θ.
- *In words:* sin²θ is reproduced.

---

**Step 3 of 10 — E² of the trial.**

$$ E^2(f\sin^2\theta)=\Big(f''-\frac{2f}{r^2}\Big)\sin^2\theta $$

- *Why we can do this:* The radial part gives f″ sin²θ; the angular part gives (1/r²)(−2 sin²θ) f.
- *In words:* E² acts on f alone.

---

**Step 4 of 10 — Apply E² again.**

$$ f^{iv}-\frac{4f''}{r^2}+\frac{8f'}{r^3}-\frac{8f}{r^4}=0 $$

- *Why we can do this:* With g = f″ − 2f/r², (8.44), $\Big[\frac{\partial^2}{\partial r^2}+\frac{\sin\theta}{r^2}\frac{\partial}{\partial\theta}\Big(\frac1{\sin\theta}\frac{\partial}{\partial\theta}\Big)\Big]^2\psi=0$ is g″ − 2g/r² = 0; expanding (f/r²)″ = f″/r² − 4f′/r³ + 6f/r⁴ gives the book's ODE.
- *In words:* a fourth-order Euler–Cauchy ODE.

---

**Step 5 of 10 — Try f = r^λ.**

$$ \lambda(\lambda-1)(\lambda-2)(\lambda-3)-4\lambda(\lambda-1)+8\lambda-8=0 $$

- *Why we can do this:* Every term is r^(λ−4) times a polynomial in λ (P187).
- *In words:* an algebraic equation.

---

**Step 6 of 10 — Factor the polynomial.**

$$ (\lambda-4)(\lambda-2)(\lambda-1)(\lambda+1)=0 $$

- *Why we can do this:* 8λ − 8 = 8(λ − 1) and the rest also carries (λ − 1); the cubic λ³ − 5λ² + 2λ + 8 has the roots 4, 2, −1.
- *In words:* four powers.

---

**Step 7 of 10 — Write the general solution.**

$$ f=Ar^4+Br^2+Cr+\frac Dr $$

- *Why we can do this:* The ODE is linear, so any combination of the four power solutions solves it.
- *In words:* four constants.

---

**Step 8 of 10 — Apply the far field (8.47).**

$$ A=0,\ \ B=\frac U2 $$

- *Why we can do this:* f must tend to ½Ur²; an r⁴ term would outgrow the stream, and the r and 1/r terms are negligible far away.
- *In words:* the uniform stream survives.

---

**Step 9 of 10 — Apply (8.45) and (8.46).**

$$ \frac{Ua^2}{2}+Ca+\frac Da=0,\ \ Ua+C-\frac{D}{a^2}=0 $$

- *Why we can do this:* ψ = 0 and ∂ψ/∂r = 0 at r = a mean f(a) = f′(a) = 0.
- *In words:* no flow through, no slip.

---

**Step 10 of 10 — Solve and assemble.**

$$ C=-\frac{3Ua}{4},\ D=\frac{Ua^3}{4}:\ \ \psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3} \Big)\ \text{(8.48)} $$

- *Why we can do this:* Divide the first by a and add the second: 3Ua/2 + 2C = 0; then D = a²(Ua + C).
- *In words:* (8.48), $\psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)$.

---

**Result**

$$ \psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)\ \text{(8.48)} $$

*In words:* stream, minus a 1/r 'Stokeslet', plus a 1/r³ doublet.

**What it means.** The −3a/4r term decays slowly: the sphere is felt tens of radii away, which is exactly why inertia returns far away (D32).

**Check it.** ψ(a) = Ua²(½ − ¾ + ¼) = 0 ✓; ∂ψ/∂r at a: Ua sin²θ(1 − ¾ − ¼) = 0 ✓; E⁴ψ = 0 (sympy).

> ⚠️ **Common confusion (traps in `D29`):** E²(f sin²θ) = (f″ − 2f/r²) sin²θ; the roots are {4, 2, 1, −1}; A = 0 from the far field, not from the surface.

📝 **Note.** **N87 [B] · Velocities** (one derivative each through (6.83), $u_r=\frac1{r^2\sin\theta}\frac{\partial\psi}{\partial\theta},\ u_\theta=-\frac1{r\sin\theta}\frac{\partial\psi}{\partial r}$):

$$ u_r=U\cos\theta\Big(1-\frac{3a}{2r}+\frac{a^3}{2r^3}\Big),\ u_\theta=-U\sin\theta\Big(1-\frac{3a}{4r}-\frac{a^3}{4r^3}\Big) \qquad \text{(8.49)} $$

In [ ]:
print("u_r =", sp.factor(S["u_r"]))                      # (8.49), factored: vanishes at r = a
print("u_θ =", sp.factor(S["u_theta"]))
print("on r = a:", S["u_r"].subs(r, a), S["u_theta"].subs(r, a))   # 0 0: no slip
print("f-roots:", S["roots"], " constants:", S["constants"])        # D29 steps 6 and 8–10

#### ✏️ Tiny example: a bead in syrup, two radii out on the side

a = 1 mm, U = 1 mm/s, ν = 10⁻³ m²/s (syrup: Re = 2aU/ν = 0.002). At r = 2a, θ = π/2: u_r = 0,
$u_\theta=-U\big(1-\frac38-\frac1{32}\big)=-0.594U$. Ideal flow (Ch. 6) would give $-U\big(1+\frac1{16}\big)=-1.063U$: the viscous
fluid is *slowed* beside the sphere instead of sped up. At r = 10a the Stokes speed is 0.925U — still a 7.5 % deficit; the
ideal-flow disturbance is already 0.05 %.

In [ ]:
for rr_ in (2.0, 10.0):                                  # two distances on the side line, in radii
    print(f"r = {rr_:g}a: Stokes |u| = {ch08.side_line_speed(rr_*1e-3, U=1e-3, a=1e-3, model='stokes')/1e-3:.4f} U, "
          f"ideal |u| = {ch08.side_line_speed(rr_*1e-3, U=1e-3, a=1e-3, model='ideal')/1e-3:.4f} U")
print("(u_r, u_θ) at r = 2a, θ = π/2:", ch08.stokes_sphere_velocity(2e-3, np.pi/2, U=1e-3, a=1e-3), "m/s")
print("fluid-frame ψ at r = 2a, θ = π/4:", ch08.stokes_sphere_streamfunction(2e-3, np.pi/4, U=1e-3, a=1e-3, frame="fluid"), "m³/s")

**What does the code above do?**

1. `side_line_speed` evaluates the speed at θ = π/2 for Stokes' (8.49), $u_r=U\cos\theta\Big(1-\frac{3a}{2r}+\frac{a^3}{2r^3}\Big),\ u_\theta=-U\sin\theta\Big(1-\frac{3a}{4r}-\frac{a^3}{4r^3}\Big)$ and for Ch. 6's ideal sphere: 0.594U against 1.0625U at
   2a, 0.925U against 1.0005U at 10a — the tiny example.
2. `stokes_sphere_velocity` gives the two components; `stokes_sphere_streamfunction(..., frame="fluid")` subtracts the stream
   (N93 below).

In [ ]:
rng = np.random.default_rng(3)                           # reproducible random points
rr = 1e-3*(1.2 + 8*rng.random(20)); tt = 0.1 + 2.9*rng.random(20)   # 20 points outside the sphere: r in (1.2a, 9.2a), θ in (0.1, 3.0)
hh = 1e-7                                                # finite-difference step [m or rad]
psi = lambda R_, T_: ch08.stokes_sphere_streamfunction(R_, T_, U=1e-3, a=1e-3)   # (8.48)
ur_fd = (psi(rr, tt + hh) - psi(rr, tt - hh))/(2*hh)/(rr**2*np.sin(tt))     # u_r = (1/r² sinθ) ∂ψ/∂θ (6.83)
ut_fd = -(psi(rr + hh, tt) - psi(rr - hh, tt))/(2*hh)/(rr*np.sin(tt))       # u_θ = −(1/r sinθ) ∂ψ/∂r
assert np.allclose((ur_fd, ut_fd), ch08.stokes_sphere_velocity(rr, tt, U=1e-3, a=1e-3), rtol=1e-6, atol=1e-12)
print("central differences of ψ through (6.83) reproduce (8.49) at 20 random points")

📝 **Note.** **N93 [B] · The sphere moving through still fluid (fluid frame).** Subtract the stream: ψ below (Fig. 8.19). The pattern is fore–aft symmetric — no wake — because (8.43), $\nabla p=\mu\nabla^2\mathbf u$ is linear: reversing U
maps u → −u and p − p∞ → −(p − p∞). ⚠️ The text calls the equation "(9.63)"; it means (8.43). Ch. 16 returns to this
reversibility (why a swimming bacterium cannot use a reciprocal stroke).

$$ \psi=Ur^2\sin^2\theta\Big(-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)  $$

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **meshgrid, contour and streamplot** — `X, Y = np.meshgrid(x, y)` gives every point; `ax.contour(X, Y, psi, levels)` draws streamlines as lines of constant ψ (Ch. 2 P78).

In [ ]:
n_ = 301 if not FAST else 151                              # grid points per side
xg = np.linspace(-6, 6, n_); yg = np.linspace(-4, 4, n_)   # in sphere radii (a = 1)
X, Y = np.meshgrid(xg, yg)
Rr = np.hypot(X, Y); Th = np.arccos(np.clip(X/np.maximum(Rr, 1e-12), -1, 1))   # spherical r and θ from +x
out = Rr > 1                                              # outside the sphere
psi_b = np.where(out, ch08.stokes_sphere_streamfunction(Rr, Th, 1.0, 1.0, frame="body"), np.nan)   # (8.48)
psi_f = np.where(out, ch08.stokes_sphere_streamfunction(Rr, Th, 1.0, 1.0, frame="fluid"), np.nan)  # N93
ur, ut = ch08.stokes_sphere_velocity(Rr, Th, 1.0, 1.0)    # (8.49) body frame
speed = np.where(out, np.hypot(ur, ut), np.nan)           # |u|/U
psi_i = np.where(out, PF.sphere(1.0, 1.0).psi(np.abs(Y), X) - 0.5*Y**2, np.nan)   # Ch. 6 ideal sphere, fluid frame
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
im = axs[0].pcolormesh(X, Y, speed, cmap="Blues", vmin=0, vmax=1.1, shading="auto")
axs[0].contour(X, Y, psi_b, np.linspace(-4, 4, 17), colors=COLORS["ink"], linewidths=0.8)
fig.colorbar(im, ax=axs[0], label="|u|/U")
axs[1].contour(X, Y, psi_f, np.linspace(-1.2, 1.2, 25), colors=COLORS["blue"], linewidths=0.9)
axs[2].contour(X, Y, psi_i, np.linspace(-0.5, 0.5, 21), colors=COLORS["muted"], linewidths=0.9)
for ax, title in zip(axs, ("Body frame: stream past the sphere", "Fluid frame: sphere moving left (Stokes)", "Fluid frame: ideal flow (Ch. 6)")):
    sphere(ax, 1.0); ax.set_xlim(-6, 6); ax.set_ylim(-4, 4); ax.set_xlabel("x/a"); ax.set_title(title, fontsize=10)
axs[0].set_ylabel("y/a")
fig.suptitle("Creeping flow: symmetric, and felt far away", fontsize=11)
savefig(fig, "ch08", "c13_stokes_sphere"); plt.show()

**What you see.** (a) the stream parting round the sphere, slowed over a wide region (pale blue); (b) closed loops that look the same in
front of and behind the moving sphere; (c) Ch. 6's ideal flow in the same frame, a much more compact disturbance — our remakes
of Figs. 8.17 (sphere) and 8.19 `N107`.

**How to read it.** Count the radii where the loops still bend: in (b) the disturbance is visible beyond 5a (it decays like a/r), in (c) it
is nearly gone by 3a (like a³/r³).

**What would change if…** …Re were 1: inertia breaks the symmetry and a wake forms behind (C15).

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **advecting many tracers in one solve_ivp call** — stack all tracer positions into one long state vector and let `solve_ivp` move them together (Ch. 5 P137).

In [ ]:
nfr = 60 if not FAST else 30                              # frames
T_end = 10.0                                              # the sphere travels 10 radii (a = 1, U = 1, time in a/U)
y0s = np.linspace(0.25, 3.0, 10)                          # ten tracers on the line x = 0, above the axis
z0 = np.r_[np.zeros(10), y0s]                              # state: all x's then all y's


def rhs_stokes(t_, z):                                    # fluid-frame Stokes velocity; the sphere centre is at x = 5 − t
    x_, y_ = z[:10] - (5.0 - t_), z[10:]                  # positions relative to the sphere
    u, v, _ = ch08.stokes_sphere_velocity_xyz(x_, y_, 0*x_, 1.0, 1.0, frame="fluid")   # (8.49) minus the stream
    return np.r_[np.nan_to_num(u), np.nan_to_num(v)]


ideal = PF.sphere(1.0, 1.0)                               # Ch. 6's ideal-flow sphere (stream along its z-axis)


def rhs_ideal(t_, z):                                     # the same for ideal flow
    x_, y_ = z[:10] - (5.0 - t_), z[10:]
    uR, uz = ideal.velocity_cyl(np.abs(y_), x_)           # body-frame (u_R, u_z) with z = our x
    return np.r_[np.nan_to_num(uz - 1.0), np.nan_to_num(uR*np.sign(y_))]   # subtract the stream: fluid frame


tf = np.linspace(0, T_end, nfr)                           # frame times
S_ = integrate.solve_ivp(rhs_stokes, (0, T_end), z0, t_eval=tf, rtol=1e-6, atol=1e-8)   # all Stokes tracers at once
I_ = integrate.solve_ivp(rhs_ideal, (0, T_end), z0, t_eval=tf, rtol=1e-6, atol=1e-8)    # all ideal-flow tracers
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.4), sharey=True)
pts, balls, trails = [], [], []
for ax, sol, c, title in ((a1, S_, COLORS["blue"], "Stokes flow"), (a2, I_, COLORS["muted"], "ideal flow")):
    pts.append(ax.plot(sol.y[:10, 0], sol.y[10:, 0], "o", color=c, ms=5)[0])
    trails.append([ax.plot([], [], color=c, lw=0.8)[0] for _ in range(10)])
    balls.append(plt.Circle((5.0, 0.0), 1.0, color=COLORS["muted"], alpha=0.6)); ax.add_patch(balls[-1])
    ax.set_xlim(-6, 6); ax.set_ylim(-0.5, 3.5); ax.set_aspect("equal"); ax.set_xlabel("x/a"); ax.set_title(title, fontsize=10)
a1.set_ylabel("y/a")


def update(i):                                            # frame i
    for sol, p_, tr, ball in zip((S_, I_), pts, trails, balls):
        p_.set_data(sol.y[:10, i], sol.y[10:, i])         # tracers now
        for k in range(10):
            tr[k].set_data(sol.y[k, :i+1], sol.y[10 + k, :i+1])   # their paths so far
        ball.center = (5.0 - tf[i], 0.0)                  # the sphere moves left at U
    return []


show_animation(animate(update, frames=nfr, fig=fig, interval=70))

**What you see.** Ten tracers on a vertical line as a sphere passes from right to left: in Stokes flow they are pushed aside and
forward, then pulled back and end up only slightly ahead of where they started; in ideal flow they barely move — the `A4`
animation.

**How to read it.** Each tracer's path is a loop: forward (left) while the sphere approaches, sideways as it passes, back as it leaves. The
Stokes loops are large even three radii away — the slow a/r decay.

**What would change if…** …the sphere stopped halfway: the tracers would stop at once — there is no inertia to coast on (reversibility).

#### 🎮 Interactive: What does creeping flow round a sphere look like?

**Why interactive rather than a static figure:** Toggle body and fluid frames and Stokes, Oseen and ideal flow, and drag Re: symmetric loops, a disturbance that decays only
like a/r, and then a wake that grows from far downstream once r reaches a/Re — three comparisons that need motion and switching
to be believed. (Its Oseen mode previews C15.)

**What to try:**
- Fluid frame, Stokes: follow one tracer — it goes round a loop and comes back.
- Read the side-line speed view: at 10a Stokes still shows a 7.5 % deficit, ideal flow none.
- Switch to Oseen and raise Re to 1: where does the circle r = a/Re_a sit, and where does the wake begin?
- Open the Derivation tab at D29 step 8: zoom out to see why the r⁴ term must go.

In [ ]:
show_viz("ch08", "stokes_sphere_flow")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …we asked for the force? The pressure and the shear stress on the surface follow from (8.49), $u_r=U\cos\theta\Big(1-\frac{3a}{2r}+\frac{a^3}{2r^3}\Big),\ u_\theta=-U\sin\theta\Big(1-\frac{3a}{4r}-\frac{a^3}{4r^3}\Big)$; integrating them gives the drag
(C14).

> 🔁 **Recap — Dimensional analysis without ρ** (Ch. 1 §1.11). If inertia does not matter, ρ cannot appear: D = f(μ, U, a); four variables and three dimensions leave one group,
D/(μUa) = constant, so D ∝ μUa and $C_D\propto1/\mathrm{Re}$ before any flow calculation (Ch. 1's Π method; Exercise 4.60).

In [ ]:
print(ch01.pi_groups({"D": "N", "mu": "Pa*s", "U": "m/s", "a": "m"}))   # one group: D μ⁻¹ U⁻¹ a⁻¹

### 🧩 Stokes drag and settling `C14`

*The question:* What force does the fluid exert on the sphere — and where does it come from, pressure or friction?

*In one line:* $D=6\pi\mu aU$ *(8.51)*

#### The problem in plain words

A cloud droplet falls at its terminal speed when the drag equals its weight minus buoyancy; Millikan used exactly this to weigh
single oil drops and count electrons; sediment settles in rivers and the ocean; aerosols stay aloft for days. All need the drag
of a slowly moving sphere — and the surprise is how it splits.

#### The idea

```
front (θ = π): high pressure +3μU/2a  ─►  ● ─►  rear (θ = 0): low pressure −3μU/2a
sides: friction σ_rθ = −(3μU/2a) sin θ drags the surface along
drag = ⅓ pressure (2πμaU) + ⅔ friction (4πμaU) = 6πμaU
```

The traction on the surface has a pressure part and a friction part; their x-components add up to a *uniform* push 3μU/2a over the whole sphere.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **line integral of a gradient** — if ∂p/∂r and (1/r)∂p/∂θ are both known, integrate one and check the other: they must come from one function p (Ch. 1 P35).
- **surface integrals on a sphere, dA = 2πa² sin θ dθ** — for an axisymmetric integrand the sphere is a stack of rings of area 2πa² sin θ dθ (Ch. 6 P163).

*Traction.* The force per unit area that the fluid exerts on the sphere is $\mathbf t=\boldsymbol\sigma\cdot\mathbf e_r$
(Cauchy, Ch. 2 §2.6); its x-component is $\sigma_{rr}\cos\theta-\sigma_{r\theta}\sin\theta$ with θ measured from +x.

#### 🧮 Derivation — The pressure $p-p_\infty=-\frac{3\mu aU\cos\theta}{2r^2}$ by integrating ∇p = μ∇²u `D30` (Eq. 8.50: $p-p_\infty=-\frac{3\mu aU\cos\theta}{2r^2}$)

**What we want to show.** Find the pressure round the sphere (the book gives only the result).

**Assumptions.** p → p∞ far away (step 9).

**The plan.**

1. Replace ∇²u by −∇×ω.
2. Compute ω and its curl.
3. Integrate ∂p/∂r; check ∂p/∂θ.
4. Evaluate extremes.

**Tools we use** (each explained before this point): curl of a curl (P122) · the azimuthal curl of D28 step 8 · line integral of a gradient (gloss).

**We start from**

$$ \nabla p=\mu\nabla^2\mathbf u\ \text{(8.43)},\qquad \psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)\ \text{(8.48)},\qquad \omega_\varphi=-\frac{E^2\psi}{r\sin\theta}\ \text{(D28)} $$

*In words:* the Stokes equations, Stokes' stream function and the vorticity written through it

---

**Step 1 of 11 — Use the vector identity.**

$$ \nabla^2\mathbf u=\nabla(\nabla\cdot\mathbf u)-\nabla\times\boldsymbol\omega=-\nabla\times\boldsymbol\omega\ \Rightarrow\ \nabla p=-\mu\nabla\times\boldsymbol\omega $$

- *Why we can do this:* Curl-of-curl (P122) with ∇·u = 0. The vorticity is easier to curl than the vector Laplacian in spherical coordinates.
- *In words:* pressure gradient = −μ curl of vorticity.

---

**Step 2 of 11 — Compute E²ψ.**

$$ f''-\frac{2f}{r^2}=\Big(U+\frac{Ua^3}{2r^3}\Big)-\Big(U-\frac{3Ua}{2r}+\frac{Ua^3}{2r^3}\Big)=\frac{3Ua}{2r} $$

- *Why we can do this:* With f = ½Ur² − (3Ua/4)r + (Ua³/4)/r and D29 step 3.
- *In words:* E²ψ = (3Ua/2r) sin²θ.

---

**Step 3 of 11 — Get the vorticity.**

$$ \omega_\varphi=-\frac{E^2\psi}{r\sin\theta}=-\frac{3Ua}{2r^2}\sin\theta $$

- *Why we can do this:* Use ω_φ = −E²ψ/(r sin θ) from D28 step 6 with step 2.
- *In words:* vorticity decays as 1/r².

---

**Step 4 of 11 — Form Ψ = r sin θ ω_φ.**

$$ \Psi=-\frac{3Ua}{2r}\sin^2\theta $$

- *Why we can do this:* The curl of an azimuthal field is read off Ψ (D28 step 8).
- *In words:* ready to curl.

---

**Step 5 of 11 — Radial component of ∇×ω.**

$$ (\nabla\times\boldsymbol\omega)_r=\frac{1}{r^2\sin\theta}\frac{\partial\Psi}{\partial\theta}=-\frac{3Ua \cos\theta}{r^3} $$

- *Why we can do this:* ∂_θ sin²θ = 2 sin θ cos θ.
- *In words:* radial part.

---

**Step 6 of 11 — Polar component of ∇×ω.**

$$ (\nabla\times\boldsymbol\omega)_\theta=-\frac{1}{r\sin\theta}\frac{\partial\Psi}{\partial r}=-\frac{3Ua \sin\theta}{2r^3} $$

- *Why we can do this:* Use the D28 step 8 formula with ∂_r(−3Ua/2r) = +3Ua/(2r²).
- *In words:* polar part.

---

**Step 7 of 11 — Write both pressure derivatives.**

$$ \frac{\partial p}{\partial r}=\frac{3\mu Ua\cos\theta}{r^3},\ \ \frac1r\frac{\partial p} {\partial\theta}=\frac{3\mu Ua\sin\theta}{2r^3} $$

- *Why we can do this:* ∇p = −μ∇×ω, component by component.
- *In words:* two equations for one function.

---

**Step 8 of 11 — Integrate the radial one.**

$$ p=-\frac{3\mu Ua\cos\theta}{2r^2}+g(\theta) $$

- *Why we can do this:* Antiderivative of r⁻³ at fixed θ; the constant may depend on θ (gloss: line integral of a gradient).
- *In words:* pressure up to a function of θ.

---

**Step 9 of 11 — Check the polar one.**

$$ \frac1r\frac{\partial p}{\partial\theta}=\frac{3\mu Ua\sin\theta}{2r^3}+\frac{g'(\theta)}{r}\ \Rightarrow\ g'=0 $$

- *Why we can do this:* Compare with step 7: both components agree only if g is constant — a real check that ∇p is a gradient.
- *In words:* consistent.

---

**Step 10 of 11 — Fix the constant far away.**

$$ p-p_\infty=-\frac{3\mu aU\cos\theta}{2r^2}\ \text{(8.50)} $$

- *Why we can do this:* The first term → 0 as r → ∞, so g = p∞.
- *In words:* (8.50), $p-p_\infty=-\frac{3\mu aU\cos\theta}{2r^2}$.

---

**Step 11 of 11 — Evaluate front and rear.**

$$ \theta=\pi:\ +\frac{3\mu U}{2a},\ \ \theta=0:\ -\frac{3\mu U}{2a} $$

- *Why we can do this:* cos π = −1 at the front stagnation point, cos 0 = 1 at the rear. The book prints the minimum without its minus sign.
- *In words:* high in front, low behind.

---

**Result**

$$ p-p_\infty=-\frac{3\mu aU\cos\theta}{2r^2}\ \text{(8.50)} $$

*In words:* the pressure disturbance is a dipole decaying as 1/r², +3μU/2a in front and −3μU/2a behind.

**What it means.** Pressure alone pushes the sphere downstream: its integral gives one third of the drag (D31).

**Check it.** Units μaU/r²: (Pa s)(m)(m/s)/m² = Pa ✓; sympy: both components of ∇p = μ∇²u hold (the check cell below). Droplet: 3μU/2a = 0.033 Pa.

In [ ]:
ur, uth, p = S["u_r"], S["u_theta"], S["p"]              # (8.49) and (8.50) from the cached engine
L_ = CL.vector_laplacian([ur, uth, 0], "spherical", X)   # ∇²u in spherical components (core.curvilinear)
print("p − p∞ =", p)                                     # −3μaU cosθ/(2r²)
assert sp.simplify(sp.diff(p, r) - mu*L_[0]) == 0        # radial component of ∇p = μ∇²u
assert sp.simplify(sp.diff(p, th)/r - mu*L_[1]) == 0     # polar component: consistent (D30 step 9)
print("rear (θ = 0):", p.subs({r: a, th: 0}), "  front (θ = π):", p.subs({r: a, th: sp.pi}))   # −3μU/2a and +3μU/2a

> ⚠️ **Common confusion (traps in `D30`):** Integrate one component and *check* the other; p → p∞ fixes the constant; the rear value is negative.

📝 **Note.** **N88 [B], N89 [B] · Pressure and surface stresses.** **N88** The pressure (8.50) has its maximum +3μU/2a at the front stagnation point (θ = π) and its minimum **−**3μU/2a at the rear
(θ = 0). ⚠️ The book prints the minimum without the minus sign (and its figure labels both extremes 1.5). **N89** Surface stresses
(ours): on r = a the viscous normal stress vanishes, $\sigma_{rr}=-p$, and $\sigma_{r\theta}=-\frac{3\mu U}{2a}\sin\theta$.

$$ p-p_\infty=-\frac{3\mu aU\cos\theta}{2r^2} \qquad \text{(8.50)} $$

#### 🧮 Derivation — Stokes drag $D=6\pi\mu aU$: 2πμaU from pressure + 4πμaU from friction `D31` (Eq. 8.51: $D=6\pi\mu aU$)

**What we want to show.** Integrate the stresses over the sphere and show where the drag comes from (Exercise 8.35).

**Assumptions.** Rigid sphere with no slip (u = 0 on r = a).

**The plan.**

1. The traction and its x-projection.
2. σ_rr and σ_rθ on r = a.
3. Integrate pressure and friction parts.

**Tools we use** (each explained before this point): Newtonian stress in spherical coordinates (Ch. 4 §4.5) · traction projection (gloss) · surface integrals on a sphere (P163).

**We start from**

$$ \begin{array}{l}\text{(8.49)}\ u_r=U\cos\theta(1-\frac{3a}{2r}+\frac{a^3}{2r^3})\ \text{,}\ u_\theta=-U\sin\theta(1-\frac{3a}{4r}-\frac{a^3}{4r^3})\ \text{, (8.50), and the Newtonian} \\ \text{stress σ = −pI + 2μS.}\end{array} $$

---

**Step 1 of 13 — Write the force on the sphere.**

$$ \mathbf F=\oint\boldsymbol\sigma\cdot\mathbf e_r\,dA,\ \ D=F_x $$

- *Why we can do this:* Cauchy (Ch. 2 §2.6): the fluid outside exerts the traction σ·n, with n = e_r pointing into the fluid.
- *In words:* drag = x-force.

---

**Step 2 of 13 — Project the traction on x.**

$$ t_x=\sigma_{rr}\cos\theta-\sigma_{r\theta}\sin\theta $$

- *Why we can do this:* e_r·e_x = cos θ and e_θ·e_x = −sin θ with θ measured from +x.
- *In words:* normal and shear parts both push in x.

---

**Step 3 of 13 — Write σ_rr.**

$$ \sigma_{rr}=-p+2\mu\frac{\partial u_r}{\partial r} $$

- *Why we can do this:* Normal stress of a Newtonian fluid in spherical coordinates.
- *In words:* pressure plus viscous normal stress.

---

**Step 4 of 13 — Evaluate ∂u_r/∂r on the sphere.**

$$ \frac{\partial u_r}{\partial r}\Big\rvert_a=U\cos\theta\Big(\frac{3}{2a}-\frac{3}{2a}\Big)=0 $$

- *Why we can do this:* ∂_r(1 − 3a/2r + a³/2r³) = 3a/2r² − 3a³/2r⁴, which vanishes at r = a.
- *In words:* on the sphere σ_rr = −p.

---

**Step 5 of 13 — Write σ_rθ.**

$$ \sigma_{r\theta}=\mu\Big[r\frac{\partial}{\partial r}\Big(\frac{u_\theta}{r}\Big)+\frac1r\frac{\partial u_r}{\partial \theta}\Big] $$

- *Why we can do this:* Shear stress in spherical coordinates (Ch. 4 §4.5).
- *In words:* the friction.

---

**Step 6 of 13 — Drop the u_r term on the sphere.**

$$ \frac1r\frac{\partial u_r}{\partial\theta}\Big\rvert_a=0 $$

- *Why we can do this:* u_r = 0 at every θ on r = a, so its θ-derivative is zero.
- *In words:* only the radial shear remains.

---

**Step 7 of 13 — Evaluate the radial shear.**

$$ \sigma_{r\theta}\rvert_a=-\frac{3\mu U}{2a}\sin\theta $$

- *Why we can do this:* u_θ/r = −U sin θ(1/r − 3a/4r² − a³/4r⁴); its r-derivative at a is −U sin θ(3/2)/a², times μr = μa.
- *In words:* friction ∝ sin θ, largest at the sides.

---

**Step 8 of 13 — Insert the surface pressure.**

$$ \sigma_{rr}\rvert_a=-p_\infty+\frac{3\mu U}{2a}\cos\theta $$

- *Why we can do this:* Evaluate the pressure (8.50), $p-p_\infty=-\frac{3\mu aU\cos\theta}{2r^2}$ on the surface r = a, where σ_rr = −p (step 4).
- *In words:* normal stress on the surface.

---

**Step 9 of 13 — Assemble t_x.**

$$ t_x=-p_\infty\cos\theta+\frac{3\mu U}{2a}\cos^2\theta+\frac{3\mu U}{2a}\sin^2\theta $$

- *Why we can do this:* Steps 7–8 into step 2 (−σ_rθ sin θ = +(3μU/2a) sin²θ).
- *In words:* pressure part ∝ cos²θ, friction part ∝ sin²θ.

---

**Step 10 of 13 — Write the area element.**

$$ dA=2\pi a^2\sin\theta\,d\theta $$

- *Why we can do this:* A ring on the sphere at angle θ (P163).
- *In words:* integrate over θ from 0 to π.

---

**Step 11 of 13 — Integrate the ambient pressure.**

$$ -p_\infty\int_0^\pi\cos\theta\,2\pi a^2\sin\theta\,d\theta=0 $$

- *Why we can do this:* cos θ sin θ is odd about θ = π/2.
- *In words:* a uniform pressure exerts no net force.

---

**Step 12 of 13 — Integrate the pressure part.**

$$ \frac{3\mu U}{2a}2\pi a^2\int_0^\pi\cos^2\theta\sin\theta\,d\theta=3\pi\mu aU\cdot\frac23=2\pi \mu aU $$

- *Why we can do this:* ∫₀^π cos²θ sin θ dθ = 2/3 (substitute c = cos θ).
- *In words:* one third of the drag.

---

**Step 13 of 13 — Integrate the friction part.**

$$ 3\pi\mu aU\int_0^\pi\sin^3\theta\,d\theta=3\pi\mu aU\cdot\frac43=4\pi\mu aU,\ \ D=6\pi\mu aU\ \text{(8.51)} $$

- *Why we can do this:* ∫₀^π sin³θ dθ = 4/3; add step 12.
- *In words:* two thirds friction; (8.51), $D=6\pi\mu aU$.

---

**Result**

$$ D=6\pi\mu aU\ \text{(8.51), ⅓ pressure + ⅔ friction} $$

*In words:* drag is proportional to speed (Stokes' law of resistance).

**What it means.** Drag ∝ U makes the fall speed ∝ a² (N90): a 10 µm droplet falls 1.2 cm/s.

**Check it.** Units (Pa s)(m)(m/s) = N ✓. Notice t_x = 3μU/2a − p∞ cos θ: the x-traction is uniform over the sphere. The running integrals πμaU(1 − cos³θ) and πμaU(2 − 3cos θ + cos³θ) (`stokes_drag_running`) reach 2π and 4π at θ = π; midpoint and Gauss–Legendre sums agree; C_D = 24/Re (8.52), $C_D=\frac{D}{\tfrac12\rho U^2\pi a^2}=\frac{24}{\mathrm{Re}}$ follows.

In [ ]:
Sm = CL.strain_rate([ur, uth, 0], "spherical", X)        # strain-rate tensor of (8.49) in spherical components
srr = sp.simplify((-p + 2*mu*Sm[0, 0]).subs(r, a))       # σ_rr on the sphere (steps 3–4, 8)
srt = sp.simplify((2*mu*Sm[0, 1]).subs(r, a))            # σ_rθ on the sphere (steps 5–7)
print("σ_rr(a) =", srr, "  σ_rθ(a) =", srt, "  viscous part of σ_rr(a):", S["sigma_rr_viscous_a"])
Dp = sp.integrate(srr*sp.cos(th)*2*sp.pi*a**2*sp.sin(th), (th, 0, sp.pi))     # pressure part of the drag (step 12)
Df = sp.integrate(-srt*sp.sin(th)*2*sp.pi*a**2*sp.sin(th), (th, 0, sp.pi))   # friction part (step 13)
print("pressure:", sp.simplify(Dp), "  friction:", sp.simplify(Df))           # 2πμaU and 4πμaU
assert sp.simplify(Dp + Df - 6*sp.pi*mu*a*U) == 0        # (8.51): D = 6πμaU

> ⚠️ **Common confusion (traps in `D31`):** The viscous normal stress vanishes on the sphere; project with θ from the downstream axis (signs!); a bubble with a slipping surface gives 4πμaU — not this case.

#### ✏️ Tiny example: the drag and fall speed of a 10 µm cloud droplet

a = 10 µm, water droplet ρ′ = 1000 kg/m³ in air ρ = 1.2 kg/m³, μ = 1.81 × 10⁻⁵ Pa s, g = 9.81 m/s².
1. Terminal balance $\frac43\pi a^3g(\rho'-\rho)=6\pi\mu aU$ ⇒ $U_t=\frac{2(\rho'-\rho)ga^2}{9\mu}=\frac{2\times998.8\times9.81\times
   10^{-10}}{9\times1.81\times10^{-5}}=1.20$ cm/s.
2. $\mathrm{Re}=2aU_t\rho/\mu=2\times10^{-5}\times0.012\times1.2/1.81\times10^{-5}=0.016$ ✓ ≪ 1.
3. $D=6\pi\mu aU_t=4.1\times10^{-11}$ N: 1.4 × 10⁻¹¹ N from pressure, 2.7 × 10⁻¹¹ N from friction.
4. $C_D=24/\mathrm{Re}\approx1.5\times10^3$.
5. A 100 µm drizzle drop: 100× faster by a² — 1.2 m/s — but then Re ≈ 16 and Stokes no longer applies.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **terminal velocity, effective weight, buoyancy** — a body falls at constant speed when drag = weight − buoyancy = (4/3)πa³(ρ′ − ρ)g (Ch. 1 P24, weight).
- **Gauss–Legendre quadrature** — `np.polynomial.legendre.leggauss(n)` gives n nodes and weights that integrate polynomials of degree < 2n exactly (Ch. 5 P143).

In [ ]:
st = ch08.settling_state(10e-6, 1000.0, 1.2, 1.81e-5)    # droplet radius [m], ρ′, ρ [kg/m³], μ of air [Pa s] (g = 9.80665)
print({k: (f"{st[k]:.4g}" if not isinstance(st[k], bool) else st[k]) for k in ("U_t", "Re", "D", "C_D", "valid", "D_pressure", "D_friction")})
print(ch08.stokes_drag(1.81e-5, 10e-6, st["U_t"], parts=True))   # (8.51) split into its two parts [N]
srr_, srt_, tx = ch08.stokes_sphere_surface_stresses(np.array([0, np.pi/2, np.pi]), U=st["U_t"], a=10e-6, mu=1.81e-5)
print("x-traction at the rear, side, front:", tx, "Pa   (3μU/2a =", 1.5*1.81e-5*st["U_t"]/10e-6, ")")
print("drag collected from the rear to the side:", ch08.stokes_drag_running(np.pi/2, 1.81e-5, 10e-6, st["U_t"]))
print("C_D:", ch08.stokes_drag_coefficient(st["Re"]), SIM.sphere_drag_coefficient(st["Re"], "stokes"))   # (8.52), and Ch. 4's

**What does the code above do?**

1. `settling_state` solves the terminal balance (N90 below) and checks Re < 0.1: U_t = 1.20 cm/s, Re = 0.016, D = 4.10 × 10⁻¹¹ N,
   C_D = 1505, valid.
2. `stokes_drag(..., parts=True)` splits (8.51), $D=6\pi\mu aU$: one third pressure, two thirds friction.
3. The surface tractions' x-component is the same everywhere, 3μU/2a = 0.0326 Pa — the pressure part and the friction part
   trade off round the sphere.
4. `stokes_drag_running` collects the drag from the rear stagnation point to θ: by the side (θ = π/2) half of each part is in.
5. (8.52), $C_D=\frac{D}{\tfrac12\rho U^2\pi a^2}=\frac{24}{\mathrm{Re}}$ agrees with Ch. 4's drag-coefficient function in its Stokes branch.

In [ ]:
M_ = 2000                                                # midpoint sum over 2000 θ-cells
thm = (np.arange(M_) + 0.5)*np.pi/M_; dth = np.pi/M_      # cell midpoints and width [rad]
a_, U_, mu_ = 10e-6, st["U_t"], 1.81e-5                  # the droplet
srr_, srt_, _ = ch08.stokes_sphere_surface_stresses(thm, U=U_, a=a_, mu=mu_)   # σ_rr = −p and σ_rθ on the sphere
dA = 2*np.pi*a_**2*np.sin(thm)*dth                        # ring areas [m²]
Dp_ = np.sum(srr_*np.cos(thm)*dA); Df_ = np.sum(-srt_*np.sin(thm)*dA)   # x-projections, summed
D_ = ch08.stokes_drag(mu_, a_, U_, parts=True)
print(f"midpoint: pressure {Dp_:.6e} N, friction {Df_:.6e} N")
assert np.allclose([Dp_, Df_], [D_["pressure"], D_["friction"]], rtol=1e-6)
tx_fn = lambda th_: ch08.stokes_sphere_surface_stresses(th_, U=U_, a=a_, mu=mu_)[2]   # x-traction as a function of θ
assert np.isclose(ch08.sphere_drag_quadrature(tx_fn, a_, n=16), D_["total"], rtol=1e-12)   # Gauss–Legendre: exact already

📝 **Note.** **N90 [B], N91 [B] · Terminal velocity and Millikan's oil drops.** **N90** The balance $(4/3)\pi a^3g(\rho'-\rho)=6\pi\mu aU$ gives $U_t=\frac{2(\rho'-\rho)ga^2}{9\mu}$ — ∝ a²: halve the droplet,
quarter the speed. This sets how long aerosols and cloud droplets stay aloft and how fast silt settles (Ch. 13). **N91**
Millikan's experiment (Fig. 8.18): with the plates uncharged the fall speed gives the radius; with the upper plate negative the
drop rises at U_u: $6\pi\mu U_ua+(4/3)\pi a^3g(\rho'-\rho)=neE$, E = −V_b/L; charges from many drops are whole multiples of e.

In [ ]:
m = ch08.synthetic_millikan(40, seed=0)                  # 40 synthetic oil drops with 1 % noise in the measured speeds
print(f"estimated e = {m['e_est']:.6e} C, relative error vs CODATA {m['e_est']/1.602176634e-19 - 1:+.1e}")
print("charges in units of e (first 10):", np.round(m["q"][:10]/m["e_est"], 2))   # whole numbers

📝 **Note.** **N92 [B] · Drag coefficient.** With the frontal area πa² in $C_D\equiv\frac{F_D}{\tfrac12\rho U^2A}$ *(4.107)*, Stokes' law gives (8.52), with Re = 2aU/ν based on the
**diameter** — the low-Re branch of the drag curve of Ch. 4.

$$ C_D=\frac{D}{\tfrac12\rho U^2\pi a^2}=\frac{24}{\mathrm{Re}} \qquad \text{(8.52)} $$

In [ ]:
import warnings                                           # to silence the "Re > 0.1" warnings of the dashed extensions
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13, 3.8))
th_ = np.linspace(0, 2*np.pi, 13)[:-1]                    # 12 points round the circle in the drawing plane
for t0 in th_:                                            # traction arrows at each point (units μU/a, U = a = μ = 1)
    c, s = np.cos(t0), np.sin(t0)                         # the point (and the outward normal e_r) on the circle
    th_s = np.arccos(c)                                   # the spherical angle θ from the downstream +x axis
    srr1, srt1, _ = ch08.stokes_sphere_surface_stresses(th_s, U=1.0, a=1.0, mu=1.0)   # σ_rr = −p and σ_rθ there
    e_th = np.array([-abs(s), np.sign(s)*c])              # unit vector of increasing θ in the drawing plane
    t_p = srr1*np.array([c, s])                           # pressure (normal) part of the traction
    t_f = srt1*e_th                                       # friction (tangential) part
    for vec, col, lw in ((t_p, COLORS["orange"], 1.0), (t_f, COLORS["rose"], 1.0), (t_p + t_f, COLORS["ink"], 1.6)):
        a1.annotate("", xy=(1.1*c + 0.5*vec[0], 1.1*s + 0.5*vec[1]), xytext=(1.1*c, 1.1*s),
                    arrowprops=dict(arrowstyle="->", color=col, lw=lw))
sphere(a1, 1.0); a1.set_xlim(-2.4, 2.8); a1.set_ylim(-2.2, 2.2); a1.axis("off")
a1.set_title("Tractions: pressure (orange), friction (rose),\nsum (black) = 3μU/2a in x everywhere", fontsize=9)
thv = np.linspace(0, np.pi, 200)                          # from the rear (0) to the front (π)
a2.plot(thv, ch08.stokes_sphere_pressure(1.0, thv, 1.0, 1.0, 1.0), color=COLORS["orange"], lw=2, label="p − p∞ [μU/a]")
a2.plot(thv, ch08.stokes_sphere_surface_stresses(thv, 1.0, 1.0, 1.0)[1], color=COLORS["rose"], lw=2, label="σ_rθ [μU/a]")
b2 = a2.twinx()                                           # running drag on a second axis
run_ = ch08.stokes_drag_running(thv, 1.0, 1.0, 1.0)       # pressure and friction parts collected up to θ [μaU]
b2.plot(thv, run_["pressure"]/np.pi, color=COLORS["orange"], ls="--", lw=1.2)
b2.plot(thv, run_["friction"]/np.pi, color=COLORS["rose"], ls="--", lw=1.2)
b2.set_ylabel("running drag [π μaU] (dashed)"); b2.set_ylim(0, 4.4)
a2.set_xlabel("θ from the rear [rad]"); a2.legend(fontsize=7, loc="lower left"); a2.set_title("Stresses and the drag they build: 2π + 4π", fontsize=9)
radii = np.geomspace(1e-6, 1e-3, 80)                      # particle radii [m]
fams = [("water droplets in air", 1000.0, 1.2, 1.81e-5, COLORS["blue"]), ("quartz sand in water", 2650.0, 1000.0, 1e-3, COLORS["amber"]),
        ("bacteria in water", 1100.0, 1000.0, 1e-3, COLORS["teal"])]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                       # beyond Re = 0.1 the law is drawn dashed, not trusted
    for name, rp, rf, mf, c in fams:
        Ut = ch08.terminal_velocity(radii, rp, rf, mf, warn=False)   # N90 [m/s]
        Re_ = 2*radii*Ut*rf/mf                            # Reynolds number with the diameter
        ok = Re_ < 0.1
        a3.loglog(1e6*radii[ok], Ut[ok], color=c, lw=2, label=name)
        a3.loglog(1e6*radii[~ok], Ut[~ok], color=c, lw=1, ls="--")
a3.set_xlabel("radius a [µm]"); a3.set_ylabel("U_t [m/s]"); a3.legend(fontsize=7); a3.set_title("Fall speed ∝ a² (dashed: Re > 0.1)", fontsize=9)
fig.suptitle("A third pressure, two thirds friction; fall speed ∝ a²", fontsize=11)
savefig(fig, "ch08", "c14_stokes_drag"); plt.show()

**What you see.** (a) arrows round the sphere: pressure (orange) pushing at the front and pulling at the rear, friction (rose) along the
surface, and their sum (black) the same everywhere; (b) the pressure and shear stress against θ and their running integrals
(dashed) reaching 2π and 4π μaU; (c) straight lines of slope 2 for droplets, sand and bacteria, dashed where Re > 0.1 — our
remake of Fig. 8.17's stress plot (`N107`).

**How to read it.** In (b) the orange dashed curve ends at 2 and the rose one at 4 (in units of πμaU): one third and two thirds of 6πμaU. In
(c) read a fall speed: a 10 µm droplet falls about 1 cm/s, a 10 µm sand grain about 0.2 mm/s.

**What would change if…** …the droplet were oil (ρ′ = 900 kg/m³): $U_t\propto(\rho'-\rho)$, so it falls about 10 % more slowly.

#### 🎮 Interactive: Where does 6πμaU come from?

**Why interactive rather than a static figure:** Sweep around the sphere and watch the local pressure and shear tractions as arrows and curves while their running integrals
build up to 2πμaU and 4πμaU in the term bars; then drag the particle radius and read U_t and Re as the Stokes law's validity limit
is crossed.

**What to try:**
- Play the sweep from the rear (θ = 0) to the front: which bar grows first?
- Pick 'drizzle 100 µm': the status turns amber — why?
- Switch the third view to C_D(Re) and find where Oseen and the correlation leave 24/Re.
- Click a point on the sphere and read the traction arithmetic.

In [ ]:
show_viz("ch08", "stokes_drag_settling")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …we looked far from the sphere? The viscous force there decays like a/r³ but inertia only like a/r²; at some distance inertia
wins even for tiny Re (C15).

### 🧩 Where Stokes fails, and Oseen's fix `C15`

*The question:* If Re is tiny, how can inertia ever matter?

*In one line:* $\frac{\text{inertia}}{\text{viscous}}\sim\frac{\rho Ua}{\mu}\frac ra=\mathrm{Re}_a\frac ra$ — small near the sphere, order one at a distance of order $a/\mathrm{Re}_a$.

#### The problem in plain words

Far behind a slowly falling bead the fluid is barely disturbed — so inertia should be even less important there. It is the
opposite: the viscous force falls off faster with distance than inertia does, so beyond some radius Stokes' solution is wrong.
For a cylinder it is so wrong that no Stokes solution exists at all.

#### The idea

| at distance r | size |
|---|---|
| disturbance velocity | ~ Ua/r |
| inertia ρu·∇u | ~ ρU²a/r² |
| viscous μ∇²u | ~ μUa/r³ |
| ratio | ~ (ρUa/μ)(r/a) = Re_a r/a → order one at r ~ a/Re_a |

The slowly decaying disturbance is swept along by the stream; its viscous force falls one power of r faster.

> 🔁 **Tools from earlier chapters used here** (see `knowledge/primers.md`)

- **asymptotic size of a term far away** — for r ≫ a keep only the slowest-decaying piece of each quantity; a derivative of a 1/r term brings another 1/r (Ch. 4 P130, order-of-magnitude scaling).
- **series of 1 − e^(−s)** — $1-e^{-s}=s-\frac{s^2}{2}+\ldots$ for small s; `sp.series` produces it (Ch. 4 P117).

#### 🧮 Derivation — Far-field breakdown: inertia/viscous ~ $\frac{\rho Ua}{\mu}\frac ra=\mathrm{Re}_a\frac ra$ `D32`

**What we want to show.** Show that however small Re is, inertia matches viscosity at some distance from the sphere.

**Assumptions.** r ≫ a; orders of magnitude only.

**The plan.**

1. Size the disturbance far away.
2. Size inertia and viscous terms.
3. Take the ratio.

**Tools we use** (each explained before this point): asymptotic size of a term (gloss, P130).

**We start from**

$$ u_r=U\cos\theta\Big(1-\frac{3a}{2r}+\frac{a^3}{2r^3}\Big),\ u_\theta=-U\sin\theta\Big(1-\frac{3a}{4r}-\frac{a^3}{4r^3}\Big)\ \text{(8.49)},\qquad r\gg a $$

*In words:* Stokes' velocities far from the sphere

---

**Step 1 of 7 — Keep the slowest-decaying disturbance.**

$$ \lvert\mathbf u-U\mathbf e_x\rvert\sim\frac{Ua}{r} $$

- *Why we can do this:* In (8.49), $u_r=U\cos\theta\Big(1-\frac{3a}{2r}+\frac{a^3}{2r^3}\Big),\ u_\theta=-U\sin\theta\Big(1-\frac{3a}{4r}-\frac{a^3}{4r^3}\Big)$ the 3a/2r and 3a/4r terms decay like 1/r; the a³/r³ terms are much smaller far away.
- *In words:* the sphere is felt as Ua/r.

---

**Step 2 of 7 — Size the disturbance's gradient.**

$$ \lvert\nabla\mathbf u'\rvert\sim\frac{Ua}{r^2},\qquad \mathbf u'=\mathbf u-U\mathbf e_x $$

- *Why we can do this:* Differentiating a 1/r term brings another 1/r; we call the part of the velocity that differs from the stream the disturbance u′.
- *In words:* gradients fall as 1/r².

---

**Step 3 of 7 — Size the inertia.**

$$ \rho\,\mathbf u\cdot\nabla\mathbf u\sim\rho U\frac{\partial\mathbf u'}{\partial x}\sim\rho U\cdot\frac{Ua}{r^2}=\frac{\rho U^2a}{r^2} $$

- *Why we can do this:* Far away the velocity is the stream U plus the small disturbance u′, so the largest part of u·∇u is the stream carrying the disturbance, U ∂u′/∂x (the term Oseen keeps, N95). In (8.49), $u_r=U\cos\theta\Big(1-\frac{3a}{2r}+\frac{a^3}{2r^3}\Big),\ u_\theta=-U\sin\theta\Big(1-\frac{3a}{4r}-\frac{a^3}{4r^3}\Big)$ this O(U²a/r²) inertia sits in the radial component, (3U²a/16r²)(8cos²θ − 4sin²θ); the θ-component has no such term.
- *In words:* the stream sweeps the disturbance past: inertia ~ ρU²a/r².

---

**Step 4 of 7 — Size the viscous force.**

$$ \mu\nabla^2\mathbf u\sim\mu\frac{Ua/r}{r^2}=\frac{\mu Ua}{r^3} $$

- *Why we can do this:* Two derivatives of the Ua/r disturbance (the uniform part has none).
- *In words:* friction ~ μUa/r³.

---

**Step 5 of 7 — Take the ratio.**

$$ \frac{\text{inertia}}{\text{viscous}}\sim\frac{\rho U^2a/r^2}{\mu Ua/r^3}=\frac{\rho Ur}{\mu} $$

- *Why we can do this:* Divide step 3 by step 4.
- *In words:* the ratio grows with r.

---

**Step 6 of 7 — Write it with a Reynolds number.**

$$ \frac{\rho Ur}{\mu}=\frac{\rho Ua}{\mu}\frac ra=\mathrm{Re}_a\frac ra $$

- *Why we can do this:* Re_a uses the radius (Re = 2aU/ν = 2Re_a).
- *In words:* small Re times a large distance.

---

**Step 7 of 7 — Find the crossover.**

$$ \mathrm{Re}_a\frac ra\sim1\ \Rightarrow\ \frac ra\sim\frac{1}{\mathrm{Re}_a} $$

- *Why we can do this:* Set the ratio to one: that is where inertia stops being negligible. This is an order of magnitude: the exact prefactor is ½ on the axis and on the side line (so the crossover is at r ≈ 2a/Re_a), and it dips near θ ≈ 55°, where the leading inertia term changes sign.
- *In words:* beyond r ~ a/Re_a Stokes' solution is wrong.

---

**Result**

$$ \frac{\text{inertia}}{\text{viscous}}\sim\frac{\rho Ua}{\mu}\frac ra=\mathrm{Re}_a\frac ra\quad\text{(order of magnitude)} $$

*In words:* the non-uniformity of Stokes flow is at infinity.

**What it means.** For a cylinder the disturbance decays even more slowly and no Stokes solution exists (Stokes' paradox); Oseen keeps U∂u′/∂x to fix the far field (N95).

**Check it.** `inertia_viscous_ratio` has log–log slope 1 ± 0.05 for r/a ∈ [50, 500] (the from-scratch cell of C15 measures it). Re_a = 0.01 → r of order 100a, and the 10 µm droplet (Re_a = 0.008) → r of order 125a = 1.25 mm — orders of magnitude: the code cell of C15 prints ratio ÷ (Re_a r/a) ≈ 0.5 on the axis and on the side line, so inertia equals friction at r ≈ 2a/Re_a (200a for Re_a = 0.01).

> ⚠️ **Common confusion (traps in `D32`):** The disturbance far away is ~ Ua/r (the Stokeslet), not a³/r³; Re here uses the radius.

📝 **Note.** **N94 [C] · A singular perturbation at infinity.** Treated as the first term of an expansion in Re, Stokes' solution is not uniformly valid: the O(Re) correction grows without
bound relative to it as r → ∞ (Whitehead's paradox); for a cylinder the Stokes equations cannot even meet the uniform stream
(Stokes' paradox, Exercise 8.37). Unlike the boundary layer (Ch. 9), where 1/Re multiplies the highest derivative near the wall,
here the trouble is far away.

📝 **Note.** **N95 [B], N96 [B] · Oseen's linearisation (1910).** **N95** Write u = U + u′ with u′ small far away. The advective term splits as
$u\frac{\partial u}{\partial x}+v\frac{\partial u}{\partial y}+w\frac{\partial u}{\partial z}=U\frac{\partial u'}{\partial x}+
\big[u'\frac{\partial u'}{\partial x}+v'\frac{\partial u'}{\partial y}+w'\frac{\partial u'}{\partial z}\big]$; dropping the bracket gives
the equation below. ⚠️ The book prints $+\partial p/\partial x_i$; the minus of (8.43), $\nabla p=\mu\nabla^2\mathbf u$ is needed. **N96** Conditions: u′, v′, w′ → 0
far away; u′ = −U, v′ = w′ = 0 on the sphere.

$$ \rho U\frac{\partial u_i'}{\partial x}=-\frac{\partial p}{\partial x_i}+\mu\nabla^2u_i'  $$

In [ ]:
ol_ = ch08.oseen_linearisation_sympy()                   # N95 in sympy
print("dropped (quadratic) part:", ol_["dropped"])
print("Oseen x-equation:", ol_["oseen_x"])               # ρU ∂u'/∂x = μ∇²u' − ∂p/∂x

📝 **Note.** **N97 [B], N98 [B] · Oseen's stream function.** **N97** Oseen's solution is (8.53), with Re = 2aU/ν (stated; no slip holds only to O(Re)). **N98** Near the sphere the
exponential's series returns Stokes' (8.48), $\psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)$. ⚠️ The text calls it "(9.68)"; it means (8.48).

$$ \frac{\psi}{Ua^2}=\Big[\frac{r^2}{2a^2}+\frac{a}{4r}\Big]\sin^2\theta-\frac{3}{\mathrm{Re}}(1+\cos\theta)\Big\{1-\exp\Big[-\frac{\mathrm{Re}}{4}\frac ra(1-\cos\theta)\Big]\Big\} \qquad \text{(8.53)} $$

#### 🧮 Derivation — Oseen's reduces to Stokes' near the sphere `D33` (Eq. 8.53: $\frac{\psi}{Ua^2}=\Big[\frac{r^2}{2a^2}+\frac{a}{4r}\Big]\sin^2\theta-\frac{3}{\mathrm{Re}}(1+\cos\theta)\Big\{1-\exp\Big[-\frac{\mathrm{Re}}{4}\frac ra(1-\cos\theta)\Big]\Big\}$)

**What we want to show.** Check that Oseen's improved solution agrees with Stokes' where Stokes is valid.

**Assumptions.** Re r/a ≪ 1 (near the sphere, small Re) — step 2.

**The plan.**

1. Name the exponent.
2. Expand the exponential for small arguments.
3. Simplify with sin²θ = (1 − cos θ)(1 + cos θ).

**Tools we use** (each explained before this point): series of 1 − $e^(-s)$ (gloss, P117).

**We start from**

$$ \frac{\psi}{Ua^2}=\big[\frac{r^2}{2a^2}+\frac a{4r}\big]\sin^2\theta-\frac3{\mathrm{Re}}(1+\cos\theta)\big\{1-\exp\big[-\frac{\mathrm{Re}}4 \frac ra(1-\cos\theta)\big]\big\}\ \text{(8.53), Re = 2aU/ν.} $$

---

**Step 1 of 7 — Name the exponent.**

$$ s=\frac{\mathrm{Re}}{4}\frac ra(1-\cos\theta) $$

- *Why we can do this:* The whole Oseen correction depends on this one combination.
- *In words:* s measures inertia's reach.

---

**Step 2 of 7 — State where s is small.**

$$ s\ll1\iff\mathrm{Re}\,\frac ra\ll1 $$

- *Why we can do this:* 0 ≤ 1 − cos θ ≤ 2, so s is small exactly where D32 says Stokes holds.
- *In words:* near the sphere at small Re.

---

**Step 3 of 7 — Expand the exponential.**

$$ 1-e^{-s}=s-\frac{s^2}{2}+\ldots\approx s $$

- *Why we can do this:* Taylor series of $e^(-s)$ (P117); keep the leading term.
- *In words:* the bracket is linear in s.

---

**Step 4 of 7 — Insert into the Oseen term.**

$$ \frac3{\mathrm{Re}}(1+\cos\theta)s=\frac3{\mathrm{Re}}(1+\cos\theta)\frac{\mathrm{Re}}4\frac ra (1-\cos\theta) $$

- *Why we can do this:* Replace 1 − e^(−s) by its leading term s from step 3.
- *In words:* Re cancels.

---

**Step 5 of 7 — Use sin²θ = (1 + cos θ)(1 − cos θ).**

$$ =\frac34\frac ra\sin^2\theta $$

- *Why we can do this:* The product of the two brackets is 1 − cos²θ.
- *In words:* the Stokeslet term appears.

---

**Step 6 of 7 — Collect the terms.**

$$ \frac{\psi}{Ua^2}\approx\Big[\frac{r^2}{2a^2}-\frac{3r}{4a}+\frac{a}{4r}\Big]\sin^2\theta $$

- *Why we can do this:* Subtract step 5 from the first bracket of (8.53), $\frac{\psi}{Ua^2}=\Big[\frac{r^2}{2a^2}+\frac{a}{4r}\Big]\sin^2\theta-\frac{3}{\mathrm{Re}}(1+\cos\theta)\Big\{1-\exp\Big[-\frac{\mathrm{Re}}{4}\frac ra(1-\cos\theta)\Big]\Big\}$.
- *In words:* three terms, like Stokes.

---

**Step 7 of 7 — Multiply by Ua².**

$$ \psi\approx Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)\ \text{(8.48)} $$

- *Why we can do this:* Ua² × [r²/2a², −3r/4a, a/4r] = Ur²[½, −3a/4r, a³/4r³]. The text calls this "(9.68)"; it is (8.48), $\psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)$.
- *In words:* Stokes' solution recovered, with an O(Re) remainder.

---

**Result**

$$ \psi_{Oseen}=\psi_{Stokes}+O\Big(\mathrm{Re}\frac ra\Big),\qquad \psi=Ur^2\sin^2\theta\Big(\frac12-\frac{3a}{4r}+\frac{a^3}{4r^3}\Big)\ \text{(8.48)} $$

*In words:* Oseen changes only the far field.

**What it means.** Near the sphere the two agree; far away Oseen's exponential sweeps the disturbance downstream into a wake, and the drag gains the factor (1 + 3Re/16).

**Check it.** `oseen_limit_sympy()["difference"]` = 0 in the limit Re → 0; numerically ψ agree to ~1e-6 at Re = 10⁻⁶. Oseen meets no slip on r = a only to O(Re).

> ⚠️ **Common confusion (traps in `D33`):** s is small only where Re r/a ≪ 1; the product (1 + cos θ)(1 − cos θ) is sin²θ; do not assert exact no slip for (8.53), $\frac{\psi}{Ua^2}=\Big[\frac{r^2}{2a^2}+\frac{a}{4r}\Big]\sin^2\theta-\frac{3}{\mathrm{Re}}(1+\cos\theta)\Big\{1-\exp\Big[-\frac{\mathrm{Re}}{4}\frac ra(1-\cos\theta)\Big]\Big\}$.

In [ ]:
ol = ch08.oseen_limit_sympy()                            # D33 in sympy: expand 1 − e^(−s) and let Re → 0
print("s =", ol["s"])
print("Oseen − Stokes as Re → 0:", sp.simplify(ol["difference"]))   # 0
print("first-order correction:", ol["first_order"])       # O(Re) and growing with r: the far-field change

#### ✏️ Tiny example: where inertia catches up

Re = 2aU/ν = 0.02, so $\mathrm{Re}_a=\mathrm{Re}/2=0.01$.
1. The estimate $\mathrm{Re}_a\,r/a\sim1$ puts the crossover at r of order 100a. With the exact prefactor ½ (the code below computes
   it on the axis and the side line) inertia equals friction at r ≈ 2a/Re_a = 200a.
2. For the 10 µm droplet (Re = 0.016, Re_a = 0.008) the same estimate gives r of order 125a = 1.25 mm, and about 250a with the
   prefactor.
3. Oseen C_D at Re = 0.5: $\frac{24}{\mathrm{Re}}\big(1+\frac{3}{16}\mathrm{Re}\big)=48(1+3/32)=52.5$ vs Stokes 48 — a 9 % correction.

In [ ]:
Re_a = 1e-3*1e-3/1e-4                                    # U a/ν with U = 1 mm/s, a = 1 mm, ν = 10⁻⁴ m²/s: 0.01
for th_, name in ((0.0, "axis θ = 0"), (np.pi/4, "θ = π/4"), (np.pi/2, "side line θ = π/2")):
    for rr_ in (10.0, 100.0, 400.0):                     # distances in radii
        ratio = ch08.inertia_viscous_ratio(rr_*1e-3, th_, U=1e-3, a=1e-3, nu=1e-4)   # |u·∇u| / |ν∇²u| on Stokes' field
        print(f"{name:18s} r = {rr_:5.0f}a: ratio = {ratio:.4f}, ratio ÷ (Re_a r/a) = {ratio/(Re_a*rr_):.3f}")
print("C_D at Re = 0.5: Oseen", ch08.oseen_drag_coefficient(0.5), " Stokes", ch08.stokes_drag_coefficient(0.5),
      " Proudman–Pearson", round(ch08.proudman_pearson_drag_coefficient(0.5), 3), " Morrison (data fit)", round(SIM.sphere_drag_coefficient(0.5), 3))
print("ψ Oseen (Re = 1e-6) vs Stokes at r = 2a, θ = π/3:", ch08.oseen_streamfunction(2e-3, np.pi/3, U=1e-3, a=1e-3, Re=1e-6),
      ch08.stokes_sphere_streamfunction(2e-3, np.pi/3, U=1e-3, a=1e-3))

**What does the code above do?**

1. `inertia_viscous_ratio` evaluates |u·∇u| / |ν∇²u| on Stokes' field by finite differences. It grows in proportion to r, and
   ratio ÷ (Re_a r/a) tends to **½ on the axis and on the side line** — so inertia equals friction at r ≈ 2a/Re_a (200a here). At
   θ = π/4 the leading inertia term nearly cancels (it changes sign near θ ≈ 55°) and the prefactor is smaller, about 0.16. The
   book's r ~ a/Re is the right order of magnitude.
2. The drag laws at Re = 0.5: Oseen 52.5, Stokes 48, Proudman–Pearson and the Morrison correlation of the measurements close to
   Oseen.
3. At a vanishing Re Oseen's ψ equals Stokes' to about 10⁻⁶ (D33).

In [ ]:
hh = 1e-4                                                # relative finite-difference step
pts = [(r_*1e-3, 0.0) for r_ in (50.0, 100.0, 200.0)]   # three points on the axis behind the sphere (x, y) [m]


def vel(x_, y_):                                          # Stokes body-frame velocity (u, v) in the plane z = 0
    u_, v_, _ = ch08.stokes_sphere_velocity_xyz(np.array(x_), np.array(y_), np.array(0.0), 1e-3, 1e-3)
    return np.array([u_, v_], float)


mine = []
for x0, y0 in pts:
    d = hh*x0                                             # step [m]
    ux = (vel(x0 + d, y0) - vel(x0 - d, y0))/(2*d); uy = (vel(x0, y0 + d) - vel(x0, y0 - d))/(2*d)   # ∂u/∂x, ∂u/∂y
    uxx = (vel(x0 + d, y0) - 2*vel(x0, y0) + vel(x0 - d, y0))/d**2
    uyy = (vel(x0, y0 + d) - 2*vel(x0, y0) + vel(x0, y0 - d))/d**2
    lap = uxx + 2*uyy                                     # ∇²u on the axis: the z-direction curves like the y-direction (axisymmetry)
    adv = vel(x0, y0)[0]*ux + vel(x0, y0)[1]*uy           # u·∇u (w = 0 in the plane z = 0)
    mine.append(np.linalg.norm(adv)/np.linalg.norm(1e-4*lap))   # |u·∇u| / |ν∇²u|
lib = [ch08.inertia_viscous_ratio(x0, 0.0, U=1e-3, a=1e-3, nu=1e-4) for x0, _ in pts]
print("by hand:", np.round(mine, 4), " library:", np.round(lib, 4))
assert np.allclose(mine, lib, rtol=1e-3)                 # the same ratio
rs = np.geomspace(50, 500, 8)                            # r/a from 50 to 500
slope = observed_order(rs, [ch08.inertia_viscous_ratio(r_*1e-3, np.pi/2, U=1e-3, a=1e-3, nu=1e-4) for r_ in rs])   # log–log slope
print(f"log–log slope of the ratio for r/a in [50, 500]: {slope:.3f}")
assert abs(slope - 1) < 0.05                             # D32: the ratio grows like r

In [ ]:
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13, 3.8))
rs = np.geomspace(1.5, 3000, 60)                          # r/a
for Rea, c in ((1e-3, COLORS["blue"]), (1e-2, COLORS["teal"]), (1e-1, COLORS["orange"])):   # three Re_a
    nu_ = 1e-3*1e-3/Rea                                    # ν that gives this Re_a with U = 1 mm/s, a = 1 mm
    a1.loglog(rs, ch08.inertia_viscous_ratio(rs*1e-3, np.pi/2, U=1e-3, a=1e-3, nu=nu_), color=c, lw=2, label=f"Re_a = {Rea:g}")
a1.axhline(1, color=COLORS["muted"], ls="--", lw=1); a1.text(2, 1.3, "inertia = friction", fontsize=8)
a1.set_xlabel("r/a"); a1.set_ylabel("|u·∇u| / |ν∇²u| (side line)"); a1.legend(fontsize=7); a1.set_title("Inertia returns far away", fontsize=10)
n_ = 200 if not FAST else 100                             # contour grid
xg = np.linspace(-15, 15, n_); yg = np.linspace(-10, 10, n_)
Xo, Yo = np.meshgrid(xg, yg); Ro = np.hypot(Xo, Yo); To = np.arccos(np.clip(Xo/np.maximum(Ro, 1e-12), -1, 1))
outo = Ro > 1
ps_ = np.where(outo, ch08.stokes_sphere_streamfunction(Ro, To, 1.0, 1.0, frame="fluid"), np.nan)
po_ = np.where(outo, ch08.oseen_streamfunction(Ro, To, 1.0, 1.0, Re=1.0, frame="fluid"), np.nan)
lev = np.linspace(-4, 4, 33)
a2.contour(Xo, Yo, ps_, lev, colors=COLORS["muted"], linewidths=0.7, linestyles="--")
a2.contour(Xo, Yo, po_, lev, colors=COLORS["blue"], linewidths=1.0)
sphere(a2, 1.0); a2.set_xlim(-15, 15); a2.set_ylim(-10, 10); a2.set_xlabel("x/a (sphere moving left)")
a2.set_title("Fluid frame, Re = 1: Oseen (blue) vs Stokes (dashed)", fontsize=9)
Res = np.geomspace(0.01, 10, 100)
a3.semilogx(Res, ch08.stokes_drag_coefficient(Res)*Res/24, color=COLORS["muted"], lw=2, label="Stokes 24/Re")
a3.semilogx(Res, ch08.oseen_drag_coefficient(Res)*Res/24, color=COLORS["blue"], lw=2, label="Oseen (1 + 3Re/16)")
a3.semilogx(Res, np.array([ch08.proudman_pearson_drag_coefficient(r_) for r_ in Res])*Res/24, color=COLORS["accent"], lw=1.5, ls="--", label="Proudman–Pearson")
a3.semilogx(Res, np.array([SIM.sphere_drag_coefficient(r_) for r_ in Res])*Res/24, color=COLORS["amber"], lw=2, label="Morrison (measurements)")
a3.set_ylim(0.9, 3); a3.set_xlabel("Re = 2aU/ν"); a3.set_ylabel("C_D·Re/24 [–]"); a3.legend(fontsize=7); a3.set_title("Drag laws", fontsize=10)
fig.suptitle("Stokes is right near the sphere and wrong far away", fontsize=11)
savefig(fig, "ch08", "c15_oseen"); plt.show()

**What you see.** (a) the inertia/friction ratio growing in proportion to r for three Re_a, crossing 1 at r ≈ 2a/Re_a; (b) at Re = 1 the
Oseen streamlines (blue) are crowded behind the moving sphere into a wake while Stokes' (dashed) stay symmetric — our remake of
Fig. 8.20 `N100`; (c) C_D·Re/24: Stokes flat at 1, Oseen rising, the Morrison correlation of measurements between them below
Re ≈ 5, and Proudman–Pearson `N101` following Oseen at first.

**How to read it.** In (c) any curve above 1 means more drag than Stokes' law: at Re = 1 Oseen adds 19 %; the measurements (amber) lie between
Stokes and Oseen up to Re ≈ 5, the range where both formulas are "fairly accurate".

**What would change if…** …Re doubled: the crossover distance in (a) halves and the wake in (b) widens and moves closer to the sphere.

📝 **Note.** **N99 [B], N100 [B], N101 [C] · Drag beyond Stokes.** **N99** Oseen's drag coefficient is below; ⚠️ with the radius-based $\mathrm{Re}_a=\mathrm{Re}/2$ the same coefficient reads 3/8.
**N100** The Oseen streamlines (panel b) are asymmetric, with a wake behind. **N101** Matched asymptotic expansions (Kaplun,
Proudman & Pearson 1957) continue the series: $D=6\pi\mu aU\big(1+\frac38\mathrm{Re}_a+\frac9{40}\mathrm{Re}_a^2\ln\mathrm{Re}_a+
\ldots\big)$ (named only).

$$ C_D=\frac{24}{\mathrm{Re}}\Big(1+\frac3{16}\mathrm{Re}\Big)  $$

In [ ]:
n_ = 200 if not FAST else 100                             # contour grid
xg = np.linspace(-15, 15, n_); yg = np.linspace(-10, 10, n_)
Xo, Yo = np.meshgrid(xg, yg); Ro = np.hypot(Xo, Yo); To = np.arccos(np.clip(Xo/np.maximum(Ro, 1e-12), -1, 1))
lev = np.linspace(-4, 4, 25)                              # streamline levels


def polylines(psi_):                                      # contour lines of ψ as one (x, y) polyline with NaN breaks
    cs = plt.figure().add_subplot().contour(Xo, Yo, psi_, lev)   # matplotlib finds the lines (figure discarded)
    xs, ys = [], []
    for path in cs.get_paths():
        for poly in path.to_polygons(closed_only=False):  # every separate piece of the line
            xs += list(poly[:, 0]) + [np.nan]; ys += list(poly[:, 1]) + [np.nan]
    plt.close("all")
    return np.array(xs), np.array(ys)


stokes_lines = polylines(np.where(Ro > 1, ch08.stokes_sphere_streamfunction(Ro, To, 1.0, 1.0, frame="fluid"), np.nan))
cache = {}                                                # Oseen lines for each Re, computed once
for Re_ in (0.1, 0.2, 0.5, 1.0, 2.0):
    cache[Re_] = polylines(np.where(Ro > 1, ch08.oseen_streamfunction(Ro, To, 1.0, 1.0, Re=Re_, frame="fluid"), np.nan))
circle = (np.cos(np.linspace(0, 2*np.pi, 60)), np.sin(np.linspace(0, 2*np.pi, 60)))   # the sphere's outline


def frame(Re_):                                           # the figure for one Re
    return {"Oseen streamlines": cache[Re_], "Stokes streamlines": stokes_lines, "sphere": circle}


Rs = [0.1, 0.2, 0.5, 1.0, 2.0]
fig = slider_figure(frame, "Re", Rs, unit="", xlabel="x/a (sphere moving left)", ylabel="y/a", title="", xrange=[-15, 15], yrange=[-10, 10])
step_titles(fig, [f"Re = {r_:g}: C_D Oseen {ch08.oseen_drag_coefficient(r_):.1f}, Stokes {ch08.stokes_drag_coefficient(r_):.1f}" for r_ in Rs])
recolor(fig, {"Oseen streamlines": COLORS["blue"], "Stokes streamlines": COLORS["muted"], "sphere": COLORS["ink"]},
        {"Stokes streamlines": "dot"})
fig.update_traces(line_width=1.2)
fig.show()

**What does the code above do?**

Precomputed Oseen streamlines (fluid frame) for five Reynolds numbers over Stokes' symmetric ones (dotted); the title gives both
drag coefficients. Slide Re up and watch the wake form behind the sphere.

🎮 The explainer of C13 (`stokes_sphere_flow`, above) has an **Oseen mode**: open it again, choose Oseen and drag Re to see the
wake grow from far downstream.

---

## 8.7 Final Remarks

**What is this section about?** Where the exact solutions stop.

📝 **Note.** **N102 [C] · After the exact solutions.** Most laminar problems that can be solved with pencil and paper have been solved; the field moves on with perturbation methods
(flows close to a known one — Ch. 9 boundary layers, Ch. 11 stability) and with numerical solution of the Navier–Stokes equations
(Ch. 10, which uses this chapter's exact solutions as test cases).

*↪ **S01** Exercises 8.1–8.38 and the literature are not reproduced. Ideas from them used here: the Reynolds equation — Exercises 8.19–8.20 → C06; the power–dissipation identity — Exercise 8.12 → R11; the switched-on vortex — Exercise 8.26 → C10; the stopped plate — Exercise 8.30 → C09; Couette start-up — Exercise 8.31 → R14; Hele-Shaw near an obstacle — Exercise 8.34 → C06; Stokes drag from the stresses — Exercise 8.35 → C14; Stokes' paradox for a cylinder — Exercise 8.37 → C15; f = 64/Re — Exercise 8.7 → C03.*

---

## ✅ What should have clicked

- **C01** $\nu=\mu/\rho$ is a diffusivity: motion spreads $\sqrt{\nu t}$, and a flow stays laminar below Re ≈ 2000.
- **C02** Between plates the profile is a Couette line plus a Poiseuille parabola, $u(y)=\frac Uh\,y-\frac1{2\mu}\frac{dp}{dx}\,y(h-y)$ *(8.5)*; the floor flow reverses when $dp/dx>2\mu U/h^2$.
- **C03** In a pipe $Q=-\frac{\pi a^4}{8\mu}\frac{dp}{dz}$: halve the radius, lose 94 % of the flow; f = 64/Re.
- **C04** Between rotating cylinders the swirl is $u_\varphi=AR+B/R$ *(8.9)*: rigid rotation plus a free vortex, chosen by the walls.
- **C05** In a thin gap inertia is weighed with $\varepsilon^2\mathrm{Re}_L$, not Re_L; pressure is uniform across the gap.
- **C06** Station by station the gap flow is Couette plus Poiseuille; integrating continuity gives the Reynolds equation $h_t+q_x=0$.
- **C07** A constant flux through a narrowing gap needs a pressure hump — that hump carries the load, but only if the pad slides the right way.
- **C08** A spreading layer obeys $h_t=(\rho g/3\mu)(h^3h_x)_x$, a diffusion that chokes itself as h shrinks.
- **C09** With no imposed scale, y and t combine as $y/\sqrt{\nu t}$: one erfc curve for every time, $\delta_{99}=3.64\sqrt{\nu t}$ *(8.31)*.
- **C10** Guess $At^{-n}F(\xi/\delta(t))$; matching powers of t and one conserved quantity fix n and δ.
- **C11** An oscillating wall drives a decaying, lagging layer of depth $\sqrt{2\nu/\omega}$ — diffusion, not a wave.
- **C12** At low Re rescale pressure by μU/L; the Stokes equations $\nabla p=\mu\nabla^2\mathbf u$ *(8.43)* are linear.
- **C13** Stokes' sphere flow is fore–aft symmetric and decays only like a/r.
- **C14** Drag $D=6\pi\mu aU$ *(8.51)*, ⅓ pressure and ⅔ friction; fall speed ∝ a².
- **C15** Inertia returns at r ~ a/Re_a (about 2a/Re_a with the exact prefactor); Oseen's linearisation adds the wake and $C_D=\frac{24}{\mathrm{Re}}(1+\frac{3}{16}\mathrm{Re})$.

**What later chapters build on**

- Ch. 9: √(νt) ↔ √(νx/U), Blasius vs the temporal layer, the lubrication scaling as the boundary-layer approximation, similarity exponents.
- Ch. 10: exact solutions as code tests; Crank–Nicolson.
- Ch. 11: Couette, Poiseuille and Taylor–Couette base states; the Rayleigh criterion in A and B.
- Ch. 12: linear total stress, τ₀ and the friction velocity, f = 64/Re on the Moody chart.
- Ch. 13: Ekman layers with the (1 + i)/δ structure of the Stokes layer, spin-up times from √(νt), cyclostrophic balance, viscous gravity currents, settling of droplets and sediment.
- Ch. 16: arteries (Q ∝ a⁴), synovial lubrication, micro-swimmers (reversibility of Stokes flow).

**Not taught here (and where to find it)**
- The exercises (S01).
- Higher-order matched expansions for the sphere (named in N101).